# PhysioNet 64ch: strict LOSO benchmark  
## Воспроизведение CCST и сравнение с нашим pipeline

### Цель эксперимента

В предыдущих экспериментах PhysioNet приводился к общему 22-канальному представлению, чтобы использовать единый pipeline вместе с BCI IV-2a и Cho2017.

В этом ноутбуке мы переходим к **нативной 64-канальной постановке PhysioNet**, чтобы максимально приблизить evaluation protocol к опубликованным subject-independent работам и провести более честное сравнение с литературой.

Сравниваем на **одних и тех же данных и одних и тех же LOSO-folds**:

1. воспроизведение **CCST**;
2. наш зафиксированный pipeline **Cho Transfer + SupCon + GRL**;
3. после этого — **task-aware denoising** поверх финального frozen classifier.

### Фиксируем протокол заранее

- Dataset: PhysioNet EEG Motor Movement/Imagery
- Subjects: 109
- EEG channels: 64
- Sampling rate: 160 Hz
- Classes: Left Hand vs Right Hand motor imagery
- Filtering: 7–30 Hz
- Evaluation: strict Leave-One-Subject-Out
- Test subject полностью исключён из train и validation
- Validation используется только для выбора эпохи
- Test не используется для выбора модели и гиперпараметров

Главная идея: сначала зафиксировать единый benchmark, а затем запускать разные методы на абсолютно одинаковых folds.

In [1]:
# ============================================================
# IMPORTS
# ============================================================

from pathlib import Path

import os
import gc
import json
import random
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
)

from scipy.stats import wilcoxon

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    TensorDataset,
)

import mne

from moabb.datasets import PhysionetMI
from moabb.paradigms import MotorImagery

warnings.filterwarnings("ignore")

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("MNE:", mne.__version__)

NumPy: 2.5.3
Pandas: 3.0.5
PyTorch: 2.14.0
MNE: 1.13.2


In [ ]:
import copy
import math

In [2]:
# ============================================================
# GLOBAL CONFIG
# ============================================================

SEED = 42

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")

elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")

else:
    DEVICE = torch.device("cpu")


# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------

def set_seed(seed=SEED):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ------------------------------------------------------------
# PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/Users/irina/Desktop/master project"
)

RESULTS_ROOT = (
    PROJECT_ROOT
    / "results_v2"
    / "physionet_64ch_strict_loso"
)

DATA_CACHE_DIR = (
    RESULTS_ROOT
    / "data_cache"
)

SPLITS_DIR = (
    RESULTS_ROOT
    / "splits"
)

CCST_DIR = (
    RESULTS_ROOT
    / "ccst"
)

OUR_DIR = (
    RESULTS_ROOT
    / "our_pipeline"
)

DENOISER_DIR = (
    RESULTS_ROOT
    / "taskaware_denoising"
)

FIGURES_DIR = (
    RESULTS_ROOT
    / "figures"
)

for directory in [
    RESULTS_ROOT,
    DATA_CACHE_DIR,
    SPLITS_DIR,
    CCST_DIR,
    OUR_DIR,
    DENOISER_DIR,
    FIGURES_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# BENCHMARK CONFIG
# ------------------------------------------------------------

SFREQ = 160.0

FMIN = 7.0
FMAX = 30.0

N_CHANNELS = 64

EVENTS = [
    "left_hand",
    "right_hand",
]

LABEL_MAP = {
    "left_hand": 0,
    "right_hand": 1,
}


print("=" * 80)
print("CONFIG")
print("=" * 80)

print("Device:", DEVICE)
print("Seed:", SEED)
print("Sampling rate:", SFREQ)
print("Band-pass:", FMIN, "-", FMAX, "Hz")
print("Channels:", N_CHANNELS)
print("Classes:", LABEL_MAP)
print("Output:", RESULTS_ROOT)

CONFIG
Device: mps
Seed: 42
Sampling rate: 160.0
Band-pass: 7.0 - 30.0 Hz
Channels: 64
Classes: {'left_hand': 0, 'right_hand': 1}
Output: /Users/irina/Desktop/master project/results_v2/physionet_64ch_strict_loso


## Загрузка PhysioNet

Используем полный PhysioNet Motor Movement/Imagery через MOABB.

В датасете:

- 109 испытуемых;
- 64 EEG-канала;
- частота дискретизации 160 Hz;
- есть motor execution и motor imagery;
- в этом эксперименте используем только **motor imagery**;
- классы — **left hand** и **right hand**.

MOABB будет использоваться только для унифицированной загрузки и epoching.  
После этого данные преобразуем в обычные `numpy`-массивы, чтобы оба pipeline — CCST и наш — работали с абсолютно одинаковыми trials.

In [3]:
# ============================================================
# PHYSIONET DATASET + PARADIGM
# ============================================================

dataset = PhysionetMI(
    imagined=True,
    executed=False,
)

subjects = dataset.subject_list

print("Dataset:", dataset)
print("Number of subjects:", len(subjects))
print("First subjects:", subjects[:10])


# ------------------------------------------------------------
# PARADIGM
# ------------------------------------------------------------

paradigm = MotorImagery(
    n_classes=2,
    events=EVENTS,

    fmin=FMIN,
    fmax=FMAX,

    resample=SFREQ,
)

print()
print("Paradigm:")
print(paradigm)

Dataset: PhysionetMotorImagery
Number of subjects: 109
First subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Paradigm:


In [4]:
# ============================================================
# SANITY CHECK — ONE SUBJECT
# ============================================================

TEST_LOAD_SUBJECT = subjects[0]

X_one, labels_one, meta_one = paradigm.get_data(
    dataset=dataset,
    subjects=[TEST_LOAD_SUBJECT],
)

print("=" * 80)
print("ONE SUBJECT SANITY CHECK")
print("=" * 80)

print("Subject:", TEST_LOAD_SUBJECT)

print("X shape:", X_one.shape)
print("X dtype:", X_one.dtype)

print(
    "Labels:",
    np.unique(
        labels_one,
        return_counts=True,
    )
)

print()
print("Metadata:")
display(
    meta_one.head()
)

This is nemar-py 0.3.1.
Preparing to download on004362 from https://data.nemar.org/
This is nemar-py 0.3.1.
Preparing to download on004362 from https://data.nemar.org/
Used Annotations descriptions: ['left_hand', 'rest', 'right_hand']
Used Annotations descriptions: ['left_hand', 'rest', 'right_hand']
Used Annotations descriptions: ['left_hand', 'rest', 'right_hand']
Used Annotations descriptions: ['feet', 'hands', 'rest']
Used Annotations descriptions: ['feet', 'hands', 'rest']
Used Annotations descriptions: ['feet', 'hands', 'rest']
ONE SUBJECT SANITY CHECK
Subject: 1
X shape: (45, 64, 481)
X dtype: float64
Labels: (array(['left_hand', 'right_hand'], dtype='<U10'), array([23, 22]))

Metadata:


,subject,session,run
0,1,0,0
1,1,0,0
2,1,0,0
3,1,0,0
4,1,0,0


In [5]:
# ============================================================
# CHECK SIGNAL DIMENSIONS
# ============================================================

n_trials = X_one.shape[0]
n_channels = X_one.shape[1]
n_samples = X_one.shape[2]

trial_duration = (
    n_samples
    / SFREQ
)

print("Trials:", n_trials)
print("Channels:", n_channels)
print("Samples:", n_samples)
print(
    f"Trial duration: {trial_duration:.3f} sec"
)

assert n_channels == 64, (
    f"Expected 64 channels, got {n_channels}"
)

assert set(
    np.unique(
        labels_one
    )
) == {
    "left_hand",
    "right_hand",
}

print()
print("64-channel binary L/R setup: OK")

Trials: 45
Channels: 64
Samples: 481
Trial duration: 3.006 sec

64-channel binary L/R setup: OK


## Подготовка полного датасета

109 subjects загружаем **по одному**, а не одним большим вызовом.

Причины:

1. меньше нагрузка на RAM;
2. если загрузка оборвётся, уже обработанные subjects останутся на диске;
3. можно повторно использовать один и тот же подготовленный датасет для CCST и нашего pipeline;
4. preprocessing выполняется один раз.

Для каждого субъекта сохраняем:

- `X`: EEG trials `[N, 64, T]`;
- `y`: `0 = left hand`, `1 = right hand`;
- номер субъекта;
- metadata.

После загрузки всех subjects соберём общий индекс, но не обязательно будем держать весь датасет в памяти одновременно.

In [7]:
# ============================================================
# LOAD + CACHE ALL SUBJECTS
# ============================================================

INDEX_PATH = (
    DATA_CACHE_DIR
    / "physionet64_index.csv"
)

index_rows = []


for i, subject in enumerate(
    subjects,
    start=1,
):

    subject_file = (
        DATA_CACHE_DIR
        / f"subject_{subject:03d}.npz"
    )

    # --------------------------------------------------------
    # RESUME
    # --------------------------------------------------------

    if subject_file.exists():

        cached = np.load(
            subject_file,
            allow_pickle=True,
        )

        X_sub = cached["X"]
        y_sub = cached["y"]

        print(
            f"[{i:03d}/{len(subjects)}] "
            f"S{subject:03d} cached "
            f"{X_sub.shape}"
        )

    else:

        # ----------------------------------------------------
        # LOAD THROUGH MOABB
        # ----------------------------------------------------

        X_sub, labels_sub, meta_sub = (
            paradigm.get_data(
                dataset=dataset,
                subjects=[subject],
            )
        )

        # float64 -> float32
        X_sub = np.asarray(
            X_sub,
            dtype=np.float32,
        )

        # ----------------------------------------------------
        # LABELS
        # ----------------------------------------------------

        y_sub = np.array(
            [
                LABEL_MAP[label]
                for label in labels_sub
            ],
            dtype=np.int64,
        )

        # ----------------------------------------------------
        # SANITY
        # ----------------------------------------------------

        assert X_sub.ndim == 3
        assert X_sub.shape[1] == 64

        assert set(
            np.unique(
                y_sub
            )
        ).issubset(
            {0, 1}
        )

        assert len(
            X_sub
        ) == len(
            y_sub
        )

        # ----------------------------------------------------
        # SAVE
        # ----------------------------------------------------

        np.savez_compressed(
            subject_file,

            X=
                X_sub,

            y=
                y_sub,

            subject=
                np.array(
                    [subject],
                    dtype=np.int64,
                ),
        )

        print(
            f"[{i:03d}/{len(subjects)}] "
            f"S{subject:03d} loaded "
            f"{X_sub.shape}"
        )


    # --------------------------------------------------------
    # INDEX
    # --------------------------------------------------------

    counts = np.bincount(
        y_sub,
        minlength=2,
    )

    index_rows.append({

        "subject":
            subject,

        "n_trials":
            len(
                y_sub
            ),

        "n_left":
            int(
                counts[0]
            ),

        "n_right":
            int(
                counts[1]
            ),

        "n_channels":
            X_sub.shape[1],

        "n_samples":
            X_sub.shape[2],

        "file":
            str(
                subject_file
            ),
    })


index_df = pd.DataFrame(
    index_rows
)

index_df.to_csv(
    INDEX_PATH,
    index=False,
)


print()
print("=" * 80)
print("DATASET CACHE COMPLETE")
print("=" * 80)

display(
    index_df.head()
)

This is nemar-py 0.3.1.
Preparing to download on004362 from https://data.nemar.org/
This is nemar-py 0.3.1.
Preparing to download on004362 from https://data.nemar.org/
Used Annotations descriptions: ['left_hand', 'rest', 'right_hand']
Used Annotations descriptions: ['left_hand', 'rest', 'right_hand']
Used Annotations descriptions: ['left_hand', 'rest', 'right_hand']
Used Annotations descriptions: ['feet', 'hands', 'rest']
Used Annotations descriptions: ['feet', 'hands', 'rest']
Used Annotations descriptions: ['feet', 'hands', 'rest']
[001/109] S001 loaded (45, 64, 481)
This is nemar-py 0.3.1.
Preparing to download on004362 from https://data.nemar.org/
This is nemar-py 0.3.1.
Preparing to download on004362 from https://data.nemar.org/
Used Annotations descriptions: ['left_hand', 'rest', 'right_hand']
Used Annotations descriptions: ['left_hand', 'rest', 'right_hand']
Used Annotations descriptions: ['left_hand', 'rest', 'right_hand']
Used Annotations descriptions: ['feet', 'hands', 'rest'

,subject,n_trials,n_left,n_right,n_channels,n_samples,file
0,1,45,23,22,64,481,/Users/irina/Desktop/master project/results_v2...
1,2,45,23,22,64,481,/Users/irina/Desktop/master project/results_v2...
2,3,45,23,22,64,481,/Users/irina/Desktop/master project/results_v2...
3,4,45,23,22,64,481,/Users/irina/Desktop/master project/results_v2...
4,5,45,21,24,64,481,/Users/irina/Desktop/master project/results_v2...


In [8]:
# ============================================================
# DATASET SUMMARY
# ============================================================

print("=" * 80)
print("PHYSIONET 64CH SUMMARY")
print("=" * 80)

print(
    "Subjects:",
    index_df["subject"].nunique(),
)

print(
    "Total trials:",
    index_df["n_trials"].sum(),
)

print(
    "Left trials:",
    index_df["n_left"].sum(),
)

print(
    "Right trials:",
    index_df["n_right"].sum(),
)

print(
    "Channels:",
    index_df["n_channels"].unique(),
)

print(
    "Samples per trial:",
    index_df["n_samples"].unique(),
)


assert (
    index_df[
        "subject"
    ].nunique()
    == 109
)

assert (
    index_df[
        "n_channels"
    ]
    ==
    64
).all()

assert (
    index_df[
        "n_samples"
    ].nunique()
    == 1
)


display(
    index_df.describe()
)

PHYSIONET 64CH SUMMARY
Subjects: 109
Total trials: 4918
Left trials: 2480
Right trials: 2438
Channels: [64]
Samples per trial: [481]


,subject,n_trials,n_left,n_right,n_channels,n_samples
count,109.000000,109.000000,109.000000,109.000000,109.0,109.0
mean,55.000000,45.119266,22.752294,22.366972,64.0,481.0
std,31.609598,1.854548,1.187559,1.302850,0.0,0.0
min,1.000000,36.000000,18.000000,18.000000,64.0,481.0
25%,28.000000,45.000000,22.000000,22.000000,64.0,481.0
50%,55.000000,45.000000,23.000000,22.000000,64.0,481.0
75%,82.000000,45.000000,23.000000,23.000000,64.0,481.0
max,109.000000,57.000000,29.000000,30.000000,64.0,481.0


In [9]:
# ============================================================
# SUBJECT LOADER
# ============================================================

def load_phys64_subjects(
    subject_ids,
):

    X_parts = []
    y_parts = []
    subject_parts = []


    for subject in subject_ids:

        path = (
            DATA_CACHE_DIR
            / f"subject_{int(subject):03d}.npz"
        )

        data = np.load(
            path,
            allow_pickle=False,
        )

        X = data["X"].astype(
            np.float32,
            copy=False,
        )

        y = data["y"].astype(
            np.int64,
            copy=False,
        )

        X_parts.append(
            X
        )

        y_parts.append(
            y
        )

        subject_parts.append(
            np.full(
                len(y),
                int(subject),
                dtype=np.int64,
            )
        )


    X = np.concatenate(
        X_parts,
        axis=0,
    )

    y = np.concatenate(
        y_parts,
        axis=0,
    )

    subject_array = np.concatenate(
        subject_parts,
        axis=0,
    )


    return (
        X,
        y,
        subject_array,
    )

In [10]:
# ============================================================
# STRICT LOSO PLAN
# ============================================================

def make_strict_loso_plan(
    subject_ids,
):

    subject_ids = np.array(
        sorted(
            subject_ids
        )
    )

    plan = []


    for i, test_subject in enumerate(
        subject_ids
    ):

        # следующий subject = validation
        val_subject = (
            subject_ids[
                (i + 1)
                %
                len(
                    subject_ids
                )
            ]
        )


        train_subjects = [
            int(s)
            for s in subject_ids
            if (
                s != test_subject
                and
                s != val_subject
            )
        ]


        plan.append({

            "fold":
                i + 1,

            "test_subject":
                int(
                    test_subject
                ),

            "val_subject":
                int(
                    val_subject
                ),

            "train_subjects":
                train_subjects,
        })


    return plan


loso_plan = make_strict_loso_plan(
    index_df[
        "subject"
    ].tolist()
)


print(
    "Number of folds:",
    len(
        loso_plan
    )
)

print()
print("Fold 1:")
print(
    loso_plan[0]
)

Number of folds: 109

Fold 1:
{'fold': 1, 'test_subject': 1, 'val_subject': 2, 'train_subjects': [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109]}


In [11]:
# ============================================================
# VERIFY STRICT LOSO
# ============================================================

def verify_strict_loso(
    plan,
    all_subjects,
):

    all_subjects = set(
        int(s)
        for s in all_subjects
    )

    test_subjects = []


    for fold in plan:

        train = set(
            fold[
                "train_subjects"
            ]
        )

        val = fold[
            "val_subject"
        ]

        test = fold[
            "test_subject"
        ]


        assert test not in train
        assert val not in train

        assert test != val

        assert (
            train
            |
            {val}
            |
            {test}
        ) == all_subjects


        test_subjects.append(
            test
        )


    assert (
        set(
            test_subjects
        )
        ==
        all_subjects
    )

    assert (
        len(
            test_subjects
        )
        ==
        len(
            all_subjects
        )
    )


    print(
        f"OK — {len(plan)} strict LOSO folds"
    )

    print(
        "Each subject appears exactly once as TEST."
    )

    print(
        "No TEST subject appears in TRAIN or VAL."
    )


verify_strict_loso(
    loso_plan,
    index_df[
        "subject"
    ].tolist(),
)

OK — 109 strict LOSO folds
Each subject appears exactly once as TEST.
No TEST subject appears in TRAIN or VAL.


In [12]:
# ============================================================
# SAVE LOSO PLAN
# ============================================================

loso_rows = []

for fold in loso_plan:

    loso_rows.append({

        "fold":
            fold["fold"],

        "test_subject":
            fold["test_subject"],

        "val_subject":
            fold["val_subject"],

        "n_train_subjects":
            len(
                fold[
                    "train_subjects"
                ]
            ),

        "train_subjects":
            ",".join(
                map(
                    str,
                    fold[
                        "train_subjects"
                    ],
                )
            ),
    })


loso_df = pd.DataFrame(
    loso_rows
)


LOSO_PATH = (
    SPLITS_DIR
    / "physionet64_strict_loso_plan.csv"
)


loso_df.to_csv(
    LOSO_PATH,
    index=False,
)


display(
    loso_df.head()
)

print(
    "Saved:",
    LOSO_PATH
)

,fold,test_subject,val_subject,n_train_subjects,train_subjects
0,1,1,2,107,"3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20..."
1,2,2,3,107,"1,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20..."
2,3,3,4,107,"1,2,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20..."
3,4,4,5,107,"1,2,3,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20..."
4,5,5,6,107,"1,2,3,4,7,8,9,10,11,12,13,14,15,16,17,18,19,20..."


Saved: /Users/irina/Desktop/master project/results_v2/physionet_64ch_strict_loso/splits/physionet64_strict_loso_plan.csv


## Зафиксированный benchmark

Дальше все модели обучаются и сравниваются на одном и том же протоколе:

- PhysioNet, 109 subjects
- 64 EEG-канала
- Left/Right motor imagery
- 160 Hz
- 7–30 Hz
- 3-секундные trials
- 109-fold strict LOSO
- 1 test subject
- 1 validation subject
- 107 train subjects
- test subject никогда не используется для выбора модели

Один и тот же LOSO-plan используется для всех методов:
1. EEGNet baseline;
2. CCST reproduction;
3. наш Cho Transfer + SupCon + GRL;
4. task-aware denoising.

## Этап 1. 64-channel EEGNet baseline

Перед воспроизведением CCST и запуском нашего subject-invariant pipeline фиксируем простой baseline.

Используем:

- те же 64 канала;
- те же L/R trials;
- тот же preprocessing 7–30 Hz;
- те же 109 strict LOSO folds;
- test subject полностью unseen;
- validation subject используется только для early stopping.

Сначала выполняем sanity-check на одном fold, затем запускаем полный 109-fold benchmark.

Этот результат станет общей контрольной точкой для дальнейшего сравнения CCST и нашего pipeline.

In [13]:
# ============================================================
# 64-CHANNEL EEGNET BASELINE
# training utilities
# ============================================================

from copy import deepcopy

BASELINE_DIR = (
    RESULTS_ROOT
    / "eegnet64_baseline"
)

BASELINE_MODEL_DIR = (
    BASELINE_DIR
    / "models"
)

BASELINE_HISTORY_DIR = (
    BASELINE_DIR
    / "histories"
)

BASELINE_RESULTS_DIR = (
    BASELINE_DIR
    / "results"
)

for p in [
    BASELINE_DIR,
    BASELINE_MODEL_DIR,
    BASELINE_HISTORY_DIR,
    BASELINE_RESULTS_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# TRAINING CONFIG
# ------------------------------------------------------------

BATCH_SIZE = 64

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

EPOCHS = 60
PATIENCE = 10


# ============================================================
# EEGNET BACKBONE — 64 CHANNELS
# ============================================================

class EEGNet64(nn.Module):

    def __init__(
        self,
        n_channels=64,
        n_samples=481,
        n_classes=2,
        dropout=0.5,
    ):
        super().__init__()

        # ----------------------------------------------------
        # temporal convolution
        # ----------------------------------------------------

        self.conv1 = nn.Conv2d(
            1,
            8,
            kernel_size=(1, 64),
            padding=(0, 32),
            bias=False,
        )

        self.bn1 = nn.BatchNorm2d(
            8
        )

        # ----------------------------------------------------
        # spatial filtering across ALL 64 EEG channels
        # ----------------------------------------------------

        self.depthwise = nn.Conv2d(
            8,
            16,
            kernel_size=(n_channels, 1),
            groups=8,
            bias=False,
        )

        self.bn2 = nn.BatchNorm2d(
            16
        )

        self.pool1 = nn.AvgPool2d(
            kernel_size=(1, 4)
        )

        self.dropout1 = nn.Dropout(
            dropout
        )

        # ----------------------------------------------------
        # separable temporal convolution
        # ----------------------------------------------------

        self.conv2 = nn.Conv2d(
            16,
            16,
            kernel_size=(1, 16),
            padding=(0, 8),
            groups=16,
            bias=False,
        )

        self.conv3 = nn.Conv2d(
            16,
            16,
            kernel_size=(1, 1),
            bias=False,
        )

        self.bn3 = nn.BatchNorm2d(
            16
        )

        self.pool2 = nn.AvgPool2d(
            kernel_size=(1, 8)
        )

        self.dropout2 = nn.Dropout(
            dropout
        )

        # ----------------------------------------------------
        # determine feature dimension automatically
        # ----------------------------------------------------

        with torch.no_grad():

            dummy = torch.zeros(
                1,
                1,
                n_channels,
                n_samples,
            )

            features = self._forward_features(
                dummy
            )

            self.feature_dim = int(
                features.flatten(
                    1
                ).shape[1]
            )

        self.fc = nn.Linear(
            self.feature_dim,
            n_classes,
        )


    def _forward_features(
        self,
        x,
    ):

        x = self.conv1(
            x
        )

        x = self.bn1(
            x
        )

        x = self.depthwise(
            x
        )

        x = self.bn2(
            x
        )

        x = F.elu(
            x
        )

        x = self.pool1(
            x
        )

        x = self.dropout1(
            x
        )


        x = self.conv2(
            x
        )

        x = self.conv3(
            x
        )

        x = self.bn3(
            x
        )

        x = F.elu(
            x
        )

        x = self.pool2(
            x
        )

        x = self.dropout2(
            x
        )

        return x


    def forward(
        self,
        x,
    ):

        # [B, 64, T]
        # ->
        # [B, 1, 64, T]

        if x.ndim == 3:

            x = x.unsqueeze(
                1
            )

        x = self._forward_features(
            x
        )

        x = x.flatten(
            1
        )

        return self.fc(
            x
        )


# ============================================================
# LOADER
# ============================================================

def make_loader64(
    X,
    y,
    batch_size=64,
    shuffle=False,
    seed=42,
):

    dataset = TensorDataset(
        torch.from_numpy(
            np.asarray(
                X,
                dtype=np.float32,
            )
        ),

        torch.from_numpy(
            np.asarray(
                y,
                dtype=np.int64,
            )
        ),
    )

    generator = torch.Generator()

    generator.manual_seed(
        seed
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        generator=generator,
    )


# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_model64(
    model,
    loader,
):

    model.eval()

    y_true = []
    y_pred = []

    total_loss = 0.0
    total_n = 0

    for xb, yb in loader:

        xb = xb.to(
            DEVICE
        )

        yb = yb.to(
            DEVICE
        )

        logits = model(
            xb
        )

        loss = F.cross_entropy(
            logits,
            yb,
        )

        pred = logits.argmax(
            dim=1
        )

        n = len(
            yb
        )

        total_loss += (
            loss.item()
            * n
        )

        total_n += n

        y_true.append(
            yb.cpu().numpy()
        )

        y_pred.append(
            pred.cpu().numpy()
        )


    y_true = np.concatenate(
        y_true
    )

    y_pred = np.concatenate(
        y_pred
    )


    return {

        "loss":
            total_loss
            / total_n,

        "accuracy":
            accuracy_score(
                y_true,
                y_pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred,
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
            ),

        "kappa":
            cohen_kappa_score(
                y_true,
                y_pred,
            ),
    }


# ============================================================
# TRAIN ONE FOLD
# ============================================================

def train_eegnet64_fold(
    X_train,
    y_train,

    X_val,
    y_val,

    seed=42,

    epochs=60,
    patience=10,

    verbose=True,
):

    set_seed(
        seed
    )


    model = EEGNet64(
        n_channels=64,
        n_samples=X_train.shape[-1],
        n_classes=2,
    ).to(
        DEVICE
    )


    train_loader = make_loader64(
        X_train,
        y_train,

        batch_size=
            BATCH_SIZE,

        shuffle=True,
        seed=seed,
    )


    val_loader = make_loader64(
        X_val,
        y_val,

        batch_size=
            128,

        shuffle=False,
    )


    optimizer = torch.optim.AdamW(
        model.parameters(),

        lr=
            LEARNING_RATE,

        weight_decay=
            WEIGHT_DECAY,
    )


    best_state = None

    best_val_accuracy = -np.inf
    best_val_f1 = -np.inf

    best_epoch = 0

    wait = 0

    history = []


    # ========================================================
    # EPOCHS
    # ========================================================

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        train_correct = 0
        train_total = 0
        train_loss_sum = 0.0


        for xb, yb in train_loader:

            xb = xb.to(
                DEVICE
            )

            yb = yb.to(
                DEVICE
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            logits = model(
                xb
            )


            loss = F.cross_entropy(
                logits,
                yb,
            )


            loss.backward()


            optimizer.step()


            pred = logits.argmax(
                dim=1
            )


            n = len(
                yb
            )


            train_loss_sum += (
                loss.item()
                * n
            )

            train_correct += (
                pred
                .eq(
                    yb
                )
                .sum()
                .item()
            )

            train_total += n


        train_accuracy = (
            train_correct
            / train_total
        )


        val_metrics = (
            evaluate_model64(
                model,
                val_loader,
            )
        )


        row = {

            "epoch":
                epoch,

            "train_loss":
                train_loss_sum
                / train_total,

            "train_accuracy":
                train_accuracy,

            "val_loss":
                val_metrics[
                    "loss"
                ],

            "val_accuracy":
                val_metrics[
                    "accuracy"
                ],

            "val_macro_f1":
                val_metrics[
                    "macro_f1"
                ],
        }


        history.append(
            row
        )


        if verbose:

            print(
                f"epoch {epoch:03d}"
                f" | train={train_accuracy:.4f}"
                f" | val={val_metrics['accuracy']:.4f}"
                f" | f1={val_metrics['macro_f1']:.4f}"
            )


        # ----------------------------------------------------
        # model selection by validation ONLY
        # ----------------------------------------------------

        improved = (
            val_metrics[
                "accuracy"
            ]
            >
            best_val_accuracy
            + 1e-8
        )


        tie_improved = (
            abs(
                val_metrics[
                    "accuracy"
                ]
                -
                best_val_accuracy
            )
            <= 1e-8

            and

            val_metrics[
                "macro_f1"
            ]
            >
            best_val_f1
        )


        if (
            improved
            or
            tie_improved
        ):

            best_val_accuracy = (
                val_metrics[
                    "accuracy"
                ]
            )

            best_val_f1 = (
                val_metrics[
                    "macro_f1"
                ]
            )

            best_epoch = epoch

            best_state = deepcopy(
                model.state_dict()
            )

            wait = 0


        else:

            wait += 1


        if wait >= patience:

            if verbose:

                print(
                    "EARLY STOP",
                    "| best epoch:",
                    best_epoch,
                    "| best val:",
                    round(
                        best_val_accuracy,
                        4,
                    ),
                )

            break


    # ========================================================
    # LOAD BEST
    # ========================================================

    model.load_state_dict(
        best_state
    )


    return {

        "model":
            model,

        "best_epoch":
            best_epoch,

        "best_val_accuracy":
            best_val_accuracy,

        "best_val_f1":
            best_val_f1,

        "history":
            pd.DataFrame(
                history
            ),
    }


print("EEGNet64 utilities ready")

EEGNet64 utilities ready


In [14]:
# ============================================================
# SANITY CHECK — FOLD 1
# ============================================================

sanity_fold = loso_plan[0]

sanity_test_subject = (
    sanity_fold[
        "test_subject"
    ]
)

sanity_val_subject = (
    sanity_fold[
        "val_subject"
    ]
)

sanity_train_subjects = (
    sanity_fold[
        "train_subjects"
    ]
)


print("=" * 80)
print("SANITY CHECK")
print("=" * 80)

print(
    "TEST:",
    sanity_test_subject
)

print(
    "VAL:",
    sanity_val_subject
)

print(
    "TRAIN SUBJECTS:",
    len(
        sanity_train_subjects
    )
)


# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

X_train, y_train, _ = (
    load_phys64_subjects(
        sanity_train_subjects
    )
)

X_val, y_val, _ = (
    load_phys64_subjects(
        [
            sanity_val_subject
        ]
    )
)

X_test, y_test, _ = (
    load_phys64_subjects(
        [
            sanity_test_subject
        ]
    )
)


print()
print(
    "train:",
    X_train.shape,
    np.bincount(
        y_train
    )
)

print(
    "val:",
    X_val.shape,
    np.bincount(
        y_val
    )
)

print(
    "test:",
    X_test.shape,
    np.bincount(
        y_test
    )
)


# ------------------------------------------------------------
# TRAIN 3 EPOCHS
# ------------------------------------------------------------

sanity_result = (
    train_eegnet64_fold(
        X_train,
        y_train,

        X_val,
        y_val,

        seed=SEED,

        epochs=3,
        patience=3,

        verbose=True,
    )
)


# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------

test_loader = make_loader64(
    X_test,
    y_test,

    batch_size=128,
    shuffle=False,
)


sanity_test_metrics = (
    evaluate_model64(
        sanity_result[
            "model"
        ],

        test_loader,
    )
)


print()
print("=" * 80)
print("SANITY RESULT")
print("=" * 80)

print(
    "best epoch:",
    sanity_result[
        "best_epoch"
    ]
)

print(
    "best val accuracy:",
    round(
        sanity_result[
            "best_val_accuracy"
        ],
        4,
    )
)

print(
    "test accuracy:",
    round(
        sanity_test_metrics[
            "accuracy"
        ],
        4,
    )
)

print(
    "test balanced accuracy:",
    round(
        sanity_test_metrics[
            "balanced_accuracy"
        ],
        4,
    )
)

print(
    "test macro F1:",
    round(
        sanity_test_metrics[
            "macro_f1"
        ],
        4,
    )
)


del X_train
del y_train
del X_val
del y_val
del X_test
del y_test

del sanity_result

gc.collect()

if DEVICE.type == "mps":
    torch.mps.empty_cache()

print()
print("SANITY CHECK COMPLETE")

SANITY CHECK
TEST: 1
VAL: 2
TRAIN SUBJECTS: 107

train: (4828, 64, 481) [2434 2394]
val: (45, 64, 481) [23 22]
test: (45, 64, 481) [23 22]
epoch 001 | train=0.4901 | val=0.4667 | f1=0.3779
epoch 002 | train=0.5137 | val=0.4889 | f1=0.4796
epoch 003 | train=0.5141 | val=0.4222 | f1=0.3855

SANITY RESULT
best epoch: 2
best val accuracy: 0.4889
test accuracy: 0.5556
test balanced accuracy: 0.5543
test macro F1: 0.5536

SANITY CHECK COMPLETE


## Ночной эксперимент: CCST vs наш subject-invariant pipeline

На одном и том же 64-канальном binary L/R benchmark сравниваем два подхода:

### 1. CCST
Compact Convolutional Swin Transformer:

- temporal convolution;
- spatial convolution по всем 64 EEG-каналам;
- Swin-style window attention;
- 3 transformer blocks;
- 4 attention heads;
- window size = 4.

### 2. Наш pipeline
Зафиксированный ранее рецепт:

**Cho2017 transfer → SupCon → GRL**

При переходе с Cho2017 (22 канала) на PhysioNet (64 канала):

- переносим только shape-compatible веса;
- temporal convolution и последующие совместимые слои инициализируются из Cho;
- spatial convolution `22 → 64` обучается заново;
- classification head обучается заново;
- subject discriminator используется только при обучении.

### Общий evaluation protocol

- PhysioNet
- 109 subjects
- 64 channels
- Left vs Right motor imagery
- 7–30 Hz
- strict LOSO
- один полностью unseen test subject
- validation берётся только из остальных subjects
- один и тот же test fold используется обоими методами
- test не используется для выбора модели

Сначала выполняется sanity-check на одном fold, затем полный resume-safe прогон 109 folds × 2 метода.

In [15]:
# ============================================================
# CCST vs OUR PIPELINE
# PhysioNet 64ch / binary L-R / strict LOSO
# ============================================================

from pathlib import Path
from copy import deepcopy

import gc
import math
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    cohen_kappa_score,
)


# ============================================================
# CONFIG
# ============================================================

NIGHT_SEED = 42

# CCST — official-paper/repo configuration
CCST_BATCH_SIZE = 72
CCST_EPOCHS = 100
CCST_PATIENCE = 10
CCST_LR = 3e-4

# OUR frozen recipe
OUR_BATCH_SIZE = 64
OUR_EPOCHS = 60
OUR_PATIENCE = 10
OUR_LR = 1e-3
OUR_WEIGHT_DECAY = 1e-4

LAMBDA_SUPCON = 0.15
LAMBDA_ADV = 0.10

CHO_CHECKPOINT = (
    PROJECT_ROOT
    / "models_v2"
    / "cho2017_binary_unified_eegnet.pt"
)

NIGHT_ROOT = (
    RESULTS_ROOT
    / "night_ccst_vs_ours"
)

MODEL_DIR = NIGHT_ROOT / "models"
HISTORY_DIR = NIGHT_ROOT / "histories"
RESULT_DIR = NIGHT_ROOT / "results"

for p in [
    NIGHT_ROOT,
    MODEL_DIR,
    HISTORY_DIR,
    RESULT_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )


def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def clear_memory():

    gc.collect()

    if DEVICE.type == "mps":
        torch.mps.empty_cache()

    elif DEVICE.type == "cuda":
        torch.cuda.empty_cache()


print("DEVICE:", DEVICE)
print("Cho checkpoint:", CHO_CHECKPOINT)
print("Output:", NIGHT_ROOT)


# ============================================================
# COMMON FOLD PREPARATION
# ============================================================

def prepare_fold_data(
    test_subject,
    seed,
):
    """
    Test = one completely unseen subject.

    Remaining 108 subjects:
      90% trials -> train
      10% trials -> validation

    Validation split is stratified by class.

    Returns train/val/test with subject ids.
    """

    all_subjects = (
        index_df[
            "subject"
        ]
        .astype(int)
        .tolist()
    )

    source_subjects = [
        s
        for s in all_subjects
        if s != int(test_subject)
    ]

    X_source, y_source, s_source = (
        load_phys64_subjects(
            source_subjects
        )
    )

    X_test, y_test, s_test = (
        load_phys64_subjects(
            [int(test_subject)]
        )
    )

    indices = np.arange(
        len(y_source)
    )

    train_idx, val_idx = (
        train_test_split(
            indices,
            test_size=0.10,
            stratify=y_source,
            random_state=seed,
        )
    )

    X_train = X_source[train_idx]
    y_train = y_source[train_idx]
    s_train = s_source[train_idx]

    X_val = X_source[val_idx]
    y_val = y_source[val_idx]
    s_val = s_source[val_idx]

    del X_source
    del y_source
    del s_source

    # --------------------------------------------------------
    # TRAIN-ONLY channel-wise z-score
    # --------------------------------------------------------
    # statistics: [1, C, 1]
    # calculated ONLY from training trials
    # --------------------------------------------------------

    mean = X_train.mean(
        axis=(0, 2),
        keepdims=True,
        dtype=np.float64,
    ).astype(np.float32)

    std = X_train.std(
        axis=(0, 2),
        keepdims=True,
        dtype=np.float64,
    ).astype(np.float32)

    std = np.maximum(
        std,
        1e-6,
    )

    # in-place to avoid huge extra arrays
    X_train -= mean
    X_train /= std

    X_val -= mean
    X_val /= std

    X_test -= mean
    X_test /= std

    return {
        "X_train": X_train.astype(np.float32, copy=False),
        "y_train": y_train.astype(np.int64, copy=False),
        "s_train": s_train.astype(np.int64, copy=False),

        "X_val": X_val.astype(np.float32, copy=False),
        "y_val": y_val.astype(np.int64, copy=False),
        "s_val": s_val.astype(np.int64, copy=False),

        "X_test": X_test.astype(np.float32, copy=False),
        "y_test": y_test.astype(np.int64, copy=False),
        "s_test": s_test.astype(np.int64, copy=False),
    }


# ============================================================
# COMMON EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_logits_model(
    model,
    loader,
    mode="plain",
):

    model.eval()

    y_true = []
    y_pred = []

    total_loss = 0.0
    total_n = 0

    for batch in loader:

        xb = batch[0].to(
            DEVICE
        )

        yb = batch[1].to(
            DEVICE
        )

        if mode == "plain":

            logits = model(
                xb
            )

        elif mode == "ours":

            output = model(
                xb,
                grl_coeff=0.0,
            )

            logits = output[
                "class_logits"
            ]

        else:

            raise ValueError(
                mode
            )

        loss = F.cross_entropy(
            logits,
            yb,
        )

        pred = logits.argmax(
            dim=1
        )

        n = len(
            yb
        )

        total_loss += (
            loss.item()
            * n
        )

        total_n += n

        y_true.append(
            yb.cpu().numpy()
        )

        y_pred.append(
            pred.cpu().numpy()
        )

    y_true = np.concatenate(
        y_true
    )

    y_pred = np.concatenate(
        y_pred
    )

    return {
        "loss":
            total_loss
            / total_n,

        "accuracy":
            accuracy_score(
                y_true,
                y_pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred,
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
            ),

        "kappa":
            cohen_kappa_score(
                y_true,
                y_pred,
            ),
    }


# ============================================================
# ============================================================
# CCST
# ============================================================
# ============================================================


# ============================================================
# CCST DATASET + AUGMENTATION
# ============================================================

class CCSTDataset(
    Dataset
):

    def __init__(
        self,
        X,
        y,
        augment=False,
        max_shift=8,
    ):

        self.X = X
        self.y = y

        self.augment = augment

        # official repository values
        self.max_shift = max_shift
        self.noise_amp = 0.01
        self.dropout_rate = 0.05


    def __len__(
        self,
    ):

        return len(
            self.X
        )


    def __getitem__(
        self,
        idx,
    ):

        x = torch.from_numpy(
            self.X[idx]
        ).float()

        y = torch.tensor(
            int(
                self.y[idx]
            ),
            dtype=torch.long,
        )

        # x [C,T] -> [1,C,T]
        x = x.unsqueeze(
            0
        )

        if self.augment:

            _, C, T = x.shape

            # temporal shift
            shift = random.randint(
                -self.max_shift,
                self.max_shift,
            )

            if shift > 0:

                x = torch.cat(
                    [
                        x[:, :, shift:],
                        torch.zeros(
                            1,
                            C,
                            shift,
                        ),
                    ],
                    dim=2,
                )

            elif shift < 0:

                sh = -shift

                x = torch.cat(
                    [
                        torch.zeros(
                            1,
                            C,
                            sh,
                        ),
                        x[:, :, :T-sh],
                    ],
                    dim=2,
                )

            # gaussian noise
            x = (
                x
                +
                torch.randn_like(x)
                * self.noise_amp
            )

            # electrode dropout
            n_drop = max(
                1,
                int(
                    C
                    * self.dropout_rate
                ),
            )

            drop_idx = torch.randperm(
                C
            )[:n_drop]

            x[
                :,
                drop_idx,
                :
            ] = 0.0

        return (
            x,
            y,
        )


# ============================================================
# SWIN HELPERS
# ============================================================

def pad_sequence_1d(
    x,
    window_size,
):

    B, L, C = x.shape

    remainder = (
        L
        %
        window_size
    )

    if remainder == 0:

        return (
            x,
            L,
            0,
        )

    pad_len = (
        window_size
        -
        remainder
    )

    pad = torch.zeros(
        B,
        pad_len,
        C,
        dtype=x.dtype,
        device=x.device,
    )

    return (
        torch.cat(
            [
                x,
                pad,
            ],
            dim=1,
        ),
        L,
        pad_len,
    )


def window_partition_1d(
    x,
    window_size,
):

    x_pad, orig_len, pad_len = (
        pad_sequence_1d(
            x,
            window_size,
        )
    )

    B, L, C = x_pad.shape

    n_windows = (
        L
        //
        window_size
    )

    windows = (
        x_pad
        .view(
            B,
            n_windows,
            window_size,
            C,
        )
        .reshape(
            B
            * n_windows,
            window_size,
            C,
        )
    )

    return (
        windows,
        (
            orig_len,
            pad_len,
            n_windows,
        ),
    )


def window_reverse_1d(
    windows,
    window_size,
    info,
):

    orig_len, pad_len, n_windows = (
        info
    )

    Bnw, _, C = windows.shape

    B = (
        Bnw
        //
        n_windows
    )

    x = (
        windows
        .view(
            B,
            n_windows,
            window_size,
            C,
        )
        .reshape(
            B,
            n_windows
            * window_size,
            C,
        )
    )

    if pad_len > 0:

        x = x[
            :,
            :orig_len,
            :
        ]

    return x


# ============================================================
# CCST ATTENTION
# ============================================================

class CCSTAttention(
    nn.Module
):

    def __init__(
        self,
        dim,
        num_heads,
        dropout=0.1,
    ):

        super().__init__()

        self.num_heads = (
            num_heads
        )

        head_dim = (
            dim
            //
            num_heads
        )

        self.scale = (
            head_dim
            ** -0.5
        )

        self.qkv = nn.Linear(
            dim,
            dim * 3,
            bias=False,
        )

        self.attn_drop = (
            nn.Dropout(
                dropout
            )
        )

        self.proj = nn.Linear(
            dim,
            dim,
        )

        self.proj_drop = (
            nn.Dropout(
                dropout
            )
        )


    def forward(
        self,
        x,
    ):

        B, L, C = x.shape

        qkv = (
            self.qkv(x)
            .reshape(
                B,
                L,
                3,
                self.num_heads,
                C
                //
                self.num_heads,
            )
            .permute(
                2,
                0,
                3,
                1,
                4,
            )
        )

        q, k, v = (
            qkv[0],
            qkv[1],
            qkv[2],
        )

        attn = (
            q
            @
            k.transpose(
                -2,
                -1,
            )
        ) * self.scale

        attn = attn.softmax(
            dim=-1
        )

        attn = self.attn_drop(
            attn
        )

        out = (
            attn
            @ v
        )

        out = (
            out
            .transpose(
                1,
                2,
            )
            .reshape(
                B,
                L,
                C,
            )
        )

        out = self.proj(
            out
        )

        return self.proj_drop(
            out
        )


class CCSTMLP(
    nn.Module
):

    def __init__(
        self,
        dim,
        hidden_dim=128,
        dropout=0.1,
    ):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                dim,
                hidden_dim,
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim,
                dim,
            ),

            nn.Dropout(
                dropout
            ),
        )


    def forward(
        self,
        x,
    ):

        return self.net(
            x
        )


class Swin1DBlock(
    nn.Module
):

    def __init__(
        self,
        dim=64,
        num_heads=4,
        window_size=4,
        shift_size=0,
        mlp_hidden=128,
        dropout=0.1,
    ):

        super().__init__()

        self.window_size = (
            window_size
        )

        self.shift_size = (
            shift_size
        )

        self.ln1 = nn.LayerNorm(
            dim
        )

        self.attn = CCSTAttention(
            dim,
            num_heads,
            dropout,
        )

        self.ln2 = nn.LayerNorm(
            dim
        )

        self.mlp = CCSTMLP(
            dim,
            mlp_hidden,
            dropout,
        )


    def forward(
        self,
        x,
    ):

        if self.shift_size > 0:

            x = torch.roll(
                x,
                shifts=
                    -self.shift_size,
                dims=1,
            )

        windows, info = (
            window_partition_1d(
                x,
                self.window_size,
            )
        )

        shortcut = windows

        windows = (
            shortcut
            +
            self.attn(
                self.ln1(
                    windows
                )
            )
        )

        shortcut = windows

        windows = (
            shortcut
            +
            self.mlp(
                self.ln2(
                    windows
                )
            )
        )

        x = window_reverse_1d(
            windows,
            self.window_size,
            info,
        )

        if self.shift_size > 0:

            x = torch.roll(
                x,
                shifts=
                    self.shift_size,
                dims=1,
            )

        return x


class CCST64(
    nn.Module
):

    def __init__(
        self,
        n_channels=64,
        n_samples=481,
        n_classes=2,
    ):

        super().__init__()

        # ----------------------------------------------------
        # Patch embedding from official CCST implementation
        # ----------------------------------------------------

        self.conv_temp = nn.Conv2d(
            1,
            40,
            kernel_size=(
                1,
                25,
            ),
            stride=(
                1,
                1,
            ),
        )

        self.conv_spat = nn.Conv2d(
            40,
            40,
            kernel_size=(
                n_channels,
                1,
            ),
        )

        self.bn = nn.BatchNorm2d(
            40
        )

        self.act = nn.ELU()

        self.pool = nn.AvgPool2d(
            kernel_size=(
                1,
                75,
            ),
            stride=(
                1,
                15,
            ),
        )

        self.dropout_patch = (
            nn.Dropout(
                0.5
            )
        )

        self.proj_conv = nn.Conv2d(
            40,
            40,
            kernel_size=(
                1,
                1,
            ),
        )

        # ----------------------------------------------------
        # determine sequence length dynamically
        # ----------------------------------------------------

        with torch.no_grad():

            dummy = torch.zeros(
                1,
                1,
                n_channels,
                n_samples,
            )

            z = self._patch_forward(
                dummy
            )

            seq_len = (
                z.shape[1]
            )

        self.embedding_projection = (
            nn.Linear(
                40,
                64,
            )
        )

        self.pos_encoding = (
            nn.Parameter(
                torch.zeros(
                    1,
                    seq_len,
                    64,
                )
            )
        )

        nn.init.trunc_normal_(
            self.pos_encoding,
            std=0.02,
        )

        self.blocks = nn.ModuleList(
            [
                Swin1DBlock(
                    dim=64,
                    num_heads=4,
                    window_size=4,
                    shift_size=(
                        2
                        if i % 2 == 1
                        else 0
                    ),
                    mlp_hidden=128,
                    dropout=0.1,
                )
                for i in range(3)
            ]
        )

        self.norm = nn.LayerNorm(
            64
        )

        self.head = nn.Sequential(

            nn.Dropout(
                0.3
            ),

            nn.Linear(
                64,
                n_classes,
            ),
        )


    def _patch_forward(
        self,
        x,
    ):

        x = self.conv_temp(
            x
        )

        x = self.conv_spat(
            x
        )

        x = self.bn(
            x
        )

        x = self.act(
            x
        )

        x = self.pool(
            x
        )

        x = self.dropout_patch(
            x
        )

        x = self.proj_conv(
            x
        )

        # [B,40,1,T'] -> [B,T',40]
        x = (
            x
            .squeeze(2)
            .transpose(
                1,
                2,
            )
        )

        return x


    def forward(
        self,
        x,
    ):

        x = self._patch_forward(
            x
        )

        x = self.embedding_projection(
            x
        )

        x = (
            x
            +
            self.pos_encoding[
                :,
                :x.shape[1],
                :
            ]
        )

        for block in self.blocks:

            x = block(
                x
            )

        x = self.norm(
            x
        )

        features = x.mean(
            dim=1
        )

        return self.head(
            features
        )


# ============================================================
# CCST TRAIN
# ============================================================

def train_ccst_fold(
    data,
    seed,
    epochs=CCST_EPOCHS,
    patience=CCST_PATIENCE,
    verbose=True,
):

    set_seed(
        seed
    )

    train_ds = CCSTDataset(
        data["X_train"],
        data["y_train"],
        augment=True,
    )

    val_ds = CCSTDataset(
        data["X_val"],
        data["y_val"],
        augment=False,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=
            CCST_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=128,
        shuffle=False,
        num_workers=0,
    )

    model = CCST64(
        n_channels=64,
        n_samples=
            data[
                "X_train"
            ].shape[-1],
        n_classes=2,
    ).to(
        DEVICE
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=CCST_LR,
        betas=(
            0.9,
            0.999,
        ),
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=5,
        )
    )

    best_state = None
    best_val = -np.inf
    best_epoch = 0

    wait = 0
    history = []

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        total = 0
        correct = 0
        loss_sum = 0.0

        for xb, yb in train_loader:

            xb = xb.to(
                DEVICE
            )

            yb = yb.to(
                DEVICE
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                xb
            )

            loss = F.cross_entropy(
                logits,
                yb,
            )

            loss.backward()

            optimizer.step()

            n = len(
                yb
            )

            loss_sum += (
                loss.item()
                * n
            )

            correct += (
                logits.argmax(1)
                .eq(yb)
                .sum()
                .item()
            )

            total += n

        train_loss = (
            loss_sum
            /
            total
        )

        train_acc = (
            correct
            /
            total
        )

        val_metrics = (
            evaluate_logits_model(
                model,
                val_loader,
                mode="plain",
            )
        )

        scheduler.step(
            val_metrics[
                "loss"
            ]
        )

        history.append({
            "epoch":
                epoch,
            "train_loss":
                train_loss,
            "train_accuracy":
                train_acc,
            "val_loss":
                val_metrics[
                    "loss"
                ],
            "val_accuracy":
                val_metrics[
                    "accuracy"
                ],
        })

        if verbose:

            print(
                f"CCST | ep={epoch:03d}"
                f" | train={train_acc:.4f}"
                f" | val={val_metrics['accuracy']:.4f}"
            )

        if (
            val_metrics[
                "accuracy"
            ]
            >
            best_val
        ):

            best_val = (
                val_metrics[
                    "accuracy"
                ]
            )

            best_epoch = epoch

            best_state = deepcopy(
                model.state_dict()
            )

            wait = 0

        else:

            wait += 1

        if wait >= patience:

            break

    model.load_state_dict(
        best_state
    )

    return {
        "model":
            model,
        "best_epoch":
            best_epoch,
        "best_val_accuracy":
            best_val,
        "history":
            pd.DataFrame(
                history
            ),
    }


# ============================================================
# ============================================================
# OUR PIPELINE — Cho Transfer + SupCon + GRL
# ============================================================
# ============================================================


# ============================================================
# EEGNET64 BACKBONE
# ============================================================

class EEGNetBackbone64(
    nn.Module
):

    def __init__(
        self,
        n_channels=64,
        n_samples=481,
        dropout=0.5,
    ):

        super().__init__()

        self.conv1 = nn.Conv2d(
            1,
            8,
            kernel_size=(
                1,
                64,
            ),
            padding=(
                0,
                32,
            ),
            bias=False,
        )

        self.bn1 = nn.BatchNorm2d(
            8
        )

        self.depthwise = nn.Conv2d(
            8,
            16,
            kernel_size=(
                n_channels,
                1,
            ),
            groups=8,
            bias=False,
        )

        self.bn2 = nn.BatchNorm2d(
            16
        )

        self.pool1 = nn.AvgPool2d(
            kernel_size=(
                1,
                4,
            )
        )

        self.dropout1 = nn.Dropout(
            dropout
        )

        self.conv2 = nn.Conv2d(
            16,
            16,
            kernel_size=(
                1,
                16,
            ),
            padding=(
                0,
                8,
            ),
            groups=16,
            bias=False,
        )

        self.conv3 = nn.Conv2d(
            16,
            16,
            kernel_size=(
                1,
                1,
            ),
            bias=False,
        )

        self.bn3 = nn.BatchNorm2d(
            16
        )

        self.pool2 = nn.AvgPool2d(
            kernel_size=(
                1,
                8,
            )
        )

        self.dropout2 = nn.Dropout(
            dropout
        )

        with torch.no_grad():

            dummy = torch.zeros(
                1,
                n_channels,
                n_samples,
            )

            out = self.forward(
                dummy
            )

            self.feature_dim = (
                out.shape[1]
            )


    def forward(
        self,
        x,
    ):

        if x.ndim == 3:

            x = x.unsqueeze(
                1
            )

        x = self.conv1(
            x
        )

        x = self.bn1(
            x
        )

        x = self.depthwise(
            x
        )

        x = self.bn2(
            x
        )

        x = F.elu(
            x
        )

        x = self.pool1(
            x
        )

        x = self.dropout1(
            x
        )

        x = self.conv2(
            x
        )

        x = self.conv3(
            x
        )

        x = self.bn3(
            x
        )

        x = F.elu(
            x
        )

        x = self.pool2(
            x
        )

        x = self.dropout2(
            x
        )

        return x.flatten(
            1
        )


# ============================================================
# GRL
# ============================================================

class GradientReversalFunction(
    torch.autograd.Function
):

    @staticmethod
    def forward(
        ctx,
        x,
        coeff,
    ):

        ctx.coeff = coeff

        return x.view_as(
            x
        )


    @staticmethod
    def backward(
        ctx,
        grad_output,
    ):

        return (
            -ctx.coeff
            * grad_output,
            None,
        )


def grad_reverse(
    x,
    coeff,
):

    return (
        GradientReversalFunction
        .apply(
            x,
            coeff,
        )
    )


# ============================================================
# SUPCON
# ============================================================

class SupConLoss(
    nn.Module
):

    def __init__(
        self,
        temperature=0.07,
    ):

        super().__init__()

        self.temperature = (
            temperature
        )


    def forward(
        self,
        features,
        labels,
    ):

        features = F.normalize(
            features,
            dim=1,
        )

        logits = (
            features
            @
            features.T
        ) / self.temperature

        labels = labels.view(
            -1,
            1,
        )

        mask = (
            labels
            ==
            labels.T
        ).float()

        eye = torch.eye(
            len(labels),
            device=
                features.device,
        )

        mask = (
            mask
            *
            (
                1
                -
                eye
            )
        )

        logits_mask = (
            1
            -
            eye
        )

        logits = (
            logits
            -
            logits.max(
                dim=1,
                keepdim=True,
            )[0].detach()
        )

        exp_logits = (
            torch.exp(logits)
            *
            logits_mask
        )

        log_prob = (
            logits
            -
            torch.log(
                exp_logits.sum(
                    dim=1,
                    keepdim=True,
                )
                +
                1e-12
            )
        )

        positives = (
            mask.sum(
                dim=1
            )
        )

        valid = (
            positives
            > 0
        )

        mean_log_prob_pos = (
            (
                mask
                *
                log_prob
            ).sum(
                dim=1
            )
            /
            (
                positives
                +
                1e-12
            )
        )

        return (
            -mean_log_prob_pos[
                valid
            ].mean()
        )


# ============================================================
# OUR MODEL
# ============================================================

class OurSubjectInvariant64(
    nn.Module
):

    def __init__(
        self,
        n_train_subjects,
        n_samples=481,
    ):

        super().__init__()

        self.backbone = (
            EEGNetBackbone64(
                n_channels=64,
                n_samples=n_samples,
            )
        )

        dim = (
            self.backbone
            .feature_dim
        )

        self.projector = nn.Sequential(

            nn.Linear(
                dim,
                256,
            ),

            nn.ReLU(),

            nn.Linear(
                256,
                128,
            ),
        )

        self.fc = nn.Linear(
            dim,
            2,
        )

        self.subject_head = (
            nn.Sequential(

                nn.Linear(
                    dim,
                    128,
                ),

                nn.ReLU(),

                nn.Dropout(
                    0.3
                ),

                nn.Linear(
                    128,
                    n_train_subjects,
                ),
            )
        )


    def forward(
        self,
        x,
        grl_coeff=1.0,
    ):

        features = self.backbone(
            x
        )

        class_logits = self.fc(
            features
        )

        projection = self.projector(
            features
        )

        subject_logits = (
            self.subject_head(
                grad_reverse(
                    features,
                    grl_coeff,
                )
            )
        )

        return {
            "class_logits":
                class_logits,
            "projection":
                projection,
            "subject_logits":
                subject_logits,
            "features":
                features,
        }


# ============================================================
# PARTIAL CHO TRANSFER
# ============================================================

def extract_state_dict(
    checkpoint,
):

    if (
        isinstance(
            checkpoint,
            dict,
        )
        and
        "model_state_dict"
        in checkpoint
    ):

        return checkpoint[
            "model_state_dict"
        ]

    if (
        isinstance(
            checkpoint,
            dict,
        )
        and
        "state_dict"
        in checkpoint
    ):

        return checkpoint[
            "state_dict"
        ]

    return checkpoint


def load_cho_partial(
    model,
):

    checkpoint = torch.load(
        CHO_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )

    source = extract_state_dict(
        checkpoint
    )

    target = model.state_dict()

    compatible = {}

    for key, value in source.items():

        key_clean = (
            key[7:]
            if key.startswith(
                "module."
            )
            else key
        )

        # direct match
        if (
            key_clean in target
            and
            target[
                key_clean
            ].shape
            ==
            value.shape
        ):

            compatible[
                key_clean
            ] = value

        # old model may have backbone keys already
        elif (
            "backbone."
            +
            key_clean
        ) in target:

            new_key = (
                "backbone."
                +
                key_clean
            )

            if (
                target[
                    new_key
                ].shape
                ==
                value.shape
            ):

                compatible[
                    new_key
                ] = value

    target.update(
        compatible
    )

    model.load_state_dict(
        target
    )

    print(
        "Transferred compatible Cho tensors:",
        len(
            compatible
        )
    )

    return len(
        compatible
    )


# ============================================================
# OUR DATASET
# ============================================================

class OurDataset(
    Dataset
):

    def __init__(
        self,
        X,
        y,
        subjects,
        subject_map,
    ):

        self.X = X
        self.y = y
        self.subjects = subjects
        self.subject_map = subject_map


    def __len__(
        self,
    ):

        return len(
            self.X
        )


    def __getitem__(
        self,
        idx,
    ):

        return (

            torch.from_numpy(
                self.X[idx]
            ).float(),

            torch.tensor(
                int(
                    self.y[idx]
                ),
                dtype=torch.long,
            ),

            torch.tensor(
                self.subject_map[
                    int(
                        self.subjects[idx]
                    )
                ],
                dtype=torch.long,
            ),
        )


# ============================================================
# OUR TRAIN
# ============================================================

def train_our_fold(
    data,
    seed,
    epochs=OUR_EPOCHS,
    patience=OUR_PATIENCE,
    verbose=True,
):

    set_seed(
        seed
    )

    train_subjects = sorted(
        np.unique(
            data[
                "s_train"
            ]
        )
    )

    subject_map = {
        int(subject):
            i
        for i, subject
        in enumerate(
            train_subjects
        )
    }

    model = (
        OurSubjectInvariant64(
            n_train_subjects=
                len(
                    train_subjects
                ),
            n_samples=
                data[
                    "X_train"
                ].shape[-1],
        )
        .to(
            DEVICE
        )
    )

    load_cho_partial(
        model
    )

    train_ds = OurDataset(
        data["X_train"],
        data["y_train"],
        data["s_train"],
        subject_map,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=
            OUR_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )

    # validation only needs X/y
    val_ds = torch.utils.data.TensorDataset(

        torch.from_numpy(
            data[
                "X_val"
            ]
        ).float(),

        torch.from_numpy(
            data[
                "y_val"
            ]
        ).long(),
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=128,
        shuffle=False,
        num_workers=0,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=OUR_LR,
        weight_decay=
            OUR_WEIGHT_DECAY,
    )

    supcon = SupConLoss()

    best_state = None
    best_val = -np.inf
    best_f1 = -np.inf
    best_epoch = 0

    wait = 0
    history = []

    n_batches = len(
        train_loader
    )

    total_steps = max(
        1,
        epochs
        *
        n_batches,
    )

    step = 0

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        total = 0
        correct = 0

        loss_sum = 0.0
        ce_sum = 0.0
        sc_sum = 0.0
        adv_sum = 0.0

        for (
            xb,
            yb,
            sb,
        ) in train_loader:

            xb = xb.to(
                DEVICE
            )

            yb = yb.to(
                DEVICE
            )

            sb = sb.to(
                DEVICE
            )

            p = (
                step
                /
                total_steps
            )

            grl_coeff = (
                2.0
                /
                (
                    1.0
                    +
                    math.exp(
                        -10.0
                        * p
                    )
                )
                -
                1.0
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            out = model(
                xb,
                grl_coeff=
                    grl_coeff,
            )

            ce = F.cross_entropy(
                out[
                    "class_logits"
                ],
                yb,
            )

            sc = supcon(
                out[
                    "projection"
                ],
                yb,
            )

            adv = F.cross_entropy(
                out[
                    "subject_logits"
                ],
                sb,
            )

            loss = (
                ce
                +
                LAMBDA_SUPCON
                * sc
                +
                LAMBDA_ADV
                * adv
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0,
            )

            optimizer.step()

            pred = (
                out[
                    "class_logits"
                ]
                .argmax(
                    dim=1
                )
            )

            n = len(
                yb
            )

            correct += (
                pred.eq(yb)
                .sum()
                .item()
            )

            total += n

            loss_sum += (
                loss.item()
                * n
            )

            ce_sum += (
                ce.item()
                * n
            )

            sc_sum += (
                sc.item()
                * n
            )

            adv_sum += (
                adv.item()
                * n
            )

            step += 1

        train_acc = (
            correct
            /
            total
        )

        val_metrics = (
            evaluate_logits_model(
                model,
                val_loader,
                mode="ours",
            )
        )

        history.append({
            "epoch":
                epoch,
            "train_loss":
                loss_sum / total,
            "train_accuracy":
                train_acc,
            "ce":
                ce_sum / total,
            "supcon":
                sc_sum / total,
            "adv":
                adv_sum / total,
            "val_accuracy":
                val_metrics[
                    "accuracy"
                ],
            "val_macro_f1":
                val_metrics[
                    "macro_f1"
                ],
        })

        if verbose:

            print(
                f"OURS | ep={epoch:03d}"
                f" | train={train_acc:.4f}"
                f" | val={val_metrics['accuracy']:.4f}"
                f" | CE={ce_sum/total:.4f}"
                f" | SC={sc_sum/total:.4f}"
                f" | ADV={adv_sum/total:.4f}"
            )

        improved = (
            val_metrics[
                "accuracy"
            ]
            >
            best_val
            +
            1e-8
        )

        tie = (
            abs(
                val_metrics[
                    "accuracy"
                ]
                -
                best_val
            )
            <=
            1e-8
            and
            val_metrics[
                "macro_f1"
            ]
            >
            best_f1
        )

        if improved or tie:

            best_val = (
                val_metrics[
                    "accuracy"
                ]
            )

            best_f1 = (
                val_metrics[
                    "macro_f1"
                ]
            )

            best_epoch = epoch

            best_state = deepcopy(
                model.state_dict()
            )

            wait = 0

        else:

            wait += 1

        if wait >= patience:

            break

    model.load_state_dict(
        best_state
    )

    return {
        "model":
            model,
        "best_epoch":
            best_epoch,
        "best_val_accuracy":
            best_val,
        "history":
            pd.DataFrame(
                history
            ),
    }


print()
print("=" * 80)
print("MODELS AND TRAINING FUNCTIONS READY")
print("=" * 80)

DEVICE: mps
Cho checkpoint: /Users/irina/Desktop/master project/models_v2/cho2017_binary_unified_eegnet.pt
Output: /Users/irina/Desktop/master project/results_v2/physionet_64ch_strict_loso/night_ccst_vs_ours

MODELS AND TRAINING FUNCTIONS READY


In [16]:
# ============================================================
# SANITY CHECK — CCST + OUR PIPELINE
# ============================================================

SANITY_TEST_SUBJECT = 1
SANITY_SEED = 1042

print("=" * 100)
print("SANITY CHECK")
print("=" * 100)

data = prepare_fold_data(
    test_subject=
        SANITY_TEST_SUBJECT,
    seed=
        SANITY_SEED,
)

print(
    "train:",
    data[
        "X_train"
    ].shape
)

print(
    "val:",
    data[
        "X_val"
    ].shape
)

print(
    "test:",
    data[
        "X_test"
    ].shape
)

print(
    "train labels:",
    np.bincount(
        data[
            "y_train"
        ]
    )
)

print(
    "val labels:",
    np.bincount(
        data[
            "y_val"
        ]
    )
)

print(
    "test labels:",
    np.bincount(
        data[
            "y_test"
        ]
    )
)


# ============================================================
# CCST SANITY
# ============================================================

print()
print("#" * 100)
print("CCST SANITY")
print("#" * 100)

ccst_sanity = train_ccst_fold(
    data,
    seed=
        SANITY_SEED,
    epochs=2,
    patience=2,
    verbose=True,
)


ccst_test_loader = DataLoader(

    CCSTDataset(
        data[
            "X_test"
        ],
        data[
            "y_test"
        ],
        augment=False,
    ),

    batch_size=128,
    shuffle=False,
)


ccst_test = (
    evaluate_logits_model(
        ccst_sanity[
            "model"
        ],
        ccst_test_loader,
        mode="plain",
    )
)

print(
    "CCST TEST:",
    ccst_test
)


# ============================================================
# OUR SANITY
# ============================================================

print()
print("#" * 100)
print("OUR PIPELINE SANITY")
print("#" * 100)

our_sanity = train_our_fold(
    data,
    seed=
        SANITY_SEED,
    epochs=2,
    patience=2,
    verbose=True,
)


our_test_loader = DataLoader(

    torch.utils.data.TensorDataset(

        torch.from_numpy(
            data[
                "X_test"
            ]
        ).float(),

        torch.from_numpy(
            data[
                "y_test"
            ]
        ).long(),
    ),

    batch_size=128,
    shuffle=False,
)


our_test = (
    evaluate_logits_model(
        our_sanity[
            "model"
        ],
        our_test_loader,
        mode="ours",
    )
)

print(
    "OURS TEST:",
    our_test
)


del ccst_sanity
del our_sanity
del data

clear_memory()

print()
print("SANITY COMPLETE")

SANITY CHECK
train: (4385, 64, 481)
val: (488, 64, 481)
test: (45, 64, 481)
train labels: [2211 2174]
val labels: [246 242]
test labels: [23 22]

####################################################################################################
CCST SANITY
####################################################################################################
CCST | ep=001 | train=0.4960 | val=0.5369
CCST | ep=002 | train=0.5357 | val=0.5615
CCST TEST: {'loss': 0.6795050501823425, 'accuracy': 0.6222222222222222, 'balanced_accuracy': 0.6254940711462451, 'macro_f1': 0.6153846153846154, 'kappa': 0.24926398429833174}

####################################################################################################
OUR PIPELINE SANITY
####################################################################################################
Transferred compatible Cho tensors: 12
OURS | ep=001 | train=0.4960 | val=0.5041 | CE=0.7061 | SC=4.1621 | ADV=4.6080
OURS | ep=002 | train=0.5056 | val=0.514

In [17]:
# ============================================================
# NIGHT RUN
#
# 109 strict LOSO folds
#
# each fold:
#   1. CCST
#   2. Cho Transfer + SupCon + GRL
#
# resume-safe by (test_subject, method)
# ============================================================

PROGRESS_PATH = (
    RESULT_DIR
    / "night_ccst_vs_ours_progress.csv"
)

FINAL_PATH = (
    RESULT_DIR
    / "night_ccst_vs_ours_final.csv"
)


# ============================================================
# RESUME
# ============================================================

if PROGRESS_PATH.exists():

    previous = pd.read_csv(
        PROGRESS_PATH
    )

    rows = previous.to_dict(
        "records"
    )

else:

    rows = []


completed = {
    (
        int(row["test_subject"]),
        row["method"],
    )
    for row in rows
}


all_subjects = (
    index_df[
        "subject"
    ]
    .astype(int)
    .tolist()
)


print("=" * 100)
print("PHYSIONET 64CH — NIGHT RUN")
print("CCST vs CHO TRANSFER + SUPCON + GRL")
print("=" * 100)

print(
    "Subjects:",
    len(
        all_subjects
    )
)

print(
    "Already completed:",
    len(
        completed
    ),
    "/",
    len(
        all_subjects
    )
    * 2,
)


# ============================================================
# FOLDS
# ============================================================

for fold_idx, test_subject in enumerate(
    all_subjects,
    start=1,
):

    print()
    print("#" * 100)
    print(
        f"FOLD {fold_idx:03d}/109"
        f" | TEST SUBJECT={test_subject}"
    )
    print("#" * 100)

    need_ccst = (
        test_subject,
        "CCST",
    ) not in completed

    need_ours = (
        test_subject,
        "ChoTransfer_SupCon_GRL",
    ) not in completed


    if (
        not need_ccst
        and
        not need_ours
    ):

        print(
            "Both methods already done — SKIP"
        )

        continue


    fold_seed = (
        NIGHT_SEED
        +
        fold_idx
    )


    # ========================================================
    # PREPARE DATA ONCE
    # ========================================================

    data = prepare_fold_data(
        test_subject=
            test_subject,
        seed=
            fold_seed,
    )


    print(
        "train/val/test:",
        len(
            data[
                "y_train"
            ]
        ),
        len(
            data[
                "y_val"
            ]
        ),
        len(
            data[
                "y_test"
            ]
        ),
    )


    # ========================================================
    # 1. CCST
    # ========================================================

    if need_ccst:

        print()
        print("=" * 80)
        print("TRAIN CCST")
        print("=" * 80)

        trained = train_ccst_fold(
            data,
            seed=
                fold_seed,
            verbose=True,
        )

        test_loader = DataLoader(

            CCSTDataset(
                data[
                    "X_test"
                ],
                data[
                    "y_test"
                ],
                augment=False,
            ),

            batch_size=128,
            shuffle=False,
            num_workers=0,
        )

        metrics = (
            evaluate_logits_model(
                trained[
                    "model"
                ],
                test_loader,
                mode="plain",
            )
        )

        print()
        print(
            "CCST TEST"
            f" | acc={metrics['accuracy']:.4f}"
            f" | bacc={metrics['balanced_accuracy']:.4f}"
            f" | f1={metrics['macro_f1']:.4f}"
        )


        # ----------------------------------------------------
        # SAVE MODEL
        # ----------------------------------------------------

        model_path = (
            MODEL_DIR
            /
            f"CCST_S{test_subject:03d}.pt"
        )

        torch.save(

            {

                "model_state_dict":
                    trained[
                        "model"
                    ].state_dict(),

                "method":
                    "CCST",

                "test_subject":
                    test_subject,

                "fold":
                    fold_idx,

                "seed":
                    fold_seed,

                "best_epoch":
                    trained[
                        "best_epoch"
                    ],

                "best_val_accuracy":
                    trained[
                        "best_val_accuracy"
                    ],
            },

            model_path,
        )


        trained[
            "history"
        ].to_csv(

            HISTORY_DIR
            /
            f"CCST_S{test_subject:03d}.csv",

            index=False,
        )


        rows.append({

            "fold":
                fold_idx,

            "test_subject":
                test_subject,

            "method":
                "CCST",

            "seed":
                fold_seed,

            "best_epoch":
                trained[
                    "best_epoch"
                ],

            "best_val_accuracy":
                trained[
                    "best_val_accuracy"
                ],

            "test_accuracy":
                metrics[
                    "accuracy"
                ],

            "test_balanced_accuracy":
                metrics[
                    "balanced_accuracy"
                ],

            "test_macro_f1":
                metrics[
                    "macro_f1"
                ],

            "test_kappa":
                metrics[
                    "kappa"
                ],
        })


        pd.DataFrame(
            rows
        ).to_csv(
            PROGRESS_PATH,
            index=False,
        )


        completed.add(
            (
                test_subject,
                "CCST",
            )
        )


        del trained
        del test_loader

        clear_memory()


    # ========================================================
    # 2. OUR PIPELINE
    # ========================================================

    if need_ours:

        print()
        print("=" * 80)
        print(
            "TRAIN CHO TRANSFER + SUPCON + GRL"
        )
        print("=" * 80)

        trained = train_our_fold(
            data,
            seed=
                fold_seed,
            verbose=True,
        )

        test_loader = DataLoader(

            torch.utils.data.TensorDataset(

                torch.from_numpy(
                    data[
                        "X_test"
                    ]
                ).float(),

                torch.from_numpy(
                    data[
                        "y_test"
                    ]
                ).long(),
            ),

            batch_size=128,
            shuffle=False,
            num_workers=0,
        )


        metrics = (
            evaluate_logits_model(
                trained[
                    "model"
                ],
                test_loader,
                mode="ours",
            )
        )


        print()
        print(
            "OURS TEST"
            f" | acc={metrics['accuracy']:.4f}"
            f" | bacc={metrics['balanced_accuracy']:.4f}"
            f" | f1={metrics['macro_f1']:.4f}"
        )


        # ----------------------------------------------------
        # SAVE
        # ----------------------------------------------------

        model_path = (
            MODEL_DIR
            /
            (
                f"OURS_GRL_"
                f"S{test_subject:03d}.pt"
            )
        )

        torch.save(

            {

                "model_state_dict":
                    trained[
                        "model"
                    ].state_dict(),

                "method":
                    "ChoTransfer_SupCon_GRL",

                "test_subject":
                    test_subject,

                "fold":
                    fold_idx,

                "seed":
                    fold_seed,

                "best_epoch":
                    trained[
                        "best_epoch"
                    ],

                "best_val_accuracy":
                    trained[
                        "best_val_accuracy"
                    ],

                "lambda_supcon":
                    LAMBDA_SUPCON,

                "lambda_adv":
                    LAMBDA_ADV,

                "cho_checkpoint":
                    str(
                        CHO_CHECKPOINT
                    ),
            },

            model_path,
        )


        trained[
            "history"
        ].to_csv(

            HISTORY_DIR
            /
            (
                f"OURS_GRL_"
                f"S{test_subject:03d}.csv"
            ),

            index=False,
        )


        rows.append({

            "fold":
                fold_idx,

            "test_subject":
                test_subject,

            "method":
                "ChoTransfer_SupCon_GRL",

            "seed":
                fold_seed,

            "best_epoch":
                trained[
                    "best_epoch"
                ],

            "best_val_accuracy":
                trained[
                    "best_val_accuracy"
                ],

            "test_accuracy":
                metrics[
                    "accuracy"
                ],

            "test_balanced_accuracy":
                metrics[
                    "balanced_accuracy"
                ],

            "test_macro_f1":
                metrics[
                    "macro_f1"
                ],

            "test_kappa":
                metrics[
                    "kappa"
                ],
        })


        pd.DataFrame(
            rows
        ).to_csv(
            PROGRESS_PATH,
            index=False,
        )


        completed.add(
            (
                test_subject,
                "ChoTransfer_SupCon_GRL",
            )
        )


        del trained
        del test_loader

        clear_memory()


    # ========================================================
    # CURRENT SUMMARY
    # ========================================================

    current = pd.DataFrame(
        rows
    )

    print()
    print("-" * 80)
    print("RUNNING SUMMARY")
    print("-" * 80)

    display(
        current
        .groupby(
            "method"
        )
        .agg(
            mean_accuracy=(
                "test_accuracy",
                "mean",
            ),
            std_accuracy=(
                "test_accuracy",
                "std",
            ),
            n=(
                "test_subject",
                "nunique",
            ),
        )
        .assign(
            mean_accuracy_pct=
                lambda d:
                    100
                    *
                    d[
                        "mean_accuracy"
                    ]
        )
        .round(
            4
        )
    )


    # ========================================================
    # CLEAN CURRENT FOLD
    # ========================================================

    del data

    clear_memory()


# ============================================================
# FINAL TABLE
# ============================================================

final_df = pd.DataFrame(
    rows
)

final_df = (
    final_df
    .drop_duplicates(
        subset=[
            "test_subject",
            "method",
        ],
        keep="last",
    )
    .sort_values(
        [
            "test_subject",
            "method",
        ]
    )
    .reset_index(
        drop=True
    )
)

final_df.to_csv(
    FINAL_PATH,
    index=False,
)


summary = (

    final_df
    .groupby(
        "method"
    )
    .agg(

        mean_accuracy=(
            "test_accuracy",
            "mean",
        ),

        std_accuracy=(
            "test_accuracy",
            "std",
        ),

        median_accuracy=(
            "test_accuracy",
            "median",
        ),

        mean_balanced_accuracy=(
            "test_balanced_accuracy",
            "mean",
        ),

        mean_macro_f1=(
            "test_macro_f1",
            "mean",
        ),

        mean_kappa=(
            "test_kappa",
            "mean",
        ),

        n_subjects=(
            "test_subject",
            "nunique",
        ),
    )
    .reset_index()
)


summary[
    "mean_accuracy_pct"
] = (
    100
    *
    summary[
        "mean_accuracy"
    ]
)


print()
print("=" * 100)
print("FINAL NIGHT SUMMARY")
print("=" * 100)

display(
    summary.round(
        4
    )
)

print()
print(
    "Published CCST reference:",
    "71.89%"
)

print(
    "Saved:",
    FINAL_PATH
)

PHYSIONET 64CH — NIGHT RUN
CCST vs CHO TRANSFER + SUPCON + GRL
Subjects: 109
Already completed: 0 / 218

####################################################################################################
FOLD 001/109 | TEST SUBJECT=1
####################################################################################################
train/val/test: 4385 488 45

TRAIN CCST
CCST | ep=001 | train=0.5017 | val=0.5389
CCST | ep=002 | train=0.5382 | val=0.5697
CCST | ep=003 | train=0.5738 | val=0.5615
CCST | ep=004 | train=0.5943 | val=0.5738
CCST | ep=005 | train=0.6212 | val=0.6004
CCST | ep=006 | train=0.6440 | val=0.5984
CCST | ep=007 | train=0.6497 | val=0.6619
CCST | ep=008 | train=0.6675 | val=0.6475
CCST | ep=009 | train=0.6652 | val=0.6557
CCST | ep=010 | train=0.6819 | val=0.5287
CCST | ep=011 | train=0.6748 | val=0.6373
CCST | ep=012 | train=0.6990 | val=0.6393
CCST | ep=013 | train=0.6949 | val=0.6783
CCST | ep=014 | train=0.7117 | val=0.6250
CCST | ep=015 | train=0.7247 | val=

,mean_accuracy,std_accuracy,n,mean_accuracy_pct
method,,,,
CCST,0.7556,NaN,1,75.5556
ChoTransfer_SupCon_GRL,0.7111,NaN,1,71.1111



####################################################################################################
FOLD 002/109 | TEST SUBJECT=2
####################################################################################################
train/val/test: 4385 488 45

TRAIN CCST
CCST | ep=001 | train=0.5099 | val=0.5348
CCST | ep=002 | train=0.5332 | val=0.5799
CCST | ep=003 | train=0.5831 | val=0.5656
CCST | ep=004 | train=0.5938 | val=0.6311
CCST | ep=005 | train=0.6144 | val=0.5902
CCST | ep=006 | train=0.6363 | val=0.6496
CCST | ep=007 | train=0.6570 | val=0.6414
CCST | ep=008 | train=0.6552 | val=0.6230
CCST | ep=009 | train=0.6661 | val=0.6824
CCST | ep=010 | train=0.6880 | val=0.6906
CCST | ep=011 | train=0.6919 | val=0.5799
CCST | ep=012 | train=0.7115 | val=0.6660
CCST | ep=013 | train=0.7044 | val=0.6967
CCST | ep=014 | train=0.7040 | val=0.6250
CCST | ep=015 | train=0.7252 | val=0.6988
CCST | ep=016 | train=0.7341 | val=0.6865
CCST | ep=017 | train=0.7345 | val=0.7049
CCST | ep=018

,mean_accuracy,std_accuracy,n,mean_accuracy_pct
method,,,,
CCST,0.8111,0.0786,2,81.1111
ChoTransfer_SupCon_GRL,0.8000,0.1257,2,80.0000



####################################################################################################
FOLD 003/109 | TEST SUBJECT=3
####################################################################################################
train/val/test: 4385 488 45

TRAIN CCST
CCST | ep=001 | train=0.5106 | val=0.5205
CCST | ep=002 | train=0.5357 | val=0.6250
CCST | ep=003 | train=0.5619 | val=0.5881
CCST | ep=004 | train=0.5979 | val=0.6127
CCST | ep=005 | train=0.6057 | val=0.5656
CCST | ep=006 | train=0.6367 | val=0.6475
CCST | ep=007 | train=0.6431 | val=0.6209
CCST | ep=008 | train=0.6502 | val=0.5922
CCST | ep=009 | train=0.6534 | val=0.6311
CCST | ep=010 | train=0.6664 | val=0.6066
CCST | ep=011 | train=0.6990 | val=0.6557
CCST | ep=012 | train=0.7081 | val=0.6803
CCST | ep=013 | train=0.7003 | val=0.6639
CCST | ep=014 | train=0.7241 | val=0.6660
CCST | ep=015 | train=0.7348 | val=0.6988
CCST | ep=016 | train=0.7304 | val=0.7090
CCST | ep=017 | train=0.7398 | val=0.6250
CCST | ep=018

,mean_accuracy,std_accuracy,n,mean_accuracy_pct
method,,,,
CCST,0.7630,0.1002,3,76.2963
ChoTransfer_SupCon_GRL,0.7407,0.1358,3,74.0741


... (211 outputs left out)

,mean_accuracy,std_accuracy,n,mean_accuracy_pct
method,,,,
CCST,0.7365,0.1041,109,73.6512
ChoTransfer_SupCon_GRL,0.6868,0.1162,109,68.6783



FINAL NIGHT SUMMARY


,method,mean_accuracy,std_accuracy,median_accuracy,mean_balanced_accuracy,mean_macro_f1,mean_kappa,n_subjects,mean_accuracy_pct
0,CCST,0.7365,0.1041,0.7333,0.7372,0.7249,0.4737,109,73.6512
1,ChoTransfer_SupCon_GRL,0.6868,0.1162,0.6889,0.6875,0.6684,0.3740,109,68.6783



Published CCST reference: 71.89%
Saved: /Users/irina/Desktop/master project/results_v2/physionet_64ch_strict_loso/night_ccst_vs_ours/results/night_ccst_vs_ours_final.csv


In [18]:
# ============================================================
# SAVE FINAL EXPERIMENT SNAPSHOT
# ============================================================

from pathlib import Path
import json
import pandas as pd

SAVE_DIR = (
    RESULTS_ROOT
    / "night_ccst_vs_ours"
    / "FINAL_SNAPSHOT"
)

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# 1. Полные результаты по всем subjects
# ------------------------------------------------------------

final_df.to_csv(
    SAVE_DIR / "all_subject_results.csv",
    index=False,
)


# ------------------------------------------------------------
# 2. Итоговая сводка
# ------------------------------------------------------------

summary.to_csv(
    SAVE_DIR / "summary.csv",
    index=False,
)


# ------------------------------------------------------------
# 3. Paired-таблица: один subject = две модели
# ------------------------------------------------------------

paired = (
    final_df
    .pivot(
        index="test_subject",
        columns="method",
        values="test_accuracy",
    )
    .reset_index()
)

paired["delta_ours_minus_ccst_pp"] = (
    (
        paired["ChoTransfer_SupCon_GRL"]
        -
        paired["CCST"]
    )
    * 100
)

paired.to_csv(
    SAVE_DIR / "paired_ccst_vs_ours.csv",
    index=False,
)


# ------------------------------------------------------------
# 4. Зафиксировать протокол
# ------------------------------------------------------------

experiment_info = {

    "dataset":
        "PhysioNet EEG Motor Movement/Imagery",

    "subjects":
        109,

    "channels":
        64,

    "classes":
        [
            "left_hand",
            "right_hand",
        ],

    "sampling_rate":
        160,

    "bandpass":
        [
            7,
            30,
        ],

    "trial_samples":
        481,

    "evaluation":
        "109-fold strict LOSO",

    "validation":
        "10% of source-subject trials, stratified",

    "ccst_mean_accuracy":
        float(
            summary.loc[
                summary["method"] == "CCST",
                "mean_accuracy"
            ].iloc[0]
        ),

    "ours_mean_accuracy":
        float(
            summary.loc[
                summary["method"] == "ChoTransfer_SupCon_GRL",
                "mean_accuracy"
            ].iloc[0]
        ),

    "published_ccst_reference":
        0.7189,

    "our_method":
        "Cho Transfer + SupCon + GRL",

    "notes":
        (
            "CCST and our pipeline evaluated on identical "
            "64-channel binary L/R LOSO benchmark."
        ),
}


with open(
    SAVE_DIR / "experiment_info.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        experiment_info,
        f,
        ensure_ascii=False,
        indent=2,
    )


print("=" * 80)
print("EXPERIMENT SAVED")
print("=" * 80)

print("Folder:", SAVE_DIR)

print()
print("CCST:")
print(
    f"{100 * experiment_info['ccst_mean_accuracy']:.2f}%"
)

print("Our pipeline:")
print(
    f"{100 * experiment_info['ours_mean_accuracy']:.2f}%"
)

print("Published CCST:")
print("71.89%")

EXPERIMENT SAVED
Folder: /Users/irina/Desktop/master project/results_v2/physionet_64ch_strict_loso/night_ccst_vs_ours/FINAL_SNAPSHOT

CCST:
73.65%
Our pipeline:
68.68%
Published CCST:
71.89%


# Эксперимент 2. CCST на 22 каналах

Цель:
сравнить CCST с нашим Cho Transfer + SupCon + GRL
в той же 22-канальной strict-LOSO постановке,
в которой наш pipeline ранее давал ~73.8–74.4%.

# Эксперимент 2. CCST на 22 каналах

В этом эксперименте воспроизводим CCST на той же 22-канальной
постановке PhysioNet, которая использовалась в основном пайплайне.

Условия:
- 106 испытуемых;
- исключены subjects 88, 92 и 100;
- Left Hand / Right Hand;
- 22 EEG-канала;
- 7–30 Hz;
- 160 Hz;
- 481 отсчёт на эпоху;
- strict LOSO.

Цель — напрямую сравнить CCST с
Cho Transfer + SupCon + GRL в одинаковой постановке.

In [19]:
# ============================================================
# PHYSIONET 22-CHANNEL SETUP
# ============================================================

PHYS22_CHANNELS = [
    "Fz",
    "FC3", "FC1", "FCz", "FC2", "FC4",
    "C5", "C3", "C1", "Cz", "C2", "C4", "C6",
    "CP3", "CP1", "CPz", "CP2", "CP4",
    "P1", "Pz", "P2",
    "POz",
]

EXCLUDED_SUBJECTS_22 = {88, 92, 100}

PHYS22_SUBJECTS = [
    s for s in range(1, 110)
    if s not in EXCLUDED_SUBJECTS_22
]

print("Каналов:", len(PHYS22_CHANNELS))
print("Subjects:", len(PHYS22_SUBJECTS))
print("Excluded:", sorted(EXCLUDED_SUBJECTS_22))

Каналов: 22
Subjects: 106
Excluded: [88, 92, 100]


In [ ]:
dataset_phys22 = PhysionetMI(
    imagined=True,
    executed=False,
)

paradigm_phys22 = MotorImagery(
    n_classes=2,
    events=[
        "left_hand",
        "right_hand",
    ],
    fmin=7,
    fmax=30,
    resample=160,
)


# Берём одного subject только для получения порядка каналов
epochs_tmp, labels_tmp, metadata_tmp = paradigm_phys22.get_data(
    dataset=dataset_phys22,
    subjects=[1],
    return_epochs=True,
)

PHYS64_CHANNELS = list(epochs_tmp.ch_names)

print("Всего каналов:", len(PHYS64_CHANNELS))
print()
print(PHYS64_CHANNELS)

This is nemar-py 0.3.1.
Preparing to download on004362 from https://data.nemar.org/
This is nemar-py 0.3.1.
Preparing to download on004362 from https://data.nemar.org/
Used Annotations descriptions: ['left_hand', 'rest', 'right_hand']
Used Annotations descriptions: ['left_hand', 'rest', 'right_hand']
Used Annotations descriptions: ['left_hand', 'rest', 'right_hand']
Used Annotations descriptions: ['feet', 'hands', 'rest']
Used Annotations descriptions: ['feet', 'hands', 'rest']
Used Annotations descriptions: ['feet', 'hands', 'rest']
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
Adding metadata with 3 columns
45 matching events found
No baseline correction applied
Всего каналов: 64

['FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6', 'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP5', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'CP6', 'Fp1', 'Fpz', 'Fp2', 'AF7', 'AF3', 'AFz', 'AF4', 'AF8', 'F7', 'F5', 'F3', 'F1', 'Fz', 'F2', 'F4', 'F6', 'F8', 'FT7', 'FT

In [21]:
# ============================================================
# 22 CHANNEL INDICES INSIDE 64CH PHYSIONET
# ============================================================

missing_channels = [
    ch for ch in PHYS22_CHANNELS
    if ch not in PHYS64_CHANNELS
]

if missing_channels:
    raise ValueError(
        f"Не найдены каналы: {missing_channels}"
    )

PHYS22_IDX = [
    PHYS64_CHANNELS.index(ch)
    for ch in PHYS22_CHANNELS
]

print("22-channel indices:")
print(PHYS22_IDX)

print()
print("Проверка порядка:")

for i, idx in enumerate(PHYS22_IDX):
    print(
        f"{i:02d}: "
        f"{PHYS64_CHANNELS[idx]:>4s} "
        f"(64ch index={idx})"
    )

22-channel indices:
[33, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 49, 50, 51, 57]

Проверка порядка:
00:   Fz (64ch index=33)
01:  FC3 (64ch index=1)
02:  FC1 (64ch index=2)
03:  FCz (64ch index=3)
04:  FC2 (64ch index=4)
05:  FC4 (64ch index=5)
06:   C5 (64ch index=7)
07:   C3 (64ch index=8)
08:   C1 (64ch index=9)
09:   Cz (64ch index=10)
10:   C2 (64ch index=11)
11:   C4 (64ch index=12)
12:   C6 (64ch index=13)
13:  CP3 (64ch index=15)
14:  CP1 (64ch index=16)
15:  CPz (64ch index=17)
16:  CP2 (64ch index=18)
17:  CP4 (64ch index=19)
18:   P1 (64ch index=49)
19:   Pz (64ch index=50)
20:   P2 (64ch index=51)
21:  POz (64ch index=57)


In [ ]:
# ============================================================
# PATHS
# ============================================================

CACHE64_DIR = (
    RESULTS_ROOT
    / "data_cache"
)

CACHE22_DIR = (
    RESULTS_ROOT
    / "data_cache_22ch"
)

CACHE22_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# CREATE 22CH CACHE FROM EXISTING 64CH CACHE
# ============================================================

summary_22 = []

for subject in PHYS22_SUBJECTS:

    src = (
        CACHE64_DIR
        / f"subject_{subject:03d}.npz"
    )

    dst = (
        CACHE22_DIR
        / f"subject_{subject:03d}.npz"
    )

    if not src.exists():
        raise FileNotFoundError(
            f"Не найден 64ch cache: {src}"
        )

    data = np.load(src)

    X64 = data["X"]
    y = data["y"]

    if X64.shape[1] != 64:
        raise ValueError(
            f"Subject {subject}: "
            f"ожидалось 64 канала, "
            f"получено {X64.shape}"
        )

    X22 = X64[:, PHYS22_IDX, :]

    np.savez_compressed(
        dst,
        X=X22.astype(np.float32),
        y=y.astype(np.int64),
    )

    summary_22.append({
        "subject": subject,
        "n_trials": len(y),
        "left": int((y == 0).sum()),
        "right": int((y == 1).sum()),
        "n_channels": X22.shape[1],
        "n_samples": X22.shape[2],
    })


summary_22_df = pd.DataFrame(summary_22)

print("=" * 70)
print("22-CHANNEL CACHE READY")
print("=" * 70)

print("Subjects:", len(summary_22_df))
print("Trials:", summary_22_df["n_trials"].sum())
print("Left:", summary_22_df["left"].sum())
print("Right:", summary_22_df["right"].sum())
print(
    "Shapes:",
    sorted(
        set(
            zip(
                summary_22_df["n_channels"],
                summary_22_df["n_samples"],
            )
        )
    )
)

display(summary_22_df.head())

22-CHANNEL CACHE READY
Subjects: 106
Trials: 4768
Left: 2406
Right: 2362
Shapes: [(22, 481)]


,subject,n_trials,left,right,n_channels,n_samples
0,1,45,23,22,22,481
1,2,45,23,22,22,481
2,3,45,23,22,22,481
3,4,45,23,22,22,481
4,5,45,21,24,22,481


In [ ]:
# ============================================================
# CCST 22ch
# ============================================================


class Swin1DBlock(nn.Module):
    def __init__(
        self,
        dim=64,
        num_heads=4,
        window_size=4,
        shift_size=0,
        mlp_ratio=2.0,
        dropout=0.1,
    ):
        super().__init__()

        self.dim = dim
        self.window_size = window_size
        self.shift_size = shift_size

        self.norm1 = nn.LayerNorm(dim)

        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.norm2 = nn.LayerNorm(dim)

        hidden_dim = int(dim * mlp_ratio)

        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):

        # x: [B, T, D]

        residual = x
        x = self.norm1(x)

        if self.shift_size > 0:
            x = torch.roll(
                x,
                shifts=-self.shift_size,
                dims=1,
            )

        B, T, D = x.shape

        pad_len = (
            self.window_size
            - T % self.window_size
        ) % self.window_size

        if pad_len > 0:
            x = F.pad(
                x,
                (0, 0, 0, pad_len),
            )

        T_pad = x.shape[1]

        x_windows = x.reshape(
            B * (T_pad // self.window_size),
            self.window_size,
            D,
        )

        attn_out, _ = self.attn(
            x_windows,
            x_windows,
            x_windows,
            need_weights=False,
        )

        x = attn_out.reshape(
            B,
            T_pad,
            D,
        )

        if pad_len > 0:
            x = x[:, :T, :]

        if self.shift_size > 0:
            x = torch.roll(
                x,
                shifts=self.shift_size,
                dims=1,
            )

        x = residual + x

        x = x + self.mlp(
            self.norm2(x)
        )

        return x


class CCST22(nn.Module):

    def __init__(
        self,
        n_channels=22,
        n_samples=481,
        n_classes=2,
    ):
        super().__init__()

        # ----------------------------------------------------
        # Convolutional front-end
        # ----------------------------------------------------

        self.temporal_conv = nn.Conv2d(
            in_channels=1,
            out_channels=40,
            kernel_size=(1, 25),
            padding=(0, 12),
            bias=False,
        )

        self.spatial_conv = nn.Conv2d(
            in_channels=40,
            out_channels=40,
            kernel_size=(n_channels, 1),
            bias=False,
        )

        self.bn = nn.BatchNorm2d(40)
        self.elu = nn.ELU()

        self.pool = nn.AvgPool2d(
            kernel_size=(1, 75),
            stride=(1, 15),
        )

        self.dropout_conv = nn.Dropout(0.5)

        self.pointwise = nn.Conv2d(
            40,
            40,
            kernel_size=(1, 1),
            bias=False,
        )

        # ----------------------------------------------------
        # Transformer projection
        # ----------------------------------------------------

        self.projection = nn.Linear(
            40,
            64,
        )

        # Длины после pooling немного удобнее узнать
        # динамически.
        with torch.no_grad():

            dummy = torch.zeros(
                1,
                1,
                n_channels,
                n_samples,
            )

            z = self.temporal_conv(dummy)
            z = self.spatial_conv(z)
            z = self.bn(z)
            z = self.elu(z)
            z = self.pool(z)
            z = self.dropout_conv(z)
            z = self.pointwise(z)

            sequence_length = z.shape[-1]

        self.pos_embedding = nn.Parameter(
            torch.zeros(
                1,
                sequence_length,
                64,
            )
        )

        nn.init.trunc_normal_(
            self.pos_embedding,
            std=0.02,
        )

        # ----------------------------------------------------
        # Swin-style blocks
        # ----------------------------------------------------

        self.blocks = nn.ModuleList([
            Swin1DBlock(
                dim=64,
                num_heads=4,
                window_size=4,
                shift_size=0,
                mlp_ratio=2.0,
            ),

            Swin1DBlock(
                dim=64,
                num_heads=4,
                window_size=4,
                shift_size=2,
                mlp_ratio=2.0,
            ),

            Swin1DBlock(
                dim=64,
                num_heads=4,
                window_size=4,
                shift_size=0,
                mlp_ratio=2.0,
            ),
        ])

        self.norm = nn.LayerNorm(64)

        self.dropout_head = nn.Dropout(0.3)

        self.classifier = nn.Linear(
            64,
            n_classes,
        )


    def forward(self, x):

        # x: [B, 22, 481]

        x = x.unsqueeze(1)

        x = self.temporal_conv(x)
        x = self.spatial_conv(x)

        x = self.bn(x)
        x = self.elu(x)

        x = self.pool(x)
        x = self.dropout_conv(x)

        x = self.pointwise(x)

        # [B,40,1,T] -> [B,T,40]
        x = x.squeeze(2)
        x = x.transpose(1, 2)

        x = self.projection(x)

        x = x + self.pos_embedding[:, :x.shape[1]]

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)

        # global temporal pooling
        x = x.mean(dim=1)

        x = self.dropout_head(x)

        return self.classifier(x)


# ============================================================
# SANITY CHECK
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model_test = CCST22().to(device)

dummy = torch.randn(
    8,
    22,
    481,
).to(device)

with torch.no_grad():
    out = model_test(dummy)

print("Device:", device)
print("Input :", dummy.shape)
print("Output:", out.shape)

n_params = sum(
    p.numel()
    for p in model_test.parameters()
)

print(
    f"Parameters: {n_params:,}"
)

Device: cpu
Input : torch.Size([8, 22, 481])
Output: torch.Size([8, 2])
Parameters: 142,970


In [24]:
# ============================================================
# STRICT LOSO DATA PREPARATION — 22ch
# ============================================================

from torch.utils.data import (
    Dataset,
    DataLoader,
)


class EEGDataset22(Dataset):

    def __init__(
        self,
        X,
        y,
        augment=False,
    ):
        self.X = torch.tensor(
            X,
            dtype=torch.float32,
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long,
        )

        self.augment = augment


    def __len__(self):
        return len(self.y)


    def __getitem__(self, idx):

        x = self.X[idx].clone()
        y = self.y[idx]

        if self.augment:

            # небольшая temporal shift
            shift = np.random.randint(
                -8,
                9,
            )

            x = torch.roll(
                x,
                shifts=shift,
                dims=-1,
            )

            # слабый Gaussian noise
            if np.random.rand() < 0.5:

                noise = (
                    torch.randn_like(x)
                    * 0.01
                    * x.std()
                )

                x = x + noise

            # случайное зануление одного электрода
            if np.random.rand() < 0.05:

                ch = np.random.randint(
                    0,
                    x.shape[0],
                )

                x[ch] = 0

        return x, y


def load_subject22(subject):

    path = (
        CACHE22_DIR
        / f"subject_{subject:03d}.npz"
    )

    data = np.load(path)

    return (
        data["X"].astype(np.float32),
        data["y"].astype(np.int64),
    )


def prepare_loso_fold22(
    test_subject,
):

    subject_list = PHYS22_SUBJECTS

    test_idx = subject_list.index(
        test_subject
    )

    # следующий subject — validation
    val_subject = subject_list[
        (test_idx + 1)
        % len(subject_list)
    ]

    train_subjects = [
        s
        for s in subject_list
        if s not in {
            test_subject,
            val_subject,
        }
    ]

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    X_train_list = []
    y_train_list = []

    for s in train_subjects:

        X_s, y_s = load_subject22(s)

        X_train_list.append(X_s)
        y_train_list.append(y_s)

    X_train = np.concatenate(
        X_train_list,
        axis=0,
    )

    y_train = np.concatenate(
        y_train_list,
        axis=0,
    )

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    X_val, y_val = load_subject22(
        val_subject
    )

    # --------------------------------------------------------
    # TEST
    # --------------------------------------------------------

    X_test, y_test = load_subject22(
        test_subject
    )

    # --------------------------------------------------------
    # TRAIN-ONLY NORMALIZATION
    # --------------------------------------------------------

    mean = X_train.mean(
        axis=(0, 2),
        keepdims=True,
    )

    std = X_train.std(
        axis=(0, 2),
        keepdims=True,
    )

    std = np.maximum(
        std,
        1e-6,
    )

    X_train = (
        X_train - mean
    ) / std

    X_val = (
        X_val - mean
    ) / std

    X_test = (
        X_test - mean
    ) / std

    return {
        "train_subjects":
            train_subjects,

        "val_subject":
            val_subject,

        "test_subject":
            test_subject,

        "X_train":
            X_train,

        "y_train":
            y_train,

        "X_val":
            X_val,

        "y_val":
            y_val,

        "X_test":
            X_test,

        "y_test":
            y_test,
    }


# ============================================================
# TEST ONE FOLD
# ============================================================

fold = prepare_loso_fold22(
    PHYS22_SUBJECTS[0]
)

print(
    "Test subject:",
    fold["test_subject"],
)

print(
    "Val subject:",
    fold["val_subject"],
)

print(
    "Train subjects:",
    len(fold["train_subjects"]),
)

print()

print(
    "Train:",
    fold["X_train"].shape,
)

print(
    "Val:",
    fold["X_val"].shape,
)

print(
    "Test:",
    fold["X_test"].shape,
)

Test subject: 1
Val subject: 2
Train subjects: 104

Train: (4678, 22, 481)
Val: (45, 22, 481)
Test: (45, 22, 481)


In [25]:
# ============================================================
# TRAIN / EVALUATE CCST22
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    cohen_kappa_score,
)


def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def evaluate_ccst22(
    model,
    loader,
    device,
):

    model.eval()

    all_true = []
    all_pred = []

    total_loss = 0.0
    n_batches = 0

    criterion = nn.CrossEntropyLoss()

    for X, y in loader:

        X = X.to(device)
        y = y.to(device)

        logits = model(X)

        loss = criterion(
            logits,
            y,
        )

        pred = logits.argmax(
            dim=1
        )

        total_loss += loss.item()
        n_batches += 1

        all_true.extend(
            y.cpu().numpy()
        )

        all_pred.extend(
            pred.cpu().numpy()
        )

    y_true = np.asarray(all_true)
    y_pred = np.asarray(all_pred)

    return {
        "loss":
            total_loss / max(n_batches, 1),

        "accuracy":
            accuracy_score(
                y_true,
                y_pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred,
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
            ),

        "kappa":
            cohen_kappa_score(
                y_true,
                y_pred,
            ),
    }


def train_ccst22_fold(
    test_subject,
    seed=42,
    max_epochs=100,
    patience=10,
    batch_size=72,
    verbose=True,
):

    set_seed(seed)

    fold = prepare_loso_fold22(
        test_subject
    )

    train_ds = EEGDataset22(
        fold["X_train"],
        fold["y_train"],
        augment=True,
    )

    val_ds = EEGDataset22(
        fold["X_val"],
        fold["y_val"],
        augment=False,
    )

    test_ds = EEGDataset22(
        fold["X_test"],
        fold["y_test"],
        augment=False,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    model = CCST22().to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=3e-4,
        betas=(0.9, 0.999),
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
    )

    best_val = -np.inf
    best_state = None
    best_epoch = 0

    bad_epochs = 0

    for epoch in range(
        1,
        max_epochs + 1,
    ):

        model.train()

        running_loss = 0.0
        n_batches = 0

        for X, y in train_loader:

            X = X.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits = model(X)

            loss = criterion(
                logits,
                y,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()

            running_loss += loss.item()
            n_batches += 1

        train_loss = (
            running_loss
            / max(n_batches, 1)
        )

        val_metrics = evaluate_ccst22(
            model,
            val_loader,
            device,
        )

        val_acc = (
            val_metrics["accuracy"]
        )

        scheduler.step(
            val_acc
        )

        if verbose:
            print(
                f"CCST22 | "
                f"ep={epoch:03d} | "
                f"train={train_loss:.4f} | "
                f"val={val_acc:.4f}"
            )

        if val_acc > best_val:

            best_val = val_acc
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            bad_epochs = 0

        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    model.load_state_dict(
        best_state
    )

    test_metrics = evaluate_ccst22(
        model,
        test_loader,
        device,
    )

    result = {
        "test_subject":
            test_subject,

        "val_subject":
            fold["val_subject"],

        "best_epoch":
            best_epoch,

        "best_val_accuracy":
            best_val,

        "test_accuracy":
            test_metrics["accuracy"],

        "balanced_accuracy":
            test_metrics["balanced_accuracy"],

        "macro_f1":
            test_metrics["macro_f1"],

        "kappa":
            test_metrics["kappa"],
    }

    return (
        result,
        model,
    )

In [27]:
import copy

In [28]:
result_test, model_test = train_ccst22_fold(
    test_subject=1,
    seed=42,
    max_epochs=5,
    patience=3,
    verbose=True,
)

print()
print("=" * 70)
print("SANITY RESULT")
print("=" * 70)

print(result_test)

CCST22 | ep=001 | train=0.7149 | val=0.5556
CCST22 | ep=002 | train=0.6806 | val=0.6667
CCST22 | ep=003 | train=0.6573 | val=0.8667
CCST22 | ep=004 | train=0.6462 | val=0.8222
CCST22 | ep=005 | train=0.6250 | val=0.7778

SANITY RESULT
{'test_subject': 1, 'val_subject': 2, 'best_epoch': 3, 'best_val_accuracy': 0.8666666666666667, 'test_accuracy': 0.6444444444444445, 'balanced_accuracy': 0.6393280632411067, 'macro_f1': 0.6218487394957983, 'kappa': 0.281437125748503}


In [30]:
import time

In [31]:
# ============================================================
# FULL 106-FOLD STRICT LOSO — CCST22
# resume-safe
# ============================================================


# ============================================================
# OUTPUT PATHS
# ============================================================

CCST22_ROOT = (
    RESULTS_ROOT
    / "ccst22_strict_loso"
)

CCST22_RESULTS_DIR = (
    CCST22_ROOT
    / "results"
)

CCST22_MODELS_DIR = (
    CCST22_ROOT
    / "models"
)

CCST22_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CCST22_MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


PROGRESS_CSV = (
    CCST22_RESULTS_DIR
    / "ccst22_loso_progress.csv"
)

FINAL_CSV = (
    CCST22_RESULTS_DIR
    / "ccst22_loso_final.csv"
)

SUMMARY_CSV = (
    CCST22_RESULTS_DIR
    / "ccst22_loso_summary.csv"
)


# ============================================================
# LOAD PREVIOUS PROGRESS IF EXISTS
# ============================================================

if PROGRESS_CSV.exists():

    results_df = pd.read_csv(
        PROGRESS_CSV
    )

    done_subjects = set(
        results_df["test_subject"]
        .astype(int)
        .tolist()
    )

    print(
        f"Найден предыдущий прогресс: "
        f"{len(done_subjects)} subjects"
    )

else:

    results_df = pd.DataFrame()

    done_subjects = set()

    print("Начинаем новый CCST22 LOSO run")


# ============================================================
# RUN ALL SUBJECTS
# ============================================================

total_subjects = len(
    PHYS22_SUBJECTS
)

for fold_idx, test_subject in enumerate(
    PHYS22_SUBJECTS,
    start=1,
):

    # --------------------------------------------------------
    # RESUME
    # --------------------------------------------------------

    if test_subject in done_subjects:

        print(
            f"[{fold_idx:03d}/{total_subjects}] "
            f"Subject {test_subject:03d} "
            f"already done — skip"
        )

        continue


    print()
    print("#" * 80)

    print(
        f"FOLD {fold_idx}/{total_subjects} "
        f"| TEST SUBJECT={test_subject}"
    )

    print("#" * 80)


    start_time = time.time()


    try:

        result, model = train_ccst22_fold(
            test_subject=test_subject,
            seed=42,
            max_epochs=100,
            patience=10,
            batch_size=72,
            verbose=True,
        )


        # ----------------------------------------------------
        # SAVE MODEL
        # ----------------------------------------------------

        model_path = (
            CCST22_MODELS_DIR
            / f"CCST22_subject_{test_subject:03d}.pt"
        )

        torch.save(
            {
                "test_subject":
                    test_subject,

                "val_subject":
                    result["val_subject"],

                "best_epoch":
                    result["best_epoch"],

                "state_dict":
                    model.state_dict(),

                "channels":
                    PHYS22_CHANNELS,

                "seed":
                    42,
            },
            model_path,
        )


        # ----------------------------------------------------
        # ADD RUNTIME
        # ----------------------------------------------------

        result["runtime_sec"] = (
            time.time() - start_time
        )

        result["model_path"] = str(
            model_path
        )


        # ----------------------------------------------------
        # APPEND RESULT
        # ----------------------------------------------------

        new_row = pd.DataFrame(
            [result]
        )

        results_df = pd.concat(
            [
                results_df,
                new_row,
            ],
            ignore_index=True,
        )


        results_df = (
            results_df
            .sort_values(
                "test_subject"
            )
            .reset_index(
                drop=True
            )
        )


        # ----------------------------------------------------
        # SAVE AFTER EVERY FOLD
        # ----------------------------------------------------

        results_df.to_csv(
            PROGRESS_CSV,
            index=False,
        )


        # ----------------------------------------------------
        # CURRENT RESULT
        # ----------------------------------------------------

        print()
        print("-" * 70)

        print(
            f"SUBJECT {test_subject:03d} DONE"
        )

        print(
            f"Best epoch: "
            f"{result['best_epoch']}"
        )

        print(
            f"Validation: "
            f"{result['best_val_accuracy'] * 100:.2f}%"
        )

        print(
            f"Test accuracy: "
            f"{result['test_accuracy'] * 100:.2f}%"
        )

        print(
            f"Balanced accuracy: "
            f"{result['balanced_accuracy'] * 100:.2f}%"
        )

        print(
            f"Macro F1: "
            f"{result['macro_f1'] * 100:.2f}%"
        )

        print("-" * 70)


        # ----------------------------------------------------
        # RUNNING SUMMARY
        # ----------------------------------------------------

        print()

        print(
            f"RUNNING MEAN "
            f"({len(results_df)}/{total_subjects}): "
            f"{results_df['test_accuracy'].mean() * 100:.2f}%"
        )


        # ----------------------------------------------------
        # CLEAN MEMORY
        # ----------------------------------------------------

        del model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    except Exception as e:

        print()
        print("!" * 80)

        print(
            f"ERROR ON SUBJECT "
            f"{test_subject}:"
        )

        print(repr(e))

        print("!" * 80)

        # Не теряем предыдущие результаты
        results_df.to_csv(
            PROGRESS_CSV,
            index=False,
        )

        raise


# ============================================================
# FINAL RESULTS
# ============================================================

results_df = (
    results_df
    .sort_values(
        "test_subject"
    )
    .reset_index(
        drop=True
    )
)

results_df.to_csv(
    FINAL_CSV,
    index=False,
)


# ============================================================
# SUMMARY
# ============================================================

summary = pd.DataFrame([
    {
        "method":
            "CCST22",

        "n_subjects":
            len(results_df),

        "mean_accuracy":
            results_df[
                "test_accuracy"
            ].mean(),

        "std_accuracy":
            results_df[
                "test_accuracy"
            ].std(),

        "median_accuracy":
            results_df[
                "test_accuracy"
            ].median(),

        "mean_balanced_accuracy":
            results_df[
                "balanced_accuracy"
            ].mean(),

        "mean_macro_f1":
            results_df[
                "macro_f1"
            ].mean(),

        "mean_kappa":
            results_df[
                "kappa"
            ].mean(),

        "mean_best_epoch":
            results_df[
                "best_epoch"
            ].mean(),
    }
])


summary[
    "mean_accuracy_pct"
] = (
    summary[
        "mean_accuracy"
    ] * 100
)


summary.to_csv(
    SUMMARY_CSV,
    index=False,
)


# ============================================================
# PRINT FINAL SUMMARY
# ============================================================

print()
print("=" * 85)
print("FINAL CCST22 STRICT LOSO SUMMARY")
print("=" * 85)

display(summary)

print()

print(
    f"CCST22 mean accuracy: "
    f"{summary.iloc[0]['mean_accuracy'] * 100:.2f}%"
)

print(
    f"Std: "
    f"{summary.iloc[0]['std_accuracy'] * 100:.2f} pp"
)

print(
    f"Subjects: "
    f"{len(results_df)}"
)

print()

print("Saved:")
print(FINAL_CSV)
print(SUMMARY_CSV)

Начинаем новый CCST22 LOSO run

################################################################################
FOLD 1/106 | TEST SUBJECT=1
################################################################################
CCST22 | ep=001 | train=0.7149 | val=0.5556
CCST22 | ep=002 | train=0.6806 | val=0.6667
CCST22 | ep=003 | train=0.6573 | val=0.8667
CCST22 | ep=004 | train=0.6462 | val=0.8222
CCST22 | ep=005 | train=0.6250 | val=0.7778
CCST22 | ep=006 | train=0.6187 | val=0.8444
CCST22 | ep=007 | train=0.6100 | val=0.7778
CCST22 | ep=008 | train=0.6068 | val=0.7778
CCST22 | ep=009 | train=0.5978 | val=0.8444
CCST22 | ep=010 | train=0.5969 | val=0.8444
CCST22 | ep=011 | train=0.5847 | val=0.8444
CCST22 | ep=012 | train=0.5827 | val=0.8667
CCST22 | ep=013 | train=0.5821 | val=0.8000

----------------------------------------------------------------------
SUBJECT 001 DONE
Best epoch: 3
Validation: 86.67%
Test accuracy: 64.44%
Balanced accuracy: 63.93%
Macro F1: 62.18%
-------------------

,method,n_subjects,mean_accuracy,std_accuracy,median_accuracy,mean_balanced_accuracy,mean_macro_f1,mean_kappa,mean_best_epoch,mean_accuracy_pct
0,CCST22,106,0.644639,0.127567,0.622222,0.644599,0.596419,0.289004,10.245283,64.463946



CCST22 mean accuracy: 64.46%
Std: 12.76 pp
Subjects: 106

Saved:
/Users/irina/Desktop/master project/results_v2/physionet_64ch_strict_loso/ccst22_strict_loso/results/ccst22_loso_final.csv
/Users/irina/Desktop/master project/results_v2/physionet_64ch_strict_loso/ccst22_strict_loso/results/ccst22_loso_summary.csv


# Эксперимент 3. Cho2017 64ch → PhysioNet 64ch transfer

Цель:
восстановить полноценный spatial transfer без потери весов 22→64.

Источник:
Cho2017 — 52 испытуемых, 64 EEG-канала, Left/Right motor imagery.

Цель:
PhysioNet — 109 испытуемых, 64 EEG-канала, Left/Right motor imagery.

Далее сравниваем:
- CCST64;
- Cho64 Transfer;
- Cho64 Transfer + SupCon;
- Cho64 Transfer + SupCon + GRL.

In [32]:
# ============================================================
# CHO2017 64ch — SANITY + CHANNEL NAMES
# ============================================================

from moabb.datasets import Cho2017
from moabb.paradigms import MotorImagery

cho64_dataset = Cho2017()

cho64_paradigm = MotorImagery(
    n_classes=2,
    events=[
        "left_hand",
        "right_hand",
    ],
    fmin=7,
    fmax=30,
    resample=160,
)

cho_epochs, cho_labels, cho_metadata = cho64_paradigm.get_data(
    dataset=cho64_dataset,
    subjects=[1],
    return_epochs=True,
)

CHO64_CHANNELS = list(
    cho_epochs.ch_names
)

print("=" * 70)
print("CHO2017 SUBJECT 1")
print("=" * 70)

print(
    "X shape:",
    cho_epochs.get_data().shape,
)

print(
    "Sampling rate:",
    cho_epochs.info["sfreq"],
)

print(
    "Channels:",
    len(CHO64_CHANNELS),
)

print(
    "Trial samples:",
    cho_epochs.get_data().shape[-1],
)

print()
print("Channel names:")
print(CHO64_CHANNELS)

print()
print(
    "Labels:",
    sorted(set(cho_labels)),
)

print(
    "Left:",
    int((cho_labels == "left_hand").sum()),
)

print(
    "Right:",
    int((cho_labels == "right_hand").sum()),
)

Trials demeaned and stacked with zero buffer to create continuous data -- edge effects present


Adding metadata with 3 columns
Adding metadata with 3 columns
200 matching events found
No baseline correction applied
CHO2017 SUBJECT 1
X shape: (200, 64, 480)
Sampling rate: 160.0
Channels: 64
Trial samples: 480

Channel names:
['Fp1', 'AF7', 'AF3', 'F1', 'F3', 'F5', 'F7', 'FT7', 'FC5', 'FC3', 'FC1', 'C1', 'C3', 'C5', 'T7', 'TP7', 'CP5', 'CP3', 'CP1', 'P1', 'P3', 'P5', 'P7', 'P9', 'PO7', 'PO3', 'O1', 'Iz', 'Oz', 'POz', 'Pz', 'CPz', 'Fpz', 'Fp2', 'AF8', 'AF4', 'AFz', 'Fz', 'F2', 'F4', 'F6', 'F8', 'FT8', 'FC6', 'FC4', 'FC2', 'FCz', 'Cz', 'C2', 'C4', 'C6', 'T8', 'TP8', 'CP6', 'CP4', 'CP2', 'P2', 'P4', 'P6', 'P8', 'P10', 'PO8', 'PO4', 'O2']

Labels: [np.str_('left_hand'), np.str_('right_hand')]
Left: 100
Right: 100


In [33]:
# ============================================================
# CHO64 vs PHYSIONET64 — CHANNEL MATCH
# ============================================================

print("=" * 80)
print("CHANNEL COMPARISON")
print("=" * 80)

print(
    "Cho channels:",
    len(CHO64_CHANNELS),
)

print(
    "PhysioNet channels:",
    len(PHYS64_CHANNELS),
)

cho_set = set(
    ch.upper()
    for ch in CHO64_CHANNELS
)

phys_set = set(
    ch.upper()
    for ch in PHYS64_CHANNELS
)

common = sorted(
    cho_set & phys_set
)

only_cho = sorted(
    cho_set - phys_set
)

only_phys = sorted(
    phys_set - cho_set
)

print()
print(
    f"Совпадают напрямую: "
    f"{len(common)}/64"
)

print()
print("Только Cho:")
print(only_cho)

print()
print("Только PhysioNet:")
print(only_phys)

CHANNEL COMPARISON
Cho channels: 64
PhysioNet channels: 64

Совпадают напрямую: 62/64

Только Cho:
['P10', 'P9']

Только PhysioNet:
['T10', 'T9']


In [34]:
# ============================================================
# DIRECT CHANNEL MAPPING
# ============================================================

phys_upper_to_name = {
    ch.upper(): ch
    for ch in PHYS64_CHANNELS
}

cho_upper_to_name = {
    ch.upper(): ch
    for ch in CHO64_CHANNELS
}

mapping_rows = []

for phys_ch in PHYS64_CHANNELS:

    key = phys_ch.upper()

    mapping_rows.append({
        "physionet_channel":
            phys_ch,

        "cho_channel":
            cho_upper_to_name.get(
                key,
                None,
            ),

        "direct_match":
            key in cho_upper_to_name,
    })

channel_mapping_df = pd.DataFrame(
    mapping_rows
)

display(channel_mapping_df)

print()
print(
    "Direct matches:",
    channel_mapping_df[
        "direct_match"
    ].sum(),
    "/",
    len(channel_mapping_df),
)

,physionet_channel,cho_channel,direct_match
0,FC5,FC5,True
1,FC3,FC3,True
2,FC1,FC1,True
3,FCz,FCz,True
4,FC2,FC2,True
...,...,...,...
59,PO8,PO8,True
60,O1,O1,True
61,Oz,Oz,True
62,O2,O2,True



Direct matches: 62 / 64


In [35]:
# ============================================================
# CHO64 -> PHYSIONET64 MONTAGE ADAPTER
# 62 direct channels + interpolate T9/T10
# ============================================================

import numpy as np
import mne


CHO_ONLY = sorted(
    set(ch.upper() for ch in CHO64_CHANNELS)
    - set(ch.upper() for ch in PHYS64_CHANNELS)
)

PHYS_ONLY = sorted(
    set(ch.upper() for ch in PHYS64_CHANNELS)
    - set(ch.upper() for ch in CHO64_CHANNELS)
)

print("Только Cho:", CHO_ONLY)
print("Только PhysioNet:", PHYS_ONLY)


# standard montage содержит T9/T10
standard_montage = mne.channels.make_standard_montage(
    "standard_1005"
)


def adapt_cho_to_phys64(
    X_cho,
    cho_channels,
    phys_channels,
    sfreq=160.0,
):
    """
    X_cho:
        [trials, 64, time]

    Возвращает:
        [trials, 64, 481]

    Каналы расположены строго в порядке PHYS64_CHANNELS.
    Недостающие T9/T10 восстанавливаются interpolation.
    """

    cho_lookup = {
        ch.upper(): i
        for i, ch in enumerate(cho_channels)
    }

    n_trials = X_cho.shape[0]
    n_time = X_cho.shape[2]

    # --------------------------------------------------------
    # Создаём target-массив в ПОРЯДКЕ PhysioNet
    # --------------------------------------------------------

    X_target = np.zeros(
        (
            n_trials,
            len(phys_channels),
            n_time,
        ),
        dtype=np.float64,
    )

    missing_target = []

    for target_idx, ch in enumerate(
        phys_channels
    ):

        key = ch.upper()

        if key in cho_lookup:

            X_target[:, target_idx, :] = (
                X_cho[
                    :,
                    cho_lookup[key],
                    :
                ]
            )

        else:
            missing_target.append(ch)

    print(
        "Интерполируем:",
        missing_target,
    )

    # --------------------------------------------------------
    # MNE EpochsArray
    # --------------------------------------------------------

    info = mne.create_info(
        ch_names=phys_channels,
        sfreq=sfreq,
        ch_types=["eeg"] * len(phys_channels),
    )

    epochs = mne.EpochsArray(
        X_target,
        info,
        verbose=False,
    )

    # Позиции электродов
    epochs.set_montage(
        standard_montage,
        match_case=False,
        on_missing="raise",
        verbose=False,
    )

    # Помечаем T9/T10 как bad
    epochs.info["bads"] = missing_target

    # spherical spline interpolation
    epochs.interpolate_bads(
        reset_bads=True,
        verbose=False,
    )

    X_out = epochs.get_data(
        copy=True
    ).astype(np.float32)

    # --------------------------------------------------------
    # Cho = 480 samples
    # PhysioNet = 481 samples
    #
    # Добавляем последнюю точку без изменения sfreq
    # --------------------------------------------------------

    if X_out.shape[-1] == 480:

        X_out = np.pad(
            X_out,
            pad_width=(
                (0, 0),
                (0, 0),
                (0, 1),
            ),
            mode="edge",
        )

    if X_out.shape[-1] != 481:
        raise ValueError(
            f"Ожидалось 481 samples, "
            f"получено {X_out.shape[-1]}"
        )

    return X_out

Только Cho: ['P10', 'P9']
Только PhysioNet: ['T10', 'T9']


In [36]:
# ============================================================
# SANITY — CHO SUBJECT 1 -> PHYS64 MONTAGE
# ============================================================

X_cho_1 = cho_epochs.get_data().astype(
    np.float32
)

X_cho_phys64_1 = adapt_cho_to_phys64(
    X_cho=X_cho_1,
    cho_channels=CHO64_CHANNELS,
    phys_channels=PHYS64_CHANNELS,
    sfreq=160.0,
)

print()
print("=" * 70)
print("CHO -> PHYS64 SANITY")
print("=" * 70)

print(
    "Before:",
    X_cho_1.shape,
)

print(
    "After :",
    X_cho_phys64_1.shape,
)

print(
    "Finite:",
    np.isfinite(
        X_cho_phys64_1
    ).all(),
)

print(
    "Mean:",
    X_cho_phys64_1.mean(),
)

print(
    "Std:",
    X_cho_phys64_1.std(),
)

Интерполируем: ['T9', 'T10']

CHO -> PHYS64 SANITY
Before: (200, 64, 480)
After : (200, 64, 481)
Finite: True
Mean: -6.469253e-08
Std: 0.0002025802


In [37]:
# ============================================================
# CACHE ALL CHO2017 SUBJECTS
# IN PHYSIONET 64-CHANNEL MONTAGE
# ============================================================

from pathlib import Path
import pandas as pd

CHO64_CACHE_DIR = (
    RESULTS_ROOT
    / "cho64_phys64_montage_cache"
)

CHO64_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CHO_SUBJECTS = list(
    range(1, 53)
)

cho_cache_summary = []


for idx, subject in enumerate(
    CHO_SUBJECTS,
    start=1,
):

    dst = (
        CHO64_CACHE_DIR
        / f"subject_{subject:03d}.npz"
    )

    print(
        f"[{idx:02d}/52] "
        f"Cho subject {subject:02d}"
    )

    # --------------------------------------------------------
    # Если уже сохранён — просто читаем summary
    # --------------------------------------------------------

    if dst.exists():

        d = np.load(dst)

        X = d["X"]
        y = d["y"]

        print(
            "  cached:",
            X.shape,
        )

    else:

        epochs_s, labels_s, metadata_s = (
            cho64_paradigm.get_data(
                dataset=cho64_dataset,
                subjects=[subject],
                return_epochs=True,
            )
        )

        X_raw = (
            epochs_s
            .get_data()
            .astype(np.float32)
        )

        # Проверяем одинаковый порядок Cho каналов
        if list(
            epochs_s.ch_names
        ) != CHO64_CHANNELS:

            raise ValueError(
                f"Subject {subject}: "
                f"порядок каналов отличается"
            )

        # --------------------------------------------
        # labels -> 0 / 1
        # --------------------------------------------

        labels_s = np.asarray(
            labels_s
        )

        y = np.where(
            labels_s == "left_hand",
            0,
            1,
        ).astype(np.int64)

        # --------------------------------------------
        # montage adaptation
        # --------------------------------------------

        X = adapt_cho_to_phys64(
            X_cho=X_raw,
            cho_channels=CHO64_CHANNELS,
            phys_channels=PHYS64_CHANNELS,
            sfreq=160.0,
        )

        np.savez_compressed(
            dst,
            X=X.astype(np.float32),
            y=y,
        )

        print(
            "  saved:",
            X.shape,
        )

    cho_cache_summary.append({
        "subject":
            subject,

        "n_trials":
            len(y),

        "left":
            int((y == 0).sum()),

        "right":
            int((y == 1).sum()),

        "channels":
            X.shape[1],

        "samples":
            X.shape[2],
    })


cho64_summary_df = pd.DataFrame(
    cho_cache_summary
)


print()
print("=" * 75)
print("CHO64 -> PHYS64 CACHE READY")
print("=" * 75)

print(
    "Subjects:",
    len(cho64_summary_df),
)

print(
    "Trials:",
    cho64_summary_df[
        "n_trials"
    ].sum(),
)

print(
    "Left:",
    cho64_summary_df[
        "left"
    ].sum(),
)

print(
    "Right:",
    cho64_summary_df[
        "right"
    ].sum(),
)

print(
    "Shapes:",
    sorted(
        set(
            zip(
                cho64_summary_df["channels"],
                cho64_summary_df["samples"],
            )
        )
    )
)

display(
    cho64_summary_df.head()
)

[01/52] Cho subject 01


Trials demeaned and stacked with zero buffer to create continuous data -- edge effects present


Adding metadata with 3 columns
Adding metadata with 3 columns
200 matching events found
No baseline correction applied
Интерполируем: ['T9', 'T10']
  saved: (200, 64, 481)
[02/52] Cho subject 02


Trials demeaned and stacked with zero buffer to create continuous data -- edge effects present


Adding metadata with 3 columns
Adding metadata with 3 columns
200 matching events found
No baseline correction applied
Интерполируем: ['T9', 'T10']
  saved: (200, 64, 481)
[03/52] Cho subject 03


Trials demeaned and stacked with zero buffer to create continuous data -- edge effects present


... (96 outputs left out)

Adding metadata with 3 columns
Adding metadata with 3 columns
200 matching events found
No baseline correction applied
Интерполируем: ['T9', 'T10']
  saved: (200, 64, 481)
[52/52] Cho subject 52


Trials demeaned and stacked with zero buffer to create continuous data -- edge effects present


Adding metadata with 3 columns
Adding metadata with 3 columns
200 matching events found
No baseline correction applied
Интерполируем: ['T9', 'T10']
  saved: (200, 64, 481)

CHO64 -> PHYS64 CACHE READY
Subjects: 52
Trials: 10520
Left: 5260
Right: 5260
Shapes: [(64, 481)]


,subject,n_trials,left,right,channels,samples
0,1,200,100,100,64,481
1,2,200,100,100,64,481
2,3,200,100,100,64,481
3,4,200,100,100,64,481
4,5,200,100,100,64,481


In [ ]:
 # ============================================================
# EEGNET64 — SOURCE PRETRAINING ON CHO2017
# ============================================================


class EEGNet64(nn.Module):
    def __init__(
        self,
        n_channels=64,
        n_samples=481,
        n_classes=2,
        F1=8,
        D=2,
        F2=16,
        dropout=0.5,
    ):
        super().__init__()

        # ----------------------------------------------------
        # Temporal convolution
        # ----------------------------------------------------

        self.temporal = nn.Conv2d(
            1,
            F1,
            kernel_size=(1, 64),
            padding=(0, 32),
            bias=False,
        )

        self.bn1 = nn.BatchNorm2d(F1)

        # ----------------------------------------------------
        # Depthwise spatial convolution
        # ----------------------------------------------------

        self.spatial = nn.Conv2d(
            F1,
            F1 * D,
            kernel_size=(n_channels, 1),
            groups=F1,
            bias=False,
        )

        self.bn2 = nn.BatchNorm2d(F1 * D)

        self.pool1 = nn.AvgPool2d(
            kernel_size=(1, 4)
        )

        self.dropout1 = nn.Dropout(dropout)

        # ----------------------------------------------------
        # Separable convolution
        # ----------------------------------------------------

        self.depthwise_temporal = nn.Conv2d(
            F1 * D,
            F1 * D,
            kernel_size=(1, 16),
            padding=(0, 8),
            groups=F1 * D,
            bias=False,
        )

        self.pointwise = nn.Conv2d(
            F1 * D,
            F2,
            kernel_size=(1, 1),
            bias=False,
        )

        self.bn3 = nn.BatchNorm2d(F2)

        self.pool2 = nn.AvgPool2d(
            kernel_size=(1, 8)
        )

        self.dropout2 = nn.Dropout(dropout)

        # ----------------------------------------------------
        # Infer feature dimension
        # ----------------------------------------------------

        with torch.no_grad():

            dummy = torch.zeros(
                1,
                1,
                n_channels,
                n_samples,
            )

            z = self._features(dummy)

            self.feature_dim = (
                z.reshape(1, -1).shape[1]
            )

        self.classifier = nn.Linear(
            self.feature_dim,
            n_classes,
        )


    def _features(self, x):

        x = self.temporal(x)
        x = self.bn1(x)

        x = self.spatial(x)
        x = self.bn2(x)

        x = F.elu(x)

        x = self.pool1(x)
        x = self.dropout1(x)

        x = self.depthwise_temporal(x)
        x = self.pointwise(x)

        x = self.bn3(x)

        x = F.elu(x)

        x = self.pool2(x)
        x = self.dropout2(x)

        return x


    def extract_features(self, x):

        if x.ndim == 3:
            x = x.unsqueeze(1)

        x = self._features(x)

        return x.flatten(
            start_dim=1
        )


    def forward(self, x):

        features = self.extract_features(x)

        return self.classifier(
            features
        )


# ============================================================
# SANITY
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model_test = EEGNet64().to(device)

dummy = torch.randn(
    8,
    64,
    481,
).to(device)

with torch.no_grad():

    logits = model_test(dummy)

    features = model_test.extract_features(
        dummy
    )

print("Input:", dummy.shape)
print("Features:", features.shape)
print("Output:", logits.shape)
print("Feature dim:", model_test.feature_dim)

Input: torch.Size([8, 64, 481])
Features: torch.Size([8, 240])
Output: torch.Size([8, 2])
Feature dim: 240


In [39]:
# ============================================================
# CHO64 SUBJECT-WISE PRETRAINING
# ============================================================

# ============================================================
# SETTINGS
# ============================================================

CHO_PRETRAIN_DIR = (
    RESULTS_ROOT
    / "cho64_pretraining"
)

CHO_PRETRAIN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CHO_PRETRAIN_MODEL_PATH = (
    CHO_PRETRAIN_DIR
    / "cho64_eegnet64_best.pt"
)

CHO_PRETRAIN_HISTORY_PATH = (
    CHO_PRETRAIN_DIR
    / "cho64_eegnet64_history.csv"
)


SEED = 42
BATCH_SIZE = 128
MAX_EPOCHS = 100
PATIENCE = 12
LR = 1e-3
WEIGHT_DECAY = 1e-4


# ============================================================
# SEED
# ============================================================

def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ============================================================
# SUBJECT SPLIT
# ============================================================

rng = np.random.default_rng(SEED)

subjects_shuffled = np.array(
    CHO_SUBJECTS
)

rng.shuffle(
    subjects_shuffled
)

n_val_subjects = 10

CHO_VAL_SUBJECTS = sorted(
    subjects_shuffled[
        :n_val_subjects
    ].tolist()
)

CHO_TRAIN_SUBJECTS = sorted(
    subjects_shuffled[
        n_val_subjects:
    ].tolist()
)

print(
    "Train subjects:",
    len(CHO_TRAIN_SUBJECTS),
    CHO_TRAIN_SUBJECTS,
)

print(
    "Val subjects:",
    len(CHO_VAL_SUBJECTS),
    CHO_VAL_SUBJECTS,
)


# ============================================================
# LOAD SUBJECTS
# ============================================================

def load_cho_subjects(subjects):

    X_list = []
    y_list = []

    for subject in subjects:

        path = (
            CHO64_CACHE_DIR
            / f"subject_{subject:03d}.npz"
        )

        data = np.load(path)

        X_list.append(
            data["X"].astype(np.float32)
        )

        y_list.append(
            data["y"].astype(np.int64)
        )

    X = np.concatenate(
        X_list,
        axis=0,
    )

    y = np.concatenate(
        y_list,
        axis=0,
    )

    return X, y


X_train_cho, y_train_cho = (
    load_cho_subjects(
        CHO_TRAIN_SUBJECTS
    )
)

X_val_cho, y_val_cho = (
    load_cho_subjects(
        CHO_VAL_SUBJECTS
    )
)


print()
print(
    "Train:",
    X_train_cho.shape,
    y_train_cho.shape,
)

print(
    "Val:",
    X_val_cho.shape,
    y_val_cho.shape,
)


# ============================================================
# TRAIN-ONLY NORMALIZATION
# ============================================================

cho_mean = X_train_cho.mean(
    axis=(0, 2),
    keepdims=True,
)

cho_std = X_train_cho.std(
    axis=(0, 2),
    keepdims=True,
)

cho_std = np.maximum(
    cho_std,
    1e-6,
)

X_train_cho = (
    X_train_cho - cho_mean
) / cho_std

X_val_cho = (
    X_val_cho - cho_mean
) / cho_std


# ============================================================
# DATASET
# ============================================================

class Cho64Dataset(Dataset):

    def __init__(
        self,
        X,
        y,
        augment=False,
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32,
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long,
        )

        self.augment = augment


    def __len__(self):

        return len(self.y)


    def __getitem__(self, idx):

        x = self.X[idx].clone()
        y = self.y[idx]

        if self.augment:

            # temporal shift
            shift = np.random.randint(
                -8,
                9,
            )

            x = torch.roll(
                x,
                shifts=shift,
                dims=-1,
            )

            # weak noise
            if np.random.rand() < 0.5:

                x = (
                    x
                    + torch.randn_like(x)
                    * 0.01
                )

            # electrode dropout
            if np.random.rand() < 0.05:

                ch = np.random.randint(
                    0,
                    x.shape[0],
                )

                x[ch] = 0

        return x, y


train_ds = Cho64Dataset(
    X_train_cho,
    y_train_cho,
    augment=True,
)

val_ds = Cho64Dataset(
    X_val_cho,
    y_val_cho,
    augment=False,
)


train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)


# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_model(
    model,
    loader,
):

    model.eval()

    criterion = nn.CrossEntropyLoss()

    total_loss = 0.0
    correct = 0
    total = 0

    for X, y in loader:

        X = X.to(device)
        y = y.to(device)

        logits = model(X)

        loss = criterion(
            logits,
            y,
        )

        total_loss += (
            loss.item()
            * len(y)
        )

        pred = logits.argmax(
            dim=1
        )

        correct += (
            pred == y
        ).sum().item()

        total += len(y)

    return {
        "loss":
            total_loss / total,

        "accuracy":
            correct / total,
    }


# ============================================================
# TRAIN
# ============================================================

model = EEGNet64().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=5,
)


best_val_acc = -np.inf
best_epoch = 0
best_state = None

bad_epochs = 0

history = []


for epoch in range(
    1,
    MAX_EPOCHS + 1,
):

    model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0


    for X, y in train_loader:

        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(X)

        loss = criterion(
            logits,
            y,
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0,
        )

        optimizer.step()

        train_loss_sum += (
            loss.item()
            * len(y)
        )

        pred = logits.argmax(
            dim=1
        )

        train_correct += (
            pred == y
        ).sum().item()

        train_total += len(y)


    train_loss = (
        train_loss_sum
        / train_total
    )

    train_acc = (
        train_correct
        / train_total
    )


    val_metrics = evaluate_model(
        model,
        val_loader,
    )

    val_loss = val_metrics[
        "loss"
    ]

    val_acc = val_metrics[
        "accuracy"
    ]


    scheduler.step(
        val_acc
    )


    history.append({
        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "train_accuracy":
            train_acc,

        "val_loss":
            val_loss,

        "val_accuracy":
            val_acc,

        "lr":
            optimizer.param_groups[0]["lr"],
    })


    print(
        f"CHO64 | "
        f"ep={epoch:03d} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_acc:.4f}"
    )


    # --------------------------------------------------------
    # BEST MODEL
    # --------------------------------------------------------

    if val_acc > best_val_acc:

        best_val_acc = val_acc
        best_epoch = epoch

        best_state = copy.deepcopy(
            model.state_dict()
        )

        bad_epochs = 0

    else:

        bad_epochs += 1


    if bad_epochs >= PATIENCE:

        print()
        print(
            f"Early stopping at epoch "
            f"{epoch}"
        )

        break


# ============================================================
# RESTORE BEST MODEL
# ============================================================

model.load_state_dict(
    best_state
)


# ============================================================
# SAVE
# ============================================================

torch.save(
    {
        "state_dict":
            model.state_dict(),

        "best_epoch":
            best_epoch,

        "best_val_accuracy":
            best_val_acc,

        "train_subjects":
            CHO_TRAIN_SUBJECTS,

        "val_subjects":
            CHO_VAL_SUBJECTS,

        "channels":
            PHYS64_CHANNELS,

        "n_channels":
            64,

        "n_samples":
            481,

        "feature_dim":
            model.feature_dim,

        "seed":
            SEED,

        "cho_mean":
            cho_mean.astype(np.float32),

        "cho_std":
            cho_std.astype(np.float32),
    },
    CHO_PRETRAIN_MODEL_PATH,
)


history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    CHO_PRETRAIN_HISTORY_PATH,
    index=False,
)


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 80)
print("CHO64 PRETRAINING FINISHED")
print("=" * 80)

print(
    "Best epoch:",
    best_epoch,
)

print(
    "Best validation accuracy:",
    f"{best_val_acc * 100:.2f}%",
)

print()
print(
    "Model saved:",
    CHO_PRETRAIN_MODEL_PATH,
)

print(
    "History saved:",
    CHO_PRETRAIN_HISTORY_PATH,
)

Train subjects: 42 [1, 2, 3, 4, 5, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23, 25, 26, 27, 29, 30, 31, 32, 33, 34, 36, 37, 38, 40, 41, 42, 43, 44, 45, 46, 47, 49, 51]
Val subjects: 10 [6, 8, 18, 24, 28, 35, 39, 48, 50, 52]

Train: (8520, 64, 481) (8520,)
Val: (2000, 64, 481) (2000,)
CHO64 | ep=001 | train_loss=0.6990 | train_acc=0.5052 | val_loss=0.7001 | val_acc=0.5185
CHO64 | ep=002 | train_loss=0.6925 | train_acc=0.5194 | val_loss=0.7179 | val_acc=0.5220
CHO64 | ep=003 | train_loss=0.6908 | train_acc=0.5329 | val_loss=0.7162 | val_acc=0.5195
CHO64 | ep=004 | train_loss=0.6874 | train_acc=0.5339 | val_loss=0.7133 | val_acc=0.5275
CHO64 | ep=005 | train_loss=0.6795 | train_acc=0.5547 | val_loss=0.7274 | val_acc=0.5585
CHO64 | ep=006 | train_loss=0.6669 | train_acc=0.5757 | val_loss=0.6843 | val_acc=0.5925
CHO64 | ep=007 | train_loss=0.6511 | train_acc=0.5973 | val_loss=0.7154 | val_acc=0.5925
CHO64 | ep=008 | train_loss=0.6370 | train_acc=0.6136 | val_loss=0.6928 | val_a

In [41]:
# ============================================================
# LOAD FULL CHO64 PRETRAINING INTO EEGNET64
# ============================================================

checkpoint = torch.load(
    CHO_PRETRAIN_MODEL_PATH,
    map_location=device,
    weights_only=False,
)

transfer_model = EEGNet64().to(device)

load_result = transfer_model.load_state_dict(
    checkpoint["state_dict"],
    strict=True,
)

print("=" * 70)
print("CHO64 -> EEGNET64 TRANSFER")
print("=" * 70)

print("Checkpoint:", CHO_PRETRAIN_MODEL_PATH)
print("Best Cho epoch:", checkpoint["best_epoch"])
print(
    "Cho validation:",
    f"{checkpoint['best_val_accuracy'] * 100:.2f}%"
)

print()
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

print()
print("Spatial weights:")
print(
    transfer_model.spatial.weight.shape
)

print()
print("FULL 64ch TRANSFER OK")

CHO64 -> EEGNET64 TRANSFER
Checkpoint: /Users/irina/Desktop/master project/results_v2/physionet_64ch_strict_loso/cho64_pretraining/cho64_eegnet64_best.pt
Best Cho epoch: 18
Cho validation: 62.40%

Missing keys: []
Unexpected keys: []

Spatial weights:
torch.Size([16, 1, 64, 1])

FULL 64ch TRANSFER OK


In [42]:
# ============================================================
# INITIALIZE FULL CHO64 TRANSFER MODEL
# ============================================================

def make_cho64_transfer_model():

    checkpoint = torch.load(
        CHO_PRETRAIN_MODEL_PATH,
        map_location=device,
        weights_only=False,
    )

    model = EEGNet64().to(device)

    result = model.load_state_dict(
        checkpoint["state_dict"],
        strict=True,
    )

    assert len(result.missing_keys) == 0
    assert len(result.unexpected_keys) == 0

    return model

In [43]:
model = make_cho64_transfer_model()

x = torch.randn(
    8,
    64,
    481,
    device=device,
)

with torch.no_grad():
    y = model(x)

print("Output:", y.shape)
print("Spatial:", model.spatial.weight.shape)

Output: torch.Size([8, 2])
Spatial: torch.Size([16, 1, 64, 1])


In [44]:
# ============================================================
# PHYSIONET64 — FULL CHO64 TRANSFER
# ONE-FOLD TRAINING
# ============================================================

import copy
import random
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    cohen_kappa_score,
)


class Phys64Dataset(Dataset):

    def __init__(
        self,
        X,
        y,
        augment=False,
    ):
        self.X = torch.tensor(
            X,
            dtype=torch.float32,
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long,
        )

        self.augment = augment


    def __len__(self):
        return len(self.y)


    def __getitem__(self, idx):

        x = self.X[idx].clone()
        y = self.y[idx]

        if self.augment:

            shift = np.random.randint(-8, 9)

            x = torch.roll(
                x,
                shifts=shift,
                dims=-1,
            )

            if np.random.rand() < 0.5:
                x = x + torch.randn_like(x) * 0.01

            if np.random.rand() < 0.05:
                ch = np.random.randint(
                    0,
                    x.shape[0],
                )
                x[ch] = 0

        return x, y


def load_phys64_subject(subject):

    path = (
        CACHE64_DIR
        / f"subject_{subject:03d}.npz"
    )

    data = np.load(path)

    return (
        data["X"].astype(np.float32),
        data["y"].astype(np.int64),
    )


def prepare_phys64_fold(
    test_subject,
):

    subject_list = list(range(1, 110))

    test_idx = subject_list.index(
        test_subject
    )

    val_subject = subject_list[
        (test_idx + 1)
        % len(subject_list)
    ]

    train_subjects = [
        s for s in subject_list
        if s not in {
            test_subject,
            val_subject,
        }
    ]

    X_train_list = []
    y_train_list = []

    for s in train_subjects:

        X_s, y_s = load_phys64_subject(s)

        X_train_list.append(X_s)
        y_train_list.append(y_s)

    X_train = np.concatenate(
        X_train_list,
        axis=0,
    )

    y_train = np.concatenate(
        y_train_list,
        axis=0,
    )

    X_val, y_val = load_phys64_subject(
        val_subject
    )

    X_test, y_test = load_phys64_subject(
        test_subject
    )

    # train-only normalization
    mean = X_train.mean(
        axis=(0, 2),
        keepdims=True,
    )

    std = X_train.std(
        axis=(0, 2),
        keepdims=True,
    )

    std = np.maximum(
        std,
        1e-6,
    )

    X_train = (X_train - mean) / std
    X_val = (X_val - mean) / std
    X_test = (X_test - mean) / std

    return {
        "train_subjects": train_subjects,
        "val_subject": val_subject,
        "test_subject": test_subject,

        "X_train": X_train,
        "y_train": y_train,

        "X_val": X_val,
        "y_val": y_val,

        "X_test": X_test,
        "y_test": y_test,
    }


@torch.no_grad()
def evaluate_phys64(
    model,
    loader,
):

    model.eval()

    criterion = nn.CrossEntropyLoss()

    losses = []
    all_true = []
    all_pred = []

    for X, y in loader:

        X = X.to(device)
        y = y.to(device)

        logits = model(X)

        loss = criterion(
            logits,
            y,
        )

        losses.append(
            loss.item()
        )

        pred = logits.argmax(
            dim=1
        )

        all_true.extend(
            y.cpu().numpy()
        )

        all_pred.extend(
            pred.cpu().numpy()
        )

    y_true = np.asarray(all_true)
    y_pred = np.asarray(all_pred)

    return {
        "loss":
            float(np.mean(losses)),

        "accuracy":
            accuracy_score(
                y_true,
                y_pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred,
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
            ),

        "kappa":
            cohen_kappa_score(
                y_true,
                y_pred,
            ),
    }


def train_cho64_transfer_fold(
    test_subject,
    seed=42,
    max_epochs=60,
    patience=10,
    batch_size=64,
):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    fold = prepare_phys64_fold(
        test_subject
    )

    train_loader = DataLoader(
        Phys64Dataset(
            fold["X_train"],
            fold["y_train"],
            augment=True,
        ),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )

    val_loader = DataLoader(
        Phys64Dataset(
            fold["X_val"],
            fold["y_val"],
            augment=False,
        ),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    test_loader = DataLoader(
        Phys64Dataset(
            fold["X_test"],
            fold["y_test"],
            augment=False,
        ),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    # FULL Cho64 pretrained weights
    model = make_cho64_transfer_model()

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
    )

    best_val = -np.inf
    best_epoch = 0
    best_state = None
    bad_epochs = 0

    for epoch in range(
        1,
        max_epochs + 1,
    ):

        model.train()

        train_loss_sum = 0.0
        n_batches = 0

        for X, y in train_loader:

            X = X.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits = model(X)

            loss = criterion(
                logits,
                y,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()

            train_loss_sum += loss.item()
            n_batches += 1

        train_loss = (
            train_loss_sum
            / max(n_batches, 1)
        )

        val_metrics = evaluate_phys64(
            model,
            val_loader,
        )

        val_acc = val_metrics[
            "accuracy"
        ]

        scheduler.step(
            val_acc
        )

        print(
            f"CHO64_TRANSFER | "
            f"ep={epoch:03d} | "
            f"train={train_loss:.4f} | "
            f"val={val_acc:.4f}"
        )

        if val_acc > best_val:

            best_val = val_acc
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            bad_epochs = 0

        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    model.load_state_dict(
        best_state
    )

    test_metrics = evaluate_phys64(
        model,
        test_loader,
    )

    result = {
        "test_subject":
            test_subject,

        "val_subject":
            fold["val_subject"],

        "best_epoch":
            best_epoch,

        "best_val_accuracy":
            best_val,

        "test_accuracy":
            test_metrics["accuracy"],

        "balanced_accuracy":
            test_metrics["balanced_accuracy"],

        "macro_f1":
            test_metrics["macro_f1"],

        "kappa":
            test_metrics["kappa"],
    }

    return result, model

In [45]:
result_test, model_test = train_cho64_transfer_fold(
    test_subject=1,
    seed=42,
    max_epochs=5,
    patience=3,
    batch_size=64,
)

print()
print("=" * 70)
print("CHO64 TRANSFER SANITY")
print("=" * 70)

print(result_test)

CHO64_TRANSFER | ep=001 | train=0.6838 | val=0.9778
CHO64_TRANSFER | ep=002 | train=0.6454 | val=0.9333
CHO64_TRANSFER | ep=003 | train=0.6408 | val=0.9333
CHO64_TRANSFER | ep=004 | train=0.6343 | val=0.8889

CHO64 TRANSFER SANITY
{'test_subject': 1, 'val_subject': 2, 'best_epoch': 1, 'best_val_accuracy': 0.9777777777777777, 'test_accuracy': 0.7111111111111111, 'balanced_accuracy': 0.7154150197628458, 'macro_f1': 0.7016828148903621, 'kappa': 0.42703232125367285}


In [46]:
# ============================================================
# FULL 109-FOLD STRICT LOSO
# FULL CHO64 -> PHYSIONET64 TRANSFER
# resume-safe
# ============================================================



# ============================================================
# PATHS
# ============================================================

CHO64_TRANSFER_ROOT = (
    RESULTS_ROOT
    / "cho64_full_transfer_physionet64"
)

CHO64_TRANSFER_RESULTS_DIR = (
    CHO64_TRANSFER_ROOT
    / "results"
)

CHO64_TRANSFER_MODELS_DIR = (
    CHO64_TRANSFER_ROOT
    / "models"
)

CHO64_TRANSFER_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CHO64_TRANSFER_MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


PROGRESS_CSV = (
    CHO64_TRANSFER_RESULTS_DIR
    / "cho64_transfer_progress.csv"
)

FINAL_CSV = (
    CHO64_TRANSFER_RESULTS_DIR
    / "cho64_transfer_final.csv"
)

SUMMARY_CSV = (
    CHO64_TRANSFER_RESULTS_DIR
    / "cho64_transfer_summary.csv"
)


# ============================================================
# RESUME
# ============================================================

if PROGRESS_CSV.exists():

    results_df = pd.read_csv(
        PROGRESS_CSV
    )

    done_subjects = set(
        results_df["test_subject"]
        .astype(int)
        .tolist()
    )

    print(
        f"Найден прогресс: "
        f"{len(done_subjects)}/109 subjects"
    )

else:

    results_df = pd.DataFrame()
    done_subjects = set()

    print(
        "Начинаем новый "
        "Cho64 -> PhysioNet64 run"
    )


# ============================================================
# ALL SUBJECTS
# ============================================================

PHYS64_SUBJECTS = list(
    range(1, 110)
)

total_subjects = len(
    PHYS64_SUBJECTS
)


for fold_idx, test_subject in enumerate(
    PHYS64_SUBJECTS,
    start=1,
):

    if test_subject in done_subjects:

        print(
            f"[{fold_idx:03d}/{total_subjects}] "
            f"Subject {test_subject:03d} "
            f"already done — skip"
        )

        continue


    print()
    print("#" * 80)

    print(
        f"FOLD {fold_idx}/{total_subjects} "
        f"| TEST SUBJECT={test_subject}"
    )

    print("#" * 80)


    start_time = time.time()


    try:

        result, model = train_cho64_transfer_fold(
            test_subject=test_subject,
            seed=42,
            max_epochs=60,
            patience=10,
            batch_size=64,
        )


        # ----------------------------------------------------
        # SAVE MODEL
        # ----------------------------------------------------

        model_path = (
            CHO64_TRANSFER_MODELS_DIR
            / f"cho64_transfer_subject_{test_subject:03d}.pt"
        )

        torch.save(
            {
                "test_subject":
                    test_subject,

                "val_subject":
                    result["val_subject"],

                "best_epoch":
                    result["best_epoch"],

                "state_dict":
                    model.state_dict(),

                "source_checkpoint":
                    str(
                        CHO_PRETRAIN_MODEL_PATH
                    ),

                "seed":
                    42,

                "channels":
                    PHYS64_CHANNELS,
            },
            model_path,
        )


        # ----------------------------------------------------
        # RESULT
        # ----------------------------------------------------

        result["runtime_sec"] = (
            time.time()
            - start_time
        )

        result["model_path"] = str(
            model_path
        )

        results_df = pd.concat(
            [
                results_df,
                pd.DataFrame([result]),
            ],
            ignore_index=True,
        )


        results_df = (
            results_df
            .sort_values(
                "test_subject"
            )
            .reset_index(
                drop=True
            )
        )


        # ----------------------------------------------------
        # SAVE AFTER EVERY SUBJECT
        # ----------------------------------------------------

        results_df.to_csv(
            PROGRESS_CSV,
            index=False,
        )


        # ----------------------------------------------------
        # RUNNING OUTPUT
        # ----------------------------------------------------

        print()
        print("-" * 72)

        print(
            f"SUBJECT "
            f"{test_subject:03d} DONE"
        )

        print(
            f"Best epoch: "
            f"{result['best_epoch']}"
        )

        print(
            f"Validation: "
            f"{result['best_val_accuracy'] * 100:.2f}%"
        )

        print(
            f"Test accuracy: "
            f"{result['test_accuracy'] * 100:.2f}%"
        )

        print(
            f"Balanced accuracy: "
            f"{result['balanced_accuracy'] * 100:.2f}%"
        )

        print(
            f"Macro F1: "
            f"{result['macro_f1'] * 100:.2f}%"
        )

        print("-" * 72)

        print()

        print(
            f"RUNNING MEAN "
            f"({len(results_df)}/{total_subjects}): "
            f"{results_df['test_accuracy'].mean() * 100:.2f}%"
        )


        # ----------------------------------------------------
        # CLEAN
        # ----------------------------------------------------

        del model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    except Exception as e:

        results_df.to_csv(
            PROGRESS_CSV,
            index=False,
        )

        print()
        print("!" * 80)

        print(
            f"ERROR SUBJECT "
            f"{test_subject}"
        )

        print(repr(e))

        print("!" * 80)

        raise


# ============================================================
# FINAL SAVE
# ============================================================

results_df = (
    results_df
    .sort_values(
        "test_subject"
    )
    .reset_index(
        drop=True
    )
)

results_df.to_csv(
    FINAL_CSV,
    index=False,
)


# ============================================================
# SUMMARY
# ============================================================

summary = pd.DataFrame([
    {
        "method":
            "Cho64_Full_Transfer",

        "n_subjects":
            len(results_df),

        "mean_accuracy":
            results_df[
                "test_accuracy"
            ].mean(),

        "std_accuracy":
            results_df[
                "test_accuracy"
            ].std(),

        "median_accuracy":
            results_df[
                "test_accuracy"
            ].median(),

        "mean_balanced_accuracy":
            results_df[
                "balanced_accuracy"
            ].mean(),

        "mean_macro_f1":
            results_df[
                "macro_f1"
            ].mean(),

        "mean_kappa":
            results_df[
                "kappa"
            ].mean(),

        "mean_best_epoch":
            results_df[
                "best_epoch"
            ].mean(),
    }
])


summary[
    "mean_accuracy_pct"
] = (
    summary[
        "mean_accuracy"
    ]
    * 100
)


summary.to_csv(
    SUMMARY_CSV,
    index=False,
)


# ============================================================
# FINAL PRINT
# ============================================================

print()
print("=" * 85)
print("FINAL CHO64 FULL TRANSFER SUMMARY")
print("=" * 85)

display(summary)

print()

print(
    f"Cho64 Full Transfer: "
    f"{summary.iloc[0]['mean_accuracy'] * 100:.2f}%"
)

print(
    f"CCST64 reference: "
    f"73.65%"
)

print(
    f"Delta vs CCST64: "
    f"{summary.iloc[0]['mean_accuracy'] * 100 - 73.65:+.2f} pp"
)

print()
print("Saved:")
print(FINAL_CSV)
print(SUMMARY_CSV)

Начинаем новый Cho64 -> PhysioNet64 run

################################################################################
FOLD 1/109 | TEST SUBJECT=1
################################################################################
CHO64_TRANSFER | ep=001 | train=0.6838 | val=0.9778
CHO64_TRANSFER | ep=002 | train=0.6454 | val=0.9333
CHO64_TRANSFER | ep=003 | train=0.6408 | val=0.9333
CHO64_TRANSFER | ep=004 | train=0.6343 | val=0.8889
CHO64_TRANSFER | ep=005 | train=0.6274 | val=0.9111
CHO64_TRANSFER | ep=006 | train=0.6244 | val=0.9333
CHO64_TRANSFER | ep=007 | train=0.6235 | val=0.8667
CHO64_TRANSFER | ep=008 | train=0.6107 | val=0.9111
CHO64_TRANSFER | ep=009 | train=0.6132 | val=0.9111
CHO64_TRANSFER | ep=010 | train=0.6086 | val=0.9333
CHO64_TRANSFER | ep=011 | train=0.6174 | val=0.9111

------------------------------------------------------------------------
SUBJECT 001 DONE
Best epoch: 1
Validation: 97.78%
Test accuracy: 71.11%
Balanced accuracy: 71.54%
Macro F1: 70.17%
--------

,method,n_subjects,mean_accuracy,std_accuracy,median_accuracy,mean_balanced_accuracy,mean_macro_f1,mean_kappa,mean_best_epoch,mean_accuracy_pct
0,Cho64_Full_Transfer,109,0.658665,0.138368,0.644444,0.659562,0.632177,0.318967,8.990826,65.866544



Cho64 Full Transfer: 65.87%
CCST64 reference: 73.65%
Delta vs CCST64: -7.78 pp

Saved:
/Users/irina/Desktop/master project/results_v2/physionet_64ch_strict_loso/cho64_full_transfer_physionet64/results/cho64_transfer_final.csv
/Users/irina/Desktop/master project/results_v2/physionet_64ch_strict_loso/cho64_full_transfer_physionet64/results/cho64_transfer_summary.csv


# Эксперимент 4. Tuned Cho64 → PhysioNet64 Transfer

## Цель

Проверить, можно ли улучшить перенос знаний с Cho2017 на PhysioNet
за счёт более аккуратного fine-tuning и более стабильной валидации.

В предыдущем эксперименте использовался прямой full fine-tuning всех весов
с одинаковым learning rate. Такой подход мог быстро разрушать полезные
pretrained-признаки.

## Постановка

**Source dataset:** Cho2017  
- 52 испытуемых;
- 64 EEG-канала;
- Left / Right motor imagery;
- 10 520 trials;
- данные приведены к montage PhysioNet64.

**Target dataset:** PhysioNet  
- 109 испытуемых;
- 64 EEG-канала;
- Left / Right motor imagery;
- strict LOSO: один испытуемый полностью используется только как test.

## Валидация

Для каждого LOSO-fold:

- test — один полностью невидимый испытуемый;
- данные остальных 108 испытуемых объединяются;
- 90% source trials используются для обучения;
- 10% source trials — для validation;
- split стратифицирован по классу;
- нормализация рассчитывается только по train-части.

Такая схема совпадает с протоколом, использованным для CCST64.

## Tuned transfer

Обучение проводится в два этапа.

### Stage 1 — frozen backbone

Сначала замораживается pretrained EEGNet64 encoder.

Обучается только классификационная голова:

- 5 эпох;
- `lr = 1e-3`.

Цель — адаптировать classifier к PhysioNet,
не разрушая representation, полученный на Cho2017.

### Stage 2 — gradual fine-tuning

Затем вся сеть размораживается.

Используются разные learning rate:

- pretrained backbone: `lr = 1e-4`;
- classifier head: `lr = 5e-4`.

Дополнительно используется early stopping по validation accuracy.

## Сравнение

Результат сравнивается с:

- CCST64 — 73.65%;
- предыдущим Cho64 Full Transfer — 65.87%.

Основной вопрос эксперимента:

**поможет ли более мягкий fine-tuning сохранить полезные признаки Cho2017
и приблизить transfer-based модель к CCST64 или превзойти её?**

In [47]:
# ============================================================
# PHYSIONET64 — SAME VALIDATION PROTOCOL AS CCST64
# test subject unseen
# remaining source trials -> 90% train / 10% validation
# ============================================================

from sklearn.model_selection import train_test_split


def prepare_phys64_fold_tuned(
    test_subject,
    seed=42,
):

    train_subjects = [
        s for s in PHYS64_SUBJECTS
        if s != test_subject
    ]

    # --------------------------------------------------------
    # SOURCE DATA
    # --------------------------------------------------------

    X_source_list = []
    y_source_list = []

    for s in train_subjects:

        X_s, y_s = load_phys64_subject(s)

        X_source_list.append(X_s)
        y_source_list.append(y_s)

    X_source = np.concatenate(
        X_source_list,
        axis=0,
    )

    y_source = np.concatenate(
        y_source_list,
        axis=0,
    )

    # --------------------------------------------------------
    # 90/10 TRAIN / VALIDATION
    # --------------------------------------------------------

    (
        X_train,
        X_val,
        y_train,
        y_val,
    ) = train_test_split(
        X_source,
        y_source,
        test_size=0.10,
        random_state=seed,
        stratify=y_source,
    )

    # --------------------------------------------------------
    # TEST = COMPLETELY UNSEEN SUBJECT
    # --------------------------------------------------------

    X_test, y_test = load_phys64_subject(
        test_subject
    )

    # --------------------------------------------------------
    # NORMALIZATION: TRAIN ONLY
    # --------------------------------------------------------

    mean = X_train.mean(
        axis=(0, 2),
        keepdims=True,
    )

    std = X_train.std(
        axis=(0, 2),
        keepdims=True,
    )

    std = np.maximum(
        std,
        1e-6,
    )

    X_train = (
        X_train - mean
    ) / std

    X_val = (
        X_val - mean
    ) / std

    X_test = (
        X_test - mean
    ) / std

    return {
        "test_subject":
            test_subject,

        "X_train":
            X_train,

        "y_train":
            y_train,

        "X_val":
            X_val,

        "y_val":
            y_val,

        "X_test":
            X_test,

        "y_test":
            y_test,
    }

In [48]:
# ============================================================
# TUNED CHO64 TRANSFER
# Stage 1: classifier only
# Stage 2: full fine-tuning with discriminative LR
# ============================================================

def set_backbone_requires_grad(
    model,
    requires_grad,
):

    backbone_modules = [
        model.temporal,
        model.bn1,
        model.spatial,
        model.bn2,
        model.depthwise_temporal,
        model.pointwise,
        model.bn3,
    ]

    for module in backbone_modules:

        for p in module.parameters():
            p.requires_grad = requires_grad


def train_cho64_transfer_tuned_fold(
    test_subject,
    seed=42,
    freeze_epochs=5,
    max_epochs=50,
    patience=10,
    batch_size=64,
):

    set_seed(seed)

    fold = prepare_phys64_fold_tuned(
        test_subject=test_subject,
        seed=seed,
    )

    train_loader = DataLoader(
        Phys64Dataset(
            fold["X_train"],
            fold["y_train"],
            augment=True,
        ),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )

    val_loader = DataLoader(
        Phys64Dataset(
            fold["X_val"],
            fold["y_val"],
            augment=False,
        ),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    test_loader = DataLoader(
        Phys64Dataset(
            fold["X_test"],
            fold["y_test"],
            augment=False,
        ),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    # --------------------------------------------------------
    # LOAD FULL CHO64 PRETRAINED MODEL
    # --------------------------------------------------------

    model = make_cho64_transfer_model()

    criterion = nn.CrossEntropyLoss()


    # ========================================================
    # STAGE 1 — FREEZE BACKBONE
    # ========================================================

    set_backbone_requires_grad(
        model,
        False,
    )

    for p in model.classifier.parameters():
        p.requires_grad = True

    optimizer_stage1 = torch.optim.AdamW(
        model.classifier.parameters(),
        lr=1e-3,
        weight_decay=1e-4,
    )

    print()
    print("STAGE 1 — classifier only")

    for epoch in range(
        1,
        freeze_epochs + 1,
    ):

        model.train()

        train_loss_sum = 0.0
        n_batches = 0

        for X, y in train_loader:

            X = X.to(device)
            y = y.to(device)

            optimizer_stage1.zero_grad()

            logits = model(X)

            loss = criterion(
                logits,
                y,
            )

            loss.backward()

            optimizer_stage1.step()

            train_loss_sum += loss.item()
            n_batches += 1

        train_loss = (
            train_loss_sum
            / max(n_batches, 1)
        )

        val_metrics = evaluate_phys64(
            model,
            val_loader,
        )

        print(
            f"FREEZE | "
            f"ep={epoch:02d} | "
            f"train={train_loss:.4f} | "
            f"val={val_metrics['accuracy']:.4f}"
        )


    # ========================================================
    # STAGE 2 — UNFREEZE
    # ========================================================

    set_backbone_requires_grad(
        model,
        True,
    )

    optimizer_stage2 = torch.optim.AdamW(
        [
            {
                "params": [
                    p
                    for name, p in model.named_parameters()
                    if not name.startswith("classifier.")
                ],
                "lr": 1e-4,
            },
            {
                "params":
                    model.classifier.parameters(),

                "lr":
                    5e-4,
            },
        ],
        weight_decay=1e-4,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_stage2,
        mode="max",
        factor=0.5,
        patience=4,
    )

    best_val = -np.inf
    best_epoch = 0
    best_state = None

    bad_epochs = 0

    print()
    print("STAGE 2 — full fine-tuning")

    for epoch in range(
        1,
        max_epochs + 1,
    ):

        model.train()

        train_loss_sum = 0.0
        n_batches = 0

        for X, y in train_loader:

            X = X.to(device)
            y = y.to(device)

            optimizer_stage2.zero_grad()

            logits = model(X)

            loss = criterion(
                logits,
                y,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer_stage2.step()

            train_loss_sum += loss.item()
            n_batches += 1

        train_loss = (
            train_loss_sum
            / max(n_batches, 1)
        )

        val_metrics = evaluate_phys64(
            model,
            val_loader,
        )

        val_acc = val_metrics[
            "accuracy"
        ]

        scheduler.step(
            val_acc
        )

        print(
            f"TUNED | "
            f"ep={epoch:03d} | "
            f"train={train_loss:.4f} | "
            f"val={val_acc:.4f} | "
            f"lr_backbone="
            f"{optimizer_stage2.param_groups[0]['lr']:.1e} | "
            f"lr_head="
            f"{optimizer_stage2.param_groups[1]['lr']:.1e}"
        )

        if val_acc > best_val:

            best_val = val_acc
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            bad_epochs = 0

        else:

            bad_epochs += 1

        if bad_epochs >= patience:
            break


    # ========================================================
    # TEST
    # ========================================================

    model.load_state_dict(
        best_state
    )

    test_metrics = evaluate_phys64(
        model,
        test_loader,
    )

    return {
        "test_subject":
            test_subject,

        "best_epoch":
            best_epoch,

        "best_val_accuracy":
            best_val,

        "test_accuracy":
            test_metrics["accuracy"],

        "balanced_accuracy":
            test_metrics["balanced_accuracy"],

        "macro_f1":
            test_metrics["macro_f1"],

        "kappa":
            test_metrics["kappa"],
    }, model

In [49]:
result_tuned, model_tuned = (
    train_cho64_transfer_tuned_fold(
        test_subject=1,
        seed=42,
        freeze_epochs=5,
        max_epochs=10,
        patience=5,
        batch_size=64,
    )
)

print()
print("=" * 70)
print("TUNED TRANSFER SANITY")
print("=" * 70)

print(result_tuned)


STAGE 1 — classifier only
FREEZE | ep=01 | train=0.7142 | val=0.6250
FREEZE | ep=02 | train=0.6688 | val=0.6393
FREEZE | ep=03 | train=0.6670 | val=0.6434
FREEZE | ep=04 | train=0.6646 | val=0.6332
FREEZE | ep=05 | train=0.6656 | val=0.6496

STAGE 2 — full fine-tuning
TUNED | ep=001 | train=0.6624 | val=0.6660 | lr_backbone=1.0e-04 | lr_head=5.0e-04
TUNED | ep=002 | train=0.6624 | val=0.6639 | lr_backbone=1.0e-04 | lr_head=5.0e-04
TUNED | ep=003 | train=0.6590 | val=0.6455 | lr_backbone=1.0e-04 | lr_head=5.0e-04
TUNED | ep=004 | train=0.6588 | val=0.6680 | lr_backbone=1.0e-04 | lr_head=5.0e-04
TUNED | ep=005 | train=0.6549 | val=0.6639 | lr_backbone=1.0e-04 | lr_head=5.0e-04
TUNED | ep=006 | train=0.6480 | val=0.6803 | lr_backbone=1.0e-04 | lr_head=5.0e-04
TUNED | ep=007 | train=0.6479 | val=0.6783 | lr_backbone=1.0e-04 | lr_head=5.0e-04
TUNED | ep=008 | train=0.6443 | val=0.6578 | lr_backbone=1.0e-04 | lr_head=5.0e-04
TUNED | ep=009 | train=0.6430 | val=0.6824 | lr_backbone=1.0e-04 |

In [50]:
# ============================================================
# SANITY 2 — CHO64 TRANSFER WITHOUT FREEZE
# discriminative learning rates
# ============================================================

def train_cho64_transfer_nofreeze_fold(
    test_subject,
    seed=42,
    max_epochs=50,
    patience=10,
    batch_size=64,
):

    set_seed(seed)

    fold = prepare_phys64_fold_tuned(
        test_subject=test_subject,
        seed=seed,
    )

    train_loader = DataLoader(
        Phys64Dataset(
            fold["X_train"],
            fold["y_train"],
            augment=True,
        ),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )

    val_loader = DataLoader(
        Phys64Dataset(
            fold["X_val"],
            fold["y_val"],
            augment=False,
        ),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    test_loader = DataLoader(
        Phys64Dataset(
            fold["X_test"],
            fold["y_test"],
            augment=False,
        ),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    # --------------------------------------------------------
    # FULL CHO64 PRETRAINED MODEL
    # --------------------------------------------------------

    model = make_cho64_transfer_model()

    # Всё сразу trainable
    for p in model.parameters():
        p.requires_grad = True

    criterion = nn.CrossEntropyLoss()

    # --------------------------------------------------------
    # DIFFERENT LR FOR BACKBONE AND HEAD
    # --------------------------------------------------------

    backbone_params = [
        p
        for name, p in model.named_parameters()
        if not name.startswith("classifier.")
    ]

    head_params = list(
        model.classifier.parameters()
    )

    optimizer = torch.optim.AdamW(
        [
            {
                "params": backbone_params,
                "lr": 1e-4,
            },
            {
                "params": head_params,
                "lr": 5e-4,
            },
        ],
        weight_decay=1e-4,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=4,
    )

    best_val = -np.inf
    best_epoch = 0
    best_state = None
    bad_epochs = 0

    history = []

    for epoch in range(
        1,
        max_epochs + 1,
    ):

        model.train()

        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0
        n_batches = 0

        for X, y in train_loader:

            X = X.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits = model(X)

            loss = criterion(
                logits,
                y,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()

            train_loss_sum += loss.item()
            n_batches += 1

            pred = logits.argmax(
                dim=1
            )

            train_correct += (
                pred == y
            ).sum().item()

            train_total += len(y)

        train_loss = (
            train_loss_sum
            / max(n_batches, 1)
        )

        train_acc = (
            train_correct
            / train_total
        )

        val_metrics = evaluate_phys64(
            model,
            val_loader,
        )

        val_acc = val_metrics[
            "accuracy"
        ]

        scheduler.step(
            val_acc
        )

        history.append({
            "epoch":
                epoch,

            "train_loss":
                train_loss,

            "train_accuracy":
                train_acc,

            "val_accuracy":
                val_acc,

            "lr_backbone":
                optimizer.param_groups[0]["lr"],

            "lr_head":
                optimizer.param_groups[1]["lr"],
        })

        print(
            f"NOFREEZE | "
            f"ep={epoch:03d} | "
            f"train={train_loss:.4f} | "
            f"train_acc={train_acc:.4f} | "
            f"val={val_acc:.4f} | "
            f"lr_b={optimizer.param_groups[0]['lr']:.1e} | "
            f"lr_h={optimizer.param_groups[1]['lr']:.1e}"
        )

        if val_acc > best_val:

            best_val = val_acc
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            bad_epochs = 0

        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            print(
                f"Early stopping at epoch {epoch}"
            )
            break

    # --------------------------------------------------------
    # BEST CHECKPOINT
    # --------------------------------------------------------

    model.load_state_dict(
        best_state
    )

    test_metrics = evaluate_phys64(
        model,
        test_loader,
    )

    result = {
        "test_subject":
            test_subject,

        "best_epoch":
            best_epoch,

        "best_val_accuracy":
            best_val,

        "test_accuracy":
            test_metrics["accuracy"],

        "balanced_accuracy":
            test_metrics["balanced_accuracy"],

        "macro_f1":
            test_metrics["macro_f1"],

        "kappa":
            test_metrics["kappa"],
    }

    return (
        result,
        model,
        pd.DataFrame(history),
    )

In [51]:
result_nofreeze, model_nofreeze, history_nofreeze = (
    train_cho64_transfer_nofreeze_fold(
        test_subject=1,
        seed=42,
        max_epochs=50,
        patience=10,
        batch_size=64,
    )
)

print()
print("=" * 70)
print("NO-FREEZE TRANSFER SANITY")
print("=" * 70)

print(result_nofreeze)

NOFREEZE | ep=001 | train=0.7259 | train_acc=0.5749 | val=0.6537 | lr_b=1.0e-04 | lr_h=5.0e-04
NOFREEZE | ep=002 | train=0.6765 | train_acc=0.5973 | val=0.6475 | lr_b=1.0e-04 | lr_h=5.0e-04
NOFREEZE | ep=003 | train=0.6662 | train_acc=0.6091 | val=0.6475 | lr_b=1.0e-04 | lr_h=5.0e-04
NOFREEZE | ep=004 | train=0.6604 | train_acc=0.6107 | val=0.6475 | lr_b=1.0e-04 | lr_h=5.0e-04
NOFREEZE | ep=005 | train=0.6547 | train_acc=0.6153 | val=0.6824 | lr_b=1.0e-04 | lr_h=5.0e-04
NOFREEZE | ep=006 | train=0.6514 | train_acc=0.6180 | val=0.6906 | lr_b=1.0e-04 | lr_h=5.0e-04
NOFREEZE | ep=007 | train=0.6521 | train_acc=0.6089 | val=0.6721 | lr_b=1.0e-04 | lr_h=5.0e-04
NOFREEZE | ep=008 | train=0.6510 | train_acc=0.6121 | val=0.6742 | lr_b=1.0e-04 | lr_h=5.0e-04
NOFREEZE | ep=009 | train=0.6528 | train_acc=0.6055 | val=0.6844 | lr_b=1.0e-04 | lr_h=5.0e-04
NOFREEZE | ep=010 | train=0.6481 | train_acc=0.6169 | val=0.6762 | lr_b=1.0e-04 | lr_h=5.0e-04
NOFREEZE | ep=011 | train=0.6427 | train_acc=0.622

# Эксперимент 5. Cho64 Transfer + SupCon + GRL

## Цель

Повторить на 64-канальном PhysioNet ту же схему,
которая использовалась в успешной 22-канальной постановке:

Cho pretraining → full-model transfer → SupCon → GRL.

В отличие от предыдущих экспериментов здесь:

- не используется freeze/unfreeze;
- вся pretrained-модель дообучается сразу;
- используется единый небольшой learning rate;
- добавляются SupCon и adversarial subject loss через GRL.

## Постановка

**Source:** Cho2017  
- 52 испытуемых;
- 64 канала;
- Left / Right motor imagery;
- 10 520 trials.

**Target:** PhysioNet  
- 109 испытуемых;
- 64 каналов;
- Left / Right motor imagery;
- strict LOSO.

Для каждого fold один испытуемый полностью используется как test.
Из trials остальных испытуемых 90% используются для train и 10% для validation.

## Loss

Итоговая функция потерь:

`Loss = CE + 0.15 × SupCon + 0.10 × SubjectLoss`

где:

- `CE` — классификация Left / Right;
- `SupCon` — формирование class-discriminative representation;
- `SubjectLoss` — классификация subject через Gradient Reversal Layer.

GRL заставляет backbone формировать признаки,
по которым хорошо определяется класс движения,
но хуже определяется личность испытуемого.

## Параметры

- AdamW;
- learning rate = `1e-4`;
- weight decay = `1e-4`;
- max epochs = 80;
- patience = 12;
- `lambda_supcon = 0.15`;
- `lambda_adv = 0.10`.

Начальная точка — pretrained `Cho64 EEGNet64`.


In [52]:
# ============================================================
# CHO64 TRANSFER + SUPCON + GRL
# ============================================================

# import math
# import copy
# import random

# import numpy as np

# import torch
# import torch.nn as nn
# import torch.nn.functional as F


# ============================================================
# GRADIENT REVERSAL
# ============================================================

class GradientReversalFunction(
    torch.autograd.Function
):

    @staticmethod
    def forward(ctx, x, lambda_grl):

        ctx.lambda_grl = lambda_grl

        return x.view_as(x)


    @staticmethod
    def backward(ctx, grad_output):

        return (
            -ctx.lambda_grl * grad_output,
            None,
        )


class GradientReversal(nn.Module):

    def forward(
        self,
        x,
        lambda_grl=1.0,
    ):

        return GradientReversalFunction.apply(
            x,
            lambda_grl,
        )


# ============================================================
# SUPERVISED CONTRASTIVE LOSS
# ============================================================

class SupConLoss(nn.Module):

    def __init__(
        self,
        temperature=0.07,
    ):
        super().__init__()

        self.temperature = temperature


    def forward(
        self,
        features,
        labels,
    ):

        features = F.normalize(
            features,
            dim=1,
        )

        labels = labels.view(-1, 1)

        batch_size = features.shape[0]

        similarity = (
            features @ features.T
        ) / self.temperature

        # исключаем diagonal
        logits_mask = (
            torch.ones_like(similarity)
            -
            torch.eye(
                batch_size,
                device=features.device,
            )
        )

        similarity = (
            similarity
            -
            similarity.max(
                dim=1,
                keepdim=True,
            )[0].detach()
        )

        positive_mask = (
            labels == labels.T
        ).float()

        positive_mask = (
            positive_mask
            * logits_mask
        )

        exp_logits = (
            torch.exp(similarity)
            * logits_mask
        )

        log_prob = (
            similarity
            -
            torch.log(
                exp_logits.sum(
                    dim=1,
                    keepdim=True,
                )
                + 1e-12
            )
        )

        positive_count = (
            positive_mask.sum(
                dim=1
            )
        )

        valid = positive_count > 0

        mean_log_prob_pos = (
            positive_mask
            * log_prob
        ).sum(
            dim=1
        ) / (
            positive_count
            + 1e-12
        )

        loss = (
            -mean_log_prob_pos[valid]
            .mean()
        )

        return loss


# ============================================================
# EEGNET64 + SUPCON + GRL
# ============================================================

class ChoTransferSupConGRL64(nn.Module):

    def __init__(
        self,
        n_subjects,
        projection_dim=64,
    ):
        super().__init__()

        self.backbone = EEGNet64()

        feature_dim = (
            self.backbone.feature_dim
        )

        # ----------------------------------------------------
        # SupCon projector
        # ----------------------------------------------------

        self.projector = nn.Sequential(
            nn.Linear(
                feature_dim,
                128,
            ),
            nn.ReLU(),
            nn.Linear(
                128,
                projection_dim,
            ),
        )

        # ----------------------------------------------------
        # GRL subject discriminator
        # ----------------------------------------------------

        self.grl = GradientReversal()

        self.subject_head = nn.Sequential(
            nn.Linear(
                feature_dim,
                128,
            ),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(
                128,
                n_subjects,
            ),
        )


    def forward(
        self,
        x,
        lambda_grl=1.0,
    ):

        features = (
            self.backbone
            .extract_features(x)
        )

        class_logits = (
            self.backbone.classifier(
                features
            )
        )

        projection = self.projector(
            features
        )

        reversed_features = self.grl(
            features,
            lambda_grl,
        )

        subject_logits = (
            self.subject_head(
                reversed_features
            )
        )

        return {
            "class_logits":
                class_logits,

            "projection":
                projection,

            "subject_logits":
                subject_logits,

            "features":
                features,
        }

In [53]:
# ============================================================
# INITIALIZE FROM CHO64 PRETRAINING
# ============================================================

def make_cho64_supcon_grl_model(
    n_subjects,
):

    checkpoint = torch.load(
        CHO_PRETRAIN_MODEL_PATH,
        map_location=device,
        weights_only=False,
    )

    model = ChoTransferSupConGRL64(
        n_subjects=n_subjects,
    ).to(device)

    result = (
        model.backbone.load_state_dict(
            checkpoint["state_dict"],
            strict=True,
        )
    )

    assert len(
        result.missing_keys
    ) == 0

    assert len(
        result.unexpected_keys
    ) == 0

    return model

In [54]:
# ============================================================
# PHYS64 DATASET WITH SUBJECT LABEL
# ============================================================

class Phys64SubjectDataset(
    torch.utils.data.Dataset
):

    def __init__(
        self,
        X,
        y,
        subject_ids,
        augment=False,
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32,
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long,
        )

        self.subject_ids = torch.tensor(
            subject_ids,
            dtype=torch.long,
        )

        self.augment = augment


    def __len__(self):

        return len(self.y)


    def __getitem__(
        self,
        idx,
    ):

        x = self.X[idx].clone()

        y = self.y[idx]

        subject_id = (
            self.subject_ids[idx]
        )

        if self.augment:

            shift = np.random.randint(
                -8,
                9,
            )

            x = torch.roll(
                x,
                shifts=shift,
                dims=-1,
            )

            if np.random.rand() < 0.5:

                x = (
                    x
                    +
                    torch.randn_like(x)
                    * 0.01
                )

            if np.random.rand() < 0.05:

                ch = np.random.randint(
                    0,
                    x.shape[0],
                )

                x[ch] = 0

        return (
            x,
            y,
            subject_id,
        )

In [55]:
# ============================================================
# PREPARE PHYSIONET64 FOLD WITH SUBJECT IDS
# ============================================================

from sklearn.model_selection import (
    train_test_split,
)


def prepare_phys64_fold_supcon_grl(
    test_subject,
    seed=42,
):

    source_subjects = [
        s
        for s in PHYS64_SUBJECTS
        if s != test_subject
    ]

    subject_to_idx = {
        subject:
            idx
        for idx, subject
        in enumerate(
            source_subjects
        )
    }

    X_list = []
    y_list = []
    subject_list = []

    for subject in source_subjects:

        X_s, y_s = (
            load_phys64_subject(
                subject
            )
        )

        X_list.append(
            X_s
        )

        y_list.append(
            y_s
        )

        subject_list.append(
            np.full(
                len(y_s),
                subject_to_idx[
                    subject
                ],
                dtype=np.int64,
            )
        )

    X_source = np.concatenate(
        X_list,
        axis=0,
    )

    y_source = np.concatenate(
        y_list,
        axis=0,
    )

    subject_source = np.concatenate(
        subject_list,
        axis=0,
    )

    indices = np.arange(
        len(y_source)
    )

    (
        idx_train,
        idx_val,
    ) = train_test_split(
        indices,
        test_size=0.10,
        random_state=seed,
        stratify=y_source,
    )

    X_train = X_source[
        idx_train
    ]

    y_train = y_source[
        idx_train
    ]

    s_train = subject_source[
        idx_train
    ]

    X_val = X_source[
        idx_val
    ]

    y_val = y_source[
        idx_val
    ]

    s_val = subject_source[
        idx_val
    ]

    X_test, y_test = (
        load_phys64_subject(
            test_subject
        )
    )

    # --------------------------------------------------------
    # TRAIN-ONLY NORMALIZATION
    # --------------------------------------------------------

    mean = X_train.mean(
        axis=(0, 2),
        keepdims=True,
    )

    std = X_train.std(
        axis=(0, 2),
        keepdims=True,
    )

    std = np.maximum(
        std,
        1e-6,
    )

    X_train = (
        X_train - mean
    ) / std

    X_val = (
        X_val - mean
    ) / std

    X_test = (
        X_test - mean
    ) / std

    return {
        "X_train":
            X_train,

        "y_train":
            y_train,

        "s_train":
            s_train,

        "X_val":
            X_val,

        "y_val":
            y_val,

        "s_val":
            s_val,

        "X_test":
            X_test,

        "y_test":
            y_test,

        "n_source_subjects":
            len(source_subjects),
    }

In [57]:
# ============================================================
# TRAIN ONE FOLD
# CHO64 TRANSFER + SUPCON + GRL
# ============================================================

def train_cho64_supcon_grl_fold(
    test_subject,
    seed=42,
    max_epochs=80,
    patience=12,
    batch_size=64,
    lr=1e-4,
    weight_decay=1e-4,
    lambda_supcon=0.15,
    lambda_adv=0.10,
):

    set_seed(seed)

    fold = (
        prepare_phys64_fold_supcon_grl(
            test_subject=test_subject,
            seed=seed,
        )
    )

    train_loader = DataLoader(
        Phys64SubjectDataset(
            fold["X_train"],
            fold["y_train"],
            fold["s_train"],
            augment=True,
        ),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )

    val_loader = DataLoader(
        Phys64SubjectDataset(
            fold["X_val"],
            fold["y_val"],
            fold["s_val"],
            augment=False,
        ),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    test_loader = DataLoader(
        Phys64Dataset(
            fold["X_test"],
            fold["y_test"],
            augment=False,
        ),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    model = (
        make_cho64_supcon_grl_model(
            n_subjects=
                fold[
                    "n_source_subjects"
                ],
        )
    )

    ce_loss_fn = (
        nn.CrossEntropyLoss()
    )

    subject_loss_fn = (
        nn.CrossEntropyLoss()
    )

    supcon_loss_fn = (
        SupConLoss(
            temperature=0.07
        )
    )

    # ВСЯ сеть сразу fine-tune
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=5,
        )
    )

    best_val = -np.inf
    best_epoch = 0
    best_state = None

    bad_epochs = 0

    history = []

    for epoch in range(
        1,
        max_epochs + 1,
    ):

        model.train()

        total_loss_sum = 0.0
        ce_sum = 0.0
        supcon_sum = 0.0
        adv_sum = 0.0

        train_correct = 0
        train_total = 0

        n_batches = 0


        # ----------------------------------------------------
        # GRL RAMP
        # ----------------------------------------------------

        progress = (
            (epoch - 1)
            /
            max(
                max_epochs - 1,
                1,
            )
        )

        lambda_grl = (
            2.0
            /
            (
                1.0
                +
                math.exp(
                    -10.0
                    * progress
                )
            )
            - 1.0
        )


        for (
            X,
            y,
            subject_id,
        ) in train_loader:

            X = X.to(device)
            y = y.to(device)

            subject_id = (
                subject_id.to(
                    device
                )
            )

            optimizer.zero_grad()

            out = model(
                X,
                lambda_grl=
                    lambda_grl,
            )

            loss_ce = ce_loss_fn(
                out["class_logits"],
                y,
            )

            loss_supcon = (
                supcon_loss_fn(
                    out["projection"],
                    y,
                )
            )

            loss_adv = (
                subject_loss_fn(
                    out[
                        "subject_logits"
                    ],
                    subject_id,
                )
            )

            loss = (
                loss_ce
                +
                lambda_supcon
                * loss_supcon
                +
                lambda_adv
                * loss_adv
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()

            total_loss_sum += (
                loss.item()
            )

            ce_sum += (
                loss_ce.item()
            )

            supcon_sum += (
                loss_supcon.item()
            )

            adv_sum += (
                loss_adv.item()
            )

            pred = (
                out["class_logits"]
                .argmax(dim=1)
            )

            train_correct += (
                pred == y
            ).sum().item()

            train_total += len(y)

            n_batches += 1


        # ----------------------------------------------------
        # VALIDATION — class accuracy only
        # ----------------------------------------------------

        model.eval()

        val_true = []
        val_pred = []

        with torch.no_grad():

            for (
                X,
                y,
                subject_id,
            ) in val_loader:

                X = X.to(device)
                y = y.to(device)

                out = model(
                    X,
                    lambda_grl=0.0,
                )

                pred = (
                    out[
                        "class_logits"
                    ]
                    .argmax(dim=1)
                )

                val_true.extend(
                    y.cpu().numpy()
                )

                val_pred.extend(
                    pred.cpu().numpy()
                )

        val_acc = accuracy_score(
            val_true,
            val_pred,
        )

        scheduler.step(
            val_acc
        )


        train_acc = (
            train_correct
            / train_total
        )

        history.append({
            "epoch":
                epoch,

            "train_accuracy":
                train_acc,

            "val_accuracy":
                val_acc,

            "total_loss":
                total_loss_sum
                / n_batches,

            "ce_loss":
                ce_sum
                / n_batches,

            "supcon_loss":
                supcon_sum
                / n_batches,

            "adv_loss":
                adv_sum
                / n_batches,

            "lambda_grl":
                lambda_grl,

            "lr":
                optimizer
                .param_groups[0]["lr"],
        })


        print(
            f"T+SC+GRL | "
            f"ep={epoch:03d} | "
            f"train={train_acc:.4f} | "
            f"val={val_acc:.4f} | "
            f"CE={ce_sum/n_batches:.3f} | "
            f"SC={supcon_sum/n_batches:.3f} | "
            f"ADV={adv_sum/n_batches:.3f} | "
            f"grl={lambda_grl:.3f}"
        )


        if val_acc > best_val:

            best_val = val_acc
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            bad_epochs = 0

        else:

            bad_epochs += 1


        if bad_epochs >= patience:

            print(
                f"Early stopping "
                f"at epoch {epoch}"
            )

            break


    # ========================================================
    # BEST MODEL
    # ========================================================

    model.load_state_dict(
        best_state
    )


    # ========================================================
    # TEST — CLASSIFICATION ONLY
    # ========================================================

    model.eval()

    test_true = []
    test_pred = []

    with torch.no_grad():

        for X, y in test_loader:

            X = X.to(device)

            out = model(
                X,
                lambda_grl=0.0,
            )

            pred = (
                out[
                    "class_logits"
                ]
                .argmax(dim=1)
                .cpu()
                .numpy()
            )

            test_true.extend(
                y.numpy()
            )

            test_pred.extend(
                pred
            )

    test_true = np.asarray(
        test_true
    )

    test_pred = np.asarray(
        test_pred
    )


    result = {
        "test_subject":
            test_subject,

        "best_epoch":
            best_epoch,

        "best_val_accuracy":
            best_val,

        "test_accuracy":
            accuracy_score(
                test_true,
                test_pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                test_true,
                test_pred,
            ),

        "macro_f1":
            f1_score(
                test_true,
                test_pred,
                average="macro",
            ),

        "kappa":
            cohen_kappa_score(
                test_true,
                test_pred,
            ),
    }


    return (
        result,
        model,
        pd.DataFrame(history),
    )

In [58]:
result_scgrl, model_scgrl, history_scgrl = (
    train_cho64_supcon_grl_fold(
        test_subject=1,

        seed=42,

        max_epochs=25,
        patience=8,

        batch_size=64,

        lr=1e-4,
        weight_decay=1e-4,

        lambda_supcon=0.15,
        lambda_adv=0.10,
    )
)

print()
print("=" * 75)
print("CHO64 TRANSFER + SUPCON + GRL SANITY")
print("=" * 75)

print(result_scgrl)

T+SC+GRL | ep=001 | train=0.5822 | val=0.6434 | CE=0.745 | SC=4.446 | ADV=4.679 | grl=0.000
T+SC+GRL | ep=002 | train=0.5811 | val=0.6537 | CE=0.714 | SC=4.211 | ADV=4.653 | grl=0.205
T+SC+GRL | ep=003 | train=0.5859 | val=0.6742 | CE=0.699 | SC=4.174 | ADV=4.631 | grl=0.394
T+SC+GRL | ep=004 | train=0.6048 | val=0.6414 | CE=0.684 | SC=4.159 | ADV=4.599 | grl=0.555
T+SC+GRL | ep=005 | train=0.5968 | val=0.6393 | CE=0.680 | SC=4.153 | ADV=4.562 | grl=0.682
T+SC+GRL | ep=006 | train=0.5927 | val=0.6557 | CE=0.668 | SC=4.148 | ADV=4.525 | grl=0.779
T+SC+GRL | ep=007 | train=0.5984 | val=0.6721 | CE=0.666 | SC=4.146 | ADV=4.488 | grl=0.848
T+SC+GRL | ep=008 | train=0.6027 | val=0.6598 | CE=0.662 | SC=4.143 | ADV=4.459 | grl=0.897
T+SC+GRL | ep=009 | train=0.6100 | val=0.6516 | CE=0.662 | SC=4.141 | ADV=4.427 | grl=0.931
T+SC+GRL | ep=010 | train=0.6068 | val=0.6537 | CE=0.656 | SC=4.139 | ADV=4.408 | grl=0.954
T+SC+GRL | ep=011 | train=0.6052 | val=0.6742 | CE=0.654 | SC=4.139 | ADV=4.395 

In [59]:
# ============================================================
# FULL 109-FOLD LOSO
# CHO64 TRANSFER + SUPCON + GRL
# resume-safe
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import torch
import gc
import time


# ============================================================
# PATHS
# ============================================================

SCGRL64_ROOT = (
    RESULTS_ROOT
    / "cho64_transfer_supcon_grl_physionet64"
)

SCGRL64_RESULTS_DIR = (
    SCGRL64_ROOT
    / "results"
)

SCGRL64_MODELS_DIR = (
    SCGRL64_ROOT
    / "models"
)

SCGRL64_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SCGRL64_MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


PROGRESS_CSV = (
    SCGRL64_RESULTS_DIR
    / "cho64_supcon_grl_progress.csv"
)

FINAL_CSV = (
    SCGRL64_RESULTS_DIR
    / "cho64_supcon_grl_final.csv"
)

SUMMARY_CSV = (
    SCGRL64_RESULTS_DIR
    / "cho64_supcon_grl_summary.csv"
)


# ============================================================
# RESUME
# ============================================================

if PROGRESS_CSV.exists():

    results_df = pd.read_csv(
        PROGRESS_CSV
    )

    done_subjects = set(
        results_df[
            "test_subject"
        ]
        .astype(int)
        .tolist()
    )

    print(
        f"Найден предыдущий прогресс: "
        f"{len(done_subjects)}/109"
    )

else:

    results_df = pd.DataFrame()

    done_subjects = set()

    print(
        "Начинаем новый "
        "Cho64 + SupCon + GRL run"
    )


# ============================================================
# SUBJECTS
# ============================================================

PHYS64_SUBJECTS = list(
    range(1, 110)
)

total_subjects = len(
    PHYS64_SUBJECTS
)


# ============================================================
# RUN
# ============================================================

for fold_idx, test_subject in enumerate(
    PHYS64_SUBJECTS,
    start=1,
):

    if test_subject in done_subjects:

        print(
            f"[{fold_idx:03d}/{total_subjects}] "
            f"Subject {test_subject:03d} "
            f"already done — skip"
        )

        continue


    print()
    print("#" * 85)

    print(
        f"FOLD {fold_idx}/{total_subjects} "
        f"| TEST SUBJECT={test_subject}"
    )

    print("#" * 85)


    start_time = time.time()


    try:

        result, model, history = (
            train_cho64_supcon_grl_fold(

                test_subject=
                    test_subject,

                seed=
                    42,

                max_epochs=
                    80,

                patience=
                    12,

                batch_size=
                    64,

                lr=
                    1e-4,

                weight_decay=
                    1e-4,

                lambda_supcon=
                    0.15,

                lambda_adv=
                    0.10,
            )
        )


        # ====================================================
        # SAVE MODEL
        # ====================================================

        model_path = (
            SCGRL64_MODELS_DIR
            / (
                f"cho64_scgrl_"
                f"subject_{test_subject:03d}.pt"
            )
        )

        torch.save(
            {
                "test_subject":
                    test_subject,

                "best_epoch":
                    result[
                        "best_epoch"
                    ],

                "best_val_accuracy":
                    result[
                        "best_val_accuracy"
                    ],

                "state_dict":
                    model.state_dict(),

                "source_checkpoint":
                    str(
                        CHO_PRETRAIN_MODEL_PATH
                    ),

                "seed":
                    42,

                "lambda_supcon":
                    0.15,

                "lambda_adv":
                    0.10,

                "lr":
                    1e-4,

                "weight_decay":
                    1e-4,

                "channels":
                    PHYS64_CHANNELS,
            },
            model_path,
        )


        # ====================================================
        # SAVE HISTORY
        # ====================================================

        history_path = (
            SCGRL64_RESULTS_DIR
            / (
                f"history_"
                f"subject_{test_subject:03d}.csv"
            )
        )

        history.to_csv(
            history_path,
            index=False,
        )


        # ====================================================
        # RESULT INFO
        # ====================================================

        result[
            "runtime_sec"
        ] = (
            time.time()
            - start_time
        )

        result[
            "model_path"
        ] = str(
            model_path
        )

        result[
            "history_path"
        ] = str(
            history_path
        )


        results_df = pd.concat(
            [
                results_df,
                pd.DataFrame(
                    [result]
                ),
            ],
            ignore_index=True,
        )


        results_df = (
            results_df
            .sort_values(
                "test_subject"
            )
            .reset_index(
                drop=True
            )
        )


        # ====================================================
        # SAVE AFTER EACH FOLD
        # ====================================================

        results_df.to_csv(
            PROGRESS_CSV,
            index=False,
        )


        # ====================================================
        # CURRENT RESULT
        # ====================================================

        print()
        print("-" * 75)

        print(
            f"SUBJECT "
            f"{test_subject:03d} DONE"
        )

        print(
            f"Best epoch: "
            f"{result['best_epoch']}"
        )

        print(
            f"Validation: "
            f"{result['best_val_accuracy'] * 100:.2f}%"
        )

        print(
            f"Test accuracy: "
            f"{result['test_accuracy'] * 100:.2f}%"
        )

        print(
            f"Balanced accuracy: "
            f"{result['balanced_accuracy'] * 100:.2f}%"
        )

        print(
            f"Macro F1: "
            f"{result['macro_f1'] * 100:.2f}%"
        )

        print(
            f"Kappa: "
            f"{result['kappa']:.3f}"
        )

        print("-" * 75)


        # ====================================================
        # RUNNING SUMMARY
        # ====================================================

        running_mean = (
            results_df[
                "test_accuracy"
            ].mean()
            * 100
        )

        running_std = (
            results_df[
                "test_accuracy"
            ].std()
            * 100
        )

        print()

        print(
            f"RUNNING MEAN "
            f"({len(results_df)}/{total_subjects}): "
            f"{running_mean:.2f}%"
        )

        if len(results_df) > 1:

            print(
                f"RUNNING STD: "
                f"{running_std:.2f} pp"
            )

        print(
            f"DELTA vs CCST64 73.65%: "
            f"{running_mean - 73.65:+.2f} pp"
        )


        # ====================================================
        # CLEAN MEMORY
        # ====================================================

        del model
        del history

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    except Exception as e:

        results_df.to_csv(
            PROGRESS_CSV,
            index=False,
        )

        print()
        print("!" * 85)

        print(
            f"ERROR ON SUBJECT "
            f"{test_subject}"
        )

        print(repr(e))

        print(
            "Предыдущие folds сохранены в:"
        )

        print(
            PROGRESS_CSV
        )

        print("!" * 85)

        raise


# ============================================================
# FINAL RESULTS
# ============================================================

results_df = (
    results_df
    .sort_values(
        "test_subject"
    )
    .reset_index(
        drop=True
    )
)

results_df.to_csv(
    FINAL_CSV,
    index=False,
)


# ============================================================
# SUMMARY
# ============================================================

summary = pd.DataFrame([
    {
        "method":
            "Cho64_Transfer_SupCon_GRL",

        "n_subjects":
            len(results_df),

        "mean_accuracy":
            results_df[
                "test_accuracy"
            ].mean(),

        "std_accuracy":
            results_df[
                "test_accuracy"
            ].std(),

        "median_accuracy":
            results_df[
                "test_accuracy"
            ].median(),

        "mean_balanced_accuracy":
            results_df[
                "balanced_accuracy"
            ].mean(),

        "mean_macro_f1":
            results_df[
                "macro_f1"
            ].mean(),

        "mean_kappa":
            results_df[
                "kappa"
            ].mean(),

        "mean_best_epoch":
            results_df[
                "best_epoch"
            ].mean(),

        "median_best_epoch":
            results_df[
                "best_epoch"
            ].median(),
    }
])


summary[
    "mean_accuracy_pct"
] = (
    summary[
        "mean_accuracy"
    ]
    * 100
)


summary.to_csv(
    SUMMARY_CSV,
    index=False,
)


# ============================================================
# FINAL PRINT
# ============================================================

mean_acc = (
    summary.iloc[0][
        "mean_accuracy"
    ]
    * 100
)


print()
print("=" * 90)
print(
    "FINAL CHO64 TRANSFER + "
    "SUPCON + GRL SUMMARY"
)
print("=" * 90)

display(summary)

print()

print(
    f"Cho64 Transfer + SupCon + GRL: "
    f"{mean_acc:.2f}%"
)

print(
    "CCST64: "
    "73.65%"
)

print(
    f"Delta vs CCST64: "
    f"{mean_acc - 73.65:+.2f} pp"
)

print()

print(
    "Previous Cho64 Full Transfer: "
    "65.87%"
)

print(
    f"Gain vs plain transfer: "
    f"{mean_acc - 65.87:+.2f} pp"
)

print()

print("Saved:")
print(FINAL_CSV)
print(SUMMARY_CSV)

Начинаем новый Cho64 + SupCon + GRL run

#####################################################################################
FOLD 1/109 | TEST SUBJECT=1
#####################################################################################
T+SC+GRL | ep=001 | train=0.5822 | val=0.6434 | CE=0.745 | SC=4.446 | ADV=4.679 | grl=0.000
T+SC+GRL | ep=002 | train=0.5811 | val=0.6516 | CE=0.714 | SC=4.211 | ADV=4.653 | grl=0.063
T+SC+GRL | ep=003 | train=0.5859 | val=0.6762 | CE=0.699 | SC=4.174 | ADV=4.631 | grl=0.126
T+SC+GRL | ep=004 | train=0.6048 | val=0.6414 | CE=0.684 | SC=4.159 | ADV=4.598 | grl=0.188
T+SC+GRL | ep=005 | train=0.5970 | val=0.6393 | CE=0.680 | SC=4.153 | ADV=4.561 | grl=0.248
T+SC+GRL | ep=006 | train=0.5934 | val=0.6557 | CE=0.668 | SC=4.149 | ADV=4.523 | grl=0.306
T+SC+GRL | ep=007 | train=0.5989 | val=0.6721 | CE=0.666 | SC=4.146 | ADV=4.486 | grl=0.362
T+SC+GRL | ep=008 | train=0.6043 | val=0.6660 | CE=0.662 | SC=4.143 | ADV=4.456 | grl=0.416
T+SC+GRL | ep=009 | tra

KeyboardInterrupt: 

# Эксперимент 6. Adaptive loss balancing

В предыдущем варианте использовались фиксированные коэффициенты:

Loss = CE + 0.15 × SupCon + 0.10 × SubjectLoss

На 64-канальной постановке значения дополнительных loss оказались
существенно выше CE:

- CE ≈ 0.6
- SupCon ≈ 4.1
- SubjectLoss ≈ 4.4

Поэтому дополнительные задачи могли оказывать слишком сильное влияние
на backbone.

В этом эксперименте веса SupCon и GRL рассчитываются автоматически
относительно текущего масштаба CE.

Целевой вклад:

- SupCon ≈ 20% от CE;
- GRL ≈ 10% от CE.

Для устойчивости используются EMA-сглаженные значения loss.

Также максимальная сила Gradient Reversal ограничена значением 0.5,
чтобы удаление subject-specific информации не разрушало
признаки, полезные для Left/Right классификации.

In [60]:
# ============================================================
# ADAPTIVE LOSS BALANCER
# ============================================================

class AdaptiveLossBalancer:

    def __init__(
        self,
        target_supcon_ratio=0.20,
        target_adv_ratio=0.10,
        ema_beta=0.95,
        min_supcon_weight=0.005,
        max_supcon_weight=0.08,
        min_adv_weight=0.003,
        max_adv_weight=0.05,
    ):

        self.target_supcon_ratio = (
            target_supcon_ratio
        )

        self.target_adv_ratio = (
            target_adv_ratio
        )

        self.ema_beta = ema_beta

        self.min_supcon_weight = (
            min_supcon_weight
        )

        self.max_supcon_weight = (
            max_supcon_weight
        )

        self.min_adv_weight = (
            min_adv_weight
        )

        self.max_adv_weight = (
            max_adv_weight
        )

        self.ema_ce = None
        self.ema_supcon = None
        self.ema_adv = None


    def _update_ema(
        self,
        old,
        new,
    ):

        new = float(new)

        if old is None:
            return new

        return (
            self.ema_beta * old
            +
            (1.0 - self.ema_beta) * new
        )


    def update(
        self,
        loss_ce,
        loss_supcon,
        loss_adv,
    ):

        self.ema_ce = self._update_ema(
            self.ema_ce,
            loss_ce.detach().item(),
        )

        self.ema_supcon = self._update_ema(
            self.ema_supcon,
            loss_supcon.detach().item(),
        )

        self.ema_adv = self._update_ema(
            self.ema_adv,
            loss_adv.detach().item(),
        )

        # ----------------------------------------------------
        # SupCon weight
        # ----------------------------------------------------

        w_supcon = (
            self.target_supcon_ratio
            * self.ema_ce
            / max(
                self.ema_supcon,
                1e-8,
            )
        )

        w_supcon = float(
            np.clip(
                w_supcon,
                self.min_supcon_weight,
                self.max_supcon_weight,
            )
        )


        # ----------------------------------------------------
        # GRL / adversarial weight
        # ----------------------------------------------------

        w_adv = (
            self.target_adv_ratio
            * self.ema_ce
            / max(
                self.ema_adv,
                1e-8,
            )
        )

        w_adv = float(
            np.clip(
                w_adv,
                self.min_adv_weight,
                self.max_adv_weight,
            )
        )


        return (
            w_supcon,
            w_adv,
        )

In [61]:
# ============================================================
# TRAIN ONE FOLD
# CHO64 TRANSFER + SUPCON + GRL
# WITH ADAPTIVE LOSS BALANCING
# ============================================================

def train_cho64_smartloss_fold(
    test_subject,
    seed=42,
    max_epochs=80,
    patience=12,
    batch_size=64,
    lr=1e-4,
    weight_decay=1e-4,
    target_supcon_ratio=0.20,
    target_adv_ratio=0.10,
    grl_max=0.5,
):

    set_seed(seed)

    fold = prepare_phys64_fold_supcon_grl(
        test_subject=test_subject,
        seed=seed,
    )

    train_loader = DataLoader(
        Phys64SubjectDataset(
            fold["X_train"],
            fold["y_train"],
            fold["s_train"],
            augment=True,
        ),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )

    val_loader = DataLoader(
        Phys64SubjectDataset(
            fold["X_val"],
            fold["y_val"],
            fold["s_val"],
            augment=False,
        ),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    test_loader = DataLoader(
        Phys64Dataset(
            fold["X_test"],
            fold["y_test"],
            augment=False,
        ),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )


    # ========================================================
    # MODEL
    # ========================================================

    model = make_cho64_supcon_grl_model(
        n_subjects=
            fold["n_source_subjects"]
    )


    ce_loss_fn = nn.CrossEntropyLoss()
    subject_loss_fn = nn.CrossEntropyLoss()

    supcon_loss_fn = SupConLoss(
        temperature=0.07
    )


    # ========================================================
    # SMART LOSS BALANCER
    # ========================================================

    balancer = AdaptiveLossBalancer(
        target_supcon_ratio=
            target_supcon_ratio,

        target_adv_ratio=
            target_adv_ratio,

        ema_beta=
            0.95,
    )


    # ========================================================
    # OPTIMIZER
    # ========================================================

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=5,
        )
    )


    best_val = -np.inf
    best_epoch = 0
    best_state = None

    bad_epochs = 0

    history = []


    # ========================================================
    # TRAINING
    # ========================================================

    for epoch in range(
        1,
        max_epochs + 1,
    ):

        model.train()

        total_loss_sum = 0.0
        ce_sum = 0.0
        sc_sum = 0.0
        adv_sum = 0.0

        w_sc_sum = 0.0
        w_adv_sum = 0.0

        train_correct = 0
        train_total = 0

        n_batches = 0


        # ----------------------------------------------------
        # SMOOTH GRL RAMP
        # 0 -> grl_max
        # ----------------------------------------------------

        progress = (
            (epoch - 1)
            /
            max(
                max_epochs - 1,
                1,
            )
        )

        lambda_grl = (
            grl_max
            *
            (
                2.0
                /
                (
                    1.0
                    +
                    math.exp(
                        -10.0
                        * progress
                    )
                )
                - 1.0
            )
        )


        for (
            X,
            y,
            subject_id,
        ) in train_loader:

            X = X.to(device)
            y = y.to(device)

            subject_id = (
                subject_id.to(device)
            )

            optimizer.zero_grad()


            out = model(
                X,
                lambda_grl=
                    lambda_grl,
            )


            # =================================================
            # THREE LOSSES
            # =================================================

            loss_ce = ce_loss_fn(
                out["class_logits"],
                y,
            )

            loss_supcon = (
                supcon_loss_fn(
                    out["projection"],
                    y,
                )
            )

            loss_adv = (
                subject_loss_fn(
                    out["subject_logits"],
                    subject_id,
                )
            )


            # =================================================
            # ADAPTIVE WEIGHTS
            # =================================================

            (
                w_supcon,
                w_adv,
            ) = balancer.update(
                loss_ce,
                loss_supcon,
                loss_adv,
            )


            # =================================================
            # TOTAL LOSS
            # =================================================

            loss = (
                loss_ce
                +
                w_supcon
                * loss_supcon
                +
                w_adv
                * loss_adv
            )


            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()


            # =================================================
            # LOGGING
            # =================================================

            total_loss_sum += (
                loss.item()
            )

            ce_sum += (
                loss_ce.item()
            )

            sc_sum += (
                loss_supcon.item()
            )

            adv_sum += (
                loss_adv.item()
            )

            w_sc_sum += (
                w_supcon
            )

            w_adv_sum += (
                w_adv
            )


            pred = (
                out["class_logits"]
                .argmax(dim=1)
            )

            train_correct += (
                pred == y
            ).sum().item()

            train_total += len(y)

            n_batches += 1


        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()

        val_true = []
        val_pred = []

        with torch.no_grad():

            for (
                X,
                y,
                subject_id,
            ) in val_loader:

                X = X.to(device)
                y = y.to(device)

                out = model(
                    X,
                    lambda_grl=0.0,
                )

                pred = (
                    out["class_logits"]
                    .argmax(dim=1)
                )

                val_true.extend(
                    y.cpu().numpy()
                )

                val_pred.extend(
                    pred.cpu().numpy()
                )


        val_acc = accuracy_score(
            val_true,
            val_pred,
        )

        scheduler.step(
            val_acc
        )


        train_acc = (
            train_correct
            / train_total
        )


        mean_ce = (
            ce_sum / n_batches
        )

        mean_sc = (
            sc_sum / n_batches
        )

        mean_adv = (
            adv_sum / n_batches
        )

        mean_w_sc = (
            w_sc_sum / n_batches
        )

        mean_w_adv = (
            w_adv_sum / n_batches
        )


        history.append({

            "epoch":
                epoch,

            "train_accuracy":
                train_acc,

            "val_accuracy":
                val_acc,

            "total_loss":
                total_loss_sum
                / n_batches,

            "ce_loss":
                mean_ce,

            "supcon_loss":
                mean_sc,

            "adv_loss":
                mean_adv,

            "w_supcon":
                mean_w_sc,

            "w_adv":
                mean_w_adv,

            "supcon_contribution":
                mean_w_sc
                * mean_sc,

            "adv_contribution":
                mean_w_adv
                * mean_adv,

            "lambda_grl":
                lambda_grl,

            "lr":
                optimizer
                .param_groups[0]["lr"],
        })


        print(
            f"SMART | "
            f"ep={epoch:03d} | "
            f"train={train_acc:.4f} | "
            f"val={val_acc:.4f} | "
            f"CE={mean_ce:.3f} | "
            f"SC={mean_sc:.3f} "
            f"(w={mean_w_sc:.4f}) | "
            f"ADV={mean_adv:.3f} "
            f"(w={mean_w_adv:.4f}) | "
            f"GRL={lambda_grl:.3f}"
        )


        # ====================================================
        # BEST CHECKPOINT
        # ====================================================

        if val_acc > best_val:

            best_val = val_acc
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            bad_epochs = 0

        else:

            bad_epochs += 1


        if bad_epochs >= patience:

            print(
                f"Early stopping "
                f"at epoch {epoch}"
            )

            break


    # ========================================================
    # RESTORE BEST MODEL
    # ========================================================

    model.load_state_dict(
        best_state
    )


    # ========================================================
    # TEST
    # ========================================================

    model.eval()

    test_true = []
    test_pred = []

    with torch.no_grad():

        for X, y in test_loader:

            X = X.to(device)

            out = model(
                X,
                lambda_grl=0.0,
            )

            pred = (
                out["class_logits"]
                .argmax(dim=1)
                .cpu()
                .numpy()
            )

            test_true.extend(
                y.numpy()
            )

            test_pred.extend(
                pred
            )


    test_true = np.asarray(
        test_true
    )

    test_pred = np.asarray(
        test_pred
    )


    result = {

        "test_subject":
            test_subject,

        "best_epoch":
            best_epoch,

        "best_val_accuracy":
            best_val,

        "test_accuracy":
            accuracy_score(
                test_true,
                test_pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                test_true,
                test_pred,
            ),

        "macro_f1":
            f1_score(
                test_true,
                test_pred,
                average="macro",
            ),

        "kappa":
            cohen_kappa_score(
                test_true,
                test_pred,
            ),
    }


    return (
        result,
        model,
        pd.DataFrame(history),
    )

In [62]:
# ============================================================
# SMART LOSS — MINI ABLATION
# ============================================================

SMART_TEST_SUBJECTS = [
    1,   # было 71.11
    2,   # было 97.78
    3,   # было 46.67
    9,   # было 42.22
    15,  # было 86.67
]

smart_results = []


for subject in SMART_TEST_SUBJECTS:

    print()
    print("=" * 80)
    print(
        f"SMART LOSS | SUBJECT {subject}"
    )
    print("=" * 80)

    result, model, history = (
        train_cho64_smartloss_fold(
            test_subject=subject,

            seed=42,

            max_epochs=60,
            patience=10,

            batch_size=64,

            lr=1e-4,
            weight_decay=1e-4,

            target_supcon_ratio=0.20,
            target_adv_ratio=0.10,

            grl_max=0.5,
        )
    )

    smart_results.append(
        result
    )

    print()
    print(result)

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


smart_results_df = pd.DataFrame(
    smart_results
)

print()
print("=" * 80)
print("SMART LOSS MINI SUMMARY")
print("=" * 80)

display(
    smart_results_df
)

print(
    "Mean accuracy:",
    f"{smart_results_df['test_accuracy'].mean()*100:.2f}%"
)


SMART LOSS | SUBJECT 1
SMART | ep=001 | train=0.5815 | val=0.6414 | CE=0.745 | SC=4.443 (w=0.0317) | ADV=4.678 (w=0.0155) | GRL=0.000
SMART | ep=002 | train=0.5815 | val=0.6475 | CE=0.714 | SC=4.205 (w=0.0341) | ADV=4.653 (w=0.0155) | GRL=0.042
SMART | ep=003 | train=0.5865 | val=0.6721 | CE=0.699 | SC=4.171 (w=0.0337) | ADV=4.631 (w=0.0152) | GRL=0.084
SMART | ep=004 | train=0.6050 | val=0.6414 | CE=0.684 | SC=4.157 (w=0.0330) | ADV=4.599 (w=0.0149) | GRL=0.124
SMART | ep=005 | train=0.5970 | val=0.6475 | CE=0.680 | SC=4.151 (w=0.0329) | ADV=4.563 (w=0.0149) | GRL=0.163
SMART | ep=006 | train=0.5925 | val=0.6619 | CE=0.668 | SC=4.147 (w=0.0322) | ADV=4.525 (w=0.0147) | GRL=0.200
SMART | ep=007 | train=0.5984 | val=0.6742 | CE=0.666 | SC=4.145 (w=0.0323) | ADV=4.488 (w=0.0149) | GRL=0.234
SMART | ep=008 | train=0.6039 | val=0.6701 | CE=0.662 | SC=4.142 (w=0.0319) | ADV=4.457 (w=0.0148) | GRL=0.266
SMART | ep=009 | train=0.6091 | val=0.6537 | CE=0.663 | SC=4.141 (w=0.0321) | ADV=4.421 

,test_subject,best_epoch,best_val_accuracy,test_accuracy,balanced_accuracy,macro_f1,kappa
0,1,14,0.700820,0.755556,0.757905,0.753609,0.513274
1,2,21,0.700820,0.977778,0.977273,0.977734,0.955490
2,3,15,0.698770,0.488889,0.498024,0.391534,-0.003880
3,9,43,0.680328,0.466667,0.470238,0.466403,-0.058824
4,15,3,0.678279,0.866667,0.868577,0.866071,0.734252


Mean accuracy: 71.11%


In [63]:
# ============================================================
# 15 REPRESENTATIVE PHYSIONET SUBJECTS
# ============================================================

REPRESENTATIVE_SUBJECTS = [
    # hard
    3, 9, 18, 24, 30,

    # medium
    1, 8, 19, 27, 32,

    # easy
    2, 12, 15, 25, 29,
]

representative_reference = pd.DataFrame({
    "subject": REPRESENTATIVE_SUBJECTS,

    "group": [
        "hard", "hard", "hard", "hard", "hard",
        "medium", "medium", "medium", "medium", "medium",
        "easy", "easy", "easy", "easy", "easy",
    ],
})

display(representative_reference)

print(
    "Subjects:",
    len(representative_reference)
)

,subject,group
0,3,hard
1,9,hard
2,18,hard
3,24,hard
4,30,hard
5,1,medium
6,8,medium
7,19,medium
8,27,medium
9,32,medium


Subjects: 15


In [64]:
# ============================================================
# UNIVERSAL 64CH ABLATION TRAINING
# ============================================================

def train_cho64_ablation_fold(
    test_subject,
    mode,
    seed=42,

    max_epochs=40,
    patience=8,

    batch_size=64,
    lr=1e-4,
    weight_decay=1e-4,

    # обычные веса как в старой 22ch постановке
    lambda_supcon=0.15,
    lambda_adv=0.10,

    # smart-loss
    target_supcon_ratio=0.20,
    target_adv_ratio=0.10,

    # GRL
    fixed_grl_max=1.0,
    smart_grl_max=0.5,
):

    VALID_MODES = {
        "transfer",
        "supcon",
        "grl",
        "scgrl",
        "smart_scgrl",
    }

    if mode not in VALID_MODES:
        raise ValueError(
            f"Unknown mode={mode}. "
            f"Use one of {VALID_MODES}"
        )

    set_seed(seed)

    # --------------------------------------------------------
    # SAME DATA SPLIT FOR EVERY METHOD
    # --------------------------------------------------------

    fold = prepare_phys64_fold_supcon_grl(
        test_subject=test_subject,
        seed=seed,
    )

    train_loader = DataLoader(
        Phys64SubjectDataset(
            fold["X_train"],
            fold["y_train"],
            fold["s_train"],
            augment=True,
        ),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )

    val_loader = DataLoader(
        Phys64SubjectDataset(
            fold["X_val"],
            fold["y_val"],
            fold["s_val"],
            augment=False,
        ),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    test_loader = DataLoader(
        Phys64Dataset(
            fold["X_test"],
            fold["y_test"],
            augment=False,
        ),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    # --------------------------------------------------------
    # SAME CHO64 INITIALIZATION
    # --------------------------------------------------------

    model = make_cho64_supcon_grl_model(
        n_subjects=fold["n_source_subjects"]
    )

    ce_loss_fn = nn.CrossEntropyLoss()

    supcon_loss_fn = SupConLoss(
        temperature=0.07
    )

    subject_loss_fn = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=4,
        )
    )

    # --------------------------------------------------------
    # SMART BALANCER ONLY FOR SMART MODE
    # --------------------------------------------------------

    balancer = None

    if mode == "smart_scgrl":

        balancer = AdaptiveLossBalancer(
            target_supcon_ratio=
                target_supcon_ratio,

            target_adv_ratio=
                target_adv_ratio,

            ema_beta=0.95,
        )

    # --------------------------------------------------------
    # TRAIN STATE
    # --------------------------------------------------------

    best_val = -np.inf
    best_epoch = 0
    best_state = None

    bad_epochs = 0

    history = []

    # ========================================================
    # TRAIN
    # ========================================================

    for epoch in range(
        1,
        max_epochs + 1,
    ):

        model.train()

        train_correct = 0
        train_total = 0

        total_sum = 0.0
        ce_sum = 0.0
        sc_sum = 0.0
        adv_sum = 0.0

        w_sc_sum = 0.0
        w_adv_sum = 0.0

        n_batches = 0

        # ----------------------------------------------------
        # GRL RAMP
        # ----------------------------------------------------

        progress = (
            (epoch - 1)
            /
            max(max_epochs - 1, 1)
        )

        base_ramp = (
            2.0
            /
            (
                1.0
                +
                math.exp(
                    -10.0 * progress
                )
            )
            - 1.0
        )

        if mode in {
            "grl",
            "scgrl",
        }:

            lambda_grl = (
                fixed_grl_max
                * base_ramp
            )

        elif mode == "smart_scgrl":

            lambda_grl = (
                smart_grl_max
                * base_ramp
            )

        else:

            lambda_grl = 0.0

        # ----------------------------------------------------
        # BATCHES
        # ----------------------------------------------------

        for (
            X,
            y,
            subject_id,
        ) in train_loader:

            X = X.to(device)
            y = y.to(device)

            subject_id = (
                subject_id.to(device)
            )

            optimizer.zero_grad()

            out = model(
                X,
                lambda_grl=lambda_grl,
            )

            # =================================================
            # MAIN CLASSIFICATION LOSS
            # =================================================

            loss_ce = ce_loss_fn(
                out["class_logits"],
                y,
            )

            # defaults
            loss_sc = torch.tensor(
                0.0,
                device=device,
            )

            loss_adv = torch.tensor(
                0.0,
                device=device,
            )

            w_sc = 0.0
            w_adv = 0.0

            # =================================================
            # SUPCON
            # =================================================

            if mode in {
                "supcon",
                "scgrl",
                "smart_scgrl",
            }:

                loss_sc = supcon_loss_fn(
                    out["projection"],
                    y,
                )

            # =================================================
            # SUBJECT ADVERSARIAL LOSS
            # =================================================

            if mode in {
                "grl",
                "scgrl",
                "smart_scgrl",
            }:

                loss_adv = (
                    subject_loss_fn(
                        out["subject_logits"],
                        subject_id,
                    )
                )

            # =================================================
            # LOSS COMBINATION
            # =================================================

            if mode == "transfer":

                loss = loss_ce

            elif mode == "supcon":

                w_sc = lambda_supcon

                loss = (
                    loss_ce
                    +
                    w_sc * loss_sc
                )

            elif mode == "grl":

                w_adv = lambda_adv

                loss = (
                    loss_ce
                    +
                    w_adv * loss_adv
                )

            elif mode == "scgrl":

                w_sc = lambda_supcon
                w_adv = lambda_adv

                loss = (
                    loss_ce
                    +
                    w_sc * loss_sc
                    +
                    w_adv * loss_adv
                )

            elif mode == "smart_scgrl":

                (
                    w_sc,
                    w_adv,
                ) = balancer.update(
                    loss_ce,
                    loss_sc,
                    loss_adv,
                )

                loss = (
                    loss_ce
                    +
                    w_sc * loss_sc
                    +
                    w_adv * loss_adv
                )

            # =================================================
            # BACKPROP
            # =================================================

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()

            # -------------------------------------------------
            # METRICS
            # -------------------------------------------------

            pred = (
                out["class_logits"]
                .argmax(dim=1)
            )

            train_correct += (
                pred == y
            ).sum().item()

            train_total += len(y)

            total_sum += loss.item()
            ce_sum += loss_ce.item()
            sc_sum += loss_sc.item()
            adv_sum += loss_adv.item()

            w_sc_sum += float(w_sc)
            w_adv_sum += float(w_adv)

            n_batches += 1

        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()

        val_true = []
        val_pred = []

        with torch.no_grad():

            for (
                X,
                y,
                subject_id,
            ) in val_loader:

                X = X.to(device)
                y = y.to(device)

                out = model(
                    X,
                    lambda_grl=0.0,
                )

                pred = (
                    out["class_logits"]
                    .argmax(dim=1)
                )

                val_true.extend(
                    y.cpu().numpy()
                )

                val_pred.extend(
                    pred.cpu().numpy()
                )

        val_acc = accuracy_score(
            val_true,
            val_pred,
        )

        train_acc = (
            train_correct
            /
            train_total
        )

        scheduler.step(
            val_acc
        )

        mean_ce = (
            ce_sum / n_batches
        )

        mean_sc = (
            sc_sum / n_batches
        )

        mean_adv = (
            adv_sum / n_batches
        )

        mean_w_sc = (
            w_sc_sum / n_batches
        )

        mean_w_adv = (
            w_adv_sum / n_batches
        )

        history.append({
            "epoch":
                epoch,

            "mode":
                mode,

            "train_accuracy":
                train_acc,

            "val_accuracy":
                val_acc,

            "total_loss":
                total_sum / n_batches,

            "ce_loss":
                mean_ce,

            "supcon_loss":
                mean_sc,

            "adv_loss":
                mean_adv,

            "w_supcon":
                mean_w_sc,

            "w_adv":
                mean_w_adv,

            "lambda_grl":
                lambda_grl,

            "lr":
                optimizer
                .param_groups[0]["lr"],
        })

        print(
            f"{mode.upper():12s} | "
            f"ep={epoch:02d} | "
            f"train={train_acc:.4f} | "
            f"val={val_acc:.4f} | "
            f"CE={mean_ce:.3f} | "
            f"SC={mean_sc:.3f} "
            f"(w={mean_w_sc:.4f}) | "
            f"ADV={mean_adv:.3f} "
            f"(w={mean_w_adv:.4f}) | "
            f"GRL={lambda_grl:.3f}"
        )

        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if val_acc > best_val:

            best_val = val_acc
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            bad_epochs = 0

        else:

            bad_epochs += 1

        if bad_epochs >= patience:

            print(
                f"Early stopping "
                f"at epoch {epoch}"
            )

            break

    # ========================================================
    # RESTORE BEST
    # ========================================================

    model.load_state_dict(
        best_state
    )

    # ========================================================
    # TEST
    # ========================================================

    model.eval()

    test_true = []
    test_pred = []

    with torch.no_grad():

        for X, y in test_loader:

            X = X.to(device)

            out = model(
                X,
                lambda_grl=0.0,
            )

            pred = (
                out["class_logits"]
                .argmax(dim=1)
                .cpu()
                .numpy()
            )

            test_true.extend(
                y.numpy()
            )

            test_pred.extend(
                pred
            )

    test_true = np.asarray(
        test_true
    )

    test_pred = np.asarray(
        test_pred
    )

    result = {
        "mode":
            mode,

        "test_subject":
            test_subject,

        "best_epoch":
            best_epoch,

        "best_val_accuracy":
            best_val,

        "test_accuracy":
            accuracy_score(
                test_true,
                test_pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                test_true,
                test_pred,
            ),

        "macro_f1":
            f1_score(
                test_true,
                test_pred,
                average="macro",
            ),

        "kappa":
            cohen_kappa_score(
                test_true,
                test_pred,
            ),
    }

    return (
        result,
        model,
        pd.DataFrame(history),
    )

In [66]:
# ============================================================
# 15-SUBJECT 64CH ABLATION SCREENING
# RESUME SAFE
# ============================================================

from pathlib import Path
import gc
import time


ABLATION64_ROOT = (
    RESULTS_ROOT
    / "physionet64_15subject_ablation"
)

ABLATION64_RESULTS = (
    ABLATION64_ROOT
    / "results"
)

ABLATION64_HISTORIES = (
    ABLATION64_ROOT
    / "histories"
)

ABLATION64_RESULTS.mkdir(
    parents=True,
    exist_ok=True,
)

ABLATION64_HISTORIES.mkdir(
    parents=True,
    exist_ok=True,
)


SCREENING_CSV = (
    ABLATION64_RESULTS
    / "screening_progress.csv"
)


ABLATION_MODES = [
    "transfer",
    "supcon",
    "grl",
    "scgrl",
    "smart_scgrl",
]


# ============================================================
# RESUME
# ============================================================

if SCREENING_CSV.exists():

    screening_df = pd.read_csv(
        SCREENING_CSV
    )

    done = set(
        zip(
            screening_df["mode"],
            screening_df[
                "test_subject"
            ].astype(int),
        )
    )

    print(
        f"Resume: "
        f"{len(screening_df)} runs already done"
    )

else:

    screening_df = pd.DataFrame()

    done = set()

    print(
        "Starting new 64ch screening"
    )


# ============================================================
# RUN
# ============================================================

total_runs = (
    len(ABLATION_MODES)
    *
    len(REPRESENTATIVE_SUBJECTS)
)

run_idx = 0


for mode in ABLATION_MODES:

    for subject in REPRESENTATIVE_SUBJECTS:

        run_idx += 1

        key = (
            mode,
            subject,
        )

        if key in done:

            print(
                f"[{run_idx}/{total_runs}] "
                f"{mode} | "
                f"S{subject:03d} "
                f"already done"
            )

            continue


        print()
        print("#" * 90)

        print(
            f"[{run_idx}/{total_runs}] "
            f"{mode.upper()} | "
            f"TEST SUBJECT={subject}"
        )

        print("#" * 90)


        start = time.time()


        result, model, history = (
            train_cho64_ablation_fold(
                test_subject=subject,

                mode=mode,

                seed=42,

                max_epochs=40,
                patience=8,

                batch_size=64,

                lr=1e-4,
                weight_decay=1e-4,

                # fixed 22ch-style weights
                lambda_supcon=0.15,
                lambda_adv=0.10,

                # smart ratios
                target_supcon_ratio=0.20,
                target_adv_ratio=0.10,

                fixed_grl_max=1.0,
                smart_grl_max=0.5,
            )
        )


        result[
            "runtime_sec"
        ] = (
            time.time()
            - start
        )


        # ----------------------------------------------------
        # SAVE HISTORY
        # ----------------------------------------------------

        history_path = (
            ABLATION64_HISTORIES
            /
            (
                f"{mode}_"
                f"subject_{subject:03d}.csv"
            )
        )

        history.to_csv(
            history_path,
            index=False,
        )

        result[
            "history_path"
        ] = str(history_path)


        # ----------------------------------------------------
        # SAVE RESULT
        # ----------------------------------------------------

        screening_df = pd.concat(
            [
                screening_df,
                pd.DataFrame(
                    [result]
                ),
            ],
            ignore_index=True,
        )

        screening_df.to_csv(
            SCREENING_CSV,
            index=False,
        )


        print()
        print(
            f"{mode} | "
            f"S{subject:03d} | "
            f"VAL="
            f"{result['best_val_accuracy']*100:.2f}% | "
            f"TEST="
            f"{result['test_accuracy']*100:.2f}%"
        )


        del model
        del history

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

Starting new 64ch screening

##########################################################################################
[1/75] TRANSFER | TEST SUBJECT=3
##########################################################################################
TRANSFER     | ep=01 | train=0.5824 | val=0.6475 | CE=0.744 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=02 | train=0.5845 | val=0.6516 | CE=0.713 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=03 | train=0.5884 | val=0.6701 | CE=0.698 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=04 | train=0.6071 | val=0.6393 | CE=0.683 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=05 | train=0.5986 | val=0.6475 | CE=0.679 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=06 | train=0.5989 | val=0.6516 | CE=0.667 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=07 | train=0.6011 | val=0.6701 | CE=

In [67]:
# ============================================================
# SCREENING SUMMARY
# ============================================================

screening_df = pd.read_csv(
    SCREENING_CSV
)

summary_15 = (
    screening_df
    .groupby("mode")
    .agg(
        n_subjects=(
            "test_subject",
            "nunique",
        ),

        mean_accuracy=(
            "test_accuracy",
            "mean",
        ),

        std_accuracy=(
            "test_accuracy",
            "std",
        ),

        median_accuracy=(
            "test_accuracy",
            "median",
        ),

        mean_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),

        mean_macro_f1=(
            "macro_f1",
            "mean",
        ),

        mean_kappa=(
            "kappa",
            "mean",
        ),

        mean_best_epoch=(
            "best_epoch",
            "mean",
        ),
    )
    .reset_index()
)


for col in [
    "mean_accuracy",
    "std_accuracy",
    "median_accuracy",
    "mean_balanced_accuracy",
    "mean_macro_f1",
]:

    summary_15[
        col + "_pct"
    ] = (
        summary_15[col]
        * 100
    )


summary_15 = (
    summary_15
    .sort_values(
        "mean_accuracy",
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    summary_15[[
        "mode",
        "n_subjects",
        "mean_accuracy_pct",
        "std_accuracy_pct",
        "median_accuracy_pct",
        "mean_balanced_accuracy_pct",
        "mean_macro_f1_pct",
        "mean_kappa",
        "mean_best_epoch",
    ]]
)

,mode,n_subjects,mean_accuracy_pct,std_accuracy_pct,median_accuracy_pct,mean_balanced_accuracy_pct,mean_macro_f1_pct,mean_kappa,mean_best_epoch
0,supcon,15,66.370370,15.688023,68.888889,66.420023,62.418124,0.329056,18.133333
1,smart_scgrl,15,64.296296,16.511803,60.000000,64.295988,58.894346,0.286711,12.733333
2,transfer,15,64.296296,16.533151,60.000000,64.289949,58.938412,0.286601,12.666667
3,scgrl,15,64.000000,15.962920,62.222222,63.959157,59.041066,0.279421,12.733333
4,grl,15,63.259259,15.844631,60.000000,63.117354,58.346382,0.263200,12.800000


In [68]:
# ============================================================
# TRANSFER LR SWEEP
# same EEGNet64, same Cho64 weights, full fine-tuning
# ============================================================

TRANSFER_LRS = [
    1e-3,
    5e-4,
    2e-4,
    1e-4,
]

TRANSFER_SEARCH_ROOT = (
    RESULTS_ROOT
    / "cho64_transfer_lr_search"
)

TRANSFER_SEARCH_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

TRANSFER_SEARCH_CSV = (
    TRANSFER_SEARCH_ROOT
    / "transfer_lr_search_progress.csv"
)


# ============================================================
# RESUME
# ============================================================

if TRANSFER_SEARCH_CSV.exists():

    transfer_search_df = pd.read_csv(
        TRANSFER_SEARCH_CSV
    )

    done = set(
        zip(
            transfer_search_df["lr"],
            transfer_search_df["test_subject"].astype(int),
        )
    )

    print(
        f"Resume: {len(transfer_search_df)} runs already done"
    )

else:

    transfer_search_df = pd.DataFrame()
    done = set()

    print("Starting transfer LR search")


# ============================================================
# RUN
# ============================================================

total_runs = (
    len(TRANSFER_LRS)
    *
    len(REPRESENTATIVE_SUBJECTS)
)

run_idx = 0


for lr in TRANSFER_LRS:

    for subject in REPRESENTATIVE_SUBJECTS:

        run_idx += 1

        key = (
            float(lr),
            int(subject),
        )

        if key in done:

            print(
                f"[{run_idx}/{total_runs}] "
                f"lr={lr:.1e} | "
                f"S{subject:03d} already done"
            )

            continue


        print()
        print("=" * 90)

        print(
            f"[{run_idx}/{total_runs}] "
            f"TRANSFER | "
            f"lr={lr:.1e} | "
            f"TEST SUBJECT={subject}"
        )

        print("=" * 90)


        result, model, history = (
            train_cho64_ablation_fold(
                test_subject=subject,

                mode="transfer",

                seed=42,

                max_epochs=40,
                patience=8,

                batch_size=64,

                lr=lr,
                weight_decay=1e-4,
            )
        )


        result["lr"] = float(lr)


        transfer_search_df = pd.concat(
            [
                transfer_search_df,
                pd.DataFrame([result]),
            ],
            ignore_index=True,
        )


        transfer_search_df.to_csv(
            TRANSFER_SEARCH_CSV,
            index=False,
        )


        print(
            f"\nRESULT | "
            f"lr={lr:.1e} | "
            f"S{subject:03d} | "
            f"VAL={result['best_val_accuracy']*100:.2f}% | "
            f"TEST={result['test_accuracy']*100:.2f}%"
        )


        del model
        del history

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

Starting transfer LR search

[1/60] TRANSFER | lr=1.0e-03 | TEST SUBJECT=3
TRANSFER     | ep=01 | train=0.5973 | val=0.6803 | CE=0.686 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=02 | train=0.6162 | val=0.6721 | CE=0.651 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=03 | train=0.6203 | val=0.6721 | CE=0.642 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=04 | train=0.6404 | val=0.6865 | CE=0.630 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=05 | train=0.6246 | val=0.6906 | CE=0.634 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=06 | train=0.6497 | val=0.6701 | CE=0.622 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=07 | train=0.6397 | val=0.6844 | CE=0.623 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=08 | train=0.6529 | val=0.6434 | CE=0.619 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) |

In [69]:
# ============================================================
# TRANSFER LR SEARCH SUMMARY
# ============================================================

transfer_search_df = pd.read_csv(
    TRANSFER_SEARCH_CSV
)


transfer_lr_summary = (
    transfer_search_df
    .groupby("lr")
    .agg(
        n_subjects=(
            "test_subject",
            "nunique",
        ),

        mean_accuracy=(
            "test_accuracy",
            "mean",
        ),

        std_accuracy=(
            "test_accuracy",
            "std",
        ),

        median_accuracy=(
            "test_accuracy",
            "median",
        ),

        mean_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),

        mean_macro_f1=(
            "macro_f1",
            "mean",
        ),

        mean_best_epoch=(
            "best_epoch",
            "mean",
        ),
    )
    .reset_index()
)


for col in [
    "mean_accuracy",
    "std_accuracy",
    "median_accuracy",
    "mean_balanced_accuracy",
    "mean_macro_f1",
]:

    transfer_lr_summary[
        col + "_pct"
    ] = (
        transfer_lr_summary[col]
        * 100
    )


transfer_lr_summary = (
    transfer_lr_summary
    .sort_values(
        "mean_accuracy",
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    transfer_lr_summary[[
        "lr",
        "n_subjects",
        "mean_accuracy_pct",
        "std_accuracy_pct",
        "median_accuracy_pct",
        "mean_balanced_accuracy_pct",
        "mean_macro_f1_pct",
        "mean_best_epoch",
    ]]
)

,lr,n_subjects,mean_accuracy_pct,std_accuracy_pct,median_accuracy_pct,mean_balanced_accuracy_pct,mean_macro_f1_pct,mean_best_epoch
0,0.0010,15,69.925926,14.689416,73.333333,70.224136,67.884823,20.733333
1,0.0005,15,67.111111,16.612611,66.666667,67.236103,65.475743,14.800000
2,0.0002,15,66.814815,16.618272,66.666667,66.629258,63.987285,15.666667
3,0.0001,15,64.296296,16.533151,60.000000,64.289949,58.938412,12.666667


In [70]:
transfer_lr_pivot = (
    transfer_search_df
    .pivot(
        index="test_subject",
        columns="lr",
        values="test_accuracy",
    )
    .mul(100)
    .reset_index()
)

display(transfer_lr_pivot)

lr,test_subject,0.0001,0.0002,0.0005,0.001
0,1,75.555556,75.555556,75.555556,77.777778
1,2,97.777778,97.777778,93.333333,91.111111
2,3,44.444444,42.222222,40.000000,37.777778
3,8,60.000000,73.333333,68.888889,84.444444
4,9,48.888889,46.666667,48.888889,55.555556
5,12,71.111111,66.666667,86.666667,82.222222
6,15,86.666667,82.222222,77.777778,82.222222
7,18,51.111111,48.888889,48.888889,57.777778
8,19,66.666667,71.111111,62.222222,73.333333
9,24,48.888889,46.666667,44.444444,48.888889


In [71]:
# ============================================================
# FINE LR SEARCH AROUND 1e-3
# ============================================================

TRANSFER_LRS_FINE = [
    7e-4,    # 0.0007
    1.5e-3,  # 0.0015
]

FINE_SEARCH_ROOT = (
    RESULTS_ROOT
    / "cho64_transfer_lr_fine_search"
)

FINE_SEARCH_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

FINE_SEARCH_CSV = (
    FINE_SEARCH_ROOT
    / "transfer_lr_fine_progress.csv"
)


# ============================================================
# RESUME
# ============================================================

if FINE_SEARCH_CSV.exists():

    fine_search_df = pd.read_csv(
        FINE_SEARCH_CSV
    )

    done = set(
        zip(
            fine_search_df["lr"],
            fine_search_df["test_subject"].astype(int),
        )
    )

    print(
        f"Resume: {len(fine_search_df)} runs already done"
    )

else:

    fine_search_df = pd.DataFrame()
    done = set()

    print("Starting fine LR search")


# ============================================================
# RUN
# ============================================================

total_runs = (
    len(TRANSFER_LRS_FINE)
    *
    len(REPRESENTATIVE_SUBJECTS)
)

run_idx = 0


for lr in TRANSFER_LRS_FINE:

    for subject in REPRESENTATIVE_SUBJECTS:

        run_idx += 1

        key = (
            float(lr),
            int(subject),
        )

        if key in done:

            print(
                f"[{run_idx}/{total_runs}] "
                f"lr={lr:.1e} | "
                f"S{subject:03d} already done"
            )

            continue


        print()
        print("=" * 90)

        print(
            f"[{run_idx}/{total_runs}] "
            f"TRANSFER | "
            f"lr={lr:.1e} | "
            f"SUBJECT={subject}"
        )

        print("=" * 90)


        result, model, history = (
            train_cho64_ablation_fold(
                test_subject=subject,

                mode="transfer",

                seed=42,

                max_epochs=40,
                patience=8,

                batch_size=64,

                lr=lr,
                weight_decay=1e-4,
            )
        )


        result["lr"] = float(lr)


        fine_search_df = pd.concat(
            [
                fine_search_df,
                pd.DataFrame([result]),
            ],
            ignore_index=True,
        )


        fine_search_df.to_csv(
            FINE_SEARCH_CSV,
            index=False,
        )


        print(
            f"\nRESULT | "
            f"lr={lr:.1e} | "
            f"S{subject:03d} | "
            f"VAL="
            f"{result['best_val_accuracy']*100:.2f}% | "
            f"TEST="
            f"{result['test_accuracy']*100:.2f}%"
        )


        del model
        del history

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

Starting fine LR search

[1/30] TRANSFER | lr=7.0e-04 | SUBJECT=3
TRANSFER     | ep=01 | train=0.5950 | val=0.6496 | CE=0.695 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=02 | train=0.6064 | val=0.6721 | CE=0.657 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=03 | train=0.6146 | val=0.6967 | CE=0.647 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=04 | train=0.6340 | val=0.6824 | CE=0.634 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=05 | train=0.6210 | val=0.6906 | CE=0.637 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=06 | train=0.6447 | val=0.6783 | CE=0.625 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=07 | train=0.6410 | val=0.6803 | CE=0.625 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.000
TRANSFER     | ep=08 | train=0.6534 | val=0.6578 | CE=0.621 | SC=0.000 (w=0.0000) | ADV=0.000 (w=0.0000) | GRL=0.00

In [72]:
# ============================================================
# COMBINE COARSE + FINE SEARCH
# ============================================================

transfer_search_df = pd.read_csv(
    TRANSFER_SEARCH_CSV
)

fine_search_df = pd.read_csv(
    FINE_SEARCH_CSV
)


all_transfer_lr_df = pd.concat(
    [
        transfer_search_df,
        fine_search_df,
    ],
    ignore_index=True,
)


transfer_lr_full_summary = (
    all_transfer_lr_df
    .groupby("lr")
    .agg(
        n_subjects=(
            "test_subject",
            "nunique",
        ),

        mean_accuracy=(
            "test_accuracy",
            "mean",
        ),

        std_accuracy=(
            "test_accuracy",
            "std",
        ),

        median_accuracy=(
            "test_accuracy",
            "median",
        ),

        mean_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),

        mean_macro_f1=(
            "macro_f1",
            "mean",
        ),

        mean_best_epoch=(
            "best_epoch",
            "mean",
        ),
    )
    .reset_index()
)


for col in [
    "mean_accuracy",
    "std_accuracy",
    "median_accuracy",
    "mean_balanced_accuracy",
    "mean_macro_f1",
]:

    transfer_lr_full_summary[
        col + "_pct"
    ] = (
        transfer_lr_full_summary[col]
        * 100
    )


transfer_lr_full_summary = (
    transfer_lr_full_summary
    .sort_values(
        "mean_accuracy",
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    transfer_lr_full_summary[[
        "lr",
        "n_subjects",
        "mean_accuracy_pct",
        "std_accuracy_pct",
        "median_accuracy_pct",
        "mean_balanced_accuracy_pct",
        "mean_macro_f1_pct",
        "mean_best_epoch",
    ]]
)

,lr,n_subjects,mean_accuracy_pct,std_accuracy_pct,median_accuracy_pct,mean_balanced_accuracy_pct,mean_macro_f1_pct,mean_best_epoch
0,0.0015,15,70.518519,16.100799,77.777778,70.805258,68.782536,19.666667
1,0.0010,15,69.925926,14.689416,73.333333,70.224136,67.884823,20.733333
2,0.0007,15,67.703704,15.933430,68.888889,67.919568,65.878001,20.733333
3,0.0005,15,67.111111,16.612611,66.666667,67.236103,65.475743,14.800000
4,0.0002,15,66.814815,16.618272,66.666667,66.629258,63.987285,15.666667
5,0.0001,15,64.296296,16.533151,60.000000,64.289949,58.938412,12.666667


In [74]:
# ============================================================
# AUTO SELECT BEST TRANSFER LR
# -> RUN TRANSFER + SUPCON ON 15 REPRESENTATIVE SUBJECTS
# ============================================================


# ------------------------------------------------------------
# 1. LOAD ALL TRANSFER LR SEARCH RESULTS
# ------------------------------------------------------------

dfs = []

if TRANSFER_SEARCH_CSV.exists():
    coarse_df = pd.read_csv(TRANSFER_SEARCH_CSV)
    dfs.append(coarse_df)

if FINE_SEARCH_CSV.exists():
    fine_df = pd.read_csv(FINE_SEARCH_CSV)
    dfs.append(fine_df)

if len(dfs) == 0:
    raise RuntimeError(
        "Не найдено результатов LR-search."
    )

all_lr_df = pd.concat(
    dfs,
    ignore_index=True,
)


# ------------------------------------------------------------
# 2. KEEP ONLY LR VALUES COMPLETED ON ALL 15 SUBJECTS
# ------------------------------------------------------------

n_required = len(REPRESENTATIVE_SUBJECTS)

lr_stats = (
    all_lr_df
    .groupby("lr")
    .agg(
        n_subjects=("test_subject", "nunique"),
        mean_accuracy=("test_accuracy", "mean"),
        median_accuracy=("test_accuracy", "median"),
        std_accuracy=("test_accuracy", "std"),
        mean_balanced_accuracy=("balanced_accuracy", "mean"),
        mean_macro_f1=("macro_f1", "mean"),
        mean_best_epoch=("best_epoch", "mean"),
    )
    .reset_index()
)

complete_lr_stats = (
    lr_stats[
        lr_stats["n_subjects"] == n_required
    ]
    .copy()
)

if len(complete_lr_stats) == 0:
    raise RuntimeError(
        f"Нет ни одного LR, рассчитанного на всех "
        f"{n_required} representative subjects."
    )


# ------------------------------------------------------------
# 3. SELECT BEST LR BY MEAN TEST ACCURACY
# ------------------------------------------------------------

complete_lr_stats = (
    complete_lr_stats
    .sort_values(
        "mean_accuracy",
        ascending=False,
    )
    .reset_index(drop=True)
)

BEST_TRANSFER_LR = float(
    complete_lr_stats.iloc[0]["lr"]
)


print("=" * 90)
print("COMPLETED TRANSFER LR SEARCH")
print("=" * 90)

display(
    complete_lr_stats.assign(
        mean_accuracy_pct=
            complete_lr_stats["mean_accuracy"] * 100,

        median_accuracy_pct=
            complete_lr_stats["median_accuracy"] * 100,

        std_accuracy_pct=
            complete_lr_stats["std_accuracy"] * 100,

        mean_balanced_accuracy_pct=
            complete_lr_stats["mean_balanced_accuracy"] * 100,

        mean_macro_f1_pct=
            complete_lr_stats["mean_macro_f1"] * 100,
    )[[
        "lr",
        "n_subjects",
        "mean_accuracy_pct",
        "median_accuracy_pct",
        "std_accuracy_pct",
        "mean_balanced_accuracy_pct",
        "mean_macro_f1_pct",
        "mean_best_epoch",
    ]]
)


print()
print("=" * 90)
print(
    f"SELECTED BEST TRANSFER LR = "
    f"{BEST_TRANSFER_LR:.1e}"
)
print("=" * 90)


# ------------------------------------------------------------
# 4. GET TRANSFER RESULTS FOR THE SELECTED LR
# ------------------------------------------------------------

best_transfer_df = (
    all_lr_df[
        np.isclose(
            all_lr_df["lr"].astype(float),
            BEST_TRANSFER_LR,
        )
    ]
    .copy()
)

print(
    f"\nTRANSFER @ {BEST_TRANSFER_LR:.1e}"
)

print(
    f"Mean test accuracy: "
    f"{best_transfer_df['test_accuracy'].mean()*100:.2f}%"
)

print(
    f"Median test accuracy: "
    f"{best_transfer_df['test_accuracy'].median()*100:.2f}%"
)


# ============================================================
# 5. TRANSFER + SUPCON SCREENING
# ============================================================

SUPCON_ROOT = (
    RESULTS_ROOT
    / "cho64_transfer_supcon_best_lr_screening"
)

SUPCON_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SUPCON_CSV = (
    SUPCON_ROOT
    / "transfer_supcon_progress.csv"
)


# ------------------------------------------------------------
# RESUME SUPPORT
# ------------------------------------------------------------

if SUPCON_CSV.exists():

    supcon_df = pd.read_csv(
        SUPCON_CSV
    )

    # Если файл был создан для другого LR —
    # не смешиваем результаты.
    if "lr" in supcon_df.columns:

        supcon_df = supcon_df[
            np.isclose(
                supcon_df["lr"].astype(float),
                BEST_TRANSFER_LR,
            )
        ].copy()

    done_subjects = set(
        supcon_df["test_subject"]
        .astype(int)
        .tolist()
    )

    print(
        f"\nResume SupCon: "
        f"{len(done_subjects)}/{n_required} subjects done"
    )

else:

    supcon_df = pd.DataFrame()

    done_subjects = set()

    print(
        "\nStarting Transfer + SupCon screening"
    )


# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

for idx, subject in enumerate(
    REPRESENTATIVE_SUBJECTS,
    start=1,
):

    subject = int(subject)

    if subject in done_subjects:

        print(
            f"[{idx}/{n_required}] "
            f"S{subject:03d} already done"
        )

        continue


    print()
    print("=" * 90)

    print(
        f"[{idx}/{n_required}] "
        f"TRANSFER + SUPCON | "
        f"LR={BEST_TRANSFER_LR:.1e} | "
        f"TEST SUBJECT={subject}"
    )

    print("=" * 90)


    result, model, history = (
        train_cho64_ablation_fold(
            test_subject=subject,

            mode="supcon",

            seed=42,

            max_epochs=40,
            patience=8,

            batch_size=64,

            lr=BEST_TRANSFER_LR,
            weight_decay=1e-4,
        )
    )


    result["lr"] = BEST_TRANSFER_LR
    result["experiment"] = "transfer_supcon"


    supcon_df = pd.concat(
        [
            supcon_df,
            pd.DataFrame([result]),
        ],
        ignore_index=True,
    )


    supcon_df.to_csv(
        SUPCON_CSV,
        index=False,
    )


    print(
        f"\nRESULT | "
        f"S{subject:03d} | "
        f"VAL="
        f"{result['best_val_accuracy']*100:.2f}% | "
        f"TEST="
        f"{result['test_accuracy']*100:.2f}%"
    )


    del model
    del history

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 6. FINAL MORNING SUMMARY
# ============================================================

supcon_df = pd.read_csv(
    SUPCON_CSV
)

supcon_df = supcon_df[
    np.isclose(
        supcon_df["lr"].astype(float),
        BEST_TRANSFER_LR,
    )
].copy()


print()
print("=" * 90)
print("TRANSFER + SUPCON SCREENING FINISHED")
print("=" * 90)


transfer_mean = (
    best_transfer_df["test_accuracy"].mean()
)

supcon_mean = (
    supcon_df["test_accuracy"].mean()
)

transfer_median = (
    best_transfer_df["test_accuracy"].median()
)

supcon_median = (
    supcon_df["test_accuracy"].median()
)


summary = pd.DataFrame({
    "method": [
        "Transfer",
        "Transfer + SupCon",
    ],

    "lr": [
        BEST_TRANSFER_LR,
        BEST_TRANSFER_LR,
    ],

    "n_subjects": [
        best_transfer_df["test_subject"].nunique(),
        supcon_df["test_subject"].nunique(),
    ],

    "mean_accuracy_pct": [
        transfer_mean * 100,
        supcon_mean * 100,
    ],

    "median_accuracy_pct": [
        transfer_median * 100,
        supcon_median * 100,
    ],

    "std_accuracy_pct": [
        best_transfer_df["test_accuracy"].std() * 100,
        supcon_df["test_accuracy"].std() * 100,
    ],

    "balanced_accuracy_pct": [
        best_transfer_df["balanced_accuracy"].mean() * 100,
        supcon_df["balanced_accuracy"].mean() * 100,
    ],

    "macro_f1_pct": [
        best_transfer_df["macro_f1"].mean() * 100,
        supcon_df["macro_f1"].mean() * 100,
    ],
})


display(summary)


# ------------------------------------------------------------
# 7. PAIRED SUBJECT COMPARISON
# ------------------------------------------------------------

paired = (
    best_transfer_df[
        [
            "test_subject",
            "test_accuracy",
        ]
    ]
    .rename(
        columns={
            "test_accuracy":
                "transfer_accuracy"
        }
    )
    .merge(
        supcon_df[
            [
                "test_subject",
                "test_accuracy",
            ]
        ].rename(
            columns={
                "test_accuracy":
                    "supcon_accuracy"
            }
        ),

        on="test_subject",

        how="inner",
    )
)


paired["delta_pp"] = (
    paired["supcon_accuracy"]
    -
    paired["transfer_accuracy"]
) * 100


paired["transfer_accuracy"] *= 100
paired["supcon_accuracy"] *= 100


print()
print("PAIRED COMPARISON")

display(
    paired.sort_values(
        "delta_pp",
        ascending=False,
    )
)


print(
    "\nMean SupCon improvement: "
    f"{paired['delta_pp'].mean():+.2f} pp"
)

print(
    "Median SupCon improvement: "
    f"{paired['delta_pp'].median():+.2f} pp"
)

print(
    "SupCon win rate: "
    f"{(paired['delta_pp'] > 0).mean()*100:.1f}%"
)


print()
print("=" * 90)

if supcon_mean > transfer_mean:

    print(
        "RESULT: Transfer + SupCon "
        f"IMPROVED mean accuracy by "
        f"{(supcon_mean-transfer_mean)*100:+.2f} pp"
    )

else:

    print(
        "RESULT: Transfer + SupCon "
        f"DID NOT improve mean accuracy: "
        f"{(supcon_mean-transfer_mean)*100:+.2f} pp"
    )

print("=" * 90)

COMPLETED TRANSFER LR SEARCH


,lr,n_subjects,mean_accuracy_pct,median_accuracy_pct,std_accuracy_pct,mean_balanced_accuracy_pct,mean_macro_f1_pct,mean_best_epoch
0,0.0015,15,70.518519,77.777778,16.100799,70.805258,68.782536,19.666667
1,0.0010,15,69.925926,73.333333,14.689416,70.224136,67.884823,20.733333
2,0.0007,15,67.703704,68.888889,15.933430,67.919568,65.878001,20.733333
3,0.0005,15,67.111111,66.666667,16.612611,67.236103,65.475743,14.800000
4,0.0002,15,66.814815,66.666667,16.618272,66.629258,63.987285,15.666667
5,0.0001,15,64.296296,60.000000,16.533151,64.289949,58.938412,12.666667



SELECTED BEST TRANSFER LR = 1.5e-03

TRANSFER @ 1.5e-03
Mean test accuracy: 70.52%
Median test accuracy: 77.78%

Starting Transfer + SupCon screening

[1/15] TRANSFER + SUPCON | LR=1.5e-03 | TEST SUBJECT=3
SUPCON       | ep=01 | train=0.5961 | val=0.6701 | CE=0.678 | SC=4.163 (w=0.1500) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=02 | train=0.6160 | val=0.6967 | CE=0.647 | SC=4.133 (w=0.1500) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=03 | train=0.6242 | val=0.6906 | CE=0.638 | SC=4.131 (w=0.1500) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=04 | train=0.6406 | val=0.6742 | CE=0.628 | SC=4.128 (w=0.1500) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=05 | train=0.6324 | val=0.6824 | CE=0.632 | SC=4.129 (w=0.1500) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=06 | train=0.6490 | val=0.6516 | CE=0.620 | SC=4.127 (w=0.1500) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=07 | train=0.6470 | val=0.7008 | CE=0.621 | SC=4.127 (w=0.1500) | ADV=0.0

,method,lr,n_subjects,mean_accuracy_pct,median_accuracy_pct,std_accuracy_pct,balanced_accuracy_pct,macro_f1_pct
0,Transfer,0.0015,15,70.518519,77.777778,16.100799,70.805258,68.782536
1,Transfer + SupCon,0.0015,15,69.037037,71.111111,15.337857,69.141179,67.325853



PAIRED COMPARISON


,test_subject,transfer_accuracy,supcon_accuracy,delta_pp
0,3,44.444444,53.333333,8.888889
3,24,42.222222,46.666667,4.444444
1,9,53.333333,55.555556,2.222222
7,19,77.777778,80.000000,2.222222
5,1,71.111111,71.111111,0.000000
6,8,82.222222,82.222222,0.000000
10,2,91.111111,91.111111,0.000000
11,12,82.222222,82.222222,0.000000
12,15,88.888889,86.666667,-2.222222
13,25,80.000000,77.777778,-2.222222



Mean SupCon improvement: -1.48 pp
Median SupCon improvement: +0.00 pp
SupCon win rate: 26.7%

RESULT: Transfer + SupCon DID NOT improve mean accuracy: -1.48 pp


In [73]:
transfer_lr_full_pivot = (
    all_transfer_lr_df
    .pivot(
        index="test_subject",
        columns="lr",
        values="test_accuracy",
    )
    .mul(100)
    .reset_index()
)

display(
    transfer_lr_full_pivot
)

lr,test_subject,0.0001,0.0002,0.0005,0.0007,0.001,0.0015
0,1,75.555556,75.555556,75.555556,71.111111,77.777778,71.111111
1,2,97.777778,97.777778,93.333333,93.333333,91.111111,91.111111
2,3,44.444444,42.222222,40.000000,40.000000,37.777778,44.444444
3,8,60.000000,73.333333,68.888889,80.000000,84.444444,82.222222
4,9,48.888889,46.666667,48.888889,48.888889,55.555556,53.333333
5,12,71.111111,66.666667,86.666667,82.222222,82.222222,82.222222
6,15,86.666667,82.222222,77.777778,80.000000,82.222222,88.888889
7,18,51.111111,48.888889,48.888889,57.777778,57.777778,48.888889
8,19,66.666667,71.111111,62.222222,68.888889,73.333333,77.777778
9,24,48.888889,46.666667,44.444444,46.666667,48.888889,42.222222


## Подбор гиперпараметров SupCon для 64-канального Transfer

На предыдущем этапе был подобран learning rate для чистого Cho64 → PhysioNet64 Transfer.

Лучший результат на screening-наборе из 15 субъектов:

- EEGNet64 без изменения архитектуры;
- веса предварительно обучены на Cho2017;
- full fine-tuning на PhysioNet;
- learning rate = `1.5e-3`;
- mean accuracy ≈ `70.52%`.

При добавлении SupCon с исходными параметрами 22-канальной версии (`weight=0.15`) средняя accuracy снизилась примерно до `69.04%`.

Поэтому на этом этапе сам Transfer больше не изменяется.

Фиксируем:

- `Transfer LR = 1.5e-3`;
- EEGNet64;
- Cho64 pretraining;
- batch size = 64;
- AdamW;
- weight decay = `1e-4`;
- seed = 42.

Подбираем только два параметра SupCon:

1. `supcon_weight` — вклад contrastive loss в общий loss;
2. `temperature` — жёсткость разделения embeddings в SupCon.

Сначала выполняется быстрый screening на 5 субъектах:

- 3 и 9 — сложные;
- 1 и 19 — средние;
- 2 — лёгкий.

Проверяем:

- SupCon weight: `0.02`, `0.05`, `0.10`, `0.15`;
- temperature: `0.05`, `0.10`, `0.20`.

Всего: 12 конфигураций × 5 субъектов = 60 запусков.

После screening выбираются лучшие конфигурации для дальнейшей проверки на 15 субъектах.

In [77]:
# ============================================================
# TRAIN ONE PHYSIONET64 FOLD
# Cho64 Transfer / SupCon / GRL / SC+GRL / Smart SC+GRL
#
# NEW:
#   supcon_weight
#   supcon_temperature
# ============================================================




def train_cho64_ablation_fold(
    test_subject,
    mode,
    seed=42,
    max_epochs=40,
    patience=8,
    batch_size=64,
    lr=1e-4,
    weight_decay=1e-4,

    # --------------------------------------------------------
    # SupCon hyperparameters
    # --------------------------------------------------------
    supcon_weight=0.15,
    supcon_temperature=0.10,

    # --------------------------------------------------------
    # GRL hyperparameters
    # --------------------------------------------------------
    adv_weight=0.10,
    fixed_grl_max=1.0,
    smart_grl_max=0.5,
):

    # ========================================================
    # CHECK MODE
    # ========================================================

    valid_modes = {
        "transfer",
        "supcon",
        "grl",
        "scgrl",
        "smart_scgrl",
    }

    if mode not in valid_modes:
        raise ValueError(
            f"Unknown mode={mode}. "
            f"Expected one of {sorted(valid_modes)}"
        )


    # ========================================================
    # REPRODUCIBILITY
    # ========================================================

    set_seed(seed)


    # ========================================================
    # DEVICE
    # ========================================================

    run_device = globals().get(
        "device",
        torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "mps"
            if torch.backends.mps.is_available()
            else "cpu"
        ),
    )


    # ========================================================
    # PREPARE STRICT LOSO FOLD
    #
    # test subject is never in train/val
    # remaining source trials -> 90/10 stratified train/val
    # ========================================================

    fold = prepare_phys64_fold_supcon_grl(
        test_subject=test_subject,
        seed=seed,
    )


    # ========================================================
    # DATASETS
    # ========================================================

    train_dataset = Phys64SubjectDataset(
        fold["X_train"],
        fold["y_train"],
        fold["s_train"],
        augment=True,
    )

    val_dataset = Phys64SubjectDataset(
        fold["X_val"],
        fold["y_val"],
        fold["s_val"],
        augment=False,
    )

    # Test subject has no source-domain ID used by GRL.
    # Dummy ID is fine because subject logits are not used
    # during test evaluation.
    s_test_dummy = np.zeros(
        len(fold["y_test"]),
        dtype=np.int64,
    )

    test_dataset = Phys64SubjectDataset(
        fold["X_test"],
        fold["y_test"],
        s_test_dummy,
        augment=False,
    )


    # ========================================================
    # LOADERS
    # ========================================================

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        drop_last=False,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        drop_last=False,
    )


    # ========================================================
    # MODEL
    #
    # Same EEGNet64 + Cho64 checkpoint
    # ========================================================

    model = make_cho64_supcon_grl_model(
        n_subjects=fold["n_source_subjects"]
    )

    model = model.to(run_device)


    # ========================================================
    # LOSSES
    # ========================================================

    ce_loss_fn = nn.CrossEntropyLoss()

    subject_loss_fn = nn.CrossEntropyLoss()

    supcon_loss_fn = SupConLoss(
        temperature=supcon_temperature
    )


    # ========================================================
    # OPTIMIZER
    # ========================================================

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )


    # ========================================================
    # SCHEDULER
    # ========================================================

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
    )


    # ========================================================
    # SMART LOSS STATE
    #
    # Used only for smart_scgrl.
    # Targets:
    #   SupCon contribution ~20% of CE
    #   ADV contribution    ~10% of CE
    # ========================================================

    ema_beta = 0.95

    ema_ce = None
    ema_sc = None
    ema_adv = None

    target_sc_ratio = 0.20
    target_adv_ratio = 0.10

    min_sc_weight = 0.005
    max_sc_weight = 0.080

    min_adv_weight = 0.003
    max_adv_weight = 0.050


    # ========================================================
    # TRAINING STATE
    # ========================================================

    best_val_accuracy = -np.inf
    best_epoch = 0
    best_state = None

    epochs_without_improvement = 0

    history = []


    # ========================================================
    # TRAIN
    # ========================================================

    for epoch in range(
        1,
        max_epochs + 1,
    ):

        # ----------------------------------------------------
        # GRL sigmoid ramp
        # ----------------------------------------------------

        progress = (
            (epoch - 1)
            /
            max(max_epochs - 1, 1)
        )

        base_ramp = (
            2.0
            /
            (
                1.0
                +
                math.exp(
                    -10.0 * progress
                )
            )
            -
            1.0
        )


        if mode in {
            "grl",
            "scgrl",
        }:

            lambda_grl = (
                fixed_grl_max
                *
                base_ramp
            )

        elif mode == "smart_scgrl":

            lambda_grl = (
                smart_grl_max
                *
                base_ramp
            )

        else:

            lambda_grl = 0.0


        # ----------------------------------------------------
        # TRAIN MODE
        # ----------------------------------------------------

        model.train()


        train_total_loss = 0.0
        train_ce_loss = 0.0
        train_sc_loss = 0.0
        train_adv_loss = 0.0

        train_correct = 0
        train_total = 0

        epoch_w_sc = []
        epoch_w_adv = []


        for X, y, subject_ids in train_loader:

            X = X.to(
                run_device,
                non_blocking=True,
            )

            y = y.to(
                run_device,
                non_blocking=True,
            )

            subject_ids = subject_ids.to(
                run_device,
                non_blocking=True,
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            # ------------------------------------------------
            # FORWARD
            # ------------------------------------------------

            out = model(
                X,
                lambda_grl=lambda_grl,
            )


            class_logits = out[
                "class_logits"
            ]

            projection = out[
                "projection"
            ]

            subject_logits = out[
                "subject_logits"
            ]


            # ------------------------------------------------
            # MAIN CE LOSS
            # ------------------------------------------------

            loss_ce = ce_loss_fn(
                class_logits,
                y,
            )


            # ------------------------------------------------
            # SUPCON LOSS
            # ------------------------------------------------

            if mode in {
                "supcon",
                "scgrl",
                "smart_scgrl",
            }:

                loss_sc = supcon_loss_fn(
                    projection,
                    y,
                )

            else:

                loss_sc = torch.zeros(
                    (),
                    device=run_device,
                )


            # ------------------------------------------------
            # SUBJECT ADVERSARIAL LOSS
            # ------------------------------------------------

            if mode in {
                "grl",
                "scgrl",
                "smart_scgrl",
            }:

                loss_adv = subject_loss_fn(
                    subject_logits,
                    subject_ids,
                )

            else:

                loss_adv = torch.zeros(
                    (),
                    device=run_device,
                )


            # ------------------------------------------------
            # LOSS WEIGHTS
            # ------------------------------------------------

            if mode == "transfer":

                w_sc = 0.0
                w_adv = 0.0


            elif mode == "supcon":

                w_sc = float(
                    supcon_weight
                )

                w_adv = 0.0


            elif mode == "grl":

                w_sc = 0.0

                w_adv = float(
                    adv_weight
                )


            elif mode == "scgrl":

                w_sc = float(
                    supcon_weight
                )

                w_adv = float(
                    adv_weight
                )


            elif mode == "smart_scgrl":

                ce_value = float(
                    loss_ce
                    .detach()
                    .cpu()
                    .item()
                )

                sc_value = float(
                    loss_sc
                    .detach()
                    .cpu()
                    .item()
                )

                adv_value = float(
                    loss_adv
                    .detach()
                    .cpu()
                    .item()
                )


                if ema_ce is None:

                    ema_ce = ce_value
                    ema_sc = sc_value
                    ema_adv = adv_value

                else:

                    ema_ce = (
                        ema_beta * ema_ce
                        +
                        (1.0 - ema_beta)
                        * ce_value
                    )

                    ema_sc = (
                        ema_beta * ema_sc
                        +
                        (1.0 - ema_beta)
                        * sc_value
                    )

                    ema_adv = (
                        ema_beta * ema_adv
                        +
                        (1.0 - ema_beta)
                        * adv_value
                    )


                w_sc = (
                    target_sc_ratio
                    *
                    ema_ce
                    /
                    max(
                        ema_sc,
                        1e-8,
                    )
                )

                w_adv = (
                    target_adv_ratio
                    *
                    ema_ce
                    /
                    max(
                        ema_adv,
                        1e-8,
                    )
                )


                w_sc = float(
                    np.clip(
                        w_sc,
                        min_sc_weight,
                        max_sc_weight,
                    )
                )

                w_adv = float(
                    np.clip(
                        w_adv,
                        min_adv_weight,
                        max_adv_weight,
                    )
                )


            # ------------------------------------------------
            # TOTAL LOSS
            # ------------------------------------------------

            loss = (
                loss_ce
                +
                w_sc * loss_sc
                +
                w_adv * loss_adv
            )


            # ------------------------------------------------
            # BACKWARD
            # ------------------------------------------------

            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )


            optimizer.step()


            # ------------------------------------------------
            # METRICS
            # ------------------------------------------------

            batch_size_real = y.size(0)

            train_total_loss += (
                loss.item()
                *
                batch_size_real
            )

            train_ce_loss += (
                loss_ce.item()
                *
                batch_size_real
            )

            train_sc_loss += (
                loss_sc.item()
                *
                batch_size_real
            )

            train_adv_loss += (
                loss_adv.item()
                *
                batch_size_real
            )


            preds = class_logits.argmax(
                dim=1
            )

            train_correct += (
                preds == y
            ).sum().item()

            train_total += (
                batch_size_real
            )


            epoch_w_sc.append(
                w_sc
            )

            epoch_w_adv.append(
                w_adv
            )


        # ====================================================
        # TRAIN METRICS
        # ====================================================

        train_accuracy = (
            train_correct
            /
            max(train_total, 1)
        )

        train_total_loss /= max(
            train_total,
            1,
        )

        train_ce_loss /= max(
            train_total,
            1,
        )

        train_sc_loss /= max(
            train_total,
            1,
        )

        train_adv_loss /= max(
            train_total,
            1,
        )


        mean_w_sc = float(
            np.mean(epoch_w_sc)
        )

        mean_w_adv = float(
            np.mean(epoch_w_adv)
        )


        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()

        val_true = []
        val_pred = []


        with torch.no_grad():

            for X, y, _ in val_loader:

                X = X.to(
                    run_device,
                    non_blocking=True,
                )

                y = y.to(
                    run_device,
                    non_blocking=True,
                )


                out = model(
                    X,
                    lambda_grl=0.0,
                )


                logits = out[
                    "class_logits"
                ]


                preds = logits.argmax(
                    dim=1
                )


                val_true.extend(
                    y.cpu().numpy()
                )

                val_pred.extend(
                    preds.cpu().numpy()
                )


        val_accuracy = accuracy_score(
            val_true,
            val_pred,
        )


        # ====================================================
        # SCHEDULER
        # ====================================================

        scheduler.step(
            val_accuracy
        )


        current_lr = optimizer.param_groups[
            0
        ]["lr"]


        # ====================================================
        # HISTORY
        # ====================================================

        history.append({
            "epoch": epoch,
            "mode": mode,

            "train_accuracy":
                train_accuracy,

            "val_accuracy":
                val_accuracy,

            "total_loss":
                train_total_loss,

            "ce_loss":
                train_ce_loss,

            "supcon_loss":
                train_sc_loss,

            "adv_loss":
                train_adv_loss,

            "w_supcon":
                mean_w_sc,

            "w_adv":
                mean_w_adv,

            "lambda_grl":
                lambda_grl,

            "lr":
                current_lr,
        })


        # ====================================================
        # LOG
        # ====================================================

        print(
            f"{mode.upper():12s} | "
            f"ep={epoch:02d} | "
            f"train={train_accuracy:.4f} | "
            f"val={val_accuracy:.4f} | "
            f"CE={train_ce_loss:.3f} | "
            f"SC={train_sc_loss:.3f} "
            f"(w={mean_w_sc:.4f}) | "
            f"ADV={train_adv_loss:.3f} "
            f"(w={mean_w_adv:.4f}) | "
            f"GRL={lambda_grl:.3f}"
        )


        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if val_accuracy > best_val_accuracy:

            best_val_accuracy = (
                val_accuracy
            )

            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0


        else:

            epochs_without_improvement += 1


        if (
            epochs_without_improvement
            >= patience
        ):

            print(
                f"Early stopping "
                f"at epoch {epoch}"
            )

            break


    # ========================================================
    # RESTORE BEST MODEL
    # ========================================================

    if best_state is None:

        raise RuntimeError(
            "Training finished without "
            "a valid checkpoint."
        )


    model.load_state_dict(
        best_state
    )

    model.eval()


    # ========================================================
    # TEST
    # ========================================================

    test_true = []
    test_pred = []


    with torch.no_grad():

        for X, y, _ in test_loader:

            X = X.to(
                run_device,
                non_blocking=True,
            )

            y = y.to(
                run_device,
                non_blocking=True,
            )


            out = model(
                X,
                lambda_grl=0.0,
            )


            logits = out[
                "class_logits"
            ]


            preds = logits.argmax(
                dim=1
            )


            test_true.extend(
                y.cpu().numpy()
            )

            test_pred.extend(
                preds.cpu().numpy()
            )


    # ========================================================
    # TEST METRICS
    # ========================================================

    test_accuracy = accuracy_score(
        test_true,
        test_pred,
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            test_true,
            test_pred,
        )
    )

    macro_f1 = f1_score(
        test_true,
        test_pred,
        average="macro",
        zero_division=0,
    )

    kappa = cohen_kappa_score(
        test_true,
        test_pred,
    )


    # ========================================================
    # RESULT
    # ========================================================

    result = {

        "test_subject":
            int(test_subject),

        "mode":
            mode,

        "best_epoch":
            int(best_epoch),

        "best_val_accuracy":
            float(best_val_accuracy),

        "test_accuracy":
            float(test_accuracy),

        "balanced_accuracy":
            float(balanced_accuracy),

        "macro_f1":
            float(macro_f1),

        "kappa":
            float(kappa),

        "supcon_weight":
            float(supcon_weight),

        "supcon_temperature":
            float(supcon_temperature),

        "initial_lr":
            float(lr),

        "seed":
            int(seed),
    }


    return (
        result,
        model,
        pd.DataFrame(history),
    )

## Подбор гиперпараметров SupCon для 64-канального Transfer

На предыдущем этапе был подобран learning rate для чистого Cho64 → PhysioNet64 Transfer.

Лучший результат на screening-наборе из 15 субъектов:

- EEGNet64 без изменения архитектуры;
- веса предварительно обучены на Cho2017;
- full fine-tuning на PhysioNet;
- learning rate = `1.5e-3`;
- mean accuracy ≈ `70.52%`.

При добавлении SupCon с исходными параметрами 22-канальной версии (`weight=0.15`) средняя accuracy снизилась примерно до `69.04%`.

Поэтому на этом этапе сам Transfer больше не изменяется.

Фиксируем:

- `Transfer LR = 1.5e-3`;
- EEGNet64;
- Cho64 pretraining;
- batch size = 64;
- AdamW;
- weight decay = `1e-4`;
- seed = 42.

Подбираем только два параметра SupCon:

1. `supcon_weight` — вклад contrastive loss в общий loss;
2. `temperature` — жёсткость разделения embeddings в SupCon.

Сначала выполняется быстрый screening на 5 субъектах:

- 3 и 9 — сложные;
- 1 и 19 — средние;
- 2 — лёгкий.

Проверяем:

- SupCon weight: `0.02`, `0.05`, `0.10`, `0.15`;
- temperature: `0.05`, `0.10`, `0.20`.

Всего: 12 конфигураций × 5 субъектов = 60 запусков.

После screening выбираются лучшие конфигурации для дальнейшей проверки на 15 субъектах.

In [79]:
import itertools

In [80]:
# ============================================================
# FIXED TRANSFER SETTINGS
# ============================================================

BEST_TRANSFER_LR = 1.5e-3

SUPCON_WEIGHTS = [
    0.02,
    0.05,
    0.10,
    0.15,
]

SUPCON_TEMPERATURES = [
    0.05,
    0.10,
    0.20,
]


# hard / hard / medium / medium / easy
SUPCON_TUNE_SUBJECTS = [
    3,
    9,
    1,
    19,
    2,
]


# ============================================================
# OUTPUT
# ============================================================

SUPCON_TUNE_ROOT = (
    RESULTS_ROOT
    / "supcon_hparam_search"
)

SUPCON_TUNE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SUPCON_TUNE_CSV = (
    SUPCON_TUNE_ROOT
    / "supcon_hparam_progress.csv"
)


# ============================================================
# RESUME
# ============================================================

if SUPCON_TUNE_CSV.exists():

    supcon_tune_df = pd.read_csv(
        SUPCON_TUNE_CSV
    )


    done = set(
        zip(
            supcon_tune_df[
                "supcon_weight"
            ].astype(float),

            supcon_tune_df[
                "supcon_temperature"
            ].astype(float),

            supcon_tune_df[
                "test_subject"
            ].astype(int),
        )
    )


    print(
        f"Resume: "
        f"{len(supcon_tune_df)} runs "
        f"already done"
    )


else:

    supcon_tune_df = (
        pd.DataFrame()
    )

    done = set()

    print(
        "Starting SupCon "
        "hyperparameter screening"
    )


# ============================================================
# CONFIGURATIONS
# ============================================================

configs = list(
    itertools.product(
        SUPCON_WEIGHTS,
        SUPCON_TEMPERATURES,
    )
)


total_runs = (
    len(configs)
    *
    len(SUPCON_TUNE_SUBJECTS)
)


print(
    f"\nConfigs: {len(configs)}"
)

print(
    f"Subjects: "
    f"{len(SUPCON_TUNE_SUBJECTS)}"
)

print(
    f"Total runs: {total_runs}"
)


# ============================================================
# RUN
# ============================================================

run_idx = 0


for (
    supcon_weight,
    supcon_temperature,
) in configs:


    for subject in SUPCON_TUNE_SUBJECTS:

        run_idx += 1


        key = (
            float(supcon_weight),
            float(supcon_temperature),
            int(subject),
        )


        if key in done:

            print(
                f"[{run_idx}/{total_runs}] "
                f"λ={supcon_weight:.3f} | "
                f"τ={supcon_temperature:.2f} | "
                f"S{subject:03d} "
                f"already done"
            )

            continue


        print()
        print("=" * 100)

        print(
            f"[{run_idx}/{total_runs}] "
            f"TRANSFER + SUPCON"
        )

        print(
            f"LR={BEST_TRANSFER_LR:.1e} | "
            f"λ={supcon_weight:.3f} | "
            f"τ={supcon_temperature:.2f} | "
            f"TEST=S{subject:03d}"
        )

        print("=" * 100)


        result, model, history = (
            train_cho64_ablation_fold(

                test_subject=subject,

                mode="supcon",

                seed=42,

                max_epochs=40,

                patience=8,

                batch_size=64,

                lr=BEST_TRANSFER_LR,

                weight_decay=1e-4,

                supcon_weight=
                    supcon_weight,

                supcon_temperature=
                    supcon_temperature,
            )
        )


        # ----------------------------------------------------
        # SAVE CONFIG
        # ----------------------------------------------------

        result["lr"] = (
            BEST_TRANSFER_LR
        )

        result[
            "supcon_weight"
        ] = float(
            supcon_weight
        )

        result[
            "supcon_temperature"
        ] = float(
            supcon_temperature
        )


        # ----------------------------------------------------
        # APPEND
        # ----------------------------------------------------

        supcon_tune_df = pd.concat(
            [
                supcon_tune_df,
                pd.DataFrame(
                    [result]
                ),
            ],
            ignore_index=True,
        )


        # ----------------------------------------------------
        # SAVE AFTER EVERY SUBJECT
        # ----------------------------------------------------

        supcon_tune_df.to_csv(
            SUPCON_TUNE_CSV,
            index=False,
        )


        done.add(key)


        print()

        print(
            f"RESULT | "
            f"λ={supcon_weight:.3f} | "
            f"τ={supcon_temperature:.2f} | "
            f"S{subject:03d} | "
            f"VAL="
            f"{result['best_val_accuracy']*100:.2f}% | "
            f"TEST="
            f"{result['test_accuracy']*100:.2f}%"
        )


        # ----------------------------------------------------
        # CLEANUP
        # ----------------------------------------------------

        del model
        del history

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


print()
print("=" * 100)
print("SUPCON SCREENING FINISHED")
print("=" * 100)

Starting SupCon hyperparameter screening

Configs: 12
Subjects: 5
Total runs: 60

[1/60] TRANSFER + SUPCON
LR=1.5e-03 | λ=0.020 | τ=0.05 | TEST=S003
SUPCON       | ep=01 | train=0.5954 | val=0.6742 | CE=0.678 | SC=4.192 (w=0.0200) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=02 | train=0.6155 | val=0.6988 | CE=0.647 | SC=4.139 (w=0.0200) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=03 | train=0.6253 | val=0.6926 | CE=0.638 | SC=4.136 (w=0.0200) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=04 | train=0.6392 | val=0.6824 | CE=0.628 | SC=4.133 (w=0.0200) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=05 | train=0.6312 | val=0.6803 | CE=0.633 | SC=4.134 (w=0.0200) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=06 | train=0.6495 | val=0.6516 | CE=0.620 | SC=4.132 (w=0.0200) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=07 | train=0.6445 | val=0.7008 | CE=0.621 | SC=4.132 (w=0.0200) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=08 | train=0.6

In [81]:
supcon_tune_df = pd.read_csv(
    SUPCON_TUNE_CSV
)


# ============================================================
# SUMMARY
# ============================================================

supcon_summary = (
    supcon_tune_df
    .groupby(
        [
            "supcon_weight",
            "supcon_temperature",
        ]
    )
    .agg(

        n_subjects=(
            "test_subject",
            "nunique",
        ),

        mean_accuracy=(
            "test_accuracy",
            "mean",
        ),

        median_accuracy=(
            "test_accuracy",
            "median",
        ),

        std_accuracy=(
            "test_accuracy",
            "std",
        ),

        mean_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),

        mean_macro_f1=(
            "macro_f1",
            "mean",
        ),

        mean_kappa=(
            "kappa",
            "mean",
        ),

        mean_best_epoch=(
            "best_epoch",
            "mean",
        ),
    )
    .reset_index()
)


# ============================================================
# KEEP ONLY COMPLETE CONFIGURATIONS
# ============================================================

n_required = len(
    SUPCON_TUNE_SUBJECTS
)

complete_supcon_summary = (
    supcon_summary[
        supcon_summary[
            "n_subjects"
        ] == n_required
    ]
    .copy()
)


if len(
    complete_supcon_summary
) == 0:

    raise RuntimeError(
        "No complete SupCon "
        "configuration found."
    )


# ============================================================
# PERCENT COLUMNS
# ============================================================

for col in [

    "mean_accuracy",

    "median_accuracy",

    "std_accuracy",

    "mean_balanced_accuracy",

    "mean_macro_f1",

]:

    complete_supcon_summary[
        col + "_pct"
    ] = (
        complete_supcon_summary[
            col
        ]
        *
        100
    )


# ============================================================
# SORT
#
# Primary criterion = mean accuracy
# Secondary = median accuracy
# ============================================================

complete_supcon_summary = (
    complete_supcon_summary
    .sort_values(
        [
            "mean_accuracy",
            "median_accuracy",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# SHOW
# ============================================================

display(
    complete_supcon_summary[
        [
            "supcon_weight",
            "supcon_temperature",
            "n_subjects",

            "mean_accuracy_pct",
            "median_accuracy_pct",
            "std_accuracy_pct",

            "mean_balanced_accuracy_pct",
            "mean_macro_f1_pct",

            "mean_kappa",
            "mean_best_epoch",
        ]
    ]
)


# ============================================================
# BEST CONFIG
# ============================================================

BEST_SUPCON_WEIGHT = float(
    complete_supcon_summary
    .iloc[0][
        "supcon_weight"
    ]
)

BEST_SUPCON_TEMPERATURE = float(
    complete_supcon_summary
    .iloc[0][
        "supcon_temperature"
    ]
)

BEST_SUPCON_SCREEN_ACC = float(
    complete_supcon_summary
    .iloc[0][
        "mean_accuracy"
    ]
)


print()
print("=" * 90)

print(
    "BEST SUPCON CONFIGURATION"
)

print("=" * 90)

print(
    f"Transfer LR : "
    f"{BEST_TRANSFER_LR:.1e}"
)

print(
    f"SupCon λ    : "
    f"{BEST_SUPCON_WEIGHT:.3f}"
)

print(
    f"Temperature : "
    f"{BEST_SUPCON_TEMPERATURE:.2f}"
)

print(
    f"Mean acc    : "
    f"{BEST_SUPCON_SCREEN_ACC*100:.2f}%"
)

print("=" * 90)

,supcon_weight,supcon_temperature,n_subjects,mean_accuracy_pct,median_accuracy_pct,std_accuracy_pct,mean_balanced_accuracy_pct,mean_macro_f1_pct,mean_kappa,mean_best_epoch
0,0.10,0.05,5,69.777778,71.111111,18.297946,70.066111,69.576433,0.399754,17.4
1,0.15,0.10,5,68.888889,73.333333,20.184336,69.194664,68.290053,0.382894,15.0
2,0.15,0.20,5,68.888889,73.333333,17.777778,69.156550,68.215481,0.381706,19.4
3,0.02,0.20,5,68.444444,75.555556,19.398358,68.839403,67.679048,0.375845,14.0
4,0.10,0.20,5,68.444444,73.333333,19.525229,68.859166,67.659570,0.375985,16.0
5,0.05,0.05,5,68.444444,71.111111,20.086234,68.779644,67.802228,0.374358,22.8
6,0.02,0.05,5,68.444444,73.333333,17.455411,68.860813,67.656213,0.375501,17.6
7,0.10,0.10,5,68.000000,71.111111,20.463759,68.362978,67.323134,0.366124,17.2
8,0.05,0.20,5,67.555556,68.888889,19.411082,68.146763,66.380910,0.361160,17.0
9,0.05,0.10,5,67.111111,68.888889,20.086234,67.751506,65.543475,0.353436,16.6



BEST SUPCON CONFIGURATION
Transfer LR : 1.5e-03
SupCon λ    : 0.100
Temperature : 0.05
Mean acc    : 69.78%


In [82]:
# ============================================================
# 15 REPRESENTATIVE SUBJECTS
#
# Already defined earlier in notebook.
# We use exactly the same screening set.
# ============================================================

FULL_SCREEN_SUBJECTS = list(
    REPRESENTATIVE_SUBJECTS
)


# ============================================================
# OUTPUT
# ============================================================

BEST_SUPCON_ROOT = (
    RESULTS_ROOT
    / "best_supcon_15_subject_screening"
)

BEST_SUPCON_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

BEST_SUPCON_CSV = (
    BEST_SUPCON_ROOT
    / "best_supcon_15_progress.csv"
)


# ============================================================
# RESUME
# ============================================================

if BEST_SUPCON_CSV.exists():

    best_supcon_df = pd.read_csv(
        BEST_SUPCON_CSV
    )


    # Keep only current winning configuration
    mask = (

        np.isclose(
            best_supcon_df[
                "supcon_weight"
            ].astype(float),

            BEST_SUPCON_WEIGHT,
        )

        &

        np.isclose(
            best_supcon_df[
                "supcon_temperature"
            ].astype(float),

            BEST_SUPCON_TEMPERATURE,
        )

        &

        np.isclose(
            best_supcon_df[
                "lr"
            ].astype(float),

            BEST_TRANSFER_LR,
        )
    )


    best_supcon_df = (
        best_supcon_df[
            mask
        ].copy()
    )


    done_subjects = set(
        best_supcon_df[
            "test_subject"
        ]
        .astype(int)
        .tolist()
    )


    print(
        f"Resume: "
        f"{len(done_subjects)}/"
        f"{len(FULL_SCREEN_SUBJECTS)} "
        f"subjects done"
    )


else:

    best_supcon_df = (
        pd.DataFrame()
    )

    done_subjects = set()


# ============================================================
# PRINT CONFIG
# ============================================================

print()
print("=" * 90)

print(
    "BEST SUPCON -> "
    "15 SUBJECT VALIDATION"
)

print("=" * 90)

print(
    f"Transfer LR : "
    f"{BEST_TRANSFER_LR:.1e}"
)

print(
    f"SupCon λ    : "
    f"{BEST_SUPCON_WEIGHT:.3f}"
)

print(
    f"Temperature : "
    f"{BEST_SUPCON_TEMPERATURE:.2f}"
)

print("=" * 90)


# ============================================================
# TRAIN
# ============================================================

for idx, subject in enumerate(
    FULL_SCREEN_SUBJECTS,
    start=1,
):

    subject = int(subject)


    if subject in done_subjects:

        print(
            f"[{idx}/"
            f"{len(FULL_SCREEN_SUBJECTS)}] "
            f"S{subject:03d} already done"
        )

        continue


    print()
    print("=" * 100)

    print(
        f"[{idx}/"
        f"{len(FULL_SCREEN_SUBJECTS)}] "
        f"BEST TRANSFER + SUPCON | "
        f"S{subject:03d}"
    )

    print("=" * 100)


    result, model, history = (
        train_cho64_ablation_fold(

            test_subject=subject,

            mode="supcon",

            seed=42,

            max_epochs=40,

            patience=8,

            batch_size=64,

            lr=BEST_TRANSFER_LR,

            weight_decay=1e-4,

            supcon_weight=
                BEST_SUPCON_WEIGHT,

            supcon_temperature=
                BEST_SUPCON_TEMPERATURE,
        )
    )


    result["lr"] = (
        BEST_TRANSFER_LR
    )


    result[
        "supcon_weight"
    ] = (
        BEST_SUPCON_WEIGHT
    )


    result[
        "supcon_temperature"
    ] = (
        BEST_SUPCON_TEMPERATURE
    )


    # ========================================================
    # SAVE
    # ========================================================

    best_supcon_df = pd.concat(
        [
            best_supcon_df,
            pd.DataFrame(
                [result]
            ),
        ],
        ignore_index=True,
    )


    best_supcon_df.to_csv(
        BEST_SUPCON_CSV,
        index=False,
    )


    done_subjects.add(
        subject
    )


    print(
        f"\nRESULT | "
        f"S{subject:03d} | "
        f"VAL="
        f"{result['best_val_accuracy']*100:.2f}% | "
        f"TEST="
        f"{result['test_accuracy']*100:.2f}%"
    )


    # ========================================================
    # CLEANUP
    # ========================================================

    del model
    del history

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print()
print("=" * 100)

print(
    "15-SUBJECT BEST SUPCON "
    "VALIDATION FINISHED"
)

print("=" * 100)


BEST SUPCON -> 15 SUBJECT VALIDATION
Transfer LR : 1.5e-03
SupCon λ    : 0.100
Temperature : 0.05

[1/15] BEST TRANSFER + SUPCON | S003
SUPCON       | ep=01 | train=0.5970 | val=0.6639 | CE=0.678 | SC=4.191 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=02 | train=0.6160 | val=0.6926 | CE=0.647 | SC=4.139 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=03 | train=0.6226 | val=0.6885 | CE=0.638 | SC=4.136 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=04 | train=0.6381 | val=0.6865 | CE=0.628 | SC=4.133 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=05 | train=0.6319 | val=0.6824 | CE=0.632 | SC=4.134 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=06 | train=0.6499 | val=0.6578 | CE=0.620 | SC=4.132 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=07 | train=0.6458 | val=0.6947 | CE=0.621 | SC=4.132 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=08 | train=0.6447 | val=0.

In [83]:
# ============================================================
# LOAD TRANSFER RESULTS
# ============================================================

transfer_dfs = []


if TRANSFER_SEARCH_CSV.exists():

    transfer_dfs.append(
        pd.read_csv(
            TRANSFER_SEARCH_CSV
        )
    )


if FINE_SEARCH_CSV.exists():

    transfer_dfs.append(
        pd.read_csv(
            FINE_SEARCH_CSV
        )
    )


all_transfer_df = pd.concat(
    transfer_dfs,
    ignore_index=True,
)


# ============================================================
# SELECT BEST TRANSFER LR
# ============================================================

transfer_best_df = (
    all_transfer_df[
        np.isclose(
            all_transfer_df[
                "lr"
            ].astype(float),

            BEST_TRANSFER_LR,
        )
    ]
    .copy()
)


# Keep the same 15 subjects
transfer_best_df = (
    transfer_best_df[
        transfer_best_df[
            "test_subject"
        ].astype(int)
        .isin(
            FULL_SCREEN_SUBJECTS
        )
    ]
    .copy()
)


# ============================================================
# LOAD BEST SUPCON
# ============================================================

best_supcon_df = pd.read_csv(
    BEST_SUPCON_CSV
)


best_supcon_df = (
    best_supcon_df[

        np.isclose(
            best_supcon_df[
                "supcon_weight"
            ].astype(float),

            BEST_SUPCON_WEIGHT,
        )

        &

        np.isclose(
            best_supcon_df[
                "supcon_temperature"
            ].astype(float),

            BEST_SUPCON_TEMPERATURE,
        )
    ]
    .copy()
)


# ============================================================
# SUMMARY
# ============================================================

summary = pd.DataFrame({

    "method": [
        "Transfer",
        "Transfer + tuned SupCon",
    ],

    "mean_accuracy_pct": [

        transfer_best_df[
            "test_accuracy"
        ].mean() * 100,

        best_supcon_df[
            "test_accuracy"
        ].mean() * 100,
    ],

    "median_accuracy_pct": [

        transfer_best_df[
            "test_accuracy"
        ].median() * 100,

        best_supcon_df[
            "test_accuracy"
        ].median() * 100,
    ],

    "std_accuracy_pct": [

        transfer_best_df[
            "test_accuracy"
        ].std() * 100,

        best_supcon_df[
            "test_accuracy"
        ].std() * 100,
    ],

    "balanced_accuracy_pct": [

        transfer_best_df[
            "balanced_accuracy"
        ].mean() * 100,

        best_supcon_df[
            "balanced_accuracy"
        ].mean() * 100,
    ],

    "macro_f1_pct": [

        transfer_best_df[
            "macro_f1"
        ].mean() * 100,

        best_supcon_df[
            "macro_f1"
        ].mean() * 100,
    ],
})


display(
    summary
)


# ============================================================
# PAIRED SUBJECT RESULTS
# ============================================================

paired = (

    transfer_best_df[
        [
            "test_subject",
            "test_accuracy",
        ]
    ]

    .rename(
        columns={
            "test_accuracy":
                "transfer_accuracy"
        }
    )

    .merge(

        best_supcon_df[
            [
                "test_subject",
                "test_accuracy",
            ]
        ]

        .rename(
            columns={
                "test_accuracy":
                    "supcon_accuracy"
            }
        ),

        on="test_subject",

        how="inner",
    )
)


paired[
    "delta_pp"
] = (

    paired[
        "supcon_accuracy"
    ]

    -

    paired[
        "transfer_accuracy"
    ]

) * 100


paired[
    "transfer_accuracy"
] *= 100

paired[
    "supcon_accuracy"
] *= 100


display(
    paired.sort_values(
        "delta_pp",
        ascending=False,
    )
)


# ============================================================
# FINAL NUMBERS
# ============================================================

mean_delta = (
    paired[
        "delta_pp"
    ].mean()
)

median_delta = (
    paired[
        "delta_pp"
    ].median()
)

win_rate = (
    (
        paired[
            "delta_pp"
        ] > 0
    )
    .mean()
    *
    100
)


print()
print("=" * 90)

print(
    f"Mean delta   : "
    f"{mean_delta:+.2f} pp"
)

print(
    f"Median delta : "
    f"{median_delta:+.2f} pp"
)

print(
    f"Win rate     : "
    f"{win_rate:.1f}%"
)

print("=" * 90)


if mean_delta > 0:

    print(
        "RESULT: tuned SupCon "
        "improves Transfer."
    )

else:

    print(
        "RESULT: tuned SupCon "
        "does not improve Transfer."
    )

,method,mean_accuracy_pct,median_accuracy_pct,std_accuracy_pct,balanced_accuracy_pct,macro_f1_pct
0,Transfer,70.518519,77.777778,16.100799,70.805258,68.782536
1,Transfer + tuned SupCon,72.740741,77.777778,13.477178,72.999561,71.516618


,test_subject,transfer_accuracy,supcon_accuracy,delta_pp
2,18,48.888889,80.000000,31.111111
3,24,42.222222,46.666667,4.444444
1,9,53.333333,57.777778,4.444444
0,3,44.444444,46.666667,2.222222
7,19,77.777778,80.000000,2.222222
9,32,77.777778,80.000000,2.222222
10,2,91.111111,93.333333,2.222222
4,30,66.666667,66.666667,0.000000
5,1,71.111111,71.111111,0.000000
8,27,68.888889,68.888889,0.000000



Mean delta   : +2.22 pp
Median delta : +0.00 pp
Win rate     : 46.7%
RESULT: tuned SupCon improves Transfer.


## Smart GRL

Фиксируем уже найденную сильную конфигурацию:

- EEGNet64;
- Cho64 Transfer;
- Transfer LR = `1.5e-3`;
- SupCon weight = `0.10`;
- SupCon temperature = `0.05`.

Добавляем адаптивный GRL.

Принцип:

1. Первые 5 эпох GRL не влияет на EEGNet.
2. Subject classifier при этом обучается распознавать субъектов.
3. После warm-up GRL включается плавно.
4. Сила reversed gradient зависит от subject accuracy:
   - если subject classifier хорошо определяет человека, GRL усиливается;
   - если subject information уже почти исчезла, GRL ослабляется.
5. Subject classifier обучается с обычной силой, а ограничивается именно reversed gradient в backbone.

Проверяем три максимальные силы GRL:

- 0.02
- 0.05
- 0.10

Сначала на тех же 5 development subjects.

In [84]:
# ============================================================
# SMART GRL
#
# Fixed:
#   Transfer LR = 1.5e-3
#   SupCon weight = 0.10
#   SupCon temperature = 0.05
#
# Smart GRL:
#   - warmup
#   - subject head learns from epoch 1
#   - reversed gradient starts after warmup
#   - GRL strength adapts to subject accuracy
# ============================================================

def train_cho64_smart_grl_fold(
    test_subject,
    seed=42,
    max_epochs=40,
    patience=8,
    batch_size=64,

    lr=1.5e-3,
    weight_decay=1e-4,

    supcon_weight=0.10,
    supcon_temperature=0.05,

    grl_max=0.05,
    grl_warmup_epochs=5,

    # subject accuracy at which adaptive GRL
    # is allowed to reach full scheduled strength
    subject_acc_target=0.05,

    # subject head itself learns normally
    adv_weight=1.0,

    ema_beta=0.90,
):

    set_seed(seed)

    run_device = globals().get(
        "device",
        torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "mps"
            if torch.backends.mps.is_available()
            else "cpu"
        ),
    )


    # ========================================================
    # DATA
    # ========================================================

    fold = prepare_phys64_fold_supcon_grl(
        test_subject=test_subject,
        seed=seed,
    )

    train_dataset = Phys64SubjectDataset(
        fold["X_train"],
        fold["y_train"],
        fold["s_train"],
        augment=True,
    )

    val_dataset = Phys64SubjectDataset(
        fold["X_val"],
        fold["y_val"],
        fold["s_val"],
        augment=False,
    )

    s_test_dummy = np.zeros(
        len(fold["y_test"]),
        dtype=np.int64,
    )

    test_dataset = Phys64SubjectDataset(
        fold["X_test"],
        fold["y_test"],
        s_test_dummy,
        augment=False,
    )


    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        drop_last=False,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        drop_last=False,
    )


    # ========================================================
    # MODEL
    # ========================================================

    model = make_cho64_supcon_grl_model(
        n_subjects=fold["n_source_subjects"]
    ).to(run_device)


    # ========================================================
    # LOSSES
    # ========================================================

    ce_loss_fn = nn.CrossEntropyLoss()

    subject_loss_fn = nn.CrossEntropyLoss()

    supcon_loss_fn = SupConLoss(
        temperature=supcon_temperature
    )


    # ========================================================
    # OPTIMIZER
    # ========================================================

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )


    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
    )


    # ========================================================
    # SMART GRL STATE
    # ========================================================

    n_subjects = int(
        fold["n_source_subjects"]
    )

    chance_subject_acc = (
        1.0 / n_subjects
    )

    ema_subject_acc = (
        chance_subject_acc
    )


    best_val_accuracy = -np.inf
    best_epoch = 0
    best_state = None

    epochs_without_improvement = 0

    history = []


    # ========================================================
    # TRAIN
    # ========================================================

    for epoch in range(
        1,
        max_epochs + 1,
    ):

        # ====================================================
        # SMART GRL SCHEDULE
        # ====================================================

        if epoch <= grl_warmup_epochs:

            scheduled_grl = 0.0

        else:

            grl_epoch = (
                epoch
                - grl_warmup_epochs
            )

            grl_total_epochs = max(
                max_epochs
                - grl_warmup_epochs,
                1,
            )

            progress = (
                grl_epoch
                /
                grl_total_epochs
            )

            # smooth DANN-style ramp
            ramp = (
                2.0
                /
                (
                    1.0
                    +
                    math.exp(
                        -10.0 * progress
                    )
                )
                -
                1.0
            )

            scheduled_grl = (
                grl_max
                *
                ramp
            )


        # ====================================================
        # ADAPT TO SUBJECT INFORMATION
        # ====================================================

        if epoch <= grl_warmup_epochs:

            subject_factor = 0.0

        else:

            denominator = max(
                subject_acc_target
                -
                chance_subject_acc,
                1e-8,
            )

            subject_factor = (
                ema_subject_acc
                -
                chance_subject_acc
            ) / denominator

            subject_factor = float(
                np.clip(
                    subject_factor,
                    0.0,
                    1.0,
                )
            )


        lambda_grl = (
            scheduled_grl
            *
            subject_factor
        )


        # ====================================================
        # TRAIN
        # ====================================================

        model.train()


        train_total = 0

        class_correct = 0
        subject_correct = 0

        total_loss_sum = 0.0
        ce_sum = 0.0
        sc_sum = 0.0
        adv_sum = 0.0


        for (
            X,
            y,
            subject_ids,
        ) in train_loader:

            X = X.to(
                run_device,
                non_blocking=True,
            )

            y = y.to(
                run_device,
                non_blocking=True,
            )

            subject_ids = subject_ids.to(
                run_device,
                non_blocking=True,
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            out = model(
                X,
                lambda_grl=lambda_grl,
            )


            class_logits = out[
                "class_logits"
            ]

            projection = out[
                "projection"
            ]

            subject_logits = out[
                "subject_logits"
            ]


            # =================================================
            # LOSSES
            # =================================================

            loss_ce = ce_loss_fn(
                class_logits,
                y,
            )

            loss_sc = supcon_loss_fn(
                projection,
                y,
            )

            loss_adv = subject_loss_fn(
                subject_logits,
                subject_ids,
            )


            loss = (
                loss_ce
                +
                supcon_weight * loss_sc
                +
                adv_weight * loss_adv
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )


            optimizer.step()


            # =================================================
            # METRICS
            # =================================================

            n = y.size(0)

            train_total += n


            class_preds = (
                class_logits.argmax(
                    dim=1
                )
            )

            subject_preds = (
                subject_logits.argmax(
                    dim=1
                )
            )


            class_correct += (
                class_preds == y
            ).sum().item()

            subject_correct += (
                subject_preds
                ==
                subject_ids
            ).sum().item()


            total_loss_sum += (
                loss.item() * n
            )

            ce_sum += (
                loss_ce.item() * n
            )

            sc_sum += (
                loss_sc.item() * n
            )

            adv_sum += (
                loss_adv.item() * n
            )


        # ====================================================
        # TRAIN METRICS
        # ====================================================

        train_accuracy = (
            class_correct
            /
            max(train_total, 1)
        )

        subject_accuracy = (
            subject_correct
            /
            max(train_total, 1)
        )


        # update subject accuracy EMA
        ema_subject_acc = (
            ema_beta
            *
            ema_subject_acc
            +
            (1.0 - ema_beta)
            *
            subject_accuracy
        )


        train_total_loss = (
            total_loss_sum
            /
            max(train_total, 1)
        )

        train_ce_loss = (
            ce_sum
            /
            max(train_total, 1)
        )

        train_sc_loss = (
            sc_sum
            /
            max(train_total, 1)
        )

        train_adv_loss = (
            adv_sum
            /
            max(train_total, 1)
        )


        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()

        val_true = []
        val_pred = []


        with torch.no_grad():

            for X, y, _ in val_loader:

                X = X.to(
                    run_device,
                    non_blocking=True,
                )

                y = y.to(
                    run_device,
                    non_blocking=True,
                )


                out = model(
                    X,
                    lambda_grl=0.0,
                )


                logits = out[
                    "class_logits"
                ]


                preds = logits.argmax(
                    dim=1
                )


                val_true.extend(
                    y.cpu().numpy()
                )

                val_pred.extend(
                    preds.cpu().numpy()
                )


        val_accuracy = accuracy_score(
            val_true,
            val_pred,
        )


        scheduler.step(
            val_accuracy
        )


        current_lr = (
            optimizer
            .param_groups[0]["lr"]
        )


        # ====================================================
        # HISTORY
        # ====================================================

        history.append({

            "epoch":
                epoch,

            "train_accuracy":
                train_accuracy,

            "val_accuracy":
                val_accuracy,

            "subject_accuracy":
                subject_accuracy,

            "ema_subject_accuracy":
                ema_subject_acc,

            "chance_subject_accuracy":
                chance_subject_acc,

            "subject_factor":
                subject_factor,

            "scheduled_grl":
                scheduled_grl,

            "lambda_grl":
                lambda_grl,

            "total_loss":
                train_total_loss,

            "ce_loss":
                train_ce_loss,

            "supcon_loss":
                train_sc_loss,

            "adv_loss":
                train_adv_loss,

            "lr":
                current_lr,
        })


        # ====================================================
        # LOG
        # ====================================================

        print(
            f"SMART_GRL | "
            f"ep={epoch:02d} | "
            f"train={train_accuracy:.4f} | "
            f"val={val_accuracy:.4f} | "
            f"SUBJ={subject_accuracy:.4f} | "
            f"SUBJ_EMA={ema_subject_acc:.4f} | "
            f"GRL={lambda_grl:.4f} | "
            f"CE={train_ce_loss:.3f} | "
            f"SC={train_sc_loss:.3f} | "
            f"ADV={train_adv_loss:.3f}"
        )


        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if val_accuracy > best_val_accuracy:

            best_val_accuracy = (
                val_accuracy
            )

            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0


        else:

            epochs_without_improvement += 1


        if (
            epochs_without_improvement
            >= patience
        ):

            print(
                f"Early stopping "
                f"at epoch {epoch}"
            )

            break


    # ========================================================
    # RESTORE BEST MODEL
    # ========================================================

    model.load_state_dict(
        best_state
    )

    model.eval()


    # ========================================================
    # TEST
    # ========================================================

    test_true = []
    test_pred = []


    with torch.no_grad():

        for X, y, _ in test_loader:

            X = X.to(
                run_device,
                non_blocking=True,
            )

            y = y.to(
                run_device,
                non_blocking=True,
            )


            out = model(
                X,
                lambda_grl=0.0,
            )


            logits = out[
                "class_logits"
            ]


            preds = logits.argmax(
                dim=1
            )


            test_true.extend(
                y.cpu().numpy()
            )

            test_pred.extend(
                preds.cpu().numpy()
            )


    test_accuracy = accuracy_score(
        test_true,
        test_pred,
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            test_true,
            test_pred,
        )
    )

    macro_f1 = f1_score(
        test_true,
        test_pred,
        average="macro",
        zero_division=0,
    )

    kappa = cohen_kappa_score(
        test_true,
        test_pred,
    )


    result = {

        "test_subject":
            int(test_subject),

        "best_epoch":
            int(best_epoch),

        "best_val_accuracy":
            float(best_val_accuracy),

        "test_accuracy":
            float(test_accuracy),

        "balanced_accuracy":
            float(balanced_accuracy),

        "macro_f1":
            float(macro_f1),

        "kappa":
            float(kappa),

        "lr":
            float(lr),

        "supcon_weight":
            float(supcon_weight),

        "supcon_temperature":
            float(supcon_temperature),

        "grl_max":
            float(grl_max),

        "grl_warmup_epochs":
            int(grl_warmup_epochs),

        "subject_acc_target":
            float(subject_acc_target),

        "seed":
            int(seed),
    }


    return (
        result,
        model,
        pd.DataFrame(history),
    )

In [85]:
# ============================================================
# SMART GRL SCREENING
# ============================================================

SMART_GRL_MAX_VALUES = [
    0.02,
    0.05,
    0.10,
]

SMART_GRL_SUBJECTS = [
    3,
    9,
    1,
    19,
    2,
]


SMART_GRL_ROOT = (
    RESULTS_ROOT
    / "smart_grl_screening"
)

SMART_GRL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SMART_GRL_CSV = (
    SMART_GRL_ROOT
    / "smart_grl_progress.csv"
)


# ============================================================
# RESUME
# ============================================================

if SMART_GRL_CSV.exists():

    smart_grl_df = pd.read_csv(
        SMART_GRL_CSV
    )

    done = set(
        zip(
            smart_grl_df[
                "grl_max"
            ].astype(float),

            smart_grl_df[
                "test_subject"
            ].astype(int),
        )
    )

else:

    smart_grl_df = pd.DataFrame()

    done = set()


# ============================================================
# RUN
# ============================================================

total_runs = (
    len(SMART_GRL_MAX_VALUES)
    *
    len(SMART_GRL_SUBJECTS)
)

run_idx = 0


for grl_max in SMART_GRL_MAX_VALUES:

    for subject in SMART_GRL_SUBJECTS:

        run_idx += 1


        key = (
            float(grl_max),
            int(subject),
        )


        if key in done:

            print(
                f"[{run_idx}/{total_runs}] "
                f"GRL={grl_max:.2f} | "
                f"S{subject:03d} already done"
            )

            continue


        print()
        print("=" * 100)

        print(
            f"[{run_idx}/{total_runs}] "
            f"SMART GRL | "
            f"MAX={grl_max:.2f} | "
            f"S{subject:03d}"
        )

        print("=" * 100)


        result, model, history = (
            train_cho64_smart_grl_fold(

                test_subject=subject,

                seed=42,

                max_epochs=40,
                patience=8,

                batch_size=64,

                lr=1.5e-3,
                weight_decay=1e-4,

                supcon_weight=0.10,
                supcon_temperature=0.05,

                grl_max=grl_max,

                grl_warmup_epochs=5,

                subject_acc_target=0.05,

                adv_weight=1.0,
            )
        )


        smart_grl_df = pd.concat(
            [
                smart_grl_df,
                pd.DataFrame(
                    [result]
                ),
            ],
            ignore_index=True,
        )


        smart_grl_df.to_csv(
            SMART_GRL_CSV,
            index=False,
        )


        done.add(
            key
        )


        print(
            f"\nRESULT | "
            f"GRL={grl_max:.2f} | "
            f"S{subject:03d} | "
            f"VAL="
            f"{result['best_val_accuracy']*100:.2f}% | "
            f"TEST="
            f"{result['test_accuracy']*100:.2f}%"
        )


        del model
        del history

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


print()
print("=" * 100)
print("SMART GRL SCREENING FINISHED")
print("=" * 100)


[1/15] SMART GRL | MAX=0.02 | S003
SMART_GRL | ep=01 | train=0.5970 | val=0.6639 | SUBJ=0.0180 | SUBJ_EMA=0.0101 | GRL=0.0000 | CE=0.678 | SC=4.191 | ADV=4.521
SMART_GRL | ep=02 | train=0.6160 | val=0.6926 | SUBJ=0.0342 | SUBJ_EMA=0.0125 | GRL=0.0000 | CE=0.647 | SC=4.139 | ADV=4.156
SMART_GRL | ep=03 | train=0.6226 | val=0.6885 | SUBJ=0.0538 | SUBJ_EMA=0.0167 | GRL=0.0000 | CE=0.638 | SC=4.136 | ADV=3.974
SMART_GRL | ep=04 | train=0.6381 | val=0.6865 | SUBJ=0.0664 | SUBJ_EMA=0.0216 | GRL=0.0000 | CE=0.628 | SC=4.133 | ADV=3.856
SMART_GRL | ep=05 | train=0.6319 | val=0.6824 | SUBJ=0.0702 | SUBJ_EMA=0.0265 | GRL=0.0000 | CE=0.632 | SC=4.134 | ADV=3.790
SMART_GRL | ep=06 | train=0.6499 | val=0.6557 | SUBJ=0.0857 | SUBJ_EMA=0.0324 | GRL=0.0012 | CE=0.620 | SC=4.132 | ADV=3.749
SMART_GRL | ep=07 | train=0.6461 | val=0.6947 | SUBJ=0.0839 | SUBJ_EMA=0.0376 | GRL=0.0032 | CE=0.621 | SC=4.132 | ADV=3.731
SMART_GRL | ep=08 | train=0.6465 | val=0.6660 | SUBJ=0.0912 | SUBJ_EMA=0.0429 | GRL=0.005

In [86]:
# ============================================================
# SMART GRL SUMMARY
# ============================================================

smart_grl_df = pd.read_csv(
    SMART_GRL_CSV
)


smart_grl_summary = (
    smart_grl_df
    .groupby(
        "grl_max"
    )
    .agg(

        n_subjects=(
            "test_subject",
            "nunique",
        ),

        mean_accuracy=(
            "test_accuracy",
            "mean",
        ),

        median_accuracy=(
            "test_accuracy",
            "median",
        ),

        std_accuracy=(
            "test_accuracy",
            "std",
        ),

        mean_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),

        mean_macro_f1=(
            "macro_f1",
            "mean",
        ),

        mean_best_epoch=(
            "best_epoch",
            "mean",
        ),
    )
    .reset_index()
)


for col in [
    "mean_accuracy",
    "median_accuracy",
    "std_accuracy",
    "mean_balanced_accuracy",
    "mean_macro_f1",
]:

    smart_grl_summary[
        col + "_pct"
    ] = (
        smart_grl_summary[col]
        * 100
    )


smart_grl_summary = (
    smart_grl_summary
    .sort_values(
        [
            "mean_accuracy",
            "median_accuracy",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    smart_grl_summary[
        [
            "grl_max",
            "n_subjects",
            "mean_accuracy_pct",
            "median_accuracy_pct",
            "std_accuracy_pct",
            "mean_balanced_accuracy_pct",
            "mean_macro_f1_pct",
            "mean_best_epoch",
        ]
    ]
)


BEST_GRL_MAX = float(
    smart_grl_summary
    .iloc[0]["grl_max"]
)


print()
print("=" * 90)

print(
    f"BEST SMART GRL MAX = "
    f"{BEST_GRL_MAX:.2f}"
)

print("=" * 90)

,grl_max,n_subjects,mean_accuracy_pct,median_accuracy_pct,std_accuracy_pct,mean_balanced_accuracy_pct,mean_macro_f1_pct,mean_best_epoch
0,0.02,5,67.111111,71.111111,19.077490,67.113448,66.949502,10.6
1,0.05,5,65.777778,68.888889,19.090428,66.128600,65.138194,14.6
2,0.10,5,65.777778,66.666667,18.365292,66.029315,65.507216,7.8



BEST SMART GRL MAX = 0.02


## Ночной эксперимент: Conflict-aware GRL

Уже зафиксировано:

- EEGNet64;
- Cho64 → PhysioNet64 Transfer;
- Transfer LR = `1.5e-3`;
- tuned SupCon:
  - weight = `0.10`;
  - temperature = `0.05`;
- Transfer + tuned SupCon на 15 subjects = `72.74%`;
- CCST64 baseline = `73.65%`.

Обычный и адаптивный GRL ухудшали результат.

### Новая идея

Вместо обычного GRL используется gradient surgery.

Для каждого batch считаются два градиента:

1. основной градиент:
   `CE + SupCon`;

2. adversarial subject-gradient.

Subject-gradient разворачивается, как в GRL.

Если reversed subject-gradient конфликтует с основным градиентом классификации,
из него удаляется конфликтующая компонента.

Таким образом GRL может уменьшать subject-specific information,
но ему запрещается непосредственно разрушать направление обучения
Left/Right classification.

### Ночной pipeline

1. На 5 development subjects проверяются:
   - GRL max = `0.02`
   - GRL max = `0.05`
   - GRL max = `0.10`

2. Автоматически выбирается лучший вариант.

3. Лучший Conflict-aware GRL сравнивается с tuned SupCon без GRL.

4. Если GRL улучшает результат хотя бы на `0.5 п.п.`:
   запускается полный 109-subject LOSO с Conflict-aware GRL.

5. Если GRL не улучшает результат:
   полный 109-subject LOSO автоматически запускается с
   `Transfer + tuned SupCon`.

То есть ночью в любом случае запускается лучший найденный вариант,
а плохой GRL не сможет испортить финальный эксперимент.

In [89]:
# ============================================================
# CONFLICT-AWARE GRL
#
# Исправленная версия для НАШЕГО EEGNet64:
# forward_features() НЕ требуется.
#
# Features перехватываются hook'ом на входе classifier.
# ============================================================


def train_cho64_conflict_grl_fold(
    test_subject,
    seed=42,
    max_epochs=40,
    patience=8,
    batch_size=64,
    lr=1.5e-3,
    weight_decay=1e-4,

    supcon_weight=0.10,
    supcon_temperature=0.05,

    grl_max=0.05,
    grl_warmup_epochs=5,

    subject_head_weight=1.0,
):

    set_seed(seed)

    run_device = globals().get(
        "device",
        torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "mps"
            if torch.backends.mps.is_available()
            else "cpu"
        ),
    )


    # ========================================================
    # DATA
    # ========================================================

    fold = prepare_phys64_fold_supcon_grl(
        test_subject=test_subject,
        seed=seed,
    )

    train_dataset = Phys64SubjectDataset(
        fold["X_train"],
        fold["y_train"],
        fold["s_train"],
        augment=True,
    )

    val_dataset = Phys64SubjectDataset(
        fold["X_val"],
        fold["y_val"],
        fold["s_val"],
        augment=False,
    )

    s_test_dummy = np.zeros(
        len(fold["y_test"]),
        dtype=np.int64,
    )

    test_dataset = Phys64SubjectDataset(
        fold["X_test"],
        fold["y_test"],
        s_test_dummy,
        augment=False,
    )


    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        drop_last=False,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        drop_last=False,
    )


    # ========================================================
    # MODEL
    # ========================================================

    model = make_cho64_supcon_grl_model(
        n_subjects=fold["n_source_subjects"]
    ).to(run_device)


    if not hasattr(model.backbone, "classifier"):
        raise RuntimeError(
            "У model.backbone не найден classifier."
        )

    if not hasattr(model, "projector"):
        raise RuntimeError(
            "У модели не найден projector."
        )

    if not hasattr(model, "subject_head"):
        raise RuntimeError(
            "У модели не найден subject_head."
        )


    # ========================================================
    # FEATURE HOOK
    #
    # classifier получает на вход flatten features EEGNet.
    # Именно их будем использовать для subject classifier
    # без GradientReversal.
    # ========================================================

    captured = {}


    def classifier_input_hook(
        module,
        inputs,
        output,
    ):
        captured["features"] = inputs[0]


    hook_handle = (
        model
        .backbone
        .classifier
        .register_forward_hook(
            classifier_input_hook
        )
    )


    # ========================================================
    # LOSSES
    # ========================================================

    ce_loss_fn = nn.CrossEntropyLoss()

    subject_loss_fn = nn.CrossEntropyLoss()

    supcon_loss_fn = SupConLoss(
        temperature=supcon_temperature
    )


    # ========================================================
    # PARAMETERS
    # ========================================================

    backbone_params = list(
        model.backbone.parameters()
    )

    projector_params = list(
        model.projector.parameters()
    )

    subject_params = list(
        model.subject_head.parameters()
    )

    main_params = (
        backbone_params
        +
        projector_params
    )


    # ========================================================
    # OPTIMIZER
    # ========================================================

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
    )


    # ========================================================
    # TRAIN STATE
    # ========================================================

    best_val_accuracy = -np.inf
    best_epoch = 0
    best_state = None

    epochs_without_improvement = 0

    history = []


    # ========================================================
    # TRAIN
    # ========================================================

    for epoch in range(
        1,
        max_epochs + 1,
    ):

        # ----------------------------------------------------
        # GRL SCHEDULE
        # ----------------------------------------------------

        if epoch <= grl_warmup_epochs:

            lambda_grl = 0.0

        else:

            progress = (
                epoch
                - grl_warmup_epochs
            ) / max(
                max_epochs
                - grl_warmup_epochs,
                1,
            )

            ramp = (
                2.0
                /
                (
                    1.0
                    +
                    math.exp(
                        -10.0 * progress
                    )
                )
                -
                1.0
            )

            lambda_grl = (
                grl_max
                *
                ramp
            )


        model.train()


        train_total = 0

        class_correct = 0
        subject_correct = 0

        ce_sum = 0.0
        sc_sum = 0.0
        adv_sum = 0.0

        conflict_batches = 0
        total_batches = 0

        cosine_sum = 0.0


        # ====================================================
        # BATCH LOOP
        # ====================================================

        for (
            X,
            y,
            subject_ids,
        ) in train_loader:

            X = X.to(
                run_device,
                non_blocking=True,
            )

            y = y.to(
                run_device,
                non_blocking=True,
            )

            subject_ids = subject_ids.to(
                run_device,
                non_blocking=True,
            )


            optimizer.zero_grad(
                set_to_none=True
            )

            captured.clear()


            # =================================================
            # NORMAL MODEL FORWARD
            #
            # lambda_grl=0:
            # обычный GRL не влияет на backbone.
            #
            # Hook сохранит features перед classifier.
            # =================================================

            out = model(
                X,
                lambda_grl=0.0,
            )


            class_logits = out[
                "class_logits"
            ]

            projection = out[
                "projection"
            ]


            if "features" not in captured:

                hook_handle.remove()

                raise RuntimeError(
                    "Не удалось перехватить EEGNet features "
                    "на входе classifier."
                )


            features = captured[
                "features"
            ]


            # =================================================
            # BYPASS GRL
            #
            # Нам нужен обычный subject gradient,
            # чтобы затем вручную его развернуть.
            # =================================================

            subject_logits = (
                model
                .subject_head(
                    features
                )
            )


            # =================================================
            # LOSSES
            # =================================================

            loss_ce = ce_loss_fn(
                class_logits,
                y,
            )

            loss_sc = supcon_loss_fn(
                projection,
                y,
            )

            loss_main = (
                loss_ce
                +
                supcon_weight
                *
                loss_sc
            )


            loss_subject = (
                subject_loss_fn(
                    subject_logits,
                    subject_ids,
                )
            )


            # =================================================
            # MAIN GRADIENT
            #
            # CE + SupCon
            # =================================================

            main_grads = torch.autograd.grad(
                loss_main,
                main_params,
                retain_graph=True,
                allow_unused=True,
            )


            # =================================================
            # SUBJECT GRADIENT -> BACKBONE
            # =================================================

            adv_backbone_grads = (
                torch.autograd.grad(
                    loss_subject,
                    backbone_params,
                    retain_graph=True,
                    allow_unused=True,
                )
            )


            # =================================================
            # SUBJECT HEAD LEARNS NORMALLY
            # =================================================

            subject_head_grads = (
                torch.autograd.grad(
                    loss_subject,
                    subject_params,
                    retain_graph=False,
                    allow_unused=True,
                )
            )


            n_backbone = len(
                backbone_params
            )


            main_backbone_grads = (
                main_grads[
                    :n_backbone
                ]
            )


            main_projector_grads = (
                main_grads[
                    n_backbone:
                ]
            )


            # =================================================
            # REVERSED SUBJECT GRADIENT
            # =================================================

            reversed_adv_grads = []


            dot = torch.tensor(
                0.0,
                device=run_device,
            )

            main_norm_sq = torch.tensor(
                0.0,
                device=run_device,
            )

            rev_norm_sq = torch.tensor(
                0.0,
                device=run_device,
            )


            for (
                g_main,
                g_adv,
            ) in zip(
                main_backbone_grads,
                adv_backbone_grads,
            ):

                if g_adv is None:

                    reversed_adv_grads.append(
                        None
                    )

                    continue


                # GRL:
                # reverse subject gradient
                g_rev = (
                    -lambda_grl
                    *
                    g_adv
                )


                reversed_adv_grads.append(
                    g_rev
                )


                if g_main is not None:

                    dot = (
                        dot
                        +
                        (
                            g_main
                            *
                            g_rev
                        ).sum()
                    )

                    main_norm_sq = (
                        main_norm_sq
                        +
                        (
                            g_main
                            *
                            g_main
                        ).sum()
                    )

                    rev_norm_sq = (
                        rev_norm_sq
                        +
                        (
                            g_rev
                            *
                            g_rev
                        ).sum()
                    )


            # =================================================
            # GRADIENT COSINE
            # =================================================

            denom = (
                torch.sqrt(
                    main_norm_sq
                    +
                    1e-12
                )
                *
                torch.sqrt(
                    rev_norm_sq
                    +
                    1e-12
                )
            )


            cosine = (
                dot
                /
                (
                    denom
                    +
                    1e-12
                )
            )


            cosine_value = float(
                cosine
                .detach()
                .cpu()
                .item()
            )


            cosine_sum += (
                cosine_value
            )

            total_batches += 1


            # =================================================
            # CONFLICT DETECTION
            #
            # dot < 0:
            # reversed subject gradient идёт против
            # основной Left/Right задачи.
            # =================================================

            has_conflict = (
                lambda_grl > 0.0
                and
                dot.item() < 0.0
            )


            if has_conflict:

                conflict_batches += 1


                projection_coeff = (
                    dot
                    /
                    (
                        main_norm_sq
                        +
                        1e-12
                    )
                )


            # =================================================
            # BACKBONE FINAL GRADIENT
            # =================================================

            for (
                param,
                g_main,
                g_rev,
            ) in zip(
                backbone_params,
                main_backbone_grads,
                reversed_adv_grads,
            ):

                if (
                    g_main is None
                    and
                    g_rev is None
                ):

                    continue


                if g_main is None:

                    final_grad = (
                        g_rev
                    )


                elif g_rev is None:

                    final_grad = (
                        g_main
                    )


                else:

                    # -----------------------------------------
                    # Gradient surgery
                    # -----------------------------------------

                    if has_conflict:

                        safe_rev = (
                            g_rev
                            -
                            projection_coeff
                            *
                            g_main
                        )

                    else:

                        safe_rev = (
                            g_rev
                        )


                    final_grad = (
                        g_main
                        +
                        safe_rev
                    )


                param.grad = (
                    final_grad
                    .detach()
                    .clone()
                )


            # =================================================
            # PROJECTOR:
            # ONLY CE/SUPCON MAIN GRADIENT
            # =================================================

            for (
                param,
                grad,
            ) in zip(
                projector_params,
                main_projector_grads,
            ):

                if grad is not None:

                    param.grad = (
                        grad
                        .detach()
                        .clone()
                    )


            # =================================================
            # SUBJECT HEAD:
            # NORMAL SUBJECT CLASSIFICATION
            # =================================================

            for (
                param,
                grad,
            ) in zip(
                subject_params,
                subject_head_grads,
            ):

                if grad is not None:

                    param.grad = (
                        subject_head_weight
                        *
                        grad
                    ).detach().clone()


            # =================================================
            # OPTIMIZER
            # =================================================

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )


            optimizer.step()


            # =================================================
            # METRICS
            # =================================================

            n = y.size(0)

            train_total += n


            class_preds = (
                class_logits.argmax(
                    dim=1
                )
            )

            subject_preds = (
                subject_logits.argmax(
                    dim=1
                )
            )


            class_correct += (
                class_preds == y
            ).sum().item()


            subject_correct += (
                subject_preds
                ==
                subject_ids
            ).sum().item()


            ce_sum += (
                loss_ce.item()
                *
                n
            )

            sc_sum += (
                loss_sc.item()
                *
                n
            )

            adv_sum += (
                loss_subject.item()
                *
                n
            )


        # ====================================================
        # TRAIN METRICS
        # ====================================================

        train_accuracy = (
            class_correct
            /
            max(train_total, 1)
        )

        subject_accuracy = (
            subject_correct
            /
            max(train_total, 1)
        )


        train_ce_loss = (
            ce_sum
            /
            max(train_total, 1)
        )

        train_sc_loss = (
            sc_sum
            /
            max(train_total, 1)
        )

        train_adv_loss = (
            adv_sum
            /
            max(train_total, 1)
        )


        conflict_rate = (
            conflict_batches
            /
            max(
                total_batches,
                1,
            )
        )


        mean_cosine = (
            cosine_sum
            /
            max(
                total_batches,
                1,
            )
        )


        # ====================================================
        # VALIDATION
        #
        # Здесь используем обычный model.forward().
        # ====================================================

        model.eval()

        val_true = []
        val_pred = []


        with torch.no_grad():

            for X, y, _ in val_loader:

                X = X.to(
                    run_device,
                    non_blocking=True,
                )

                y = y.to(
                    run_device,
                    non_blocking=True,
                )


                out = model(
                    X,
                    lambda_grl=0.0,
                )


                logits = out[
                    "class_logits"
                ]


                preds = (
                    logits.argmax(
                        dim=1
                    )
                )


                val_true.extend(
                    y.cpu().numpy()
                )

                val_pred.extend(
                    preds.cpu().numpy()
                )


        val_accuracy = accuracy_score(
            val_true,
            val_pred,
        )


        scheduler.step(
            val_accuracy
        )


        current_lr = (
            optimizer
            .param_groups[0]["lr"]
        )


        # ====================================================
        # HISTORY
        # ====================================================

        history.append({

            "epoch":
                epoch,

            "train_accuracy":
                train_accuracy,

            "val_accuracy":
                val_accuracy,

            "subject_accuracy":
                subject_accuracy,

            "lambda_grl":
                lambda_grl,

            "conflict_rate":
                conflict_rate,

            "mean_gradient_cosine":
                mean_cosine,

            "ce_loss":
                train_ce_loss,

            "supcon_loss":
                train_sc_loss,

            "subject_loss":
                train_adv_loss,

            "lr":
                current_lr,
        })


        # ====================================================
        # LOG
        # ====================================================

        print(
            f"CONFLICT_GRL | "
            f"ep={epoch:02d} | "
            f"train={train_accuracy:.4f} | "
            f"val={val_accuracy:.4f} | "
            f"SUBJ={subject_accuracy:.4f} | "
            f"GRL={lambda_grl:.4f} | "
            f"conflict={conflict_rate:.2f} | "
            f"cos={mean_cosine:+.3f} | "
            f"CE={train_ce_loss:.3f} | "
            f"SC={train_sc_loss:.3f} | "
            f"ADV={train_adv_loss:.3f}"
        )


        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if val_accuracy > best_val_accuracy:

            best_val_accuracy = (
                val_accuracy
            )

            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0


        else:

            epochs_without_improvement += 1


        if (
            epochs_without_improvement
            >= patience
        ):

            print(
                f"Early stopping "
                f"at epoch {epoch}"
            )

            break


    # ========================================================
    # RESTORE BEST
    # ========================================================

    if best_state is None:

        hook_handle.remove()

        raise RuntimeError(
            "No valid checkpoint produced."
        )


    model.load_state_dict(
        best_state
    )

    model.eval()


    # ========================================================
    # TEST
    # ========================================================

    test_true = []
    test_pred = []


    with torch.no_grad():

        for X, y, _ in test_loader:

            X = X.to(
                run_device,
                non_blocking=True,
            )

            y = y.to(
                run_device,
                non_blocking=True,
            )


            out = model(
                X,
                lambda_grl=0.0,
            )


            logits = out[
                "class_logits"
            ]


            preds = (
                logits.argmax(
                    dim=1
                )
            )


            test_true.extend(
                y.cpu().numpy()
            )

            test_pred.extend(
                preds.cpu().numpy()
            )


    # ========================================================
    # REMOVE HOOK
    # ========================================================

    hook_handle.remove()


    # ========================================================
    # TEST METRICS
    # ========================================================

    test_accuracy = accuracy_score(
        test_true,
        test_pred,
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            test_true,
            test_pred,
        )
    )

    macro_f1 = f1_score(
        test_true,
        test_pred,
        average="macro",
        zero_division=0,
    )

    kappa = cohen_kappa_score(
        test_true,
        test_pred,
    )


    result = {

        "test_subject":
            int(test_subject),

        "best_epoch":
            int(best_epoch),

        "best_val_accuracy":
            float(best_val_accuracy),

        "test_accuracy":
            float(test_accuracy),

        "balanced_accuracy":
            float(balanced_accuracy),

        "macro_f1":
            float(macro_f1),

        "kappa":
            float(kappa),

        "lr":
            float(lr),

        "supcon_weight":
            float(supcon_weight),

        "supcon_temperature":
            float(supcon_temperature),

        "grl_max":
            float(grl_max),

        "grl_warmup_epochs":
            int(grl_warmup_epochs),

        "seed":
            int(seed),
    }


    return (
        result,
        model,
        pd.DataFrame(history),
    )

In [90]:
# ============================================================
# NIGHT PIPELINE
#
# Phase 1:
#   Conflict-aware GRL tuning on 5 development subjects
#
# Phase 2:
#   Automatically choose:
#
#       tuned SupCon
#           OR
#       tuned SupCon + Conflict-aware GRL
#
# Phase 3:
#   Full 109-subject LOSO
# ============================================================


# ============================================================
# FIXED CONFIGURATION
# ============================================================

NIGHT_TRANSFER_LR = 1.5e-3

NIGHT_SUPCON_WEIGHT = 0.10

NIGHT_SUPCON_TEMPERATURE = 0.05


NIGHT_GRL_VALUES = [
    0.02,
    0.05,
    0.10,
]


NIGHT_DEV_SUBJECTS = [
    3,
    9,
    1,
    19,
    2,
]


# GRL must beat tuned SupCon by at least 0.5 pp
# to be selected for the expensive full run.

MIN_GRL_GAIN_PP = 0.50


# ============================================================
# OUTPUT
# ============================================================

NIGHT_ROOT = (
    RESULTS_ROOT
    / "conflict_grl_night"
)

NIGHT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


NIGHT_TUNE_CSV = (
    NIGHT_ROOT
    / "conflict_grl_tuning.csv"
)


NIGHT_FULL_CSV = (
    NIGHT_ROOT
    / "full_109_loso_progress.csv"
)


# ============================================================
# PHASE 1 — GRL SEARCH
# ============================================================

print()
print("=" * 100)
print("PHASE 1 — CONFLICT-AWARE GRL SEARCH")
print("=" * 100)


if NIGHT_TUNE_CSV.exists():

    night_tune_df = pd.read_csv(
        NIGHT_TUNE_CSV
    )

    done = set(
        zip(
            night_tune_df[
                "grl_max"
            ].astype(float),

            night_tune_df[
                "test_subject"
            ].astype(int),
        )
    )

else:

    night_tune_df = pd.DataFrame()

    done = set()


total_tune_runs = (
    len(NIGHT_GRL_VALUES)
    *
    len(NIGHT_DEV_SUBJECTS)
)


run_idx = 0


for grl_max in NIGHT_GRL_VALUES:

    for subject in NIGHT_DEV_SUBJECTS:

        run_idx += 1


        key = (
            float(grl_max),
            int(subject),
        )


        if key in done:

            print(
                f"[{run_idx}/{total_tune_runs}] "
                f"GRL={grl_max:.2f} | "
                f"S{subject:03d} already done"
            )

            continue


        print()
        print("=" * 100)

        print(
            f"[{run_idx}/{total_tune_runs}] "
            f"CONFLICT GRL | "
            f"MAX={grl_max:.2f} | "
            f"S{subject:03d}"
        )

        print("=" * 100)


        result, model, history = (
            train_cho64_conflict_grl_fold(

                test_subject=subject,

                seed=42,

                max_epochs=40,

                patience=8,

                batch_size=64,

                lr=NIGHT_TRANSFER_LR,

                weight_decay=1e-4,

                supcon_weight=
                    NIGHT_SUPCON_WEIGHT,

                supcon_temperature=
                    NIGHT_SUPCON_TEMPERATURE,

                grl_max=grl_max,

                grl_warmup_epochs=5,

                subject_head_weight=1.0,
            )
        )


        night_tune_df = pd.concat(
            [
                night_tune_df,
                pd.DataFrame(
                    [result]
                ),
            ],
            ignore_index=True,
        )


        night_tune_df.to_csv(
            NIGHT_TUNE_CSV,
            index=False,
        )


        done.add(
            key
        )


        print(
            f"\nRESULT | "
            f"GRL={grl_max:.2f} | "
            f"S{subject:03d} | "
            f"TEST="
            f"{result['test_accuracy']*100:.2f}%"
        )


        del model
        del history

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ============================================================
# PHASE 1 SUMMARY
# ============================================================

grl_summary = (
    night_tune_df
    .groupby(
        "grl_max"
    )
    .agg(

        n_subjects=(
            "test_subject",
            "nunique",
        ),

        mean_accuracy=(
            "test_accuracy",
            "mean",
        ),

        median_accuracy=(
            "test_accuracy",
            "median",
        ),

        mean_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),

        mean_macro_f1=(
            "macro_f1",
            "mean",
        ),
    )
    .reset_index()
)


grl_summary = (
    grl_summary[
        grl_summary[
            "n_subjects"
        ] == len(
            NIGHT_DEV_SUBJECTS
        )
    ]
    .copy()
)


grl_summary = (
    grl_summary
    .sort_values(
        [
            "mean_accuracy",
            "median_accuracy",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


if len(grl_summary) == 0:

    raise RuntimeError(
        "No complete GRL configuration."
    )


BEST_NIGHT_GRL = float(
    grl_summary
    .iloc[0]["grl_max"]
)


BEST_NIGHT_GRL_ACC = float(
    grl_summary
    .iloc[0]["mean_accuracy"]
)


print()
print("=" * 100)
print("GRL SEARCH SUMMARY")
print("=" * 100)

display(
    grl_summary.assign(

        mean_accuracy_pct=
            grl_summary[
                "mean_accuracy"
            ] * 100,

        median_accuracy_pct=
            grl_summary[
                "median_accuracy"
            ] * 100,

        balanced_accuracy_pct=
            grl_summary[
                "mean_balanced_accuracy"
            ] * 100,

        macro_f1_pct=
            grl_summary[
                "mean_macro_f1"
            ] * 100,

    )[[
        "grl_max",
        "n_subjects",
        "mean_accuracy_pct",
        "median_accuracy_pct",
        "balanced_accuracy_pct",
        "macro_f1_pct",
    ]]
)


# ============================================================
# GET TUNED SUPCON CONTROL ON SAME 5 SUBJECTS
# ============================================================

control_rows = (
    supcon_tune_df[

        np.isclose(
            supcon_tune_df[
                "supcon_weight"
            ].astype(float),

            NIGHT_SUPCON_WEIGHT,
        )

        &

        np.isclose(
            supcon_tune_df[
                "supcon_temperature"
            ].astype(float),

            NIGHT_SUPCON_TEMPERATURE,
        )

        &

        supcon_tune_df[
            "test_subject"
        ]
        .astype(int)
        .isin(
            NIGHT_DEV_SUBJECTS
        )
    ]
    .copy()
)


if (
    control_rows[
        "test_subject"
    ].nunique()
    ==
    len(NIGHT_DEV_SUBJECTS)
):

    CONTROL_SUPCON_ACC = float(
        control_rows[
            "test_accuracy"
        ].mean()
    )

else:

    # Known result from our completed
    # 5-subject tuned SupCon screening.
    CONTROL_SUPCON_ACC = (
        0.69777778
    )


grl_gain_pp = (
    BEST_NIGHT_GRL_ACC
    -
    CONTROL_SUPCON_ACC
) * 100


print()
print("=" * 100)

print(
    f"Tuned SupCon control : "
    f"{CONTROL_SUPCON_ACC*100:.2f}%"
)

print(
    f"Best Conflict GRL     : "
    f"{BEST_NIGHT_GRL_ACC*100:.2f}%"
)

print(
    f"GRL gain              : "
    f"{grl_gain_pp:+.2f} pp"
)

print("=" * 100)


# ============================================================
# AUTOMATIC METHOD SELECTION
# ============================================================

if grl_gain_pp >= MIN_GRL_GAIN_PP:

    NIGHT_METHOD = (
        "conflict_grl"
    )

    print(
        "\nSELECTED FOR FULL LOSO: "
        "Transfer + tuned SupCon "
        "+ Conflict-aware GRL"
    )

    print(
        f"GRL max = "
        f"{BEST_NIGHT_GRL:.2f}"
    )


else:

    NIGHT_METHOD = (
        "supcon"
    )

    print(
        "\nSELECTED FOR FULL LOSO: "
        "Transfer + tuned SupCon"
    )

    print(
        "Conflict-aware GRL did not "
        "provide sufficient gain."
    )


# ============================================================
# PHASE 2 — FULL 109 SUBJECT LOSO
# ============================================================

print()
print("=" * 100)
print("PHASE 2 — FULL 109-SUBJECT LOSO")
print("=" * 100)

print(
    f"METHOD = {NIGHT_METHOD}"
)


FULL_SUBJECTS = list(
    range(
        1,
        110,
    )
)


# ============================================================
# RESUME
# ============================================================

if NIGHT_FULL_CSV.exists():

    full_df = pd.read_csv(
        NIGHT_FULL_CSV
    )


    same_method = (
        full_df[
            "night_method"
        ]
        ==
        NIGHT_METHOD
    )


    full_df = (
        full_df[
            same_method
        ]
        .copy()
    )


    done_subjects = set(
        full_df[
            "test_subject"
        ]
        .astype(int)
        .tolist()
    )


    print(
        f"Resume: "
        f"{len(done_subjects)}/109 "
        f"subjects completed"
    )


else:

    full_df = pd.DataFrame()

    done_subjects = set()


# ============================================================
# FULL LOSO LOOP
# ============================================================

for idx, subject in enumerate(
    FULL_SUBJECTS,
    start=1,
):

    if subject in done_subjects:

        print(
            f"[{idx}/109] "
            f"S{subject:03d} already done"
        )

        continue


    print()
    print("=" * 100)

    print(
        f"[{idx}/109] "
        f"{NIGHT_METHOD.upper()} | "
        f"TEST SUBJECT=S{subject:03d}"
    )

    print("=" * 100)


    # ========================================================
    # SELECTED: CONFLICT GRL
    # ========================================================

    if NIGHT_METHOD == "conflict_grl":

        result, model, history = (
            train_cho64_conflict_grl_fold(

                test_subject=subject,

                seed=42,

                max_epochs=40,

                patience=8,

                batch_size=64,

                lr=NIGHT_TRANSFER_LR,

                weight_decay=1e-4,

                supcon_weight=
                    NIGHT_SUPCON_WEIGHT,

                supcon_temperature=
                    NIGHT_SUPCON_TEMPERATURE,

                grl_max=
                    BEST_NIGHT_GRL,

                grl_warmup_epochs=5,

                subject_head_weight=1.0,
            )
        )


    # ========================================================
    # SELECTED: TUNED SUPCON ONLY
    # ========================================================

    else:

        result, model, history = (
            train_cho64_ablation_fold(

                test_subject=subject,

                mode="supcon",

                seed=42,

                max_epochs=40,

                patience=8,

                batch_size=64,

                lr=NIGHT_TRANSFER_LR,

                weight_decay=1e-4,

                supcon_weight=
                    NIGHT_SUPCON_WEIGHT,

                supcon_temperature=
                    NIGHT_SUPCON_TEMPERATURE,
            )
        )


    # ========================================================
    # SAVE RESULT
    # ========================================================

    result[
        "night_method"
    ] = NIGHT_METHOD


    result[
        "selected_grl_max"
    ] = (
        BEST_NIGHT_GRL
        if NIGHT_METHOD
        ==
        "conflict_grl"
        else
        0.0
    )


    full_df = pd.concat(
        [
            full_df,
            pd.DataFrame(
                [result]
            ),
        ],
        ignore_index=True,
    )


    full_df.to_csv(
        NIGHT_FULL_CSV,
        index=False,
    )


    done_subjects.add(
        int(subject)
    )


    current_mean = (
        full_df[
            "test_accuracy"
        ].mean()
        *
        100
    )


    print()
    print(
        f"SUBJECT RESULT = "
        f"{result['test_accuracy']*100:.2f}%"
    )

    print(
        f"RUNNING MEAN   = "
        f"{current_mean:.2f}%"
    )

    print(
        f"PROGRESS       = "
        f"{len(done_subjects)}/109"
    )


    del model
    del history

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 100)
print("FULL LOSO FINISHED")
print("=" * 100)


final_summary = pd.DataFrame({

    "method": [
        NIGHT_METHOD
    ],

    "n_subjects": [
        full_df[
            "test_subject"
        ].nunique()
    ],

    "mean_accuracy_pct": [
        full_df[
            "test_accuracy"
        ].mean()
        *
        100
    ],

    "median_accuracy_pct": [
        full_df[
            "test_accuracy"
        ].median()
        *
        100
    ],

    "std_accuracy_pct": [
        full_df[
            "test_accuracy"
        ].std()
        *
        100
    ],

    "balanced_accuracy_pct": [
        full_df[
            "balanced_accuracy"
        ].mean()
        *
        100
    ],

    "macro_f1_pct": [
        full_df[
            "macro_f1"
        ].mean()
        *
        100
    ],

    "mean_kappa": [
        full_df[
            "kappa"
        ].mean()
    ],
})


display(
    final_summary
)


FINAL_ACC = (
    full_df[
        "test_accuracy"
    ].mean()
    *
    100
)


CCST64_ACC = 73.65


print()
print(
    f"Our final accuracy : "
    f"{FINAL_ACC:.2f}%"
)

print(
    f"CCST64             : "
    f"{CCST64_ACC:.2f}%"
)

print(
    f"Difference         : "
    f"{FINAL_ACC-CCST64_ACC:+.2f} pp"
)

print("=" * 100)


PHASE 1 — CONFLICT-AWARE GRL SEARCH

[1/15] CONFLICT GRL | MAX=0.02 | S003
CONFLICT_GRL | ep=01 | train=0.5865 | val=0.6742 | SUBJ=0.0246 | GRL=0.0000 | conflict=0.00 | cos=+0.000 | CE=0.681 | SC=4.196 | ADV=4.510
CONFLICT_GRL | ep=02 | train=0.6208 | val=0.6844 | SUBJ=0.0342 | GRL=0.0000 | conflict=0.00 | cos=+0.000 | CE=0.640 | SC=4.138 | ADV=4.139
CONFLICT_GRL | ep=03 | train=0.6308 | val=0.6803 | SUBJ=0.0545 | GRL=0.0000 | conflict=0.00 | cos=+0.000 | CE=0.633 | SC=4.135 | ADV=3.956
CONFLICT_GRL | ep=04 | train=0.6255 | val=0.7029 | SUBJ=0.0712 | GRL=0.0000 | conflict=0.00 | cos=+0.000 | CE=0.627 | SC=4.134 | ADV=3.840
CONFLICT_GRL | ep=05 | train=0.6385 | val=0.7193 | SUBJ=0.0789 | GRL=0.0000 | conflict=0.00 | cos=+0.000 | CE=0.620 | SC=4.133 | ADV=3.760
CONFLICT_GRL | ep=06 | train=0.6513 | val=0.6373 | SUBJ=0.0869 | GRL=0.0028 | conflict=0.51 | cos=-0.028 | CE=0.620 | SC=4.132 | ADV=3.723
CONFLICT_GRL | ep=07 | train=0.6458 | val=0.7090 | SUBJ=0.1013 | GRL=0.0056 | conflict=0.7

,grl_max,n_subjects,mean_accuracy_pct,median_accuracy_pct,balanced_accuracy_pct,macro_f1_pct
0,0.05,5,65.333333,71.111111,65.833804,64.335665
1,0.02,5,64.888889,71.111111,65.379258,63.868244
2,0.10,5,64.444444,68.888889,64.807312,63.570243



Tuned SupCon control : 69.78%
Best Conflict GRL     : 65.33%
GRL gain              : -4.44 pp

SELECTED FOR FULL LOSO: Transfer + tuned SupCon
Conflict-aware GRL did not provide sufficient gain.

PHASE 2 — FULL 109-SUBJECT LOSO
METHOD = supcon

[1/109] SUPCON | TEST SUBJECT=S001
SUPCON       | ep=01 | train=0.5909 | val=0.6537 | CE=0.679 | SC=4.191 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=02 | train=0.6105 | val=0.7008 | CE=0.649 | SC=4.139 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=03 | train=0.6178 | val=0.6721 | CE=0.642 | SC=4.137 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=04 | train=0.6408 | val=0.6783 | CE=0.629 | SC=4.133 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=05 | train=0.6285 | val=0.6803 | CE=0.634 | SC=4.134 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=06 | train=0.6436 | val=0.6475 | CE=0.620 | SC=4.132 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep

,method,n_subjects,mean_accuracy_pct,median_accuracy_pct,std_accuracy_pct,balanced_accuracy_pct,macro_f1_pct,mean_kappa
0,supcon,109,69.177612,68.888889,13.185277,69.189541,67.020558,0.383762



Our final accuracy : 69.18%
CCST64             : 73.65%
Difference         : -4.47 pp


В полном эксперименте на PhysioNet с 64 каналами была проведена строгая LOSO-оценка на всех 109 испытуемых. Модель EEGNet с предварительным обучением на Cho2017, последующим transfer learning и настроенным Supervised Contrastive Learning показала среднюю точность 69,18%, медианную точность 68,89% и balanced accuracy 69,19%. Macro-F1 составил 67,02%, Cohen’s κ — 0,384.

Полученный результат оказался на 4,47 п.п. ниже воспроизведённого на том же протоколе CCST64 (73,65%). При этом стандартное отклонение точности между испытуемыми составило 13,19 п.п., а для части испытуемых точность находилась около уровня случайного угадывания.

Таким образом, основным ограничением текущего метода является не только средняя точность классификации, но и высокая межсубъектная вариативность: модель хорошо переносится на часть новых испытуемых, но существенно теряет качество на других.

Эксперименты с GRL, включая ослабленный, адаптивный и conflict-aware варианты, не устранили эту проблему и в среднем ухудшали качество. Это указывает на то, что прямое удаление информации о субъекте может одновременно удалять признаки, полезные для распознавания моторного воображения.

Поэтому следующим экспериментом выбран более направленный вариант contrastive learning: вместо подавления индивидуальных особенностей предлагается явно сближать представления одного класса, принадлежащие разным испытуемым. Такой подход сохраняет текущую архитектуру EEGNet, transfer learning и общую схему SupCon, но непосредственно оптимизирует межсубъектную переносимость признаков.

In [91]:
# ============================================================
# CROSS-SUBJECT SUPERVISED CONTRASTIVE LOSS
#
# Positive:
#   same class
#   AND different subject
#
# Examples:
#
#   Left S001 <-> Left S017    positive
#   Left S001 <-> Left S001    NOT positive
#   Left S001 <-> Right S017   NOT positive
#
# ============================================================


class CrossSubjectSupConLoss(nn.Module):

    def __init__(
        self,
        temperature=0.05,
    ):
        super().__init__()

        self.temperature = temperature


    def forward(
        self,
        features,
        labels,
        subject_ids,
    ):

        features = torch.nn.functional.normalize(
            features,
            dim=1,
        )

        labels = labels.view(-1)

        subject_ids = subject_ids.view(-1)


        batch_size = features.size(0)


        # ----------------------------------------------------
        # COSINE SIMILARITY
        # ----------------------------------------------------

        logits = (
            torch.matmul(
                features,
                features.T,
            )
            /
            self.temperature
        )


        # ----------------------------------------------------
        # REMOVE SELF-CONTRAST
        # ----------------------------------------------------

        self_mask = torch.eye(
            batch_size,
            dtype=torch.bool,
            device=features.device,
        )


        # ----------------------------------------------------
        # SAME CLASS
        # ----------------------------------------------------

        same_class = (
            labels[:, None]
            ==
            labels[None, :]
        )


        # ----------------------------------------------------
        # DIFFERENT SUBJECT
        # ----------------------------------------------------

        different_subject = (
            subject_ids[:, None]
            !=
            subject_ids[None, :]
        )


        # ----------------------------------------------------
        # CROSS-SUBJECT POSITIVES
        # ----------------------------------------------------

        positive_mask = (
            same_class
            &
            different_subject
            &
            (~self_mask)
        )


        # ----------------------------------------------------
        # ALL OTHER SAMPLES EXCEPT SELF ARE IN DENOMINATOR
        # ----------------------------------------------------

        valid_mask = ~self_mask


        # Numerical stability
        logits = (
            logits
            -
            logits.max(
                dim=1,
                keepdim=True,
            ).values.detach()
        )


        masked_logits = logits.masked_fill(
            ~valid_mask,
            float("-inf"),
        )


        log_denominator = torch.logsumexp(
            masked_logits,
            dim=1,
            keepdim=True,
        )


        log_prob = (
            logits
            -
            log_denominator
        )


        positive_count = positive_mask.sum(
            dim=1
        )


        valid_anchor = (
            positive_count > 0
        )


        if not valid_anchor.any():

            # Keep graph connected
            loss = (
                features.sum()
                *
                0.0
            )

            valid_anchor_fraction = 0.0
            mean_positive_count = 0.0

            return (
                loss,
                valid_anchor_fraction,
                mean_positive_count,
            )


        positive_log_prob = (
            log_prob
            *
            positive_mask.float()
        ).sum(dim=1)


        mean_positive_log_prob = (
            positive_log_prob[
                valid_anchor
            ]
            /
            positive_count[
                valid_anchor
            ].float()
        )


        loss = (
            -mean_positive_log_prob.mean()
        )


        valid_anchor_fraction = float(
            valid_anchor.float()
            .mean()
            .detach()
            .cpu()
            .item()
        )


        mean_positive_count = float(
            positive_count[
                valid_anchor
            ]
            .float()
            .mean()
            .detach()
            .cpu()
            .item()
        )


        return (
            loss,
            valid_anchor_fraction,
            mean_positive_count,
        )

In [92]:
# ============================================================
# SUBJECT-AWARE BATCH SAMPLER
#
# Default:
#   16 subjects per batch
#   2 Left + 2 Right trials from each subject
#
# Batch size:
#   16 * 2 classes * 2 trials = 64
# ============================================================


class SubjectAwareBatchSampler:

    def __init__(
        self,
        labels,
        subject_ids,
        subjects_per_batch=16,
        samples_per_class=2,
        seed=42,
    ):

        self.labels = np.asarray(
            labels
        ).astype(int)

        self.subject_ids = np.asarray(
            subject_ids
        ).astype(int)

        self.subjects_per_batch = (
            subjects_per_batch
        )

        self.samples_per_class = (
            samples_per_class
        )

        self.seed = seed


        self.subjects = np.unique(
            self.subject_ids
        )


        self.index_map = {}


        for subject in self.subjects:

            for label in [0, 1]:

                indices = np.where(
                    (
                        self.subject_ids
                        ==
                        subject
                    )
                    &
                    (
                        self.labels
                        ==
                        label
                    )
                )[0]


                self.index_map[
                    (
                        int(subject),
                        int(label),
                    )
                ] = indices


        # Only subjects containing both classes
        self.valid_subjects = []


        for subject in self.subjects:

            has_left = (
                len(
                    self.index_map[
                        (
                            int(subject),
                            0,
                        )
                    ]
                )
                >
                0
            )

            has_right = (
                len(
                    self.index_map[
                        (
                            int(subject),
                            1,
                        )
                    ]
                )
                >
                0
            )


            if (
                has_left
                and
                has_right
            ):

                self.valid_subjects.append(
                    int(subject)
                )


        self.batch_size = (
            self.subjects_per_batch
            *
            2
            *
            self.samples_per_class
        )


        self.n_batches = max(
            1,
            len(self.labels)
            //
            self.batch_size,
        )


        self.epoch = 0


    def __len__(self):

        return self.n_batches


    def __iter__(self):

        rng = np.random.default_rng(
            self.seed
            +
            self.epoch
        )

        self.epoch += 1


        for _ in range(
            self.n_batches
        ):

            replace_subjects = (
                len(
                    self.valid_subjects
                )
                <
                self.subjects_per_batch
            )


            selected_subjects = (
                rng.choice(
                    self.valid_subjects,
                    size=self.subjects_per_batch,
                    replace=replace_subjects,
                )
            )


            batch_indices = []


            for subject in selected_subjects:

                for label in [0, 1]:

                    pool = (
                        self.index_map[
                            (
                                int(subject),
                                label,
                            )
                        ]
                    )


                    replace_trials = (
                        len(pool)
                        <
                        self.samples_per_class
                    )


                    selected_trials = (
                        rng.choice(
                            pool,
                            size=self.samples_per_class,
                            replace=replace_trials,
                        )
                    )


                    batch_indices.extend(
                        selected_trials.tolist()
                    )


            rng.shuffle(
                batch_indices
            )


            yield batch_indices

In [93]:
# ============================================================
# CROSS-SUBJECT SUPCON TRAINING
# ============================================================


def train_cho64_cross_subject_supcon_fold(
    test_subject,
    seed=42,
    max_epochs=40,
    patience=8,
    batch_size=64,
    lr=1.5e-3,
    weight_decay=1e-4,
    supcon_weight=0.10,
    supcon_temperature=0.05,
    subject_aware_sampling=False,
):

    set_seed(seed)


    run_device = globals().get(
        "device",
        torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "mps"
            if torch.backends.mps.is_available()
            else "cpu"
        ),
    )


    # ========================================================
    # DATA
    # ========================================================

    fold = prepare_phys64_fold_supcon_grl(
        test_subject=test_subject,
        seed=seed,
    )


    train_dataset = Phys64SubjectDataset(
        fold["X_train"],
        fold["y_train"],
        fold["s_train"],
        augment=True,
    )


    val_dataset = Phys64SubjectDataset(
        fold["X_val"],
        fold["y_val"],
        fold["s_val"],
        augment=False,
    )


    s_test_dummy = np.zeros(
        len(
            fold["y_test"]
        ),
        dtype=np.int64,
    )


    test_dataset = Phys64SubjectDataset(
        fold["X_test"],
        fold["y_test"],
        s_test_dummy,
        augment=False,
    )


    # ========================================================
    # TRAIN LOADER
    # ========================================================

    if subject_aware_sampling:

        batch_sampler = (
            SubjectAwareBatchSampler(
                labels=fold["y_train"],
                subject_ids=fold["s_train"],
                subjects_per_batch=16,
                samples_per_class=2,
                seed=seed,
            )
        )


        train_loader = DataLoader(
            train_dataset,
            batch_sampler=batch_sampler,
            num_workers=0,
        )


    else:

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            drop_last=False,
        )


    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        drop_last=False,
    )


    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        drop_last=False,
    )


    # ========================================================
    # MODEL
    # ========================================================

    model = make_cho64_supcon_grl_model(
        n_subjects=
            fold[
                "n_source_subjects"
            ]
    ).to(
        run_device
    )


    # ========================================================
    # LOSSES
    # ========================================================

    ce_loss_fn = (
        nn.CrossEntropyLoss()
    )


    cross_subject_supcon = (
        CrossSubjectSupConLoss(
            temperature=
                supcon_temperature
        )
    )


    # ========================================================
    # OPTIMIZER
    # ========================================================

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )


    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=5,
        )
    )


    # ========================================================
    # TRAINING STATE
    # ========================================================

    best_val_accuracy = -np.inf
    best_epoch = 0
    best_state = None

    epochs_without_improvement = 0

    history = []


    # ========================================================
    # TRAIN
    # ========================================================

    for epoch in range(
        1,
        max_epochs + 1,
    ):

        model.train()


        train_total = 0
        train_correct = 0

        ce_sum = 0.0
        sc_sum = 0.0

        valid_anchor_sum = 0.0
        positive_count_sum = 0.0

        n_batches = 0


        for (
            X,
            y,
            subject_ids,
        ) in train_loader:

            X = X.to(
                run_device,
                non_blocking=True,
            )

            y = y.to(
                run_device,
                non_blocking=True,
            )

            subject_ids = (
                subject_ids.to(
                    run_device,
                    non_blocking=True,
                )
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            # =================================================
            # FORWARD
            # =================================================

            out = model(
                X,
                lambda_grl=0.0,
            )


            class_logits = (
                out[
                    "class_logits"
                ]
            )


            projection = (
                out[
                    "projection"
                ]
            )


            # =================================================
            # LOSSES
            # =================================================

            loss_ce = ce_loss_fn(
                class_logits,
                y,
            )


            (
                loss_sc,
                valid_anchor_fraction,
                mean_positive_count,
            ) = cross_subject_supcon(
                projection,
                y,
                subject_ids,
            )


            loss = (
                loss_ce
                +
                supcon_weight
                *
                loss_sc
            )


            # =================================================
            # BACKWARD
            # =================================================

            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )


            optimizer.step()


            # =================================================
            # METRICS
            # =================================================

            n = y.size(0)


            preds = (
                class_logits.argmax(
                    dim=1
                )
            )


            train_correct += (
                preds
                ==
                y
            ).sum().item()


            train_total += n


            ce_sum += (
                loss_ce.item()
                *
                n
            )


            sc_sum += (
                loss_sc.item()
                *
                n
            )


            valid_anchor_sum += (
                valid_anchor_fraction
            )


            positive_count_sum += (
                mean_positive_count
            )


            n_batches += 1


        # ====================================================
        # TRAIN SUMMARY
        # ====================================================

        train_accuracy = (
            train_correct
            /
            train_total
        )


        train_ce = (
            ce_sum
            /
            train_total
        )


        train_sc = (
            sc_sum
            /
            train_total
        )


        valid_anchor_fraction = (
            valid_anchor_sum
            /
            max(
                n_batches,
                1,
            )
        )


        mean_positive_count = (
            positive_count_sum
            /
            max(
                n_batches,
                1,
            )
        )


        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()

        val_true = []
        val_pred = []


        with torch.no_grad():

            for X, y, _ in val_loader:

                X = X.to(
                    run_device,
                    non_blocking=True,
                )

                y = y.to(
                    run_device,
                    non_blocking=True,
                )


                out = model(
                    X,
                    lambda_grl=0.0,
                )


                logits = (
                    out[
                        "class_logits"
                    ]
                )


                preds = logits.argmax(
                    dim=1
                )


                val_true.extend(
                    y
                    .cpu()
                    .numpy()
                )

                val_pred.extend(
                    preds
                    .cpu()
                    .numpy()
                )


        val_accuracy = accuracy_score(
            val_true,
            val_pred,
        )


        scheduler.step(
            val_accuracy
        )


        current_lr = (
            optimizer
            .param_groups[0]["lr"]
        )


        # ====================================================
        # LOG
        # ====================================================

        sampling_name = (
            "AWARE"
            if subject_aware_sampling
            else "RANDOM"
        )


        print(
            f"XSUPCON-{sampling_name} | "
            f"ep={epoch:02d} | "
            f"train={train_accuracy:.4f} | "
            f"val={val_accuracy:.4f} | "
            f"CE={train_ce:.3f} | "
            f"XSC={train_sc:.3f} | "
            f"anchors={valid_anchor_fraction:.2f} | "
            f"pos={mean_positive_count:.1f} | "
            f"lr={current_lr:.6f}"
        )


        history.append({

            "epoch":
                epoch,

            "train_accuracy":
                train_accuracy,

            "val_accuracy":
                val_accuracy,

            "ce_loss":
                train_ce,

            "cross_subject_supcon_loss":
                train_sc,

            "valid_anchor_fraction":
                valid_anchor_fraction,

            "mean_positive_count":
                mean_positive_count,

            "lr":
                current_lr,
        })


        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if (
            val_accuracy
            >
            best_val_accuracy
        ):

            best_val_accuracy = (
                val_accuracy
            )

            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0


        else:

            epochs_without_improvement += 1


        if (
            epochs_without_improvement
            >=
            patience
        ):

            print(
                f"Early stopping "
                f"at epoch {epoch}"
            )

            break


    # ========================================================
    # RESTORE BEST MODEL
    # ========================================================

    if best_state is None:

        raise RuntimeError(
            "No valid checkpoint produced."
        )


    model.load_state_dict(
        best_state
    )

    model.eval()


    # ========================================================
    # TEST
    # ========================================================

    test_true = []
    test_pred = []


    with torch.no_grad():

        for X, y, _ in test_loader:

            X = X.to(
                run_device,
                non_blocking=True,
            )

            y = y.to(
                run_device,
                non_blocking=True,
            )


            out = model(
                X,
                lambda_grl=0.0,
            )


            logits = (
                out[
                    "class_logits"
                ]
            )


            preds = logits.argmax(
                dim=1
            )


            test_true.extend(
                y
                .cpu()
                .numpy()
            )

            test_pred.extend(
                preds
                .cpu()
                .numpy()
            )


    # ========================================================
    # METRICS
    # ========================================================

    test_accuracy = accuracy_score(
        test_true,
        test_pred,
    )


    balanced_accuracy = (
        balanced_accuracy_score(
            test_true,
            test_pred,
        )
    )


    macro_f1 = f1_score(
        test_true,
        test_pred,
        average="macro",
        zero_division=0,
    )


    kappa = cohen_kappa_score(
        test_true,
        test_pred,
    )


    result = {

        "test_subject":
            int(test_subject),

        "method":
            (
                "cross_subject_supcon_aware"
                if subject_aware_sampling
                else
                "cross_subject_supcon"
            ),

        "best_epoch":
            int(best_epoch),

        "best_val_accuracy":
            float(best_val_accuracy),

        "test_accuracy":
            float(test_accuracy),

        "balanced_accuracy":
            float(balanced_accuracy),

        "macro_f1":
            float(macro_f1),

        "kappa":
            float(kappa),

        "lr":
            float(lr),

        "supcon_weight":
            float(supcon_weight),

        "supcon_temperature":
            float(supcon_temperature),

        "subject_aware_sampling":
            bool(
                subject_aware_sampling
            ),

        "seed":
            int(seed),
    }


    return (
        result,
        model,
        pd.DataFrame(
            history
        ),
    )

In [94]:
# ============================================================
# 10 WORST SUBJECTS FROM COMPLETED 109-SUBJECT LOSO
# ============================================================


WORST_N = 10


worst_baseline_df = (
    full_df
    .sort_values(
        "test_accuracy",
        ascending=True,
    )
    .head(
        WORST_N
    )
    [[
        "test_subject",
        "test_accuracy",
        "balanced_accuracy",
        "macro_f1",
    ]]
    .copy()
)


worst_baseline_df[
    "test_subject"
] = (
    worst_baseline_df[
        "test_subject"
    ]
    .astype(int)
)


WORST_SUBJECTS = (
    worst_baseline_df[
        "test_subject"
    ]
    .tolist()
)


print(
    "Worst subjects:",
    WORST_SUBJECTS,
)


display(
    worst_baseline_df.assign(
        accuracy_pct=
            worst_baseline_df[
                "test_accuracy"
            ] * 100
    )
)


print(
    "\nBaseline worst-10 mean:",
    f"{worst_baseline_df['test_accuracy'].mean()*100:.2f}%"
)

Worst subjects: [99, 24, 87, 3, 109, 67, 5, 88, 37, 69]


,test_subject,test_accuracy,balanced_accuracy,macro_f1,accuracy_pct
98,99,0.400000,0.405138,0.368831,40.000000
23,24,0.466667,0.477273,0.318182,46.666667
86,87,0.466667,0.460474,0.418103,46.666667
2,3,0.466667,0.468379,0.464286,46.666667
108,109,0.488889,0.500000,0.328358,48.888889
66,67,0.488889,0.500000,0.328358,48.888889
4,5,0.488889,0.464286,0.391534,48.888889
87,88,0.508772,0.510468,0.504963,50.877193
36,37,0.511111,0.500000,0.338235,51.111111
68,69,0.533333,0.556548,0.483324,53.333333



Baseline worst-10 mean: 48.20%


In [95]:
# ============================================================
# HARD-SUBJECT EXPERIMENT
#
# Baseline:
#   existing tuned SupCon result
#
# Variant 1:
#   Cross-subject SupCon + random batches
#
# Variant 2:
#   Cross-subject SupCon + subject-aware batches
# ============================================================


XSUPCON_ROOT = (
    RESULTS_ROOT
    /
    "cross_subject_supcon_hard_subjects"
)


XSUPCON_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


XSUPCON_CSV = (
    XSUPCON_ROOT
    /
    "hard_subject_experiment.csv"
)


if XSUPCON_CSV.exists():

    xsupcon_results = pd.read_csv(
        XSUPCON_CSV
    )

else:

    xsupcon_results = pd.DataFrame()


EXPERIMENT_VARIANTS = [

    {
        "name":
            "cross_subject_supcon",

        "subject_aware_sampling":
            False,
    },

    {
        "name":
            "cross_subject_supcon_aware",

        "subject_aware_sampling":
            True,
    },
]


done = set()


if len(
    xsupcon_results
) > 0:

    done = set(
        zip(
            xsupcon_results[
                "method"
            ].astype(str),

            xsupcon_results[
                "test_subject"
            ].astype(int),
        )
    )


total_runs = (
    len(
        WORST_SUBJECTS
    )
    *
    len(
        EXPERIMENT_VARIANTS
    )
)


run_idx = 0


for variant in EXPERIMENT_VARIANTS:

    for subject in WORST_SUBJECTS:

        run_idx += 1


        key = (
            variant[
                "name"
            ],
            int(subject),
        )


        if key in done:

            print(
                f"[{run_idx}/{total_runs}] "
                f"{variant['name']} | "
                f"S{subject:03d} already done"
            )

            continue


        print()
        print("=" * 100)

        print(
            f"[{run_idx}/{total_runs}] "
            f"{variant['name'].upper()} | "
            f"TEST SUBJECT=S{subject:03d}"
        )

        print("=" * 100)


        result, model, history = (
            train_cho64_cross_subject_supcon_fold(

                test_subject=subject,

                seed=42,

                max_epochs=40,

                patience=8,

                batch_size=64,

                lr=1.5e-3,

                weight_decay=1e-4,

                supcon_weight=0.10,

                supcon_temperature=0.05,

                subject_aware_sampling=
                    variant[
                        "subject_aware_sampling"
                    ],
            )
        )


        xsupcon_results = pd.concat(
            [
                xsupcon_results,
                pd.DataFrame(
                    [result]
                ),
            ],
            ignore_index=True,
        )


        xsupcon_results.to_csv(
            XSUPCON_CSV,
            index=False,
        )


        done.add(
            key
        )


        baseline_acc = float(
            worst_baseline_df.loc[
                worst_baseline_df[
                    "test_subject"
                ]
                ==
                subject,

                "test_accuracy",
            ]
            .iloc[0]
        )


        delta_pp = (
            result[
                "test_accuracy"
            ]
            -
            baseline_acc
        ) * 100


        print()

        print(
            f"BASELINE = "
            f"{baseline_acc*100:.2f}%"
        )

        print(
            f"NEW      = "
            f"{result['test_accuracy']*100:.2f}%"
        )

        print(
            f"DELTA    = "
            f"{delta_pp:+.2f} pp"
        )


        del model
        del history

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


[1/20] CROSS_SUBJECT_SUPCON | TEST SUBJECT=S099
XSUPCON-RANDOM | ep=01 | train=0.5954 | val=0.5840 | CE=0.674 | XSC=4.196 | anchors=1.00 | pos=31.0 | lr=0.001500
XSUPCON-RANDOM | ep=02 | train=0.6201 | val=0.6148 | CE=0.643 | XSC=4.138 | anchors=1.00 | pos=30.9 | lr=0.001500
XSUPCON-RANDOM | ep=03 | train=0.6333 | val=0.6025 | CE=0.629 | XSC=4.136 | anchors=1.00 | pos=31.1 | lr=0.001500
XSUPCON-RANDOM | ep=04 | train=0.6328 | val=0.6250 | CE=0.632 | XSC=4.135 | anchors=1.00 | pos=30.9 | lr=0.001500
XSUPCON-RANDOM | ep=05 | train=0.6353 | val=0.6496 | CE=0.625 | XSC=4.133 | anchors=1.00 | pos=31.2 | lr=0.001500
XSUPCON-RANDOM | ep=06 | train=0.6406 | val=0.6270 | CE=0.627 | XSC=4.133 | anchors=1.00 | pos=31.0 | lr=0.001500
XSUPCON-RANDOM | ep=07 | train=0.6426 | val=0.6455 | CE=0.616 | XSC=4.131 | anchors=1.00 | pos=31.1 | lr=0.001500
XSUPCON-RANDOM | ep=08 | train=0.6499 | val=0.6619 | CE=0.609 | XSC=4.130 | anchors=1.00 | pos=31.0 | lr=0.001500
XSUPCON-RANDOM | ep=09 | train=0.6509 |

In [96]:
# ============================================================
# COMPARE HARD-SUBJECT RESULTS
# ============================================================


baseline_compare = (
    worst_baseline_df[
        [
            "test_subject",
            "test_accuracy",
        ]
    ]
    .rename(
        columns={
            "test_accuracy":
                "baseline_supcon"
        }
    )
)


random_compare = (
    xsupcon_results[
        xsupcon_results[
            "method"
        ]
        ==
        "cross_subject_supcon"
    ]
    [
        [
            "test_subject",
            "test_accuracy",
        ]
    ]
    .rename(
        columns={
            "test_accuracy":
                "cross_subject_supcon"
        }
    )
)


aware_compare = (
    xsupcon_results[
        xsupcon_results[
            "method"
        ]
        ==
        "cross_subject_supcon_aware"
    ]
    [
        [
            "test_subject",
            "test_accuracy",
        ]
    ]
    .rename(
        columns={
            "test_accuracy":
                "cross_subject_aware"
        }
    )
)


comparison = (
    baseline_compare
    .merge(
        random_compare,
        on="test_subject",
        how="left",
    )
    .merge(
        aware_compare,
        on="test_subject",
        how="left",
    )
)


comparison[
    "delta_random_pp"
] = (
    comparison[
        "cross_subject_supcon"
    ]
    -
    comparison[
        "baseline_supcon"
    ]
) * 100


comparison[
    "delta_aware_pp"
] = (
    comparison[
        "cross_subject_aware"
    ]
    -
    comparison[
        "baseline_supcon"
    ]
) * 100


comparison_display = (
    comparison.copy()
)


for column in [
    "baseline_supcon",
    "cross_subject_supcon",
    "cross_subject_aware",
]:

    comparison_display[
        column
    ] = (
        comparison_display[
            column
        ]
        *
        100
    )


display(
    comparison_display
    .sort_values(
        "baseline_supcon"
    )
)


print()
print("=" * 80)

print(
    "BASELINE WORST-10 MEAN      : "
    f"{comparison['baseline_supcon'].mean()*100:.2f}%"
)

print(
    "CROSS-SUBJECT SUPCON        : "
    f"{comparison['cross_subject_supcon'].mean()*100:.2f}%"
)

print(
    "CROSS-SUBJECT + AWARE       : "
    f"{comparison['cross_subject_aware'].mean()*100:.2f}%"
)


print()

print(
    "DELTA X-SUPCON              : "
    f"{(
        comparison['cross_subject_supcon'].mean()
        -
        comparison['baseline_supcon'].mean()
    )*100:+.2f} pp"
)

print(
    "DELTA X-SUPCON + AWARE      : "
    f"{(
        comparison['cross_subject_aware'].mean()
        -
        comparison['baseline_supcon'].mean()
    )*100:+.2f} pp"
)


print()

print(
    "SUBJECTS < 55% BASELINE     :",
    int(
        (
            comparison[
                "baseline_supcon"
            ]
            <
            0.55
        ).sum()
    ),
)


print(
    "SUBJECTS < 55% X-SUPCON     :",
    int(
        (
            comparison[
                "cross_subject_supcon"
            ]
            <
            0.55
        ).sum()
    ),
)


print(
    "SUBJECTS < 55% XSC+AWARE    :",
    int(
        (
            comparison[
                "cross_subject_aware"
            ]
            <
            0.55
        ).sum()
    ),
)

print("=" * 80)

,test_subject,baseline_supcon,cross_subject_supcon,cross_subject_aware,delta_random_pp,delta_aware_pp
0,99,40.000000,46.666667,42.222222,6.666667,2.222222
1,24,46.666667,46.666667,53.333333,0.000000,6.666667
2,87,46.666667,46.666667,51.111111,0.000000,4.444444
3,3,46.666667,42.222222,48.888889,-4.444444,2.222222
4,109,48.888889,48.888889,48.888889,0.000000,0.000000
5,67,48.888889,48.888889,51.111111,0.000000,2.222222
6,5,48.888889,57.777778,51.111111,8.888889,2.222222
7,88,50.877193,50.877193,45.614035,0.000000,-5.263158
8,37,51.111111,51.111111,64.444444,0.000000,13.333333
9,69,53.333333,57.777778,64.444444,4.444444,11.111111



BASELINE WORST-10 MEAN      : 48.20%
CROSS-SUBJECT SUPCON        : 49.75%
CROSS-SUBJECT + AWARE       : 52.12%

DELTA X-SUPCON              : +1.56 pp
DELTA X-SUPCON + AWARE      : +3.92 pp

SUBJECTS < 55% BASELINE     : 10
SUBJECTS < 55% X-SUPCON     : 8
SUBJECTS < 55% XSC+AWARE    : 8


In [97]:
# ============================================================
# CLASS-WISE SOURCE-DOMAIN ALIGNMENT
#
# Для каждого класса:
#
#   subject A Left  -> centroid
#   subject B Left  -> centroid
#   subject C Left  -> centroid
#
# И сближаем эти центры.
#
# То же независимо для Right.
#
# Target subject здесь НЕ используется.
# ============================================================


class ClassWiseSourceAlignmentLoss(nn.Module):

    def __init__(
        self,
        min_samples_per_centroid=2,
    ):
        super().__init__()

        self.min_samples_per_centroid = (
            min_samples_per_centroid
        )


    def forward(
        self,
        features,
        labels,
        subject_ids,
    ):

        # ----------------------------------------------------
        # Нормируем feature space, чтобы alignment
        # работал по направлению признаков, а не по масштабу.
        # ----------------------------------------------------

        features = (
            torch.nn.functional.normalize(
                features,
                dim=1,
            )
        )

        labels = labels.view(-1)

        subject_ids = (
            subject_ids.view(-1)
        )


        class_losses = []

        total_centroids = 0


        # ====================================================
        # LEFT / RIGHT
        # ====================================================

        for class_id in [0, 1]:

            class_mask = (
                labels == class_id
            )


            class_subjects = torch.unique(
                subject_ids[
                    class_mask
                ]
            )


            centroids = []


            # ------------------------------------------------
            # centroid(class, subject)
            # ------------------------------------------------

            for subject in class_subjects:

                mask = (
                    class_mask
                    &
                    (
                        subject_ids
                        ==
                        subject
                    )
                )


                n_samples = int(
                    mask.sum().item()
                )


                if (
                    n_samples
                    <
                    self.min_samples_per_centroid
                ):
                    continue


                centroid = (
                    features[
                        mask
                    ]
                    .mean(
                        dim=0
                    )
                )


                centroid = (
                    torch.nn.functional.normalize(
                        centroid.unsqueeze(0),
                        dim=1,
                    )
                    .squeeze(0)
                )


                centroids.append(
                    centroid
                )


            # ------------------------------------------------
            # Нужно минимум 2 source domains
            # ------------------------------------------------

            if len(centroids) < 2:
                continue


            centroids = torch.stack(
                centroids,
                dim=0,
            )


            total_centroids += (
                centroids.size(0)
            )


            # ------------------------------------------------
            # Общий class centroid
            # ------------------------------------------------

            global_centroid = (
                centroids.mean(
                    dim=0
                )
            )


            global_centroid = (
                torch.nn.functional.normalize(
                    global_centroid.unsqueeze(0),
                    dim=1,
                )
                .squeeze(0)
            )


            # ------------------------------------------------
            # Cosine distance:
            #
            # 0 = идеально совпадают
            # ------------------------------------------------

            similarity = (
                centroids
                @
                global_centroid
            )


            class_loss = (
                1.0
                -
                similarity
            ).mean()


            class_losses.append(
                class_loss
            )


        # ====================================================
        # NO VALID CENTROIDS
        # ====================================================

        if len(class_losses) == 0:

            return (
                features.sum() * 0.0,
                0,
            )


        loss = torch.stack(
            class_losses
        ).mean()


        return (
            loss,
            total_centroids,
        )

In [98]:
# ============================================================
# TRANSFER
# + TUNED SUPCON
# + CLASS-WISE SOURCE ALIGNMENT
#
# STRICT LOSO
# ============================================================


def train_cho64_class_alignment_fold(
    test_subject,
    seed=42,
    max_epochs=40,
    patience=8,

    lr=1.5e-3,
    weight_decay=1e-4,

    supcon_weight=0.10,
    supcon_temperature=0.05,

    alignment_weight=0.10,

    subjects_per_batch=16,
    samples_per_class=2,
):

    set_seed(seed)


    run_device = globals().get(
        "device",
        torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "mps"
            if torch.backends.mps.is_available()
            else "cpu"
        ),
    )


    # ========================================================
    # DATA
    # ========================================================

    fold = prepare_phys64_fold_supcon_grl(
        test_subject=test_subject,
        seed=seed,
    )


    train_dataset = Phys64SubjectDataset(
        fold["X_train"],
        fold["y_train"],
        fold["s_train"],
        augment=True,
    )


    val_dataset = Phys64SubjectDataset(
        fold["X_val"],
        fold["y_val"],
        fold["s_val"],
        augment=False,
    )


    s_test_dummy = np.zeros(
        len(
            fold["y_test"]
        ),
        dtype=np.int64,
    )


    test_dataset = Phys64SubjectDataset(
        fold["X_test"],
        fold["y_test"],
        s_test_dummy,
        augment=False,
    )


    # ========================================================
    # SUBJECT-AWARE TRAIN BATCHES
    #
    # 16 subjects × 2 classes × 2 trials = batch 64
    # ========================================================

    batch_sampler = (
        SubjectAwareBatchSampler(
            labels=fold["y_train"],
            subject_ids=fold["s_train"],
            subjects_per_batch=
                subjects_per_batch,
            samples_per_class=
                samples_per_class,
            seed=seed,
        )
    )


    train_loader = DataLoader(
        train_dataset,
        batch_sampler=batch_sampler,
        num_workers=0,
    )


    val_loader = DataLoader(
        val_dataset,
        batch_size=64,
        shuffle=False,
        num_workers=0,
        drop_last=False,
    )


    test_loader = DataLoader(
        test_dataset,
        batch_size=64,
        shuffle=False,
        num_workers=0,
        drop_last=False,
    )


    # ========================================================
    # MODEL
    # ========================================================

    model = make_cho64_supcon_grl_model(
        n_subjects=
            fold[
                "n_source_subjects"
            ]
    ).to(
        run_device
    )


    # ========================================================
    # HOOK:
    # берем 240-d EEGNet features перед classifier
    # ========================================================

    captured = {}


    def classifier_input_hook(
        module,
        inputs,
        output,
    ):

        captured[
            "features"
        ] = inputs[0]


    hook_handle = (
        model
        .backbone
        .classifier
        .register_forward_hook(
            classifier_input_hook
        )
    )


    # ========================================================
    # LOSSES
    # ========================================================

    ce_loss_fn = (
        nn.CrossEntropyLoss()
    )


    supcon_loss_fn = (
        SupConLoss(
            temperature=
                supcon_temperature
        )
    )


    alignment_loss_fn = (
        ClassWiseSourceAlignmentLoss(
            min_samples_per_centroid=2
        )
    )


    # ========================================================
    # OPTIMIZER
    # ========================================================

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )


    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=5,
        )
    )


    # ========================================================
    # STATE
    # ========================================================

    best_val_accuracy = -np.inf
    best_epoch = 0
    best_state = None

    epochs_without_improvement = 0

    history = []


    # ========================================================
    # TRAIN
    # ========================================================

    for epoch in range(
        1,
        max_epochs + 1,
    ):

        model.train()


        train_total = 0
        train_correct = 0

        ce_sum = 0.0
        sc_sum = 0.0
        alignment_sum = 0.0

        centroid_sum = 0.0
        n_batches = 0


        for (
            X,
            y,
            subject_ids,
        ) in train_loader:

            X = X.to(
                run_device,
                non_blocking=True,
            )

            y = y.to(
                run_device,
                non_blocking=True,
            )

            subject_ids = (
                subject_ids.to(
                    run_device,
                    non_blocking=True,
                )
            )


            optimizer.zero_grad(
                set_to_none=True
            )

            captured.clear()


            # =================================================
            # FORWARD
            # =================================================

            out = model(
                X,
                lambda_grl=0.0,
            )


            class_logits = (
                out[
                    "class_logits"
                ]
            )


            projection = (
                out[
                    "projection"
                ]
            )


            if (
                "features"
                not in captured
            ):

                hook_handle.remove()

                raise RuntimeError(
                    "EEGNet features were not captured."
                )


            features = (
                captured[
                    "features"
                ]
            )


            # =================================================
            # CE
            # =================================================

            loss_ce = ce_loss_fn(
                class_logits,
                y,
            )


            # =================================================
            # ORIGINAL TUNED SUPCON
            # =================================================

            loss_sc = supcon_loss_fn(
                projection,
                y,
            )


            # =================================================
            # CLASS-WISE SOURCE ALIGNMENT
            # =================================================

            (
                loss_alignment,
                n_centroids,
            ) = alignment_loss_fn(
                features,
                y,
                subject_ids,
            )


            # =================================================
            # TOTAL
            # =================================================

            loss = (
                loss_ce
                +
                supcon_weight
                *
                loss_sc
                +
                alignment_weight
                *
                loss_alignment
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )


            optimizer.step()


            # =================================================
            # METRICS
            # =================================================

            n = y.size(0)


            preds = (
                class_logits.argmax(
                    dim=1
                )
            )


            train_correct += (
                preds == y
            ).sum().item()


            train_total += n


            ce_sum += (
                loss_ce.item()
                *
                n
            )


            sc_sum += (
                loss_sc.item()
                *
                n
            )


            alignment_sum += (
                loss_alignment.item()
                *
                n
            )


            centroid_sum += (
                n_centroids
            )


            n_batches += 1


        # ====================================================
        # TRAIN SUMMARY
        # ====================================================

        train_accuracy = (
            train_correct
            /
            max(
                train_total,
                1,
            )
        )


        train_ce = (
            ce_sum
            /
            max(
                train_total,
                1,
            )
        )


        train_sc = (
            sc_sum
            /
            max(
                train_total,
                1,
            )
        )


        train_alignment = (
            alignment_sum
            /
            max(
                train_total,
                1,
            )
        )


        mean_centroids = (
            centroid_sum
            /
            max(
                n_batches,
                1,
            )
        )


        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()

        val_true = []
        val_pred = []


        with torch.no_grad():

            for X, y, _ in val_loader:

                X = X.to(
                    run_device,
                    non_blocking=True,
                )

                y = y.to(
                    run_device,
                    non_blocking=True,
                )


                out = model(
                    X,
                    lambda_grl=0.0,
                )


                logits = (
                    out[
                        "class_logits"
                    ]
                )


                preds = logits.argmax(
                    dim=1
                )


                val_true.extend(
                    y.cpu().numpy()
                )

                val_pred.extend(
                    preds.cpu().numpy()
                )


        val_accuracy = accuracy_score(
            val_true,
            val_pred,
        )


        scheduler.step(
            val_accuracy
        )


        current_lr = (
            optimizer
            .param_groups[0]["lr"]
        )


        # ====================================================
        # LOG
        # ====================================================

        print(
            f"CLASS-ALIGN | "
            f"ep={epoch:02d} | "
            f"train={train_accuracy:.4f} | "
            f"val={val_accuracy:.4f} | "
            f"CE={train_ce:.3f} | "
            f"SC={train_sc:.3f} | "
            f"ALIGN={train_alignment:.4f} | "
            f"centroids={mean_centroids:.1f} | "
            f"w={alignment_weight:.3f} | "
            f"lr={current_lr:.6f}"
        )


        history.append({

            "epoch":
                epoch,

            "train_accuracy":
                train_accuracy,

            "val_accuracy":
                val_accuracy,

            "ce_loss":
                train_ce,

            "supcon_loss":
                train_sc,

            "alignment_loss":
                train_alignment,

            "mean_centroids":
                mean_centroids,

            "alignment_weight":
                alignment_weight,

            "lr":
                current_lr,
        })


        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if (
            val_accuracy
            >
            best_val_accuracy
        ):

            best_val_accuracy = (
                val_accuracy
            )

            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0


        else:

            epochs_without_improvement += 1


        if (
            epochs_without_improvement
            >=
            patience
        ):

            print(
                f"Early stopping "
                f"at epoch {epoch}"
            )

            break


    # ========================================================
    # RESTORE BEST
    # ========================================================

    if best_state is None:

        hook_handle.remove()

        raise RuntimeError(
            "No valid checkpoint produced."
        )


    model.load_state_dict(
        best_state
    )

    model.eval()


    # ========================================================
    # TEST — UNSEEN SUBJECT
    # ========================================================

    test_true = []
    test_pred = []


    with torch.no_grad():

        for X, y, _ in test_loader:

            X = X.to(
                run_device,
                non_blocking=True,
            )

            y = y.to(
                run_device,
                non_blocking=True,
            )


            out = model(
                X,
                lambda_grl=0.0,
            )


            logits = (
                out[
                    "class_logits"
                ]
            )


            preds = logits.argmax(
                dim=1
            )


            test_true.extend(
                y.cpu().numpy()
            )

            test_pred.extend(
                preds.cpu().numpy()
            )


    hook_handle.remove()


    # ========================================================
    # METRICS
    # ========================================================

    test_accuracy = accuracy_score(
        test_true,
        test_pred,
    )


    balanced_accuracy = (
        balanced_accuracy_score(
            test_true,
            test_pred,
        )
    )


    macro_f1 = f1_score(
        test_true,
        test_pred,
        average="macro",
        zero_division=0,
    )


    kappa = cohen_kappa_score(
        test_true,
        test_pred,
    )


    result = {

        "test_subject":
            int(test_subject),

        "method":
            "class_source_alignment",

        "best_epoch":
            int(best_epoch),

        "best_val_accuracy":
            float(best_val_accuracy),

        "test_accuracy":
            float(test_accuracy),

        "balanced_accuracy":
            float(balanced_accuracy),

        "macro_f1":
            float(macro_f1),

        "kappa":
            float(kappa),

        "lr":
            float(lr),

        "supcon_weight":
            float(supcon_weight),

        "supcon_temperature":
            float(supcon_temperature),

        "alignment_weight":
            float(alignment_weight),

        "seed":
            int(seed),
    }


    return (
        result,
        model,
        pd.DataFrame(
            history
        ),
    )

In [99]:
# ============================================================
# NIGHT PLAN B
#
# Phase 1:
#   alignment weight search
#   на 5 худших subjects
#
# Phase 2:
#   лучший weight
#   на всех worst-10
#
# STRICT LOSO
# ============================================================


ALIGN_ROOT = (
    RESULTS_ROOT
    /
    "class_source_alignment_hard_subjects"
)


ALIGN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


ALIGN_SEARCH_CSV = (
    ALIGN_ROOT
    /
    "alignment_weight_search.csv"
)


ALIGN_FINAL_CSV = (
    ALIGN_ROOT
    /
    "alignment_worst10.csv"
)


ALIGN_WEIGHTS = [
    0.05,
    0.10,
    0.20,
]


# ============================================================
# WORST SUBJECTS
# ============================================================

worst_sorted = (
    worst_baseline_df
    .sort_values(
        "test_accuracy",
        ascending=True,
    )
    .copy()
)


ALIGN_DEV_SUBJECTS = (
    worst_sorted
    .head(5)[
        "test_subject"
    ]
    .astype(int)
    .tolist()
)


ALIGN_WORST10 = (
    worst_sorted[
        "test_subject"
    ]
    .astype(int)
    .tolist()
)


print(
    "Alignment dev subjects:",
    ALIGN_DEV_SUBJECTS,
)


print(
    "Worst-10:",
    ALIGN_WORST10,
)


# ============================================================
# PHASE 1 — WEIGHT SEARCH
# ============================================================

if ALIGN_SEARCH_CSV.exists():

    align_search_df = pd.read_csv(
        ALIGN_SEARCH_CSV
    )

else:

    align_search_df = pd.DataFrame()


done = set()


if len(
    align_search_df
) > 0:

    done = set(
        zip(
            align_search_df[
                "alignment_weight"
            ].astype(float),

            align_search_df[
                "test_subject"
            ].astype(int),
        )
    )


total_runs = (
    len(ALIGN_WEIGHTS)
    *
    len(ALIGN_DEV_SUBJECTS)
)


run_idx = 0


print()
print("=" * 100)
print("PHASE 1 — CLASS ALIGNMENT WEIGHT SEARCH")
print("=" * 100)


for weight in ALIGN_WEIGHTS:

    for subject in ALIGN_DEV_SUBJECTS:

        run_idx += 1


        key = (
            float(weight),
            int(subject),
        )


        if key in done:

            print(
                f"[{run_idx}/{total_runs}] "
                f"w={weight:.2f} | "
                f"S{subject:03d} already done"
            )

            continue


        print()
        print("=" * 100)

        print(
            f"[{run_idx}/{total_runs}] "
            f"CLASS ALIGN | "
            f"W={weight:.2f} | "
            f"S{subject:03d}"
        )

        print("=" * 100)


        result, model, history = (
            train_cho64_class_alignment_fold(

                test_subject=subject,

                seed=42,

                max_epochs=40,

                patience=8,

                lr=1.5e-3,

                weight_decay=1e-4,

                supcon_weight=0.10,

                supcon_temperature=0.05,

                alignment_weight=weight,

                subjects_per_batch=16,

                samples_per_class=2,
            )
        )


        baseline_acc = float(
            worst_baseline_df.loc[
                worst_baseline_df[
                    "test_subject"
                ]
                ==
                subject,

                "test_accuracy",
            ]
            .iloc[0]
        )


        result[
            "baseline_accuracy"
        ] = baseline_acc


        result[
            "delta_pp"
        ] = (
            result[
                "test_accuracy"
            ]
            -
            baseline_acc
        ) * 100


        align_search_df = pd.concat(
            [
                align_search_df,
                pd.DataFrame(
                    [result]
                ),
            ],
            ignore_index=True,
        )


        align_search_df.to_csv(
            ALIGN_SEARCH_CSV,
            index=False,
        )


        done.add(
            key
        )


        print()

        print(
            f"BASELINE = "
            f"{baseline_acc*100:.2f}%"
        )

        print(
            f"ALIGN    = "
            f"{result['test_accuracy']*100:.2f}%"
        )

        print(
            f"DELTA    = "
            f"{result['delta_pp']:+.2f} pp"
        )


        del model
        del history

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ============================================================
# SEARCH SUMMARY
# ============================================================

align_summary = (
    align_search_df
    .groupby(
        "alignment_weight"
    )
    .agg(

        n_subjects=(
            "test_subject",
            "nunique",
        ),

        mean_accuracy=(
            "test_accuracy",
            "mean",
        ),

        median_accuracy=(
            "test_accuracy",
            "median",
        ),

        mean_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),

        mean_macro_f1=(
            "macro_f1",
            "mean",
        ),

        mean_delta_pp=(
            "delta_pp",
            "mean",
        ),
    )
    .reset_index()
)


align_summary = (
    align_summary[
        align_summary[
            "n_subjects"
        ]
        ==
        len(
            ALIGN_DEV_SUBJECTS
        )
    ]
    .sort_values(
        [
            "mean_accuracy",
            "mean_balanced_accuracy",
        ],
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


if len(
    align_summary
) == 0:

    raise RuntimeError(
        "No complete alignment configuration."
    )


BEST_ALIGN_WEIGHT = float(
    align_summary.iloc[0][
        "alignment_weight"
    ]
)


print()
print("=" * 100)
print("ALIGNMENT SEARCH SUMMARY")
print("=" * 100)


display(
    align_summary.assign(

        mean_accuracy_pct=
            align_summary[
                "mean_accuracy"
            ] * 100,

        median_accuracy_pct=
            align_summary[
                "median_accuracy"
            ] * 100,

        balanced_accuracy_pct=
            align_summary[
                "mean_balanced_accuracy"
            ] * 100,

        macro_f1_pct=
            align_summary[
                "mean_macro_f1"
            ] * 100,

    )[[
        "alignment_weight",
        "n_subjects",
        "mean_accuracy_pct",
        "median_accuracy_pct",
        "balanced_accuracy_pct",
        "macro_f1_pct",
        "mean_delta_pp",
    ]]
)


print(
    "\nBEST ALIGNMENT WEIGHT =",
    BEST_ALIGN_WEIGHT,
)


# ============================================================
# PHASE 2 — BEST WEIGHT ON WORST-10
# ============================================================

print()
print("=" * 100)
print("PHASE 2 — BEST ALIGNMENT ON WORST-10")
print("=" * 100)


# ============================================================
# Уже рассчитанные лучшие folds из search переиспользуем
# ============================================================

best_search_rows = (
    align_search_df[
        np.isclose(
            align_search_df[
                "alignment_weight"
            ].astype(float),

            BEST_ALIGN_WEIGHT,
        )
    ]
    .copy()
)


if ALIGN_FINAL_CSV.exists():

    align_final_df = pd.read_csv(
        ALIGN_FINAL_CSV
    )

else:

    align_final_df = pd.DataFrame()


# Добавляем результаты dev subjects,
# если их ещё нет в final CSV.

existing_final_subjects = set()


if len(
    align_final_df
) > 0:

    existing_final_subjects = set(
        align_final_df[
            "test_subject"
        ]
        .astype(int)
        .tolist()
    )


rows_to_add = (
    best_search_rows[
        ~best_search_rows[
            "test_subject"
        ]
        .astype(int)
        .isin(
            existing_final_subjects
        )
    ]
)


if len(
    rows_to_add
) > 0:

    align_final_df = pd.concat(
        [
            align_final_df,
            rows_to_add,
        ],
        ignore_index=True,
    )


    align_final_df.to_csv(
        ALIGN_FINAL_CSV,
        index=False,
    )


done_final = set(
    align_final_df[
        "test_subject"
    ]
    .astype(int)
    .tolist()
)


# ============================================================
# REMAINING WORST SUBJECTS
# ============================================================

for idx, subject in enumerate(
    ALIGN_WORST10,
    start=1,
):

    if subject in done_final:

        print(
            f"[{idx}/{len(ALIGN_WORST10)}] "
            f"S{subject:03d} already done"
        )

        continue


    print()
    print("=" * 100)

    print(
        f"[{idx}/{len(ALIGN_WORST10)}] "
        f"BEST CLASS ALIGN | "
        f"W={BEST_ALIGN_WEIGHT:.2f} | "
        f"S{subject:03d}"
    )

    print("=" * 100)


    result, model, history = (
        train_cho64_class_alignment_fold(

            test_subject=subject,

            seed=42,

            max_epochs=40,

            patience=8,

            lr=1.5e-3,

            weight_decay=1e-4,

            supcon_weight=0.10,

            supcon_temperature=0.05,

            alignment_weight=
                BEST_ALIGN_WEIGHT,

            subjects_per_batch=16,

            samples_per_class=2,
        )
    )


    baseline_acc = float(
        worst_baseline_df.loc[
            worst_baseline_df[
                "test_subject"
            ]
            ==
            subject,

            "test_accuracy",
        ]
        .iloc[0]
    )


    result[
        "baseline_accuracy"
    ] = baseline_acc


    result[
        "delta_pp"
    ] = (
        result[
            "test_accuracy"
        ]
        -
        baseline_acc
    ) * 100


    align_final_df = pd.concat(
        [
            align_final_df,
            pd.DataFrame(
                [result]
            ),
        ],
        ignore_index=True,
    )


    align_final_df.to_csv(
        ALIGN_FINAL_CSV,
        index=False,
    )


    done_final.add(
        int(subject)
    )


    print()

    print(
        f"BASELINE = "
        f"{baseline_acc*100:.2f}%"
    )

    print(
        f"ALIGN    = "
        f"{result['test_accuracy']*100:.2f}%"
    )

    print(
        f"DELTA    = "
        f"{result['delta_pp']:+.2f} pp"
    )


    del model
    del history

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# FINAL PLAN B SUMMARY
# ============================================================

print()
print("=" * 100)
print("PLAN B — FINAL SUMMARY")
print("=" * 100)


final_compare = (
    worst_baseline_df[
        [
            "test_subject",
            "test_accuracy",
        ]
    ]
    .rename(
        columns={
            "test_accuracy":
                "baseline"
        }
    )
    .merge(

        align_final_df[
            [
                "test_subject",
                "test_accuracy",
            ]
        ]
        .rename(
            columns={
                "test_accuracy":
                    "class_alignment"
            }
        ),

        on="test_subject",
        how="left",
    )
)


final_compare[
    "delta_pp"
] = (
    final_compare[
        "class_alignment"
    ]
    -
    final_compare[
        "baseline"
    ]
) * 100


display_df = (
    final_compare.copy()
)


display_df[
    "baseline"
] *= 100


display_df[
    "class_alignment"
] *= 100


display(
    display_df.sort_values(
        "baseline"
    )
)


baseline_mean = (
    final_compare[
        "baseline"
    ].mean()
    *
    100
)


alignment_mean = (
    final_compare[
        "class_alignment"
    ].mean()
    *
    100
)


print()

print(
    f"BASELINE WORST-10 MEAN : "
    f"{baseline_mean:.2f}%"
)

print(
    f"CLASS ALIGN WORST-10    : "
    f"{alignment_mean:.2f}%"
)

print(
    f"DELTA                   : "
    f"{alignment_mean-baseline_mean:+.2f} pp"
)


print()


baseline_under_55 = int(
    (
        final_compare[
            "baseline"
        ]
        <
        0.55
    ).sum()
)


alignment_under_55 = int(
    (
        final_compare[
            "class_alignment"
        ]
        <
        0.55
    ).sum()
)


print(
    f"SUBJECTS <55% BASELINE  : "
    f"{baseline_under_55}"
)

print(
    f"SUBJECTS <55% ALIGNMENT : "
    f"{alignment_under_55}"
)


print("=" * 100)

Alignment dev subjects: [99, 24, 87, 3, 109]
Worst-10: [99, 24, 87, 3, 109, 67, 5, 88, 37, 69]

PHASE 1 — CLASS ALIGNMENT WEIGHT SEARCH

[1/15] CLASS ALIGN | W=0.05 | S099
CLASS-ALIGN | ep=01 | train=0.5963 | val=0.6230 | CE=0.665 | SC=4.205 | ALIGN=0.5715 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIGN | ep=02 | train=0.6232 | val=0.6250 | CE=0.634 | SC=4.141 | ALIGN=0.4502 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIGN | ep=03 | train=0.6388 | val=0.6455 | CE=0.622 | SC=4.139 | ALIGN=0.3933 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIGN | ep=04 | train=0.6307 | val=0.6414 | CE=0.627 | SC=4.139 | ALIGN=0.3546 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIGN | ep=05 | train=0.6505 | val=0.6701 | CE=0.614 | SC=4.136 | ALIGN=0.3405 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIGN | ep=06 | train=0.6484 | val=0.6721 | CE=0.619 | SC=4.136 | ALIGN=0.3259 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIGN | ep=07 | train=0.6484 | val=0.6537 | CE=0.614 | SC=4.136 | 

,alignment_weight,n_subjects,mean_accuracy_pct,median_accuracy_pct,balanced_accuracy_pct,macro_f1_pct,mean_delta_pp
0,0.05,5,50.666667,51.111111,50.849802,44.733749,4.888889
1,0.10,5,48.444444,48.888889,48.715415,39.640393,2.666667
2,0.20,5,48.444444,48.888889,48.715415,42.065266,2.666667



BEST ALIGNMENT WEIGHT = 0.05

PHASE 2 — BEST ALIGNMENT ON WORST-10
[1/10] S099 already done
[2/10] S024 already done
[3/10] S087 already done
[4/10] S003 already done
[5/10] S109 already done

[6/10] BEST CLASS ALIGN | W=0.05 | S067
CLASS-ALIGN | ep=01 | train=0.6089 | val=0.6311 | CE=0.661 | SC=4.205 | ALIGN=0.5722 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIGN | ep=02 | train=0.6351 | val=0.6291 | CE=0.632 | SC=4.141 | ALIGN=0.4458 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIGN | ep=03 | train=0.6448 | val=0.6639 | CE=0.622 | SC=4.139 | ALIGN=0.3848 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIGN | ep=04 | train=0.6383 | val=0.6373 | CE=0.623 | SC=4.138 | ALIGN=0.3555 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIGN | ep=05 | train=0.6613 | val=0.6516 | CE=0.605 | SC=4.134 | ALIGN=0.3374 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIGN | ep=06 | train=0.6565 | val=0.6414 | CE=0.612 | SC=4.136 | ALIGN=0.3244 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIG

,test_subject,baseline,class_alignment,delta_pp
0,99,40.000000,42.222222,2.222222
1,24,46.666667,51.111111,4.444444
2,87,46.666667,53.333333,6.666667
3,3,46.666667,57.777778,11.111111
4,109,48.888889,48.888889,0.000000
5,67,48.888889,48.888889,0.000000
6,5,48.888889,55.555556,6.666667
7,88,50.877193,47.368421,-3.508772
8,37,51.111111,51.111111,0.000000
9,69,53.333333,57.777778,4.444444



BASELINE WORST-10 MEAN : 48.20%
CLASS ALIGN WORST-10    : 51.40%
DELTA                   : +3.20 pp

SUBJECTS <55% BASELINE  : 10
SUBJECTS <55% ALIGNMENT : 7


In [100]:
# ============================================================
# FULL 109-SUBJECT LOSO
#
# EEGNet64
# + Cho Transfer
# + tuned SupCon
# + Class-wise Source Alignment
#
# FIXED CONFIG:
#   lr = 1.5e-3
#   SupCon weight = 0.10
#   temperature = 0.05
#   alignment weight = 0.05
#
# STRICT LOSO
# ============================================================


FULL_ALIGN_ROOT = (
    RESULTS_ROOT
    /
    "class_alignment_full_109"
)


FULL_ALIGN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


FULL_ALIGN_CSV = (
    FULL_ALIGN_ROOT
    /
    "class_alignment_full_109.csv"
)


# ============================================================
# FIXED HYPERPARAMETERS
# ============================================================

FULL_ALIGN_LR = 1.5e-3

FULL_ALIGN_SUPCON_WEIGHT = 0.10

FULL_ALIGN_TEMPERATURE = 0.05

FULL_ALIGN_WEIGHT = 0.05


FULL_SUBJECTS = list(
    range(
        1,
        110,
    )
)


# ============================================================
# LOAD / RESUME
# ============================================================

if FULL_ALIGN_CSV.exists():

    full_align_df = pd.read_csv(
        FULL_ALIGN_CSV
    )

    done_subjects = set(
        full_align_df[
            "test_subject"
        ]
        .astype(int)
        .tolist()
    )

    print(
        f"Resume: "
        f"{len(done_subjects)}/109 "
        f"subjects completed"
    )

else:

    full_align_df = pd.DataFrame()

    done_subjects = set()


# ============================================================
# FULL LOSO
# ============================================================

print()
print("=" * 100)
print("FULL 109-SUBJECT LOSO — CLASS ALIGNMENT")
print("=" * 100)

print(
    f"LR                 = {FULL_ALIGN_LR}"
)

print(
    f"SupCon weight      = {FULL_ALIGN_SUPCON_WEIGHT}"
)

print(
    f"SupCon temperature = {FULL_ALIGN_TEMPERATURE}"
)

print(
    f"Alignment weight   = {FULL_ALIGN_WEIGHT}"
)

print("=" * 100)


for idx, subject in enumerate(
    FULL_SUBJECTS,
    start=1,
):

    if subject in done_subjects:

        print(
            f"[{idx}/109] "
            f"S{subject:03d} already done"
        )

        continue


    print()
    print("=" * 100)

    print(
        f"[{idx}/109] "
        f"CLASS ALIGNMENT | "
        f"TEST SUBJECT=S{subject:03d}"
    )

    print("=" * 100)


    result, model, history = (
        train_cho64_class_alignment_fold(

            test_subject=subject,

            seed=42,

            max_epochs=40,

            patience=8,

            lr=FULL_ALIGN_LR,

            weight_decay=1e-4,

            supcon_weight=
                FULL_ALIGN_SUPCON_WEIGHT,

            supcon_temperature=
                FULL_ALIGN_TEMPERATURE,

            alignment_weight=
                FULL_ALIGN_WEIGHT,

            subjects_per_batch=16,

            samples_per_class=2,
        )
    )


    # ========================================================
    # SAVE METHOD INFO
    # ========================================================

    result[
        "method"
    ] = (
        "transfer_supcon_class_alignment"
    )


    result[
        "full_loso"
    ] = True


    # ========================================================
    # APPEND + SAVE
    # ========================================================

    full_align_df = pd.concat(
        [
            full_align_df,
            pd.DataFrame(
                [result]
            ),
        ],
        ignore_index=True,
    )


    full_align_df.to_csv(
        FULL_ALIGN_CSV,
        index=False,
    )


    done_subjects.add(
        int(subject)
    )


    # ========================================================
    # RUNNING METRICS
    # ========================================================

    running_mean = (
        full_align_df[
            "test_accuracy"
        ]
        .mean()
        *
        100
    )


    running_median = (
        full_align_df[
            "test_accuracy"
        ]
        .median()
        *
        100
    )


    running_std = (
        full_align_df[
            "test_accuracy"
        ]
        .std()
        *
        100
    )


    running_under_55 = int(
        (
            full_align_df[
                "test_accuracy"
            ]
            <
            0.55
        ).sum()
    )


    running_under_60 = int(
        (
            full_align_df[
                "test_accuracy"
            ]
            <
            0.60
        ).sum()
    )


    print()
    print(
        f"SUBJECT RESULT = "
        f"{result['test_accuracy']*100:.2f}%"
    )

    print(
        f"RUNNING MEAN   = "
        f"{running_mean:.2f}%"
    )

    print(
        f"RUNNING MEDIAN = "
        f"{running_median:.2f}%"
    )

    print(
        f"RUNNING STD    = "
        f"{running_std:.2f} pp"
    )

    print(
        f"SUBJECTS <55%  = "
        f"{running_under_55}"
    )

    print(
        f"SUBJECTS <60%  = "
        f"{running_under_60}"
    )

    print(
        f"PROGRESS       = "
        f"{len(done_subjects)}/109"
    )


    # ========================================================
    # CLEANUP
    # ========================================================

    del model
    del history

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 100)
print("FULL 109 LOSO FINISHED")
print("=" * 100)


final_mean = (
    full_align_df[
        "test_accuracy"
    ]
    .mean()
    *
    100
)


final_median = (
    full_align_df[
        "test_accuracy"
    ]
    .median()
    *
    100
)


final_std = (
    full_align_df[
        "test_accuracy"
    ]
    .std()
    *
    100
)


final_balanced = (
    full_align_df[
        "balanced_accuracy"
    ]
    .mean()
    *
    100
)


final_macro_f1 = (
    full_align_df[
        "macro_f1"
    ]
    .mean()
    *
    100
)


final_kappa = (
    full_align_df[
        "kappa"
    ]
    .mean()
)


final_p10 = (
    full_align_df[
        "test_accuracy"
    ]
    .quantile(
        0.10
    )
    *
    100
)


worst10_df = (
    full_align_df
    .sort_values(
        "test_accuracy",
        ascending=True,
    )
    .head(10)
)


worst10_mean = (
    worst10_df[
        "test_accuracy"
    ]
    .mean()
    *
    100
)


subjects_under_55 = int(
    (
        full_align_df[
            "test_accuracy"
        ]
        <
        0.55
    ).sum()
)


subjects_under_60 = int(
    (
        full_align_df[
            "test_accuracy"
        ]
        <
        0.60
    ).sum()
)


# ============================================================
# REFERENCE RESULTS
# ============================================================

BASELINE_MEAN = 69.177612

BASELINE_STD = 13.185277

BASELINE_WORST10 = 48.20

CCST64_MEAN = 73.65


summary_df = pd.DataFrame({

    "method": [
        "Transfer + SupCon + Class Alignment"
    ],

    "n_subjects": [
        full_align_df[
            "test_subject"
        ]
        .nunique()
    ],

    "mean_accuracy_pct": [
        final_mean
    ],

    "median_accuracy_pct": [
        final_median
    ],

    "std_accuracy_pct": [
        final_std
    ],

    "balanced_accuracy_pct": [
        final_balanced
    ],

    "macro_f1_pct": [
        final_macro_f1
    ],

    "mean_kappa": [
        final_kappa
    ],

    "p10_accuracy_pct": [
        final_p10
    ],

    "worst10_mean_pct": [
        worst10_mean
    ],

    "subjects_under_55": [
        subjects_under_55
    ],

    "subjects_under_60": [
        subjects_under_60
    ],
})


display(
    summary_df
)


# ============================================================
# WORST SUBJECTS
# ============================================================

worst10_display = (
    worst10_df[
        [
            "test_subject",
            "test_accuracy",
            "balanced_accuracy",
            "macro_f1",
        ]
    ]
    .copy()
)


worst10_display[
    "test_accuracy"
] *= 100


worst10_display[
    "balanced_accuracy"
] *= 100


worst10_display[
    "macro_f1"
] *= 100


display(
    worst10_display
)


# ============================================================
# COMPARISON
# ============================================================

print()
print("=" * 100)

print(
    f"BASELINE MEAN           : "
    f"{BASELINE_MEAN:.2f}%"
)

print(
    f"CLASS ALIGNMENT MEAN    : "
    f"{final_mean:.2f}%"
)

print(
    f"DELTA VS BASELINE       : "
    f"{final_mean-BASELINE_MEAN:+.2f} pp"
)

print()


print(
    f"BASELINE STD            : "
    f"{BASELINE_STD:.2f} pp"
)

print(
    f"CLASS ALIGNMENT STD     : "
    f"{final_std:.2f} pp"
)

print(
    f"STD DELTA               : "
    f"{final_std-BASELINE_STD:+.2f} pp"
)

print()


print(
    f"BASELINE WORST-10       : "
    f"{BASELINE_WORST10:.2f}%"
)

print(
    f"CLASS ALIGN WORST-10    : "
    f"{worst10_mean:.2f}%"
)

print(
    f"WORST-10 DELTA          : "
    f"{worst10_mean-BASELINE_WORST10:+.2f} pp"
)

print()


print(
    f"10th PERCENTILE         : "
    f"{final_p10:.2f}%"
)

print(
    f"SUBJECTS <55%           : "
    f"{subjects_under_55}"
)

print(
    f"SUBJECTS <60%           : "
    f"{subjects_under_60}"
)

print()


print(
    f"CCST64                   : "
    f"{CCST64_MEAN:.2f}%"
)

print(
    f"DELTA VS CCST64         : "
    f"{final_mean-CCST64_MEAN:+.2f} pp"
)

print("=" * 100)


FULL 109-SUBJECT LOSO — CLASS ALIGNMENT
LR                 = 0.0015
SupCon weight      = 0.1
SupCon temperature = 0.05
Alignment weight   = 0.05

[1/109] CLASS ALIGNMENT | TEST SUBJECT=S001
CLASS-ALIGN | ep=01 | train=0.6027 | val=0.6885 | CE=0.668 | SC=4.207 | ALIGN=0.5692 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIGN | ep=02 | train=0.6243 | val=0.7111 | CE=0.635 | SC=4.142 | ALIGN=0.4519 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIGN | ep=03 | train=0.6229 | val=0.6537 | CE=0.636 | SC=4.141 | ALIGN=0.3846 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIGN | ep=04 | train=0.6271 | val=0.6639 | CE=0.636 | SC=4.140 | ALIGN=0.3513 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIGN | ep=05 | train=0.6402 | val=0.6680 | CE=0.622 | SC=4.139 | ALIGN=0.3372 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIGN | ep=06 | train=0.6317 | val=0.6988 | CE=0.625 | SC=4.138 | ALIGN=0.3247 | centroids=32.0 | w=0.050 | lr=0.001500
CLASS-ALIGN | ep=07 | train=0.6390 | val=0.6865 | CE=

,method,n_subjects,mean_accuracy_pct,median_accuracy_pct,std_accuracy_pct,balanced_accuracy_pct,macro_f1_pct,mean_kappa,p10_accuracy_pct,worst10_mean_pct,subjects_under_55,subjects_under_60
0,Transfer + SupCon + Class Alignment,109,68.853598,68.888889,12.504296,68.864849,67.054687,0.377008,53.333333,49.570175,15,28


,test_subject,test_accuracy,balanced_accuracy,macro_f1
98,99,42.222222,42.687747,39.814815
87,88,47.368421,47.167488,46.562500
108,109,48.888889,50.000000,32.835821
66,67,48.888889,49.703557,41.492369
23,24,51.111111,51.679842,48.004202
35,36,51.111111,51.383399,50.500000
36,37,51.111111,50.000000,33.823529
39,40,51.111111,51.488095,51.086957
57,58,51.111111,52.976190,48.004202
99,100,52.777778,52.777778,52.447552



BASELINE MEAN           : 69.18%
CLASS ALIGNMENT MEAN    : 68.85%
DELTA VS BASELINE       : -0.32 pp

BASELINE STD            : 13.19 pp
CLASS ALIGNMENT STD     : 12.50 pp
STD DELTA               : -0.68 pp

BASELINE WORST-10       : 48.20%
CLASS ALIGN WORST-10    : 49.57%
WORST-10 DELTA          : +1.37 pp

10th PERCENTILE         : 53.33%
SUBJECTS <55%           : 15
SUBJECTS <60%           : 28

CCST64                   : 73.65%
DELTA VS CCST64         : -4.80 pp


OUTER LOSO:
S003 = TEST — вообще не трогаем

из остальных 108 subjects:
98 subjects → TRAIN
10 целых subjects → VALIDATION

checkpoint выбирается по accuracy
на невиданных validation-subjects

In [101]:
# ============================================================
# SUBJECT-HELD-OUT VALIDATION
#
# Outer:
#   1 subject = TEST
#
# Inner:
#   N subjects = VALIDATION
#   остальные = TRAIN
#
# Ни одного trial validation-subject
# нет в train.
# ============================================================


# Сохраняем настоящую исходную prepare-функцию.
# Если ранее была создана *_original для EA,
# используем именно её.

if "prepare_phys64_fold_supcon_grl_original" in globals():

    BASE_PHYS64_PREPARE = (
        prepare_phys64_fold_supcon_grl_original
    )

else:

    BASE_PHYS64_PREPARE = (
        prepare_phys64_fold_supcon_grl
    )


def prepare_phys64_fold_subject_val(
    test_subject,
    seed=42,
    n_val_subjects=10,
):

    # ========================================================
    # Сначала создаём обычный outer LOSO fold
    # ========================================================

    fold = BASE_PHYS64_PREPARE(
        test_subject=test_subject,
        seed=seed,
    )


    # ========================================================
    # СОБИРАЕМ ВСЕ SOURCE TRIALS ОБРАТНО
    #
    # Старый train/val разделён trial-wise.
    # Для inner subject-validation нам нужен общий source pool.
    # ========================================================

    X_source = np.concatenate(
        [
            fold["X_train"],
            fold["X_val"],
        ],
        axis=0,
    )


    y_source = np.concatenate(
        [
            fold["y_train"],
            fold["y_val"],
        ],
        axis=0,
    )


    s_source = np.concatenate(
        [
            fold["s_train"],
            fold["s_val"],
        ],
        axis=0,
    )


    # ========================================================
    # SOURCE SUBJECTS
    # ========================================================

    source_subjects = np.unique(
        s_source
    )


    if (
        n_val_subjects
        >=
        len(source_subjects)
    ):

        raise ValueError(
            "n_val_subjects must be smaller "
            "than number of source subjects."
        )


    # ========================================================
    # DETERMINISTIC SUBJECT SPLIT
    #
    # Для каждого outer test subject split фиксирован.
    # ========================================================

    split_seed = (
        seed
        +
        int(test_subject)
        *
        1000
    )


    rng = np.random.default_rng(
        split_seed
    )


    shuffled_subjects = (
        source_subjects.copy()
    )


    rng.shuffle(
        shuffled_subjects
    )


    val_subjects = np.sort(
        shuffled_subjects[
            :n_val_subjects
        ]
    )


    train_subjects = np.sort(
        shuffled_subjects[
            n_val_subjects:
        ]
    )


    # ========================================================
    # MASKS
    # ========================================================

    val_mask = np.isin(
        s_source,
        val_subjects,
    )


    train_mask = ~val_mask


    # ========================================================
    # NEW TRAIN / VAL
    # ========================================================

    X_train = X_source[
        train_mask
    ]

    y_train = y_source[
        train_mask
    ]

    s_train = s_source[
        train_mask
    ]


    X_val = X_source[
        val_mask
    ]

    y_val = y_source[
        val_mask
    ]

    s_val = s_source[
        val_mask
    ]


    # ========================================================
    # SANITY CHECKS
    # ========================================================

    train_subject_set = set(
        np.unique(
            s_train
        ).tolist()
    )


    val_subject_set = set(
        np.unique(
            s_val
        ).tolist()
    )


    overlap = (
        train_subject_set
        &
        val_subject_set
    )


    if len(overlap) > 0:

        raise RuntimeError(
            f"Subject leakage detected: {overlap}"
        )


    # Оба класса должны присутствовать
    # и в train, и в val.

    if len(
        np.unique(
            y_train
        )
    ) < 2:

        raise RuntimeError(
            "Train contains fewer than 2 classes."
        )


    if len(
        np.unique(
            y_val
        )
    ) < 2:

        raise RuntimeError(
            "Validation contains fewer than 2 classes."
        )


    # ========================================================
    # UPDATE FOLD
    # ========================================================

    new_fold = dict(
        fold
    )


    new_fold[
        "X_train"
    ] = X_train

    new_fold[
        "y_train"
    ] = y_train

    new_fold[
        "s_train"
    ] = s_train


    new_fold[
        "X_val"
    ] = X_val

    new_fold[
        "y_val"
    ] = y_val

    new_fold[
        "s_val"
    ] = s_val


    new_fold[
        "group_val_subjects"
    ] = val_subjects


    new_fold[
        "group_train_subjects"
    ] = train_subjects


    # ========================================================
    # INFO
    # ========================================================

    print(
        f"SUBJECT-VAL SPLIT | "
        f"outer_test=S{int(test_subject):03d} | "
        f"train_subjects={len(train_subjects)} | "
        f"val_subjects={len(val_subjects)} | "
        f"train_trials={len(y_train)} | "
        f"val_trials={len(y_val)}"
    )


    return new_fold

In [102]:
# ============================================================
# TRAIN:
# SAME TRANSFER + TUNED SUPCON
# BUT SUBJECT-HELD-OUT VALIDATION
# ============================================================


def train_cho64_subject_val_supcon_fold(
    test_subject,
    seed=42,
    n_val_subjects=10,
):

    global prepare_phys64_fold_supcon_grl


    old_prepare = (
        prepare_phys64_fold_supcon_grl
    )


    def grouped_prepare(
        test_subject,
        seed=42,
    ):

        return prepare_phys64_fold_subject_val(
            test_subject=test_subject,
            seed=seed,
            n_val_subjects=n_val_subjects,
        )


    try:

        # --------------------------------------------
        # На время вызова train-функции
        # меняем только подготовку fold.
        # --------------------------------------------

        prepare_phys64_fold_supcon_grl = (
            grouped_prepare
        )


        result, model, history = (
            train_cho64_ablation_fold(

                test_subject=test_subject,

                mode="supcon",

                seed=seed,

                max_epochs=40,

                patience=8,

                batch_size=64,

                lr=1.5e-3,

                weight_decay=1e-4,

                supcon_weight=0.10,

                supcon_temperature=0.05,
            )
        )


        result[
            "method"
        ] = (
            "transfer_supcon_subject_val"
        )


        result[
            "n_val_subjects"
        ] = int(
            n_val_subjects
        )


        return (
            result,
            model,
            history,
        )


    finally:

        # --------------------------------------------
        # Обязательно возвращаем старую функцию,
        # даже если обучение упало.
        # --------------------------------------------

        prepare_phys64_fold_supcon_grl = (
            old_prepare
        )

In [103]:
# ============================================================
# WORST-10 FROM ORIGINAL FULL SUPCON LOSO
# ============================================================


GROUPVAL_WORST_N = 10


groupval_baseline_df = (
    full_df
    .sort_values(
        "test_accuracy",
        ascending=True,
    )
    .head(
        GROUPVAL_WORST_N
    )
    [
        [
            "test_subject",
            "test_accuracy",
            "balanced_accuracy",
            "macro_f1",
        ]
    ]
    .copy()
)


groupval_baseline_df[
    "test_subject"
] = (
    groupval_baseline_df[
        "test_subject"
    ]
    .astype(int)
)


GROUPVAL_WORST_SUBJECTS = (
    groupval_baseline_df[
        "test_subject"
    ]
    .tolist()
)


print(
    "Worst-10 baseline subjects:",
    GROUPVAL_WORST_SUBJECTS,
)


display(
    groupval_baseline_df.assign(

        accuracy_pct=
            groupval_baseline_df[
                "test_accuracy"
            ]
            *
            100
    )
)


print(
    "\nBaseline worst-10 mean = "
    f"{groupval_baseline_df['test_accuracy'].mean()*100:.2f}%"
)

Worst-10 baseline subjects: [99, 24, 87, 3, 109, 67, 5, 88, 37, 69]


,test_subject,test_accuracy,balanced_accuracy,macro_f1,accuracy_pct
98,99,0.400000,0.405138,0.368831,40.000000
23,24,0.466667,0.477273,0.318182,46.666667
86,87,0.466667,0.460474,0.418103,46.666667
2,3,0.466667,0.468379,0.464286,46.666667
108,109,0.488889,0.500000,0.328358,48.888889
66,67,0.488889,0.500000,0.328358,48.888889
4,5,0.488889,0.464286,0.391534,48.888889
87,88,0.508772,0.510468,0.504963,50.877193
36,37,0.511111,0.500000,0.338235,51.111111
68,69,0.533333,0.556548,0.483324,53.333333



Baseline worst-10 mean = 48.20%


In [104]:
# ============================================================
# SUBJECT-HELD-OUT VALIDATION EXPERIMENT
#
# 10 hardest baseline subjects
# ============================================================


GROUPVAL_ROOT = (
    RESULTS_ROOT
    /
    "subject_heldout_validation"
)


GROUPVAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


GROUPVAL_CSV = (
    GROUPVAL_ROOT
    /
    "subject_val_worst10.csv"
)


if GROUPVAL_CSV.exists():

    groupval_results = pd.read_csv(
        GROUPVAL_CSV
    )

else:

    groupval_results = pd.DataFrame()


done_subjects = set()


if len(
    groupval_results
) > 0:

    done_subjects = set(
        groupval_results[
            "test_subject"
        ]
        .astype(int)
        .tolist()
    )


print()
print("=" * 100)
print("SUBJECT-HELD-OUT VALIDATION — WORST-10")
print("=" * 100)


for idx, subject in enumerate(
    GROUPVAL_WORST_SUBJECTS,
    start=1,
):

    if subject in done_subjects:

        print(
            f"[{idx}/10] "
            f"S{subject:03d} already done"
        )

        continue


    print()
    print("=" * 100)

    print(
        f"[{idx}/10] "
        f"SUBJECT-VAL SUPCON | "
        f"TEST SUBJECT=S{subject:03d}"
    )

    print("=" * 100)


    result, model, history = (
        train_cho64_subject_val_supcon_fold(

            test_subject=subject,

            seed=42,

            n_val_subjects=10,
        )
    )


    baseline_acc = float(
        groupval_baseline_df.loc[

            groupval_baseline_df[
                "test_subject"
            ]
            ==
            subject,

            "test_accuracy",
        ]
        .iloc[0]
    )


    result[
        "baseline_accuracy"
    ] = baseline_acc


    result[
        "delta_pp"
    ] = (
        result[
            "test_accuracy"
        ]
        -
        baseline_acc
    ) * 100


    groupval_results = pd.concat(
        [
            groupval_results,

            pd.DataFrame(
                [result]
            ),
        ],

        ignore_index=True,
    )


    groupval_results.to_csv(
        GROUPVAL_CSV,
        index=False,
    )


    done_subjects.add(
        int(subject)
    )


    print()

    print(
        f"BASELINE         = "
        f"{baseline_acc*100:.2f}%"
    )

    print(
        f"SUBJECT-VAL      = "
        f"{result['test_accuracy']*100:.2f}%"
    )

    print(
        f"DELTA            = "
        f"{result['delta_pp']:+.2f} pp"
    )


    del model
    del history

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


SUBJECT-HELD-OUT VALIDATION — WORST-10

[1/10] SUBJECT-VAL SUPCON | TEST SUBJECT=S099
SUBJECT-VAL SPLIT | outer_test=S099 | train_subjects=98 | val_subjects=10 | train_trials=4423 | val_trials=450
SUPCON       | ep=01 | train=0.5980 | val=0.6156 | CE=0.662 | SC=4.202 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=02 | train=0.6263 | val=0.6200 | CE=0.635 | SC=4.141 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=03 | train=0.6322 | val=0.6689 | CE=0.632 | SC=4.137 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=04 | train=0.6459 | val=0.6556 | CE=0.620 | SC=4.135 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=05 | train=0.6419 | val=0.6689 | CE=0.617 | SC=4.134 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=06 | train=0.6383 | val=0.6244 | CE=0.619 | SC=4.134 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=07 | train=0.6500 | val=0.6844 | CE=0.613 | SC=4.132 (w=0.1000) | ADV=0.000 (w=0.0

In [105]:
# ============================================================
# SUBJECT-VAL SUMMARY
# ============================================================


subject_val_compare = (
    groupval_baseline_df[
        [
            "test_subject",
            "test_accuracy",
        ]
    ]
    .rename(
        columns={
            "test_accuracy":
                "baseline"
        }
    )
    .merge(

        groupval_results[
            [
                "test_subject",
                "test_accuracy",
                "best_epoch",
                "best_val_accuracy",
            ]
        ]
        .rename(
            columns={
                "test_accuracy":
                    "subject_val"
            }
        ),

        on="test_subject",
        how="left",
    )
)


subject_val_compare[
    "delta_pp"
] = (
    subject_val_compare[
        "subject_val"
    ]
    -
    subject_val_compare[
        "baseline"
    ]
) * 100


display_df = (
    subject_val_compare.copy()
)


display_df[
    "baseline"
] *= 100


display_df[
    "subject_val"
] *= 100


display_df[
    "best_val_accuracy"
] *= 100


display(
    display_df.sort_values(
        "baseline"
    )
)


# ============================================================
# AGGREGATES
# ============================================================

baseline_mean = (
    subject_val_compare[
        "baseline"
    ]
    .mean()
    *
    100
)


subject_val_mean = (
    subject_val_compare[
        "subject_val"
    ]
    .mean()
    *
    100
)


delta_mean = (
    subject_val_mean
    -
    baseline_mean
)


wins = int(
    (
        subject_val_compare[
            "delta_pp"
        ]
        >
        0
    ).sum()
)


ties = int(
    np.isclose(
        subject_val_compare[
            "delta_pp"
        ],
        0.0,
    ).sum()
)


losses = int(
    (
        subject_val_compare[
            "delta_pp"
        ]
        <
        0
    ).sum()
)


baseline_under_55 = int(
    (
        subject_val_compare[
            "baseline"
        ]
        <
        0.55
    ).sum()
)


subject_val_under_55 = int(
    (
        subject_val_compare[
            "subject_val"
        ]
        <
        0.55
    ).sum()
)


print()
print("=" * 100)

print(
    f"BASELINE WORST-10 MEAN : "
    f"{baseline_mean:.2f}%"
)

print(
    f"SUBJECT-VAL WORST-10    : "
    f"{subject_val_mean:.2f}%"
)

print(
    f"DELTA                   : "
    f"{delta_mean:+.2f} pp"
)


print()

print(
    f"WINS / TIES / LOSSES    : "
    f"{wins} / {ties} / {losses}"
)


print(
    f"MEDIAN DELTA            : "
    f"{subject_val_compare['delta_pp'].median():+.2f} pp"
)


print()

print(
    f"SUBJECTS <55% BASELINE  : "
    f"{baseline_under_55}"
)

print(
    f"SUBJECTS <55% NEW       : "
    f"{subject_val_under_55}"
)

print("=" * 100)

,test_subject,baseline,subject_val,best_epoch,best_val_accuracy,delta_pp
0,99,40.000000,42.222222,13,70.666667,2.222222
1,24,46.666667,42.222222,31,76.888889,-4.444444
2,87,46.666667,51.111111,9,74.000000,4.444444
3,3,46.666667,44.444444,2,66.000000,-2.222222
4,109,48.888889,48.888889,19,76.666667,0.000000
5,67,48.888889,51.111111,10,76.000000,2.222222
6,5,48.888889,53.333333,17,65.079365,4.444444
7,88,50.877193,56.140351,25,66.888889,5.263158
8,37,51.111111,68.888889,14,75.111111,17.777778
9,69,53.333333,55.555556,23,73.333333,2.222222



BASELINE WORST-10 MEAN : 48.20%
SUBJECT-VAL WORST-10    : 51.39%
DELTA                   : +3.19 pp

WINS / TIES / LOSSES    : 7 / 1 / 2
MEDIAN DELTA            : +2.22 pp

SUBJECTS <55% BASELINE  : 10
SUBJECTS <55% NEW       : 7


In [106]:
# ============================================================
# FULL 109-SUBJECT LOSO
#
# EEGNet64
# + Cho Transfer
# + tuned SupCon
# + SUBJECT-HELD-OUT VALIDATION
#
# FIXED CONFIG:
#   lr = 1.5e-3
#   SupCon weight = 0.10
#   temperature = 0.05
#   validation subjects = 10
#
# STRICT OUTER LOSO
# ============================================================


SUBJVAL_FULL_ROOT = (
    RESULTS_ROOT
    /
    "subject_heldout_validation_full_109"
)


SUBJVAL_FULL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


SUBJVAL_FULL_CSV = (
    SUBJVAL_FULL_ROOT
    /
    "subject_val_full_109.csv"
)


# ============================================================
# FIXED CONFIG
# ============================================================

SUBJVAL_LR = 1.5e-3

SUBJVAL_SUPCON_WEIGHT = 0.10

SUBJVAL_TEMPERATURE = 0.05

SUBJVAL_N_VAL_SUBJECTS = 10

SUBJVAL_SEED = 42


FULL_SUBJECTS = list(
    range(
        1,
        110,
    )
)


# ============================================================
# LOAD / RESUME
# ============================================================

if SUBJVAL_FULL_CSV.exists():

    subject_val_full_df = pd.read_csv(
        SUBJVAL_FULL_CSV
    )

    done_subjects = set(
        subject_val_full_df[
            "test_subject"
        ]
        .astype(int)
        .tolist()
    )

    print(
        f"Resume: "
        f"{len(done_subjects)}/109 "
        f"subjects completed"
    )

else:

    subject_val_full_df = pd.DataFrame()

    done_subjects = set()


# ============================================================
# START
# ============================================================

print()
print("=" * 100)

print(
    "FULL 109-SUBJECT LOSO — "
    "SUBJECT-HELD-OUT VALIDATION"
)

print("=" * 100)

print(
    f"LR                   = {SUBJVAL_LR}"
)

print(
    f"SupCon weight        = {SUBJVAL_SUPCON_WEIGHT}"
)

print(
    f"SupCon temperature   = {SUBJVAL_TEMPERATURE}"
)

print(
    f"Validation subjects  = {SUBJVAL_N_VAL_SUBJECTS}"
)

print(
    f"Seed                 = {SUBJVAL_SEED}"
)

print("=" * 100)


# ============================================================
# FULL LOSO LOOP
# ============================================================

for idx, subject in enumerate(
    FULL_SUBJECTS,
    start=1,
):

    if subject in done_subjects:

        print(
            f"[{idx}/109] "
            f"S{subject:03d} already done"
        )

        continue


    print()
    print("=" * 100)

    print(
        f"[{idx}/109] "
        f"SUBJECT-VAL SUPCON | "
        f"TEST SUBJECT=S{subject:03d}"
    )

    print("=" * 100)


    result, model, history = (
        train_cho64_subject_val_supcon_fold(

            test_subject=subject,

            seed=SUBJVAL_SEED,

            n_val_subjects=
                SUBJVAL_N_VAL_SUBJECTS,
        )
    )


    # ========================================================
    # EXTRA INFO
    # ========================================================

    result[
        "method"
    ] = (
        "transfer_supcon_subject_heldout_val"
    )


    result[
        "full_loso"
    ] = True


    # ========================================================
    # BASELINE SAME SUBJECT
    # ========================================================

    baseline_rows = (
        full_df[
            full_df[
                "test_subject"
            ]
            .astype(int)
            ==
            int(subject)
        ]
    )


    if len(
        baseline_rows
    ) > 0:

        baseline_acc = float(
            baseline_rows[
                "test_accuracy"
            ]
            .iloc[0]
        )

        result[
            "baseline_accuracy"
        ] = baseline_acc

        result[
            "delta_vs_baseline_pp"
        ] = (
            result[
                "test_accuracy"
            ]
            -
            baseline_acc
        ) * 100

    else:

        result[
            "baseline_accuracy"
        ] = np.nan

        result[
            "delta_vs_baseline_pp"
        ] = np.nan


    # ========================================================
    # SAVE
    # ========================================================

    subject_val_full_df = pd.concat(
        [
            subject_val_full_df,
            pd.DataFrame(
                [result]
            ),
        ],
        ignore_index=True,
    )


    subject_val_full_df.to_csv(
        SUBJVAL_FULL_CSV,
        index=False,
    )


    done_subjects.add(
        int(subject)
    )


    # ========================================================
    # RUNNING SUMMARY
    # ========================================================

    running_mean = (
        subject_val_full_df[
            "test_accuracy"
        ]
        .mean()
        *
        100
    )


    running_median = (
        subject_val_full_df[
            "test_accuracy"
        ]
        .median()
        *
        100
    )


    running_std = (
        subject_val_full_df[
            "test_accuracy"
        ]
        .std()
        *
        100
    )


    running_p10 = (
        subject_val_full_df[
            "test_accuracy"
        ]
        .quantile(
            0.10
        )
        *
        100
    )


    running_under_55 = int(
        (
            subject_val_full_df[
                "test_accuracy"
            ]
            <
            0.55
        ).sum()
    )


    running_under_60 = int(
        (
            subject_val_full_df[
                "test_accuracy"
            ]
            <
            0.60
        ).sum()
    )


    print()

    print(
        f"SUBJECT RESULT = "
        f"{result['test_accuracy']*100:.2f}%"
    )


    if not np.isnan(
        result[
            "baseline_accuracy"
        ]
    ):

        print(
            f"BASELINE       = "
            f"{result['baseline_accuracy']*100:.2f}%"
        )

        print(
            f"DELTA          = "
            f"{result['delta_vs_baseline_pp']:+.2f} pp"
        )


    print(
        f"RUNNING MEAN   = "
        f"{running_mean:.2f}%"
    )

    print(
        f"RUNNING MEDIAN = "
        f"{running_median:.2f}%"
    )

    print(
        f"RUNNING STD    = "
        f"{running_std:.2f} pp"
    )

    print(
        f"RUNNING P10    = "
        f"{running_p10:.2f}%"
    )

    print(
        f"SUBJECTS <55%  = "
        f"{running_under_55}"
    )

    print(
        f"SUBJECTS <60%  = "
        f"{running_under_60}"
    )

    print(
        f"PROGRESS       = "
        f"{len(done_subjects)}/109"
    )


    # ========================================================
    # CLEANUP
    # ========================================================

    del model
    del history

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 100)
print("FULL SUBJECT-VAL LOSO FINISHED")
print("=" * 100)


final_mean = (
    subject_val_full_df[
        "test_accuracy"
    ]
    .mean()
    *
    100
)


final_median = (
    subject_val_full_df[
        "test_accuracy"
    ]
    .median()
    *
    100
)


final_std = (
    subject_val_full_df[
        "test_accuracy"
    ]
    .std()
    *
    100
)


final_balanced = (
    subject_val_full_df[
        "balanced_accuracy"
    ]
    .mean()
    *
    100
)


final_macro_f1 = (
    subject_val_full_df[
        "macro_f1"
    ]
    .mean()
    *
    100
)


final_kappa = (
    subject_val_full_df[
        "kappa"
    ]
    .mean()
)


final_p10 = (
    subject_val_full_df[
        "test_accuracy"
    ]
    .quantile(
        0.10
    )
    *
    100
)


worst10_df = (
    subject_val_full_df
    .sort_values(
        "test_accuracy",
        ascending=True,
    )
    .head(10)
)


worst10_mean = (
    worst10_df[
        "test_accuracy"
    ]
    .mean()
    *
    100
)


subjects_under_55 = int(
    (
        subject_val_full_df[
            "test_accuracy"
        ]
        <
        0.55
    ).sum()
)


subjects_under_60 = int(
    (
        subject_val_full_df[
            "test_accuracy"
        ]
        <
        0.60
    ).sum()
)


# ============================================================
# BASELINE
# ============================================================

BASELINE_MEAN = 69.177612

BASELINE_MEDIAN = 68.888889

BASELINE_STD = 13.185277

BASELINE_WORST10 = 48.20

CCST64_MEAN = 73.65


# ============================================================
# SUMMARY TABLE
# ============================================================

summary_df = pd.DataFrame({

    "method": [
        (
            "Transfer + SupCon + "
            "Subject-held-out validation"
        )
    ],

    "n_subjects": [
        subject_val_full_df[
            "test_subject"
        ]
        .nunique()
    ],

    "mean_accuracy_pct": [
        final_mean
    ],

    "median_accuracy_pct": [
        final_median
    ],

    "std_accuracy_pct": [
        final_std
    ],

    "balanced_accuracy_pct": [
        final_balanced
    ],

    "macro_f1_pct": [
        final_macro_f1
    ],

    "mean_kappa": [
        final_kappa
    ],

    "p10_accuracy_pct": [
        final_p10
    ],

    "worst10_mean_pct": [
        worst10_mean
    ],

    "subjects_under_55": [
        subjects_under_55
    ],

    "subjects_under_60": [
        subjects_under_60
    ],
})


display(
    summary_df
)


# ============================================================
# PAIRED COMPARISON AGAINST ORIGINAL FULL SUPCON
# ============================================================

paired = (
    full_df[
        [
            "test_subject",
            "test_accuracy",
        ]
    ]
    .rename(
        columns={
            "test_accuracy":
                "baseline"
        }
    )
    .merge(

        subject_val_full_df[
            [
                "test_subject",
                "test_accuracy",
            ]
        ]
        .rename(
            columns={
                "test_accuracy":
                    "subject_val"
            }
        ),

        on="test_subject",
        how="inner",
    )
)


paired[
    "delta_pp"
] = (
    paired[
        "subject_val"
    ]
    -
    paired[
        "baseline"
    ]
) * 100


wins = int(
    (
        paired[
            "delta_pp"
        ] > 0
    ).sum()
)


ties = int(
    np.isclose(
        paired[
            "delta_pp"
        ],
        0.0,
    ).sum()
)


losses = int(
    (
        paired[
            "delta_pp"
        ] < 0
    ).sum()
)


paired_median_delta = (
    paired[
        "delta_pp"
    ]
    .median()
)


# ============================================================
# WORST-10 DISPLAY
# ============================================================

worst10_display = (
    worst10_df[
        [
            "test_subject",
            "test_accuracy",
            "balanced_accuracy",
            "macro_f1",
        ]
    ]
    .copy()
)


worst10_display[
    "test_accuracy"
] *= 100


worst10_display[
    "balanced_accuracy"
] *= 100


worst10_display[
    "macro_f1"
] *= 100


display(
    worst10_display
)


# ============================================================
# FINAL PRINT
# ============================================================

print()
print("=" * 100)

print(
    f"BASELINE MEAN           : "
    f"{BASELINE_MEAN:.2f}%"
)

print(
    f"SUBJECT-VAL MEAN        : "
    f"{final_mean:.2f}%"
)

print(
    f"DELTA VS BASELINE       : "
    f"{final_mean-BASELINE_MEAN:+.2f} pp"
)

print()


print(
    f"BASELINE MEDIAN         : "
    f"{BASELINE_MEDIAN:.2f}%"
)

print(
    f"SUBJECT-VAL MEDIAN      : "
    f"{final_median:.2f}%"
)

print()


print(
    f"BASELINE STD            : "
    f"{BASELINE_STD:.2f} pp"
)

print(
    f"SUBJECT-VAL STD         : "
    f"{final_std:.2f} pp"
)

print(
    f"STD DELTA               : "
    f"{final_std-BASELINE_STD:+.2f} pp"
)

print()


print(
    f"BASELINE WORST-10       : "
    f"{BASELINE_WORST10:.2f}%"
)

print(
    f"SUBJECT-VAL WORST-10    : "
    f"{worst10_mean:.2f}%"
)

print(
    f"WORST-10 DELTA          : "
    f"{worst10_mean-BASELINE_WORST10:+.2f} pp"
)

print()


print(
    f"10th PERCENTILE         : "
    f"{final_p10:.2f}%"
)

print(
    f"SUBJECTS <55%           : "
    f"{subjects_under_55}"
)

print(
    f"SUBJECTS <60%           : "
    f"{subjects_under_60}"
)

print()


print(
    f"WINS / TIES / LOSSES    : "
    f"{wins} / {ties} / {losses}"
)

print(
    f"MEDIAN PAIRED DELTA     : "
    f"{paired_median_delta:+.2f} pp"
)

print()


print(
    f"CCST64                  : "
    f"{CCST64_MEAN:.2f}%"
)

print(
    f"DELTA VS CCST64         : "
    f"{final_mean-CCST64_MEAN:+.2f} pp"
)

print("=" * 100)


FULL 109-SUBJECT LOSO — SUBJECT-HELD-OUT VALIDATION
LR                   = 0.0015
SupCon weight        = 0.1
SupCon temperature   = 0.05
Validation subjects  = 10
Seed                 = 42

[1/109] SUBJECT-VAL SUPCON | TEST SUBJECT=S001
SUBJECT-VAL SPLIT | outer_test=S001 | train_subjects=98 | val_subjects=10 | train_trials=4423 | val_trials=450
SUPCON       | ep=01 | train=0.5989 | val=0.5822 | CE=0.670 | SC=4.198 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=02 | train=0.6159 | val=0.6089 | CE=0.642 | SC=4.139 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=03 | train=0.6249 | val=0.6400 | CE=0.638 | SC=4.138 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=04 | train=0.6285 | val=0.6044 | CE=0.631 | SC=4.136 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=05 | train=0.6437 | val=0.6178 | CE=0.624 | SC=4.134 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=06 | train=0.6279 | val=0.6244 | CE=0.622 | SC=4

,method,n_subjects,mean_accuracy_pct,median_accuracy_pct,std_accuracy_pct,balanced_accuracy_pct,macro_f1_pct,mean_kappa,p10_accuracy_pct,worst10_mean_pct,subjects_under_55,subjects_under_60
0,Transfer + SupCon + Subject-held-out validation,109,68.421832,68.888889,12.899139,68.496827,66.685235,0.369501,53.333333,46.888889,14,27


,test_subject,test_accuracy,balanced_accuracy,macro_f1
23,24,42.222222,42.984190,35.000000
98,99,42.222222,42.094862,41.964286
2,3,44.444444,45.256917,36.404748
35,36,44.444444,44.762846,43.438914
57,58,44.444444,47.321429,33.862434
100,101,48.888889,49.802372,39.153439
108,109,48.888889,50.000000,32.835821
66,67,51.111111,52.075099,40.504808
86,87,51.111111,50.592885,48.004202
78,79,51.111111,51.284585,50.892857



BASELINE MEAN           : 69.18%
SUBJECT-VAL MEAN        : 68.42%
DELTA VS BASELINE       : -0.76 pp

BASELINE MEDIAN         : 68.89%
SUBJECT-VAL MEDIAN      : 68.89%

BASELINE STD            : 13.19 pp
SUBJECT-VAL STD         : 12.90 pp
STD DELTA               : -0.29 pp

BASELINE WORST-10       : 48.20%
SUBJECT-VAL WORST-10    : 46.89%
WORST-10 DELTA          : -1.31 pp

10th PERCENTILE         : 53.33%
SUBJECTS <55%           : 14
SUBJECTS <60%           : 27

WINS / TIES / LOSSES    : 39 / 14 / 56
MEDIAN PAIRED DELTA     : -2.22 pp

CCST64                  : 73.65%
DELTA VS CCST64         : -5.23 pp


In [107]:
# ============================================================
# FIXED-EPOCH TRAINING
#
# EEGNet64
# + Cho Transfer
# + tuned SupCon
#
# NO:
#   - validation checkpoint selection
#   - early stopping
#   - ReduceLROnPlateau
#
# Все 108 source subjects используются для обучения.
# Test subject используется ТОЛЬКО один раз после обучения.
# ============================================================


def train_cho64_fixed_epochs_fold(
    test_subject,
    fixed_epochs,
    seed=42,
    batch_size=64,
    lr=1.5e-3,
    weight_decay=1e-4,
    supcon_weight=0.10,
    supcon_temperature=0.05,
):

    set_seed(seed)


    run_device = globals().get(
        "device",
        torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "mps"
            if torch.backends.mps.is_available()
            else "cpu"
        ),
    )


    # ========================================================
    # DATA
    # ========================================================

    # Используем именно исходную prepare-функцию,
    # если ранее создавали wrapper'ы для EA / subject-val.
    if "prepare_phys64_fold_supcon_grl_original" in globals():

        prepare_fn = (
            prepare_phys64_fold_supcon_grl_original
        )

    elif "BASE_PHYS64_PREPARE" in globals():

        prepare_fn = (
            BASE_PHYS64_PREPARE
        )

    else:

        prepare_fn = (
            prepare_phys64_fold_supcon_grl
        )


    fold = prepare_fn(
        test_subject=test_subject,
        seed=seed,
    )


    # ========================================================
    # ОБЪЕДИНЯЕМ СТАРЫЕ TRAIN + VAL
    #
    # Validation теперь вообще не нужна.
    # Используем ВСЕ source trials.
    # ========================================================

    X_train = np.concatenate(
        [
            fold["X_train"],
            fold["X_val"],
        ],
        axis=0,
    )


    y_train = np.concatenate(
        [
            fold["y_train"],
            fold["y_val"],
        ],
        axis=0,
    )


    s_train = np.concatenate(
        [
            fold["s_train"],
            fold["s_val"],
        ],
        axis=0,
    )


    # ========================================================
    # TEST
    # ========================================================

    X_test = fold[
        "X_test"
    ]

    y_test = fold[
        "y_test"
    ]


    s_test_dummy = np.zeros(
        len(y_test),
        dtype=np.int64,
    )


    # ========================================================
    # SANITY
    # ========================================================

    n_train_subjects = len(
        np.unique(
            s_train
        )
    )


    if n_train_subjects != fold[
        "n_source_subjects"
    ]:

        raise RuntimeError(
            f"Expected "
            f"{fold['n_source_subjects']} "
            f"source subjects, got "
            f"{n_train_subjects}."
        )


    print(
        f"FIXED-EPOCH DATA | "
        f"test=S{int(test_subject):03d} | "
        f"train_subjects={n_train_subjects} | "
        f"train_trials={len(y_train)} | "
        f"test_trials={len(y_test)}"
    )


    # ========================================================
    # DATASETS
    # ========================================================

    train_dataset = Phys64SubjectDataset(
        X_train,
        y_train,
        s_train,
        augment=True,
    )


    test_dataset = Phys64SubjectDataset(
        X_test,
        y_test,
        s_test_dummy,
        augment=False,
    )


    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        drop_last=False,
    )


    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        drop_last=False,
    )


    # ========================================================
    # MODEL
    # ========================================================

    model = make_cho64_supcon_grl_model(
        n_subjects=
            fold["n_source_subjects"]
    ).to(
        run_device
    )


    # ========================================================
    # LOSSES
    # ========================================================

    ce_loss_fn = nn.CrossEntropyLoss()


    supcon_loss_fn = SupConLoss(
        temperature=
            supcon_temperature
    )


    # ========================================================
    # OPTIMIZER
    #
    # LR фиксированный.
    # Scheduler не зависит от validation,
    # потому что validation больше нет.
    # ========================================================

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )


    # ========================================================
    # HISTORY
    # ========================================================

    history = []


    # ========================================================
    # TRAIN EXACTLY N EPOCHS
    # ========================================================

    for epoch in range(
        1,
        fixed_epochs + 1,
    ):

        model.train()


        total = 0
        correct = 0

        ce_sum = 0.0
        sc_sum = 0.0
        total_loss_sum = 0.0


        for (
            X,
            y,
            subject_ids,
        ) in train_loader:

            X = X.to(
                run_device,
                non_blocking=True,
            )

            y = y.to(
                run_device,
                non_blocking=True,
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            # =================================================
            # FORWARD
            # =================================================

            out = model(
                X,
                lambda_grl=0.0,
            )


            class_logits = (
                out["class_logits"]
            )


            projection = (
                out["projection"]
            )


            # =================================================
            # LOSSES
            # =================================================

            loss_ce = ce_loss_fn(
                class_logits,
                y,
            )


            loss_sc = supcon_loss_fn(
                projection,
                y,
            )


            loss = (
                loss_ce
                +
                supcon_weight
                *
                loss_sc
            )


            # =================================================
            # BACKWARD
            # =================================================

            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )


            optimizer.step()


            # =================================================
            # TRAIN METRICS
            # =================================================

            n = y.size(0)


            preds = class_logits.argmax(
                dim=1
            )


            correct += (
                preds == y
            ).sum().item()


            total += n


            ce_sum += (
                loss_ce.item()
                *
                n
            )


            sc_sum += (
                loss_sc.item()
                *
                n
            )


            total_loss_sum += (
                loss.item()
                *
                n
            )


        # ====================================================
        # EPOCH SUMMARY
        # ====================================================

        train_accuracy = (
            correct
            /
            max(total, 1)
        )


        train_ce = (
            ce_sum
            /
            max(total, 1)
        )


        train_sc = (
            sc_sum
            /
            max(total, 1)
        )


        train_loss = (
            total_loss_sum
            /
            max(total, 1)
        )


        history.append({

            "epoch":
                epoch,

            "train_accuracy":
                train_accuracy,

            "train_loss":
                train_loss,

            "ce_loss":
                train_ce,

            "supcon_loss":
                train_sc,

            "lr":
                lr,
        })


        print(
            f"FIXED-{fixed_epochs:02d} | "
            f"ep={epoch:02d}/{fixed_epochs:02d} | "
            f"train={train_accuracy:.4f} | "
            f"CE={train_ce:.3f} | "
            f"SC={train_sc:.3f} | "
            f"loss={train_loss:.3f}"
        )


    # ========================================================
    # IMPORTANT:
    #
    # Никакого best_state.
    # Берём модель ИМЕННО ПОСЛЕ последней эпохи.
    # ========================================================

    model.eval()


    # ========================================================
    # TEST
    # ========================================================

    test_true = []
    test_pred = []


    with torch.no_grad():

        for X, y, _ in test_loader:

            X = X.to(
                run_device,
                non_blocking=True,
            )

            y = y.to(
                run_device,
                non_blocking=True,
            )


            out = model(
                X,
                lambda_grl=0.0,
            )


            logits = (
                out["class_logits"]
            )


            preds = logits.argmax(
                dim=1
            )


            test_true.extend(
                y.cpu().numpy()
            )

            test_pred.extend(
                preds.cpu().numpy()
            )


    # ========================================================
    # METRICS
    # ========================================================

    test_accuracy = accuracy_score(
        test_true,
        test_pred,
    )


    balanced_accuracy = (
        balanced_accuracy_score(
            test_true,
            test_pred,
        )
    )


    macro_f1 = f1_score(
        test_true,
        test_pred,
        average="macro",
        zero_division=0,
    )


    kappa = cohen_kappa_score(
        test_true,
        test_pred,
    )


    result = {

        "test_subject":
            int(test_subject),

        "method":
            "transfer_supcon_fixed_epochs",

        "fixed_epochs":
            int(fixed_epochs),

        "test_accuracy":
            float(test_accuracy),

        "balanced_accuracy":
            float(balanced_accuracy),

        "macro_f1":
            float(macro_f1),

        "kappa":
            float(kappa),

        "lr":
            float(lr),

        "supcon_weight":
            float(supcon_weight),

        "supcon_temperature":
            float(supcon_temperature),

        "n_train_subjects":
            int(n_train_subjects),

        "n_train_trials":
            int(len(y_train)),

        "seed":
            int(seed),
    }


    return (
        result,
        model,
        pd.DataFrame(
            history
        ),
    )

In [108]:
# ============================================================
# ORIGINAL BASELINE WORST-10
# ============================================================


FIXED_WORST_N = 10


fixed_baseline_df = (
    full_df
    .sort_values(
        "test_accuracy",
        ascending=True,
    )
    .head(
        FIXED_WORST_N
    )
    [
        [
            "test_subject",
            "test_accuracy",
            "balanced_accuracy",
            "macro_f1",
        ]
    ]
    .copy()
)


fixed_baseline_df[
    "test_subject"
] = (
    fixed_baseline_df[
        "test_subject"
    ]
    .astype(int)
)


FIXED_WORST_SUBJECTS = (
    fixed_baseline_df[
        "test_subject"
    ]
    .tolist()
)


print(
    "Worst-10:",
    FIXED_WORST_SUBJECTS,
)


print(
    "Baseline worst-10 mean:",
    f"{fixed_baseline_df['test_accuracy'].mean()*100:.2f}%"
)


display(
    fixed_baseline_df.assign(
        accuracy_pct=
            fixed_baseline_df[
                "test_accuracy"
            ]
            *
            100
    )
)

Worst-10: [99, 24, 87, 3, 109, 67, 5, 88, 37, 69]
Baseline worst-10 mean: 48.20%


,test_subject,test_accuracy,balanced_accuracy,macro_f1,accuracy_pct
98,99,0.400000,0.405138,0.368831,40.000000
23,24,0.466667,0.477273,0.318182,46.666667
86,87,0.466667,0.460474,0.418103,46.666667
2,3,0.466667,0.468379,0.464286,46.666667
108,109,0.488889,0.500000,0.328358,48.888889
66,67,0.488889,0.500000,0.328358,48.888889
4,5,0.488889,0.464286,0.391534,48.888889
87,88,0.508772,0.510468,0.504963,50.877193
36,37,0.511111,0.500000,0.338235,51.111111
68,69,0.533333,0.556548,0.483324,53.333333


In [109]:
# ============================================================
# FIXED-EPOCH SCREENING
#
# 15 vs 20 vs 25 epochs
# on original worst-10 subjects
# ============================================================


FIXED_EPOCH_ROOT = (
    RESULTS_ROOT
    /
    "fixed_epoch_screening"
)


FIXED_EPOCH_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


FIXED_EPOCH_CSV = (
    FIXED_EPOCH_ROOT
    /
    "fixed_epoch_worst10.csv"
)


FIXED_EPOCH_OPTIONS = [
    15,
    20,
    25,
]


# ============================================================
# LOAD / RESUME
# ============================================================

if FIXED_EPOCH_CSV.exists():

    fixed_epoch_results = pd.read_csv(
        FIXED_EPOCH_CSV
    )

else:

    fixed_epoch_results = pd.DataFrame()


done = set()


if len(
    fixed_epoch_results
) > 0:

    done = set(
        zip(
            fixed_epoch_results[
                "fixed_epochs"
            ]
            .astype(int),

            fixed_epoch_results[
                "test_subject"
            ]
            .astype(int),
        )
    )


total_runs = (
    len(FIXED_EPOCH_OPTIONS)
    *
    len(FIXED_WORST_SUBJECTS)
)


run_idx = 0


print()
print("=" * 100)
print("FIXED-EPOCH SCREENING")
print("=" * 100)


for fixed_epochs in FIXED_EPOCH_OPTIONS:

    for subject in FIXED_WORST_SUBJECTS:

        run_idx += 1


        key = (
            int(fixed_epochs),
            int(subject),
        )


        if key in done:

            print(
                f"[{run_idx}/{total_runs}] "
                f"E={fixed_epochs} | "
                f"S{subject:03d} already done"
            )

            continue


        print()
        print("=" * 100)

        print(
            f"[{run_idx}/{total_runs}] "
            f"FIXED EPOCHS={fixed_epochs} | "
            f"TEST SUBJECT=S{subject:03d}"
        )

        print("=" * 100)


        result, model, history = (
            train_cho64_fixed_epochs_fold(

                test_subject=subject,

                fixed_epochs=
                    fixed_epochs,

                seed=42,

                batch_size=64,

                lr=1.5e-3,

                weight_decay=1e-4,

                supcon_weight=0.10,

                supcon_temperature=0.05,
            )
        )


        # ====================================================
        # BASELINE SAME SUBJECT
        # ====================================================

        baseline_acc = float(
            fixed_baseline_df.loc[

                fixed_baseline_df[
                    "test_subject"
                ]
                ==
                subject,

                "test_accuracy",
            ]
            .iloc[0]
        )


        result[
            "baseline_accuracy"
        ] = baseline_acc


        result[
            "delta_pp"
        ] = (
            result[
                "test_accuracy"
            ]
            -
            baseline_acc
        ) * 100


        # ====================================================
        # SAVE
        # ====================================================

        fixed_epoch_results = pd.concat(
            [
                fixed_epoch_results,

                pd.DataFrame(
                    [result]
                ),
            ],

            ignore_index=True,
        )


        fixed_epoch_results.to_csv(
            FIXED_EPOCH_CSV,
            index=False,
        )


        done.add(
            key
        )


        print()

        print(
            f"BASELINE = "
            f"{baseline_acc*100:.2f}%"
        )

        print(
            f"FIXED    = "
            f"{result['test_accuracy']*100:.2f}%"
        )

        print(
            f"DELTA    = "
            f"{result['delta_pp']:+.2f} pp"
        )


        # ====================================================
        # CLEANUP
        # ====================================================

        del model
        del history

        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


        if (
            hasattr(torch.backends, "mps")
            and
            torch.backends.mps.is_available()
        ):

            try:
                torch.mps.empty_cache()
            except Exception:
                pass


FIXED-EPOCH SCREENING

[1/30] FIXED EPOCHS=15 | TEST SUBJECT=S099
FIXED-EPOCH DATA | test=S099 | train_subjects=108 | train_trials=4873 | test_trials=45
FIXED-15 | ep=01/15 | train=0.5922 | CE=0.677 | SC=4.192 | loss=1.096
FIXED-15 | ep=02/15 | train=0.6218 | CE=0.642 | SC=4.140 | loss=1.056
FIXED-15 | ep=03/15 | train=0.6298 | CE=0.634 | SC=4.137 | loss=1.048
FIXED-15 | ep=04/15 | train=0.6417 | CE=0.626 | SC=4.135 | loss=1.040
FIXED-15 | ep=05/15 | train=0.6308 | CE=0.629 | SC=4.135 | loss=1.043
FIXED-15 | ep=06/15 | train=0.6405 | CE=0.622 | SC=4.133 | loss=1.035
FIXED-15 | ep=07/15 | train=0.6534 | CE=0.615 | SC=4.132 | loss=1.028
FIXED-15 | ep=08/15 | train=0.6561 | CE=0.615 | SC=4.131 | loss=1.028
FIXED-15 | ep=09/15 | train=0.6364 | CE=0.619 | SC=4.133 | loss=1.033
FIXED-15 | ep=10/15 | train=0.6624 | CE=0.606 | SC=4.128 | loss=1.019
FIXED-15 | ep=11/15 | train=0.6657 | CE=0.599 | SC=4.128 | loss=1.012
FIXED-15 | ep=12/15 | train=0.6710 | CE=0.595 | SC=4.127 | loss=1.008
FIXED-

In [110]:
# ============================================================
# FIXED-EPOCH SUMMARY
# ============================================================


fixed_summary = (
    fixed_epoch_results
    .groupby(
        "fixed_epochs"
    )
    .agg(

        n_subjects=(
            "test_subject",
            "nunique",
        ),

        mean_accuracy=(
            "test_accuracy",
            "mean",
        ),

        median_accuracy=(
            "test_accuracy",
            "median",
        ),

        std_accuracy=(
            "test_accuracy",
            "std",
        ),

        balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),

        macro_f1=(
            "macro_f1",
            "mean",
        ),

        mean_delta_pp=(
            "delta_pp",
            "mean",
        ),

        median_delta_pp=(
            "delta_pp",
            "median",
        ),
    )
    .reset_index()
)


# Только полностью законченные конфигурации
fixed_summary = (
    fixed_summary[
        fixed_summary[
            "n_subjects"
        ]
        ==
        len(
            FIXED_WORST_SUBJECTS
        )
    ]
    .copy()
)


fixed_summary[
    "mean_accuracy_pct"
] = (
    fixed_summary[
        "mean_accuracy"
    ]
    *
    100
)


fixed_summary[
    "median_accuracy_pct"
] = (
    fixed_summary[
        "median_accuracy"
    ]
    *
    100
)


fixed_summary[
    "std_accuracy_pct"
] = (
    fixed_summary[
        "std_accuracy"
    ]
    *
    100
)


fixed_summary[
    "balanced_accuracy_pct"
] = (
    fixed_summary[
        "balanced_accuracy"
    ]
    *
    100
)


fixed_summary[
    "macro_f1_pct"
] = (
    fixed_summary[
        "macro_f1"
    ]
    *
    100
)


# ============================================================
# Дополнительные показатели
# ============================================================

extra_rows = []


for fixed_epochs in sorted(
    fixed_epoch_results[
        "fixed_epochs"
    ].unique()
):

    part = (
        fixed_epoch_results[
            fixed_epoch_results[
                "fixed_epochs"
            ]
            ==
            fixed_epochs
        ]
    )


    if len(part) < len(
        FIXED_WORST_SUBJECTS
    ):
        continue


    wins = int(
        (
            part[
                "delta_pp"
            ] > 0
        ).sum()
    )


    ties = int(
        np.isclose(
            part[
                "delta_pp"
            ],
            0.0,
        ).sum()
    )


    losses = int(
        (
            part[
                "delta_pp"
            ] < 0
        ).sum()
    )


    under_55 = int(
        (
            part[
                "test_accuracy"
            ]
            <
            0.55
        ).sum()
    )


    under_60 = int(
        (
            part[
                "test_accuracy"
            ]
            <
            0.60
        ).sum()
    )


    extra_rows.append({

        "fixed_epochs":
            int(fixed_epochs),

        "wins":
            wins,

        "ties":
            ties,

        "losses":
            losses,

        "subjects_under_55":
            under_55,

        "subjects_under_60":
            under_60,
    })


extra_df = pd.DataFrame(
    extra_rows
)


fixed_summary = (
    fixed_summary
    .merge(
        extra_df,
        on="fixed_epochs",
        how="left",
    )
)


fixed_summary = (
    fixed_summary
    .sort_values(
        [
            "mean_accuracy",
            "median_accuracy",
            "balanced_accuracy",
        ],
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


display(
    fixed_summary[
        [
            "fixed_epochs",
            "n_subjects",
            "mean_accuracy_pct",
            "median_accuracy_pct",
            "std_accuracy_pct",
            "balanced_accuracy_pct",
            "macro_f1_pct",
            "mean_delta_pp",
            "median_delta_pp",
            "wins",
            "ties",
            "losses",
            "subjects_under_55",
            "subjects_under_60",
        ]
    ]
)


# ============================================================
# BEST
# ============================================================

if len(
    fixed_summary
) == 0:

    raise RuntimeError(
        "No complete fixed-epoch configuration yet."
    )


BEST_FIXED_EPOCHS = int(
    fixed_summary.iloc[0][
        "fixed_epochs"
    ]
)


BEST_FIXED_MEAN = float(
    fixed_summary.iloc[0][
        "mean_accuracy_pct"
    ]
)


baseline_worst10_mean = (
    fixed_baseline_df[
        "test_accuracy"
    ]
    .mean()
    *
    100
)


print()
print("=" * 100)

print(
    f"BASELINE WORST-10 MEAN : "
    f"{baseline_worst10_mean:.2f}%"
)

print(
    f"BEST FIXED EPOCHS       : "
    f"{BEST_FIXED_EPOCHS}"
)

print(
    f"BEST FIXED WORST-10     : "
    f"{BEST_FIXED_MEAN:.2f}%"
)

print(
    f"DELTA                   : "
    f"{BEST_FIXED_MEAN-baseline_worst10_mean:+.2f} pp"
)

print("=" * 100)

,fixed_epochs,n_subjects,mean_accuracy_pct,median_accuracy_pct,std_accuracy_pct,balanced_accuracy_pct,macro_f1_pct,mean_delta_pp,median_delta_pp,wins,ties,losses,subjects_under_55,subjects_under_60
0,20,10,52.023392,51.111111,5.838016,51.839101,46.012199,3.824561,3.333333,6,2,2,6,9
1,15,10,49.532164,49.883041,6.370483,49.623228,43.869003,1.333333,0.000000,4,4,2,8,9
2,25,10,49.181287,50.000000,7.193009,49.134962,42.633653,0.982456,0.000000,4,2,4,9,9



BASELINE WORST-10 MEAN : 48.20%
BEST FIXED EPOCHS       : 20
BEST FIXED WORST-10     : 52.02%
DELTA                   : +3.82 pp


In [111]:
# ============================================================
# FULL 109-SUBJECT LOSO
#
# EEGNet64
# + Cho Transfer
# + tuned SupCon
# + FIXED 20 EPOCHS
#
# NO validation
# NO early stopping
# NO best-checkpoint selection
#
# Каждый fold:
#   108 source subjects -> train
#   1 unseen subject    -> test
#
# Финальная модель = состояние ПОСЛЕ 20-й эпохи.
# ============================================================


FIXED20_ROOT = (
    RESULTS_ROOT
    /
    "fixed20_full_109"
)


FIXED20_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


FIXED20_CSV = (
    FIXED20_ROOT
    /
    "fixed20_full_109.csv"
)


# ============================================================
# FIXED CONFIG
# ============================================================

FIXED_EPOCHS = 20

FIXED_LR = 1.5e-3

FIXED_SUPCON_WEIGHT = 0.10

FIXED_SUPCON_TEMPERATURE = 0.05

FIXED_SEED = 42


FULL_SUBJECTS = list(
    range(
        1,
        110,
    )
)


# ============================================================
# LOAD / RESUME
# ============================================================

if FIXED20_CSV.exists():

    fixed20_full_df = pd.read_csv(
        FIXED20_CSV
    )

    done_subjects = set(
        fixed20_full_df[
            "test_subject"
        ]
        .astype(int)
        .tolist()
    )

    print(
        f"Resume: "
        f"{len(done_subjects)}/109 "
        f"subjects completed"
    )

else:

    fixed20_full_df = pd.DataFrame()

    done_subjects = set()


# ============================================================
# START
# ============================================================

print()
print("=" * 100)

print(
    "FULL 109-SUBJECT LOSO — "
    "FIXED 20 EPOCHS"
)

print("=" * 100)

print(
    f"Epochs               = {FIXED_EPOCHS}"
)

print(
    f"LR                   = {FIXED_LR}"
)

print(
    f"SupCon weight        = {FIXED_SUPCON_WEIGHT}"
)

print(
    f"SupCon temperature   = {FIXED_SUPCON_TEMPERATURE}"
)

print(
    f"Seed                 = {FIXED_SEED}"
)

print("=" * 100)


# ============================================================
# FULL LOSO
# ============================================================

for idx, subject in enumerate(
    FULL_SUBJECTS,
    start=1,
):

    if subject in done_subjects:

        print(
            f"[{idx}/109] "
            f"S{subject:03d} already done"
        )

        continue


    print()
    print("=" * 100)

    print(
        f"[{idx}/109] "
        f"FIXED-20 SUPCON | "
        f"TEST SUBJECT=S{subject:03d}"
    )

    print("=" * 100)


    result, model, history = (
        train_cho64_fixed_epochs_fold(

            test_subject=subject,

            fixed_epochs=
                FIXED_EPOCHS,

            seed=
                FIXED_SEED,

            batch_size=64,

            lr=
                FIXED_LR,

            weight_decay=1e-4,

            supcon_weight=
                FIXED_SUPCON_WEIGHT,

            supcon_temperature=
                FIXED_SUPCON_TEMPERATURE,
        )
    )


    # ========================================================
    # METHOD INFO
    # ========================================================

    result[
        "method"
    ] = (
        "transfer_supcon_fixed20"
    )


    result[
        "full_loso"
    ] = True


    # ========================================================
    # PAIRED BASELINE
    # ========================================================

    baseline_rows = (
        full_df[
            full_df[
                "test_subject"
            ]
            .astype(int)
            ==
            int(subject)
        ]
    )


    if len(
        baseline_rows
    ) > 0:

        baseline_acc = float(
            baseline_rows[
                "test_accuracy"
            ]
            .iloc[0]
        )


        result[
            "baseline_accuracy"
        ] = baseline_acc


        result[
            "delta_vs_baseline_pp"
        ] = (
            result[
                "test_accuracy"
            ]
            -
            baseline_acc
        ) * 100

    else:

        result[
            "baseline_accuracy"
        ] = np.nan


        result[
            "delta_vs_baseline_pp"
        ] = np.nan


    # ========================================================
    # SAVE AFTER EVERY SUBJECT
    # ========================================================

    fixed20_full_df = pd.concat(
        [
            fixed20_full_df,

            pd.DataFrame(
                [result]
            ),
        ],

        ignore_index=True,
    )


    fixed20_full_df.to_csv(
        FIXED20_CSV,
        index=False,
    )


    done_subjects.add(
        int(subject)
    )


    # ========================================================
    # RUNNING SUMMARY
    # ========================================================

    running_mean = (
        fixed20_full_df[
            "test_accuracy"
        ]
        .mean()
        *
        100
    )


    running_median = (
        fixed20_full_df[
            "test_accuracy"
        ]
        .median()
        *
        100
    )


    running_std = (
        fixed20_full_df[
            "test_accuracy"
        ]
        .std()
        *
        100
    )


    running_p10 = (
        fixed20_full_df[
            "test_accuracy"
        ]
        .quantile(
            0.10
        )
        *
        100
    )


    running_under_55 = int(
        (
            fixed20_full_df[
                "test_accuracy"
            ]
            <
            0.55
        ).sum()
    )


    running_under_60 = int(
        (
            fixed20_full_df[
                "test_accuracy"
            ]
            <
            0.60
        ).sum()
    )


    print()

    print(
        f"SUBJECT RESULT = "
        f"{result['test_accuracy']*100:.2f}%"
    )


    if not np.isnan(
        result[
            "baseline_accuracy"
        ]
    ):

        print(
            f"BASELINE       = "
            f"{result['baseline_accuracy']*100:.2f}%"
        )

        print(
            f"DELTA          = "
            f"{result['delta_vs_baseline_pp']:+.2f} pp"
        )


    print(
        f"RUNNING MEAN   = "
        f"{running_mean:.2f}%"
    )

    print(
        f"RUNNING MEDIAN = "
        f"{running_median:.2f}%"
    )

    print(
        f"RUNNING STD    = "
        f"{running_std:.2f} pp"
    )

    print(
        f"RUNNING P10    = "
        f"{running_p10:.2f}%"
    )

    print(
        f"SUBJECTS <55%  = "
        f"{running_under_55}"
    )

    print(
        f"SUBJECTS <60%  = "
        f"{running_under_60}"
    )

    print(
        f"PROGRESS       = "
        f"{len(done_subjects)}/109"
    )


    # ========================================================
    # CLEANUP
    # ========================================================

    del model
    del history

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    if (
        hasattr(torch.backends, "mps")
        and
        torch.backends.mps.is_available()
    ):

        try:
            torch.mps.empty_cache()
        except Exception:
            pass


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 100)

print(
    "FULL FIXED-20 LOSO FINISHED"
)

print("=" * 100)


final_mean = (
    fixed20_full_df[
        "test_accuracy"
    ]
    .mean()
    *
    100
)


final_median = (
    fixed20_full_df[
        "test_accuracy"
    ]
    .median()
    *
    100
)


final_std = (
    fixed20_full_df[
        "test_accuracy"
    ]
    .std()
    *
    100
)


final_balanced = (
    fixed20_full_df[
        "balanced_accuracy"
    ]
    .mean()
    *
    100
)


final_macro_f1 = (
    fixed20_full_df[
        "macro_f1"
    ]
    .mean()
    *
    100
)


final_kappa = (
    fixed20_full_df[
        "kappa"
    ]
    .mean()
)


final_p10 = (
    fixed20_full_df[
        "test_accuracy"
    ]
    .quantile(
        0.10
    )
    *
    100
)


worst10_df = (
    fixed20_full_df
    .sort_values(
        "test_accuracy",
        ascending=True,
    )
    .head(10)
)


worst10_mean = (
    worst10_df[
        "test_accuracy"
    ]
    .mean()
    *
    100
)


subjects_under_55 = int(
    (
        fixed20_full_df[
            "test_accuracy"
        ]
        <
        0.55
    ).sum()
)


subjects_under_60 = int(
    (
        fixed20_full_df[
            "test_accuracy"
        ]
        <
        0.60
    ).sum()
)


# ============================================================
# ORIGINAL BASELINE
# ============================================================

BASELINE_MEAN = 69.177612

BASELINE_MEDIAN = 68.888889

BASELINE_STD = 13.185277

BASELINE_WORST10 = 48.20

CCST64_MEAN = 73.65


# ============================================================
# PAIRED COMPARISON
# ============================================================

paired = (
    full_df[
        [
            "test_subject",
            "test_accuracy",
        ]
    ]
    .rename(
        columns={
            "test_accuracy":
                "baseline"
        }
    )
    .merge(

        fixed20_full_df[
            [
                "test_subject",
                "test_accuracy",
            ]
        ]
        .rename(
            columns={
                "test_accuracy":
                    "fixed20"
            }
        ),

        on="test_subject",
        how="inner",
    )
)


paired[
    "delta_pp"
] = (
    paired[
        "fixed20"
    ]
    -
    paired[
        "baseline"
    ]
) * 100


wins = int(
    (
        paired[
            "delta_pp"
        ] > 0
    ).sum()
)


ties = int(
    np.isclose(
        paired[
            "delta_pp"
        ],
        0.0,
    ).sum()
)


losses = int(
    (
        paired[
            "delta_pp"
        ] < 0
    ).sum()
)


median_delta = (
    paired[
        "delta_pp"
    ]
    .median()
)


# ============================================================
# SUMMARY TABLE
# ============================================================

summary_df = pd.DataFrame({

    "method": [
        "Transfer + SupCon + Fixed 20 epochs"
    ],

    "n_subjects": [
        fixed20_full_df[
            "test_subject"
        ]
        .nunique()
    ],

    "mean_accuracy_pct": [
        final_mean
    ],

    "median_accuracy_pct": [
        final_median
    ],

    "std_accuracy_pct": [
        final_std
    ],

    "balanced_accuracy_pct": [
        final_balanced
    ],

    "macro_f1_pct": [
        final_macro_f1
    ],

    "mean_kappa": [
        final_kappa
    ],

    "p10_accuracy_pct": [
        final_p10
    ],

    "worst10_mean_pct": [
        worst10_mean
    ],

    "subjects_under_55": [
        subjects_under_55
    ],

    "subjects_under_60": [
        subjects_under_60
    ],
})


display(
    summary_df
)


# ============================================================
# WORST-10
# ============================================================

worst10_display = (
    worst10_df[
        [
            "test_subject",
            "test_accuracy",
            "balanced_accuracy",
            "macro_f1",
        ]
    ]
    .copy()
)


worst10_display[
    "test_accuracy"
] *= 100


worst10_display[
    "balanced_accuracy"
] *= 100


worst10_display[
    "macro_f1"
] *= 100


display(
    worst10_display
)


# ============================================================
# FINAL COMPARISON
# ============================================================

print()
print("=" * 100)

print(
    f"BASELINE MEAN        : "
    f"{BASELINE_MEAN:.2f}%"
)

print(
    f"FIXED-20 MEAN        : "
    f"{final_mean:.2f}%"
)

print(
    f"DELTA VS BASELINE    : "
    f"{final_mean-BASELINE_MEAN:+.2f} pp"
)

print()


print(
    f"BASELINE MEDIAN      : "
    f"{BASELINE_MEDIAN:.2f}%"
)

print(
    f"FIXED-20 MEDIAN      : "
    f"{final_median:.2f}%"
)

print()


print(
    f"BASELINE STD         : "
    f"{BASELINE_STD:.2f} pp"
)

print(
    f"FIXED-20 STD         : "
    f"{final_std:.2f} pp"
)

print(
    f"STD DELTA            : "
    f"{final_std-BASELINE_STD:+.2f} pp"
)

print()


print(
    f"BASELINE WORST-10    : "
    f"{BASELINE_WORST10:.2f}%"
)

print(
    f"FIXED-20 WORST-10    : "
    f"{worst10_mean:.2f}%"
)

print(
    f"WORST-10 DELTA       : "
    f"{worst10_mean-BASELINE_WORST10:+.2f} pp"
)

print()


print(
    f"10th PERCENTILE      : "
    f"{final_p10:.2f}%"
)

print(
    f"SUBJECTS <55%        : "
    f"{subjects_under_55}"
)

print(
    f"SUBJECTS <60%        : "
    f"{subjects_under_60}"
)

print()


print(
    f"WINS / TIES / LOSSES : "
    f"{wins} / {ties} / {losses}"
)

print(
    f"MEDIAN PAIRED DELTA  : "
    f"{median_delta:+.2f} pp"
)

print()


print(
    f"CCST64               : "
    f"{CCST64_MEAN:.2f}%"
)

print(
    f"DELTA VS CCST64      : "
    f"{final_mean-CCST64_MEAN:+.2f} pp"
)

print("=" * 100)


FULL 109-SUBJECT LOSO — FIXED 20 EPOCHS
Epochs               = 20
LR                   = 0.0015
SupCon weight        = 0.1
SupCon temperature   = 0.05
Seed                 = 42

[1/109] FIXED-20 SUPCON | TEST SUBJECT=S001
FIXED-EPOCH DATA | test=S001 | train_subjects=108 | train_trials=4873 | test_trials=45
FIXED-20 | ep=01/20 | train=0.5812 | CE=0.678 | SC=4.192 | loss=1.098
FIXED-20 | ep=02/20 | train=0.6187 | CE=0.645 | SC=4.140 | loss=1.059
FIXED-20 | ep=03/20 | train=0.6245 | CE=0.635 | SC=4.138 | loss=1.049
FIXED-20 | ep=04/20 | train=0.6353 | CE=0.630 | SC=4.136 | loss=1.044
FIXED-20 | ep=05/20 | train=0.6392 | CE=0.623 | SC=4.135 | loss=1.037
FIXED-20 | ep=06/20 | train=0.6329 | CE=0.621 | SC=4.134 | loss=1.034
FIXED-20 | ep=07/20 | train=0.6513 | CE=0.616 | SC=4.133 | loss=1.029
FIXED-20 | ep=08/20 | train=0.6559 | CE=0.623 | SC=4.132 | loss=1.036
FIXED-20 | ep=09/20 | train=0.6548 | CE=0.616 | SC=4.131 | loss=1.029
FIXED-20 | ep=10/20 | train=0.6454 | CE=0.611 | SC=4.131 | l

,method,n_subjects,mean_accuracy_pct,median_accuracy_pct,std_accuracy_pct,balanced_accuracy_pct,macro_f1_pct,mean_kappa,p10_accuracy_pct,worst10_mean_pct,subjects_under_55,subjects_under_60
0,Transfer + SupCon + Fixed 20 epochs,109,68.53745,68.888889,13.531951,68.109876,65.256781,0.363204,51.111111,48.356725,19,32


,test_subject,test_accuracy,balanced_accuracy,macro_f1
23,24,42.222222,42.588933,40.789474
35,36,46.666667,46.739130,46.640316
98,99,46.666667,46.837945,46.428571
108,109,48.888889,50.000000,32.835821
57,58,48.888889,49.702381,48.481832
10,11,48.888889,49.901186,36.307692
87,88,49.122807,49.076355,49.060092
99,100,50.000000,50.000000,48.571429
75,76,51.111111,50.197628,40.504808
49,50,51.111111,50.098814,37.500000



BASELINE MEAN        : 69.18%
FIXED-20 MEAN        : 68.54%
DELTA VS BASELINE    : -0.64 pp

BASELINE MEDIAN      : 68.89%
FIXED-20 MEDIAN      : 68.89%

BASELINE STD         : 13.19 pp
FIXED-20 STD         : 13.53 pp
STD DELTA            : +0.35 pp

BASELINE WORST-10    : 48.20%
FIXED-20 WORST-10    : 48.36%
WORST-10 DELTA       : +0.16 pp

10th PERCENTILE      : 51.11%
SUBJECTS <55%        : 19
SUBJECTS <60%        : 32

WINS / TIES / LOSSES : 43 / 15 / 51
MEDIAN PAIRED DELTA  : +0.00 pp

CCST64               : 73.65%
DELTA VS CCST64      : -5.11 pp


In [112]:
# ============================================================
# 64-CHANNEL TASK-AWARE DENOISER
#
# Loss:
#   MSE
#   + 0.10 * PSD
#   + 0.25 * classifier CE
#
# Train SNR:
#   random [-5, +5] dB
#
# Validation / Test:
#   -5 dB
# ============================================================


if "MultichannelResidualDenoiser" not in globals():

    class MultichannelResidualDenoiser(nn.Module):

        def __init__(
            self,
            n_channels=64,
            hidden=64,
        ):
            super().__init__()

            self.net = nn.Sequential(

                nn.Conv1d(
                    n_channels,
                    hidden,
                    kernel_size=7,
                    padding=3,
                ),
                nn.BatchNorm1d(hidden),
                nn.ELU(),

                nn.Conv1d(
                    hidden,
                    hidden,
                    kernel_size=5,
                    padding=2,
                ),
                nn.BatchNorm1d(hidden),
                nn.ELU(),

                nn.Conv1d(
                    hidden,
                    hidden,
                    kernel_size=5,
                    padding=2,
                ),
                nn.BatchNorm1d(hidden),
                nn.ELU(),

                nn.Conv1d(
                    hidden,
                    n_channels,
                    kernel_size=7,
                    padding=3,
                ),
            )


        def forward(
            self,
            noisy,
        ):

            predicted_noise = self.net(
                noisy
            )

            return (
                noisy
                -
                predicted_noise
            )


def add_awgn_torch(
    clean,
    snr_db,
):

    signal_power = (
        clean.pow(2)
        .mean(
            dim=(-2, -1),
            keepdim=True,
        )
        .clamp_min(1e-8)
    )


    snr_linear = (
        10.0
        **
        (
            snr_db
            /
            10.0
        )
    )


    if not torch.is_tensor(
        snr_linear
    ):

        snr_linear = torch.tensor(
            snr_linear,
            dtype=clean.dtype,
            device=clean.device,
        )


    while (
        snr_linear.ndim
        <
        clean.ndim
    ):

        snr_linear = (
            snr_linear.unsqueeze(-1)
        )


    noise_power = (
        signal_power
        /
        snr_linear
    )


    noise = torch.randn_like(
        clean
    )


    noise_current_power = (
        noise.pow(2)
        .mean(
            dim=(-2, -1),
            keepdim=True,
        )
        .clamp_min(1e-8)
    )


    noise = (
        noise
        *
        torch.sqrt(
            noise_power
            /
            noise_current_power
        )
    )


    return (
        clean
        +
        noise
    )


def mi_psd_loss_64(
    predicted,
    target,
    fs=160.0,
    fmin=8.0,
    fmax=30.0,
    eps=1e-8,
):

    n_times = predicted.shape[-1]


    freqs = torch.fft.rfftfreq(
        n_times,
        d=1.0 / fs,
        device=predicted.device,
    )


    band_mask = (
        (freqs >= fmin)
        &
        (freqs <= fmax)
    )


    pred_fft = torch.fft.rfft(
        predicted,
        dim=-1,
    )


    target_fft = torch.fft.rfft(
        target,
        dim=-1,
    )


    pred_power = (
        pred_fft.abs().pow(2)[
            ...,
            band_mask
        ]
    )


    target_power = (
        target_fft.abs().pow(2)[
            ...,
            band_mask
        ]
    )


    # normalized PSD inside MI band

    pred_power = (
        pred_power
        /
        (
            pred_power.sum(
                dim=-1,
                keepdim=True,
            )
            +
            eps
        )
    )


    target_power = (
        target_power
        /
        (
            target_power.sum(
                dim=-1,
                keepdim=True,
            )
            +
            eps
        )
    )


    return torch.nn.functional.mse_loss(
        torch.log(
            pred_power + eps
        ),
        torch.log(
            target_power + eps
        ),
    )

In [113]:
# ============================================================
# FROZEN CLASSIFIER FOR TASK LOSS
# ============================================================


def frozen_classifier_logits(
    classifier,
    signals,
):

    # denoiser:
    # [B, 64, 481]
    #
    # EEGNet:
    # [B, 1, 64, 481]

    classifier_input = (
        signals.unsqueeze(1)
    )


    out = classifier(
        classifier_input,
        lambda_grl=0.0,
    )


    return out[
        "class_logits"
    ]

In [114]:
# ============================================================
# FROZEN CLASSIFIER FOR TASK LOSS
# ============================================================


def frozen_classifier_logits(
    classifier,
    signals,
):

    # denoiser:
    # [B, 64, 481]
    #
    # EEGNet:
    # [B, 1, 64, 481]

    classifier_input = (
        signals.unsqueeze(1)
    )


    out = classifier(
        classifier_input,
        lambda_grl=0.0,
    )


    return out[
        "class_logits"
    ]

In [115]:
# ============================================================
# ONE LOSO FOLD:
#
# ORIGINAL BEST CLASSIFIER
# Transfer + tuned SupCon
#
# +
#
# TASK-AWARE DENOISER
# ============================================================


def train_taskaware_denoiser64_fold(
    test_subject,
    seed=42,

    denoiser_epochs=80,
    denoiser_patience=15,

    lambda_psd=0.10,
    lambda_task=0.25,

    denoiser_lr=1e-3,
    denoiser_weight_decay=1e-4,

    batch_size=64,
):

    set_seed(
        seed
    )


    run_device = globals().get(
        "device",
        torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "mps"
            if torch.backends.mps.is_available()
            else "cpu"
        ),
    )


    print()
    print("=" * 100)
    print(
        f"STAGE 1 — BEST CLASSIFIER | "
        f"S{int(test_subject):03d}"
    )
    print("=" * 100)


    # ========================================================
    # ORIGINAL BEST 64ch CLASSIFIER
    #
    # НЕ fixed20
    # НЕ class alignment
    # НЕ subject-val
    # ========================================================

    classifier_result, classifier, _ = (
        train_cho64_ablation_fold(

            test_subject=
                test_subject,

            mode="supcon",

            seed=seed,

            max_epochs=40,

            patience=8,

            batch_size=64,

            lr=1.5e-3,

            weight_decay=1e-4,

            supcon_weight=0.10,

            supcon_temperature=0.05,
        )
    )


    classifier = classifier.to(
        run_device
    )


    classifier.eval()


    for param in classifier.parameters():

        param.requires_grad = False


    # ========================================================
    # SAME FOLD DATA
    # ========================================================

    if (
        "prepare_phys64_fold_supcon_grl_original"
        in globals()
    ):

        prepare_fn = (
            prepare_phys64_fold_supcon_grl_original
        )

    elif (
        "BASE_PHYS64_PREPARE"
        in globals()
    ):

        prepare_fn = (
            BASE_PHYS64_PREPARE
        )

    else:

        prepare_fn = (
            prepare_phys64_fold_supcon_grl
        )


    fold = prepare_fn(
        test_subject=test_subject,
        seed=seed,
    )


    # ========================================================
    # DENOISER DATA
    # ========================================================

    train_dataset = (
        FinalTaskAwareDenoisingDataset(

            fold["X_train"],
            fold["y_train"],

            train=True,

            seed=seed,
        )
    )


    val_dataset = (
        FinalTaskAwareDenoisingDataset(

            fold["X_val"],
            fold["y_val"],

            train=False,

            fixed_snr_db=-5.0,

            seed=
                seed
                +
                100000,
        )
    )


    train_loader = DataLoader(

        train_dataset,

        batch_size=batch_size,

        shuffle=True,

        num_workers=0,
    )


    val_loader = DataLoader(

        val_dataset,

        batch_size=batch_size,

        shuffle=False,

        num_workers=0,
    )


    # ========================================================
    # DENOISER
    # ========================================================

    denoiser = (
        MultichannelResidualDenoiser(

            n_channels=64,

            hidden=64,
        )
        .to(
            run_device
        )
    )


    optimizer = torch.optim.AdamW(

        denoiser.parameters(),

        lr=denoiser_lr,

        weight_decay=
            denoiser_weight_decay,
    )


    ce_fn = nn.CrossEntropyLoss()


    # ========================================================
    # BEST DENOISER STATE
    #
    # Primary criterion:
    # classifier accuracy after denoising
    # ========================================================

    best_val_accuracy = -np.inf

    best_val_loss = np.inf

    best_epoch = 0

    best_state = None

    no_improvement = 0

    history = []


    print()
    print("=" * 100)
    print(
        f"STAGE 2 — TASK-AWARE DENOISER | "
        f"S{int(test_subject):03d}"
    )
    print("=" * 100)


    # ========================================================
    # TRAIN
    # ========================================================

    for epoch in range(
        1,
        denoiser_epochs + 1,
    ):

        denoiser.train()


        total_sum = 0.0
        mse_sum = 0.0
        psd_sum = 0.0
        task_sum = 0.0

        n_train = 0


        for (
            noisy_norm,
            clean_norm,
            scale,
            labels,
        ) in train_loader:

            noisy_norm = noisy_norm.to(
                run_device
            )

            clean_norm = clean_norm.to(
                run_device
            )

            scale = scale.to(
                run_device
            )

            labels = labels.to(
                run_device
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            # =================================================
            # DENOISE
            # =================================================

            clean_hat_norm = denoiser(
                noisy_norm
            )


            # =================================================
            # RECONSTRUCTION
            # =================================================

            loss_mse = (
                torch.nn.functional
                .mse_loss(
                    clean_hat_norm,
                    clean_norm,
                )
            )


            # =================================================
            # MI PSD
            # =================================================

            loss_psd = (
                mi_psd_loss_64(
                    clean_hat_norm,
                    clean_norm,
                    fs=160.0,
                )
            )


            # =================================================
            # RESTORE ORIGINAL SCALE
            # =================================================

            restored = (
                clean_hat_norm
                *
                scale.view(
                    -1,
                    1,
                    1,
                )
            )


            # =================================================
            # TASK LOSS
            #
            # classifier frozen,
            # but gradient flows THROUGH it
            # into denoiser.
            # =================================================

            logits = (
                frozen_classifier_logits(
                    classifier,
                    restored,
                )
            )


            loss_task = ce_fn(
                logits,
                labels,
            )


            # =================================================
            # FINAL LOSS
            # =================================================

            loss = (
                loss_mse
                +
                lambda_psd
                *
                loss_psd
                +
                lambda_task
                *
                loss_task
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                denoiser.parameters(),
                max_norm=5.0,
            )


            optimizer.step()


            batch_n = (
                labels.size(0)
            )


            n_train += batch_n


            total_sum += (
                loss.item()
                *
                batch_n
            )


            mse_sum += (
                loss_mse.item()
                *
                batch_n
            )


            psd_sum += (
                loss_psd.item()
                *
                batch_n
            )


            task_sum += (
                loss_task.item()
                *
                batch_n
            )


        # ====================================================
        # VALIDATION @ -5 dB
        # ====================================================

        denoiser.eval()


        val_true = []

        val_pred = []

        val_loss_sum = 0.0

        n_val = 0


        with torch.no_grad():

            for (
                noisy_norm,
                clean_norm,
                scale,
                labels,
            ) in val_loader:

                noisy_norm = noisy_norm.to(
                    run_device
                )

                clean_norm = clean_norm.to(
                    run_device
                )

                scale = scale.to(
                    run_device
                )

                labels = labels.to(
                    run_device
                )


                clean_hat_norm = denoiser(
                    noisy_norm
                )


                loss_mse = (
                    torch.nn.functional
                    .mse_loss(
                        clean_hat_norm,
                        clean_norm,
                    )
                )


                loss_psd = (
                    mi_psd_loss_64(
                        clean_hat_norm,
                        clean_norm,
                        fs=160.0,
                    )
                )


                restored = (
                    clean_hat_norm
                    *
                    scale.view(
                        -1,
                        1,
                        1,
                    )
                )


                logits = (
                    frozen_classifier_logits(
                        classifier,
                        restored,
                    )
                )


                loss_task = ce_fn(
                    logits,
                    labels,
                )


                val_loss = (
                    loss_mse
                    +
                    lambda_psd
                    *
                    loss_psd
                    +
                    lambda_task
                    *
                    loss_task
                )


                preds = logits.argmax(
                    dim=1
                )


                val_true.extend(
                    labels
                    .cpu()
                    .numpy()
                )


                val_pred.extend(
                    preds
                    .cpu()
                    .numpy()
                )


                batch_n = (
                    labels.size(0)
                )


                val_loss_sum += (
                    val_loss.item()
                    *
                    batch_n
                )


                n_val += batch_n


        val_accuracy = accuracy_score(
            val_true,
            val_pred,
        )


        val_loss_mean = (
            val_loss_sum
            /
            max(
                n_val,
                1,
            )
        )


        train_total = (
            total_sum
            /
            n_train
        )


        train_mse = (
            mse_sum
            /
            n_train
        )


        train_psd = (
            psd_sum
            /
            n_train
        )


        train_task = (
            task_sum
            /
            n_train
        )


        print(
            f"DENOISER | "
            f"ep={epoch:02d} | "
            f"loss={train_total:.4f} | "
            f"MSE={train_mse:.4f} | "
            f"PSD={train_psd:.4f} | "
            f"TASK={train_task:.4f} | "
            f"val_acc={val_accuracy:.4f}"
        )


        history.append({

            "epoch":
                epoch,

            "train_loss":
                train_total,

            "mse":
                train_mse,

            "psd":
                train_psd,

            "task":
                train_task,

            "val_accuracy":
                val_accuracy,

            "val_loss":
                val_loss_mean,
        })


        # ====================================================
        # CHECKPOINT
        # ====================================================

        better = (
            val_accuracy
            >
            best_val_accuracy
        )


        tie_better_loss = (
            np.isclose(
                val_accuracy,
                best_val_accuracy,
            )
            and
            val_loss_mean
            <
            best_val_loss
        )


        if (
            better
            or
            tie_better_loss
        ):

            best_val_accuracy = (
                val_accuracy
            )

            best_val_loss = (
                val_loss_mean
            )

            best_epoch = (
                epoch
            )

            best_state = copy.deepcopy(
                denoiser.state_dict()
            )

            no_improvement = 0

        else:

            no_improvement += 1


        if (
            no_improvement
            >=
            denoiser_patience
        ):

            print(
                f"Early stopping "
                f"at epoch {epoch}"
            )

            break


    # ========================================================
    # RESTORE DENOISER
    # ========================================================

    denoiser.load_state_dict(
        best_state
    )


    denoiser.eval()


    return (
        classifier_result,
        classifier,
        denoiser,
        fold,
        pd.DataFrame(
            history
        ),
        best_epoch,
    )

In [116]:
# ============================================================
# EVALUATION
# ============================================================


def create_fixed_noisy_numpy(
    X,
    snr_db,
    seed,
):

    rng = np.random.default_rng(
        seed
    )


    X = np.asarray(
        X,
        dtype=np.float32,
    )


    noise = rng.standard_normal(
        X.shape
    ).astype(
        np.float32
    )


    signal_power = np.mean(
        X ** 2,
        axis=(1, 2),
        keepdims=True,
    )


    noise_power_current = np.mean(
        noise ** 2,
        axis=(1, 2),
        keepdims=True,
    )


    target_noise_power = (
        signal_power
        /
        (
            10.0
            **
            (
                snr_db
                /
                10.0
            )
        )
    )


    noise = (
        noise
        *
        np.sqrt(
            target_noise_power
            /
            (
                noise_power_current
                +
                1e-8
            )
        )
    )


    return (
        X
        +
        noise
    ).astype(
        np.float32
    )


def evaluate_signal64(
    classifier,
    X,
    y,
    denoiser=None,
    batch_size=64,
):

    run_device = next(
        classifier.parameters()
    ).device


    y = np.asarray(
        y,
        dtype=np.int64,
    )


    predictions = []


    classifier.eval()


    if denoiser is not None:
        denoiser.eval()


    with torch.no_grad():

        for start in range(
            0,
            len(X),
            batch_size,
        ):

            xb = torch.tensor(
                X[
                    start:
                    start + batch_size
                ],
                dtype=torch.float32,
                device=run_device,
            )


            if denoiser is not None:

                scale = (
                    xb.std(
                        dim=(-2, -1),
                        keepdim=True,
                    )
                    .clamp_min(
                        1e-6
                    )
                )


                xb_norm = (
                    xb
                    /
                    scale
                )


                xb = (
                    denoiser(
                        xb_norm
                    )
                    *
                    scale
                )


            logits = (
                frozen_classifier_logits(
                    classifier,
                    xb,
                )
            )


            predictions.extend(
                logits.argmax(
                    dim=1
                )
                .cpu()
                .numpy()
            )


    predictions = np.asarray(
        predictions
    )


    return {

        "accuracy":
            accuracy_score(
                y,
                predictions,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y,
                predictions,
            ),

        "macro_f1":
            f1_score(
                y,
                predictions,
                average="macro",
                zero_division=0,
            ),
    }


In [118]:
# ============================================================
# TASK-AWARE DENOISING DATASET
# ============================================================


class FinalTaskAwareDenoisingDataset(
    torch.utils.data.Dataset
):

    def __init__(
        self,
        X,
        y,
        train=True,
        fixed_snr_db=-5.0,
        seed=42,
    ):

        self.X = np.asarray(
            X,
            dtype=np.float32,
        )

        self.y = np.asarray(
            y,
            dtype=np.int64,
        )

        self.train = train

        self.fixed_snr_db = (
            fixed_snr_db
        )

        self.seed = seed


    def __len__(
        self,
    ):

        return len(
            self.X
        )


    def __getitem__(
        self,
        index,
    ):

        clean = torch.tensor(
            self.X[index],
            dtype=torch.float32,
        )


        label = int(
            self.y[index]
        )


        # ====================================================
        # TRAIN: RANDOM SNR [-5, +5]
        # ====================================================

        if self.train:

            snr_db = float(
                torch.empty(1)
                .uniform_(
                    -5.0,
                    5.0,
                )
                .item()
            )


            noisy = add_awgn_torch(
                clean.unsqueeze(0),
                snr_db,
            ).squeeze(0)


        # ====================================================
        # VAL: FIXED -5 dB
        # ====================================================

        else:

            generator = (
                torch.Generator()
                .manual_seed(
                    self.seed
                    +
                    int(index)
                )
            )


            signal_power = (
                clean.pow(2)
                .mean()
                .clamp_min(1e-8)
            )


            noise_power = (
                signal_power
                /
                (
                    10.0
                    **
                    (
                        self.fixed_snr_db
                        /
                        10.0
                    )
                )
            )


            noise = torch.randn(
                clean.shape,
                generator=generator,
                dtype=clean.dtype,
            )


            noise = (
                noise
                /
                noise.pow(2)
                .mean()
                .sqrt()
                .clamp_min(1e-8)
            )


            noise = (
                noise
                *
                torch.sqrt(
                    noise_power
                )
            )


            noisy = (
                clean
                +
                noise
            )


        # ====================================================
        # SAME SCALE FOR CLEAN AND NOISY
        # ====================================================

        scale = (
            noisy.std()
            .clamp_min(
                1e-6
            )
        )


        noisy_norm = (
            noisy
            /
            scale
        )


        clean_norm = (
            clean
            /
            scale
        )


        return (
            noisy_norm,
            clean_norm,
            scale,
            label,
        )

In [ ]:
# ============================================================
# TASK-AWARE DENOISER64
# FIRST SCREENING ON ORIGINAL WORST-10
# ============================================================


DENOISER64_ROOT = (
    RESULTS_ROOT
    /
    "taskaware_denoiser64_worst10"
)


DENOISER64_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


DENOISER64_CSV = (
    DENOISER64_ROOT
    /
    "taskaware_denoiser64_worst10.csv"
)


WORST10_DENOISER = (
    full_df
    .sort_values(
        "test_accuracy"
    )
    .head(10)[
        "test_subject"
    ]
    .astype(int)
    .tolist()
)


if DENOISER64_CSV.exists():

    denoiser64_results = (
        pd.read_csv(
            DENOISER64_CSV
        )
    )

else:

    denoiser64_results = (
        pd.DataFrame()
    )


done_subjects = set()


if len(
    denoiser64_results
) > 0:

    done_subjects = set(
        denoiser64_results[
            "test_subject"
        ]
        .astype(int)
        .tolist()
    )


print(
    "Subjects:",
    WORST10_DENOISER
)


for idx, subject in enumerate(
    WORST10_DENOISER,
    start=1,
):

    if subject in done_subjects:
        continue


    print()
    print("=" * 100)

    print(
        f"[{idx}/10] TASK-AWARE DENOISER64 | "
        f"S{subject:03d}"
    )

    print("=" * 100)


    (
        classifier_result,
        classifier,
        denoiser,
        fold,
        history,
        best_denoiser_epoch,
    ) = train_taskaware_denoiser64_fold(

        test_subject=subject,

        seed=42,

        denoiser_epochs=80,

        denoiser_patience=15,

        lambda_psd=0.10,

        lambda_task=0.25,
    )


    X_test = fold[
        "X_test"
    ]


    y_test = fold[
        "y_test"
    ]


    X_test_noisy = (
        create_fixed_noisy_numpy(

            X_test,

            snr_db=-5.0,

            seed=
                420000
                +
                subject,
        )
    )


    # ========================================================
    # FOUR CONDITIONS
    # ========================================================

    clean = evaluate_signal64(
        classifier,
        X_test,
        y_test,
    )


    clean_denoised = (
        evaluate_signal64(
            classifier,
            X_test,
            y_test,
            denoiser=denoiser,
        )
    )


    noisy = evaluate_signal64(
        classifier,
        X_test_noisy,
        y_test,
    )


    noisy_denoised = (
        evaluate_signal64(
            classifier,
            X_test_noisy,
            y_test,
            denoiser=denoiser,
        )
    )


    row = {

        "test_subject":
            subject,

        "classifier_clean":
            clean["accuracy"],

        "clean_denoised":
            clean_denoised[
                "accuracy"
            ],

        "noisy_minus5":
            noisy["accuracy"],

        "noisy_denoised_minus5":
            noisy_denoised[
                "accuracy"
            ],

        "clean_delta_pp":
            (
                clean_denoised[
                    "accuracy"
                ]
                -
                clean[
                    "accuracy"
                ]
            ) * 100,

        "noisy_recovery_pp":
            (
                noisy_denoised[
                    "accuracy"
                ]
                -
                noisy[
                    "accuracy"
                ]
            ) * 100,

        "best_denoiser_epoch":
            best_denoiser_epoch,
    }


    denoiser64_results = pd.concat(
        [
            denoiser64_results,
            pd.DataFrame(
                [row]
            ),
        ],
        ignore_index=True,
    )


    denoiser64_results.to_csv(
        DENOISER64_CSV,
        index=False,
    )


    done_subjects.add(
        subject
    )


    print()

    print(
        f"CLEAN              : "
        f"{clean['accuracy']*100:.2f}%"
    )

    print(
        f"CLEAN + DENOISER   : "
        f"{clean_denoised['accuracy']*100:.2f}%"
    )

    print(
        f"NOISY -5 dB        : "
        f"{noisy['accuracy']*100:.2f}%"
    )

    print(
        f"NOISY + DENOISER   : "
        f"{noisy_denoised['accuracy']*100:.2f}%"
    )

    print(
        f"NOISY RECOVERY     : "
        f"{row['noisy_recovery_pp']:+.2f} pp"
    )


    del classifier
    del denoiser
    del history

    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Subjects: [99, 24, 87, 3, 109, 67, 5, 88, 37, 69]

[1/10] TASK-AWARE DENOISER64 | S099

STAGE 1 — BEST CLASSIFIER | S099
SUPCON       | ep=01 | train=0.5948 | val=0.5881 | CE=0.674 | SC=4.195 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=02 | train=0.6201 | val=0.6250 | CE=0.643 | SC=4.138 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=03 | train=0.6326 | val=0.5963 | CE=0.629 | SC=4.135 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=04 | train=0.6342 | val=0.6209 | CE=0.632 | SC=4.135 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=05 | train=0.6347 | val=0.6393 | CE=0.626 | SC=4.133 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=06 | train=0.6401 | val=0.6332 | CE=0.627 | SC=4.133 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=07 | train=0.6438 | val=0.6496 | CE=0.616 | SC=4.131 (w=0.1000) | ADV=0.000 (w=0.0000) | GRL=0.000
SUPCON       | ep=08 | train=0.6488 | val=0.6598 | CE=0.609 

In [ ]:
# ============================================================
# DENOISER64 SUMMARY
# ============================================================


summary = pd.DataFrame({

    "condition": [

        "Clean",

        "Clean + denoiser",

        "Noisy -5 dB",

        "Noisy -5 dB + denoiser",
    ],

    "mean_accuracy_pct": [

        denoiser64_results[
            "classifier_clean"
        ].mean() * 100,

        denoiser64_results[
            "clean_denoised"
        ].mean() * 100,

        denoiser64_results[
            "noisy_minus5"
        ].mean() * 100,

        denoiser64_results[
            "noisy_denoised_minus5"
        ].mean() * 100,
    ],
})


display(
    summary
)


print()
print("=" * 80)

print(
    "CLEAN DELTA       : "
    f"{denoiser64_results['clean_delta_pp'].mean():+.2f} pp"
)

print(
    "NOISY RECOVERY    : "
    f"{denoiser64_results['noisy_recovery_pp'].mean():+.2f} pp"
)

print("=" * 80)